In [ ]:
# Close and then Open VS Code for packages such as 'python-dotenv'
# !uv add ipykernel IPython python-dotenv openai openai-agents pydantic asyncio requests
!uv add qdrant_client
!uv add openpyxl

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

In [ ]:
# import irw

# # a Redivis account is needed.
# # Get database information
# irw.info()

# # View available tables
# tables = irw.list_tables()
# tables_with_metadata = irw.list_tables(include_metadata=True)

## TIMSS 2011 grade 4 released mathematics items

Build `data/timss11_g4_math_released_items.jsonl` (one item per line) from NCES's
[Complete 2011 TIMSS 4 mathematics set](https://nces.ed.gov/timss/released-questions.asp).

The PDF draws much of its math (fractions, number sentences, some scoring guides) as vector
outlines, so the text layer alone is incomplete. Extraction is therefore hybrid:

- **Deterministic, from the PDF:** item ID, label, content domain, main topic, cognitive domain,
  multiple-choice answer key, percent correct by education system (with the significance arrows),
  and the official accessibility transcription (`ActualText`) that NCES attached to each item.
- **Gemini via OpenRouter:** `stem`, `options`, `scoring_guide`, and `figure_description`, read
  from rendered images of the item page plus its accessibility transcription, then cross-checked
  against the deterministic fields. Responses are cached, so reruns make no API calls.

`figure_description` is model-generated; `accessibility_text` is the official NCES wording.

In [ ]:
import base64
import json
import re
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import pymupdf
import requests
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, Field

TIMSS_PDF_URL = "https://nces.ed.gov/timss/pdf/TIMSS2011_G4_Math.pdf"
DATA_DIR = Path.cwd() / "data"
PDF_PATH = DATA_DIR / "raw" / "TIMSS2011_G4_Math.pdf"
OUTPUT_PATH = DATA_DIR / "timss11_g4_math_released_items.jsonl"

if not PDF_PATH.exists():
    response = requests.get(TIMSS_PDF_URL, timeout=120)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"{TIMSS_PDF_URL} did not return a PDF")
    PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
    PDF_PATH.write_bytes(response.content)

timss_pdf = pymupdf.open(PDF_PATH)
print(f"{PDF_PATH.name}: {timss_pdf.page_count} pages")

### Deterministic parse of item pages

Every item starts on a page with the same layout (coordinates in points, origin top-left):
a classification header (y 75-112), the item box (y 150-520), the item number and answer key or
scoring guide below it, and the percent-correct table in the right column (x > 415).
Continuation pages with sample student responses are skipped.

In [ ]:
# Text as drawn on the page. Without TEXT_IGNORE_ACTUALTEXT, PyMuPDF substitutes the tagged-PDF
# accessibility text for the glyphs, which scrambles the visible text.
VISIBLE_WORDS = pymupdf.TEXTFLAGS_WORDS | pymupdf.TEXT_IGNORE_ACTUALTEXT
VISIBLE_TEXT = pymupdf.TEXTFLAGS_TEXT | pymupdf.TEXT_IGNORE_ACTUALTEXT

RIGHT_COLUMN_X = 415
HEADER_Y = (75, 112)
CONTENT_DOMAIN_X_MAX = 220
MAIN_TOPIC_X_MAX = 410
ITEM_BOX_Y = (150, 520)
TABLE_LEGEND_Y = 728
PERCENT_X_MIN = 525

CONTENT_DOMAINS = {
    "NUMBER": "Number",
    "GEOMETRIC SHAPES AND MEASURES": "Geometric Shapes and Measures",
    "DATA DISPLAY": "Data Display",
}
# TIMSS 2011 grade 4 mathematics framework topics, used to validate the parsed header.
MAIN_TOPICS = {
    "Number": {
        "Whole Numbers",
        "Fractions and Decimals",
        "Number Sentences with Whole Numbers",
        "Patterns and Relationships",
    },
    "Geometric Shapes and Measures": {
        "Points, Lines, and Angles",
        "Two- and Three-dimensional Shapes",
    },
    "Data Display": {"Reading and Interpreting", "Organizing and Representing"},
}
COGNITIVE_DOMAINS = {"Knowing", "Applying", "Reasoning"}

# Percent-correct table: "—" is "Not applicable" and "#" is "Rounds to zero".
PCT_SYMBOLS = {"—": None, "#": 0}
SYSTEM_NAME_FIXES = {"United Arab Emiraes": "United Arab Emirates"}  # typo on one page of the PDF


def words_in(page, x0, y0, x1, y1):
    """Visible words whose top-left corner falls inside the rectangle, in reading order."""
    return [
        w
        for w in page.get_text("words", flags=VISIBLE_WORDS, sort=True)
        if x0 <= w[0] < x1 and y0 <= w[1] < y1
    ]


def joined(words):
    return " ".join(w[4] for w in words)


def parse_classification(page):
    header = words_in(page, 0, HEADER_Y[0], page.rect.width, HEADER_Y[1])
    content = joined(w for w in header if w[0] < CONTENT_DOMAIN_X_MAX)
    topic = joined(w for w in header if CONTENT_DOMAIN_X_MAX <= w[0] < MAIN_TOPIC_X_MAX)
    cognitive = joined(w for w in header if w[0] >= MAIN_TOPIC_X_MAX)
    content_domain = CONTENT_DOMAINS[content]
    if topic not in MAIN_TOPICS[content_domain] or cognitive not in COGNITIVE_DOMAINS:
        raise ValueError(f"Unexpected classification on page {page.number + 1}: {header}")
    return content_domain, topic, cognitive


def center_y(bbox):
    return (bbox[1] + bbox[3]) / 2


def table_lines(words, tolerance=3):
    """Group words into (y center, words left to right) lines by vertical position."""
    lines = []
    for w in sorted(words, key=center_y):
        if lines and center_y(w) - lines[-1][0] < tolerance:
            lines[-1][1].append(w)
        else:
            lines.append((center_y(w), [w]))
    return [(y_mid, sorted(line, key=lambda w: w[0])) for y_mid, line in lines]


def parse_pct_correct(page):
    """Rows of the "Overall Percent Correct" table, with the significance marker for each row."""
    words = [
        w
        for w in words_in(page, RIGHT_COLUMN_X, 150, page.rect.width, TABLE_LEGEND_Y)
        if w[3] - w[1] > 5  # drops 2pt-high hidden "Not applicable." text
    ]
    benchmark_y = next(w[1] for w in words if w[4] == "Benchmarking")

    # The legend's filled circle ("Percent higher than International average") tells which
    # circle fill means "higher"; the other fill means "lower".
    circles = [
        d
        for d in page.get_drawings()
        if d["type"] == "fs" and d["rect"].x0 > RIGHT_COLUMN_X and 5 < d["rect"].width < 7
    ]
    legend_higher_y = next(
        w[1]
        for w in page.get_text("words", flags=VISIBLE_WORDS)
        if w[4] == "higher" and w[1] > TABLE_LEGEND_Y
    )
    higher_fill = next(
        d["fill"]
        for d in circles
        if d["rect"].y0 > TABLE_LEGEND_Y and abs(d["rect"].y0 - legend_higher_y) < 3
    )

    rows = []
    for y_mid, line in table_lines(words):
        name = joined(w for w in line if w[0] < PERCENT_X_MIN)
        values = [w[4] for w in line if w[0] >= PERCENT_X_MIN]
        if len(values) != 1 or not (values[0].isdigit() or values[0] in PCT_SYMBOLS):
            continue  # section headers such as "Education system  Percent"
        marker = next(
            (d for d in circles if d["rect"].x0 > PERCENT_X_MIN and abs(center_y(d["rect"]) - y_mid) < 3),
            None,
        )
        rows.append(
            {
                "education_system": SYSTEM_NAME_FIXES.get(name, name),
                "pct_correct": int(values[0]) if values[0].isdigit() else PCT_SYMBOLS[values[0]],
                "vs_intl_avg": None if marker is None else ("higher" if marker["fill"] == higher_fill else "lower"),
                "benchmarking": y_mid > benchmark_y,
            }
        )
    return rows


def parse_item_page(page):
    text = page.get_text(flags=VISIBLE_TEXT)
    item_id = re.search(r"Item Number:\s*(M\d{6}[A-Z]?)", text).group(1)
    content_domain, main_topic, cognitive_domain = parse_classification(page)
    mc_key = re.search(r"Correct Response:\s+([A-E])\b", text)
    pct_rows = parse_pct_correct(page)
    intl_avg = next(r for r in pct_rows if r["education_system"] == "International average")
    usa = next(r for r in pct_rows if r["education_system"] == "United States")
    booklet_page = next(w[4] for w in words_in(page, 300, 760, 400, page.rect.height) if w[4].isdigit())
    return {
        "item_id": item_id,
        "item_label": re.search(r"Item label:\s*(.+)", text).group(1).strip(),
        "content_domain": content_domain,
        "main_topic": main_topic,
        "cognitive_domain": cognitive_domain,
        "item_type": "multiple_choice" if mc_key else "constructed_response",
        "correct_option": mc_key.group(1) if mc_key else None,
        "pct_correct_intl_avg": intl_avg["pct_correct"],
        "pct_correct_usa": usa["pct_correct"],
        "pct_correct": [r for r in pct_rows if r is not intl_avg],
        "booklet_page": int(booklet_page),
        "pdf_page": page.number + 1,
        # Visible text in the item box, used later to check the model's transcription.
        "_item_box_text": joined(words_in(page, 0, ITEM_BOX_Y[0], RIGHT_COLUMN_X, ITEM_BOX_Y[1])),
    }


def accessibility_texts(doc):
    """Official ActualText of the tagged-PDF structure elements, as {page index: [(top y, text)]}."""
    page_index = {doc.page_xref(i): i for i in range(doc.page_count)}
    texts = defaultdict(list)
    for xref in range(1, doc.xref_length()):
        kind, text = doc.xref_get_key(xref, "ActualText")
        page_kind, page_ref = doc.xref_get_key(xref, "Pg")
        if kind != "string" or page_kind != "xref":
            continue  # not a structure element with ActualText, or not attached to a page
        page = doc[page_index[int(page_ref.split()[0])]]
        bbox = [float(v) for v in doc.xref_get_key(xref, "A/BBox")[1].strip("[]").split()]
        text = re.sub(r"[ \u00a0\u2009\u200a]+", " ", text.replace("\r\n", "\n").replace("\r", "\n"))
        texts[page.number].append((page.rect.height - bbox[3], text.strip()))
    return {p: sorted(entries) for p, entries in texts.items()}


item_pages = [page for page in timss_pdf if "Item Number:" in page.get_text(flags=VISIBLE_TEXT)]
alt_texts = accessibility_texts(timss_pdf)
items = []
for page in item_pages:
    item = parse_item_page(page)
    entries = alt_texts.get(page.number, [])
    item["accessibility_text"] = "\n".join(t for y, t in entries if y < ITEM_BOX_Y[1])
    item["_scoring_accessibility_text"] = "\n".join(t for y, t in entries if y >= ITEM_BOX_Y[1])
    items.append(item)

index_ids = set(re.findall(r"\b(M\d{6}[A-Z]?)\b", timss_pdf[2].get_text(flags=VISIBLE_TEXT)))
parsed_ids = [item["item_id"] for item in items]
assert len(parsed_ids) == len(set(parsed_ids)), "duplicate item IDs"
assert set(parsed_ids) == index_ids, f"item pages vs. index mismatch: {set(parsed_ids) ^ index_ids}"
assert all(item["accessibility_text"] for item in items), "item without accessibility text"
system_sets = {frozenset(r["education_system"] for r in item["pct_correct"]) for item in items}
assert len(system_sets) == 1, "percent-correct tables list different education systems"

print(f"Parsed {len(items)} items")
display(pd.DataFrame(items).groupby(["content_domain", "item_type"]).size().unstack(fill_value=0))

### Transcribe stems, options, and scoring guides with Gemini

For each item, the item box and the answer-key/scoring area are rendered as two PNGs (separate
images keep small figure details legible) and sent to Gemini through OpenRouter with the item's
official accessibility transcription. Responses are parsed into a Pydantic schema and cached per
item under `data/cache/`; bump `PROMPT_VERSION` after editing the prompt or schema to re-extract.

**Ze's notes:** Code changed to use `gemini-3.8-flash` directly with OpenAI SDK instead of through OpenRouter because the model is available under the free Gemini plan.

In [ ]:
LLM_MODEL = "google/gemini-3.8-flash"
# LLM_MODEL = "gemini-3.8-flash"
PROMPT_VERSION = 2
LLM_CACHE_DIR = DATA_DIR / "cache" / "item_transcriptions" / f"{LLM_MODEL.replace('/', '--')}-v{PROMPT_VERSION}"
ITEM_BOX_CLIP = pymupdf.Rect(40, 120, RIGHT_COLUMN_X, 522)  # item label and item box
SCORING_CLIP = pymupdf.Rect(40, 518, RIGHT_COLUMN_X, 770)  # item number and answer key or scoring guide
IMAGE_DPI = 300


class ResponseOption(BaseModel):
    label: str = Field(description="Option letter as printed, e.g. 'A'.")
    text: str = Field(description="Option content in the same notation as the stem.")


class ScoringCategory(BaseModel):
    category: str = Field(description="Category heading as printed, e.g. 'Correct Response'.")
    descriptors: list[str] = Field(description="Each bullet under the heading, in printed order.")


class ItemTranscription(BaseModel):
    stem: str
    options: list[ResponseOption]
    answer_key: str | None
    scoring_guide: list[ScoringCategory]
    scoring_notes: str | None
    has_figure: bool
    figure_description: str | None


TRANSCRIPTION_INSTRUCTIONS = """\
You transcribe items from the TIMSS 2011 grade 4 released mathematics booklet into structured data.

For each item you receive:
1. Two images of the item page. The first shows the item label and the boxed item exactly as \
students saw it. The second shows the item number and either the multiple-choice answer key \
("Correct Response: X") or the scoring guide.
2. The official accessibility transcription of the item, written for screen readers (math is spelled \
out, e.g. "one-half", "4 times box equals 28", "Answer colon blank"), and accessibility descriptions \
of any pictures in the scoring guide.

The images are the authority for wording, numbers, and layout. Use the accessibility text to confirm \
hard-to-read content and as the source for picture descriptions. Never add content found in neither.

Fields:
- stem: Everything inside the item box except multiple-choice response options, in reading order, \
with the wording verbatim. Write math in plain-text notation: fractions as 1/2, mixed numbers as \
1 1/2, and the symbols ×, ÷, −, =, °. Write an empty box in a number sentence as □ and an answer \
blank as ____, keeping labels such as "Answer:". Write tables as Markdown tables and bulleted lists \
as "- " lines. Keep part labels of multi-part items (e.g. "A.", "B.") in the stem. Replace each \
picture, diagram, graph, grid, or map with "[Image: <description>]", using the wording of the \
matching accessibility description where one exists. Separate paragraphs with a newline.
- options: For multiple-choice items, one entry per response option in printed order, in the same \
notation as the stem; a pictured option is "[Image: <description>]". Empty for constructed-response \
items.
- answer_key: The letter printed in the "Correct Response:" box of a multiple-choice item; null for \
constructed-response items.
- scoring_guide: For constructed-response items, each category under "SCORING" in printed order, \
with its heading (e.g. "Correct Response", "Partially Correct Response", "Incorrect Response") and \
each bullet verbatim as a descriptor, in the same notation as the stem. Write a picture of a sample \
answer, whether it is a bullet of its own or follows a bullet's text, as "[Image: <description>]", \
using the scoring-guide accessibility text where available. Empty for multiple-choice items.
- scoring_notes: Any scoring-guide text that is not a heading or bullet (e.g. "Note: Part A not \
scored"); null if none.
- has_figure: Whether the item box (including the options) contains any picture, diagram, graph, \
grid, map, or drawn shape. A table of text or numbers alone does not count.
- figure_description: If has_figure, a precise, objective description of every visual element, \
giving all the information a student would need to answer without seeing it: counts, labels, \
measurements, positions, grid coordinates, bar heights, shading, and orientation. Describe what is \
drawn; do not solve the item or reveal the answer. Null if has_figure is false.
"""


def png_data_url(page, clip):
    png = page.get_pixmap(dpi=IMAGE_DPI, clip=clip).tobytes("png")
    return "data:image/png;base64," + base64.b64encode(png).decode()


def transcription_messages(item):
    page = timss_pdf[item["pdf_page"] - 1]
    item_type = item["item_type"].replace("_", "-")
    prompt = (
        f"Item {item['item_id']} ({item_type}), labeled \"{item['item_label']}\".\n\n"
        f"Accessibility transcription of the item:\n<<<\n{item['accessibility_text']}\n>>>\n\n"
        "Accessibility descriptions of pictures in the scoring guide:\n"
        f"<<<\n{item['_scoring_accessibility_text'] or '(none)'}\n>>>"
    )
    return [
        {"role": "system", "content": TRANSCRIPTION_INSTRUCTIONS},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": png_data_url(page, ITEM_BOX_CLIP)}},
                {"type": "image_url", "image_url": {"url": png_data_url(page, SCORING_CLIP)}},
            ],
        },
    ]


def transcription_path(item_id):
    return LLM_CACHE_DIR / f"{item_id}.json"


def fetch_transcription(item_id, messages, attempts=3):
    # OpenRouter reports some upstream failures as HTTP 200 with finish_reason="error", which the
    # OpenAI client does not retry, so retry those here.
    for _ in range(attempts):
        choice = llm_client.chat.completions.parse(
            model=LLM_MODEL, messages=messages, response_format=ItemTranscription
        ).choices[0]
        if choice.message.parsed is not None:
            transcription_path(item_id).write_text(choice.message.parsed.model_dump_json(indent=2))
            return
    raise RuntimeError(
        f"{item_id}: no parsed response after {attempts} attempts (finish_reason={choice.finish_reason})"
    )


llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key, max_retries=5, timeout=600
)

# llm_client = OpenAI(
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=gemini_api_key, max_retries=5, timeout=600
# ) 
LLM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# PyMuPDF is not thread-safe, so render every page image up front and parallelize only the API calls.
pending = {
    item["item_id"]: transcription_messages(item)
    for item in items
    if not transcription_path(item["item_id"]).exists()
}
print(f"Transcribing {len(pending)} of {len(items)} items with {LLM_MODEL}")
with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(fetch_transcription, pending.keys(), pending.values()))

transcriptions = {
    item["item_id"]: ItemTranscription.model_validate_json(transcription_path(item["item_id"]).read_text())
    for item in items
}

### Validate and write the JSONL

Hard checks compare the transcriptions with the deterministic parse. The coverage report lists
words drawn in the item box that are missing from the transcribed stem and options; misses are
usually figure labels, but anything else deserves a look at the page.

In [ ]:
SCORING_CATEGORIES = ["Correct Response", "Partially Correct Response", "Incorrect Response"]

problems = []
for item in items:
    t = transcriptions[item["item_id"]]
    where = f"{item['item_id']} (PDF page {item['pdf_page']})"
    if item["item_type"] == "multiple_choice":
        if [o.label for o in t.options] != ["A", "B", "C", "D"]:
            problems.append(f"{where}: option labels {[o.label for o in t.options]}")
        if t.answer_key != item["correct_option"]:
            problems.append(f"{where}: answer key {t.answer_key!r} != {item['correct_option']!r}")
        if t.scoring_guide:
            problems.append(f"{where}: multiple-choice item has a scoring guide")
    else:
        if t.options or t.answer_key:
            problems.append(f"{where}: constructed-response item has options or an answer key")
        categories = [c.category for c in t.scoring_guide]
        if categories not in (SCORING_CATEGORIES, [SCORING_CATEGORIES[0], SCORING_CATEGORIES[2]]):
            problems.append(f"{where}: scoring categories {categories}")
    if t.has_figure != bool(t.figure_description):
        problems.append(f"{where}: has_figure={t.has_figure}, figure_description={t.figure_description!r}")
assert not problems, "\n".join(problems)


def normalized(text):
    return re.sub(r"[^0-9a-z]", "", text.lower())


coverage_rows = []
for item in items:
    t = transcriptions[item["item_id"]]
    transcribed = normalized(" ".join([t.stem, *(o.text for o in t.options)]))
    drawn = [w for w in map(normalized, item["_item_box_text"].split()) if len(w) >= 3]
    missing = [w for w in drawn if w not in transcribed]
    if missing:
        coverage_rows.append(
            {
                "item_id": item["item_id"],
                "coverage": 1 - len(missing) / len(drawn),
                "missing_words": " ".join(missing),
            }
        )
print(f"{len(coverage_rows)} of {len(items)} items have drawn words missing from the transcription")
if coverage_rows:
    display(pd.DataFrame(coverage_rows).sort_values("coverage"))

In [ ]:
def tidy(text):
    """Even out the model's formatting noise: answer blanks become ____ and runs of spaces one space."""
    return re.sub(r"_{2,}", "____", re.sub(r" {2,}", " ", text))


records = []
for item in items:
    t = transcriptions[item["item_id"]]
    categories = [c.category for c in t.scoring_guide]
    records.append(
        {
            "item_id": item["item_id"],
            "item_label": item["item_label"],
            "assessment": "TIMSS 2011",
            "grade": 4,
            "subject": "Mathematics",
            "content_domain": item["content_domain"],
            "main_topic": item["main_topic"],
            "cognitive_domain": item["cognitive_domain"],
            "item_type": item["item_type"],
            "max_points": 2 if "Partially Correct Response" in categories else 1,
            "stem": tidy(t.stem),
            "options": [{"label": o.label, "text": tidy(o.text)} for o in t.options] or None,
            "correct_option": item["correct_option"],
            "scoring_guide": [
                {"category": c.category, "descriptors": [tidy(d) for d in c.descriptors]} for c in t.scoring_guide
            ]
            or None,
            "scoring_notes": t.scoring_notes,
            "has_figure": t.has_figure,
            "figure_description": t.figure_description,
            "accessibility_text": item["accessibility_text"],
            "pct_correct_intl_avg": item["pct_correct_intl_avg"],
            "pct_correct_usa": item["pct_correct_usa"],
            "pct_correct": item["pct_correct"],
            "booklet_page": item["booklet_page"],
            "pdf_page": item["pdf_page"],
            "source_url": TIMSS_PDF_URL,
            "transcription_model": LLM_MODEL,
        }
    )

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

released_items = pd.read_json(OUTPUT_PATH, lines=True)
print(f"Wrote {len(released_items)} items to {OUTPUT_PATH.resolve()}")
preview_columns = ["item_id", "content_domain", "main_topic", "cognitive_domain", "item_type", "stem", "options"]
display(released_items[preview_columns].head())

### Summarize and analyze released items

Review item types, domains, answer rates, and missingness. Then create an item-level dataframe without the full question stem, option text, or scoring-guide descriptions, retaining correct answers and derived word-count features.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
)
released_items_path = next(
    path
    for path in (
        project_root / "data" / "timss11_g4_math_released_items.jsonl",
        project_root / "notebooks" / "data" / "timss11_g4_math_released_items.jsonl",
    )
    if path.is_file()
)
released_items = pd.read_json(released_items_path, lines=True)

print(f"{len(released_items):,} items × {released_items.shape[1]:,} source columns")
print("Item counts by type:")
display(released_items["item_type"].value_counts(dropna=False).rename("items").to_frame())

print("Item counts by content and cognitive domain:")
display(
    pd.crosstab(
        [released_items["content_domain"], released_items["main_topic"]],
        released_items["cognitive_domain"],
        margins=True,
    )
)

print("Item counts by content domain and item type:")
display(
    pd.crosstab(released_items["content_domain"], released_items["item_type"], margins=True)
)

print("Percent-correct summary:")
display(
    released_items[["pct_correct_intl_avg", "pct_correct_usa"]]
    .describe()
    .round(1)
)

missing_summary = released_items.isna().sum().rename("missing").to_frame()
missing_summary["missing_pct"] = (released_items.isna().mean() * 100).round(1)
print("Columns with missing values:")
display(missing_summary[missing_summary["missing"] > 0].sort_values("missing", ascending=False))

In [ ]:
import re


def word_count(text):
    """Count word-like tokens while ignoring underscores used for answer blanks."""
    return len(re.findall(r"[^\W_]+(?:['’][^\W_]+)*", text or "", flags=re.UNICODE))


def as_list(value):
    return value if isinstance(value, list) else []


option_labels = sorted(
    {
        option["label"]
        for options in released_items["options"]
        for option in as_list(options)
    }
)
text_columns_to_drop = {"stem", "options", "scoring_guide"}
metadata_columns = [
    column for column in released_items.columns if column not in text_columns_to_drop
]
analysis_rows = []

for _, item in released_items.iterrows():
    options = as_list(item["options"])
    scoring_guide = as_list(item["scoring_guide"])
    correct_category = next(
        (category for category in scoring_guide if category["category"] == "Correct Response"),
        None,
    )
    correct_response = (
        "; ".join(correct_category["descriptors"])
        if correct_category is not None
        else None
    )
    option_word_counts = {
        option["label"]: word_count(option["text"])
        for option in options
    }
    is_multiple_choice = item["item_type"] == "multiple_choice"
    correct_label = item["correct_option"] if is_multiple_choice else None

    row = {column: item[column] for column in metadata_columns}
    row["scoring_categories"] = [category["category"] for category in scoring_guide] or None
    row["correct_answer"] = correct_label if is_multiple_choice else correct_response
    row["stem_word_count"] = word_count(item["stem"])
    for label in option_labels:
        row[f"option_{label}_word_count"] = option_word_counts.get(label)

    if is_multiple_choice:
        row["correct_option_word_count"] = option_word_counts.get(correct_label)
        row["distractors_word_count"] = sum(
            count for label, count in option_word_counts.items() if label != correct_label
        )
    else:
        row["correct_option_word_count"] = None
        row["distractors_word_count"] = None
    analysis_rows.append(row)

released_items_analysis = pd.DataFrame(analysis_rows)
print(
    f"Analysis dataframe: {len(released_items_analysis):,} items × "
    f"{released_items_analysis.shape[1]:,} columns"
)
print("Text columns omitted: stem, option text, and scoring-guide descriptors.")
display(
    released_items_analysis[
        [
            "item_id",
            "item_type",
            "correct_option",
            "correct_answer",
            "stem_word_count",
            *[f"option_{label}_word_count" for label in option_labels],
            "correct_option_word_count",
            "distractors_word_count",
        ]
    ].head(10)
)

In [ ]:
xlsx_path = released_items_path.with_name(f"{released_items_path.stem}_analysis.xlsx")
released_items_analysis.to_excel(xlsx_path, index=False)
print(f"Exported {len(released_items_analysis):,} rows to {xlsx_path.resolve()}")

### Build semantic-search item chunks

Each released item is one retrieval chunk. The embedding input combines only the question stem and response-option text, while metadata remains separate for keyword filters and hybrid retrieval. Correct-answer fields, scoring-guide content, and scoring notes are never sent to the embedding model. Running the embedding cell calls the Gemini API.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

if "released_items_path" not in globals():
    project_root = next(
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "pyproject.toml").is_file()
    )
    released_items_path = next(
        path
        for path in (
            project_root / "data" / "timss11_g4_math_released_items.jsonl",
            project_root / "notebooks" / "data" / "timss11_g4_math_released_items.jsonl",
        )
        if path.is_file()
    )
if "released_items" not in globals():
    released_items = pd.read_json(released_items_path, lines=True)


def as_options(value):
    return value if isinstance(value, list) else []


def make_embedding_text(item):
    """Build searchable text exclusively from the stem and response-option contents."""
    option_text = "\n".join(
        f"{option['label']}: {option['text']}"
        for option in as_options(item["options"])
    )
    return "\n\n".join(part for part in (item["stem"], option_text) if part)


excluded_retrieval_text = {
    "stem",
    "options",
    "scoring_guide",
    "scoring_notes",
    "accessibility_text",
    "figure_description",
}
retrieval_metadata_columns = [
    column for column in released_items.columns if column not in excluded_retrieval_text
]
retrieval_metadata = released_items[retrieval_metadata_columns].copy()
retrieval_metadata["embedding_text"] = released_items.apply(make_embedding_text, axis=1)

# One row/chunk per item; answer and scoring fields are separate metadata, never in embedding_text.
assert retrieval_metadata["item_id"].is_unique
assert retrieval_metadata["embedding_text"].notna().all()
print(f"Prepared {len(retrieval_metadata)} one-item chunks with {len(retrieval_metadata_columns)} metadata fields")
display(retrieval_metadata[["item_id", "assessment", "grade", "subject", "item_type", "embedding_text"]].head())

In [ ]:
xlsx_path = released_items_path.with_name(f"{released_items_path.stem}_retrieval_metadata.xlsx")
retrieval_metadata.to_excel(xlsx_path, index=False)
print(f"Exported {len(retrieval_metadata):,} rows to {xlsx_path.resolve()}")

### Item embeddings

In [ ]:
import time

import numpy as np
import requests

EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSION = 768
EMBEDDING_BATCH_SIZE = 50

if not gemini_api_key:
    raise RuntimeError("Set GEMINI_API_KEY in the environment before requesting embeddings.")

embedding_texts = retrieval_metadata["embedding_text"].tolist()
embedding_vectors = []
embedding_url = (
    "https://generativelanguage.googleapis.com/v1beta/models/"
    f"{EMBEDDING_MODEL}:batchEmbedContents"
)

for start in range(0, len(embedding_texts), EMBEDDING_BATCH_SIZE):
    batch_texts = embedding_texts[start : start + EMBEDDING_BATCH_SIZE]
    payload = {
        "requests": [
            {
                "model": f"models/{EMBEDDING_MODEL}",
                "content": {"parts": [{"text": text}]},
                "taskType": "RETRIEVAL_DOCUMENT",
                "outputDimensionality": EMBEDDING_DIMENSION,
            }
            for text in batch_texts
        ]
    }
    response = requests.post(
        embedding_url,
        headers={"x-goog-api-key": gemini_api_key},
        json=payload,
        timeout=120,
    )
    response.raise_for_status()
    batch_embeddings = response.json()["embeddings"]
    if len(batch_embeddings) != len(batch_texts):
        raise RuntimeError("Embedding API returned a different number of vectors than input items.")
    embedding_vectors.extend(embedding["values"] for embedding in batch_embeddings)
    print(f"Embedded {min(start + len(batch_texts), len(embedding_texts))}/{len(embedding_texts)} items")
    if start + EMBEDDING_BATCH_SIZE < len(embedding_texts):
        time.sleep(0.5)

item_embeddings = np.asarray(embedding_vectors, dtype=np.float32)
if item_embeddings.shape != (len(retrieval_metadata), EMBEDDING_DIMENSION):
    raise RuntimeError(f"Unexpected embedding array shape: {item_embeddings.shape}")

# Unit-normalize once so cosine similarity at retrieval time is a fast matrix-vector product.
item_embeddings /= np.linalg.norm(item_embeddings, axis=1, keepdims=True)
embedding_path = released_items_path.with_name("timss11_g4_math_item_embeddings.npz")
retrieval_records_path = released_items_path.with_name("timss11_g4_math_retrieval_records.jsonl")
np.savez_compressed(
    embedding_path,
    item_ids=retrieval_metadata["item_id"].to_numpy(),
    embeddings=item_embeddings,
    model=EMBEDDING_MODEL,
)
retrieval_metadata.to_json(retrieval_records_path, orient="records", lines=True, force_ascii=False)
print(f"Saved {len(item_embeddings)} normalized vectors to {embedding_path.resolve()}")
print(f"Saved item text and metadata to {retrieval_records_path.resolve()}")

In [ ]:
# Check the first embedding vector and its shape.
# item_embeddings[0].shape
# item_embeddings[0]

In [ ]:
# Load the saved embeddings and verify their contents.
import numpy as np
from pathlib import Path

embedding_path = Path("data/timss11_g4_math_item_embeddings.npz")

data = np.load(embedding_path, allow_pickle=True)

# 1. See what "keys" or array names are inside the file
print("Arrays in this file:", data.files)

# data["item_ids"]  # This will show the item IDs
data["embeddings"][0]  # This will show the first embedding vector

### Deploy item embeddings to local Qdrant

Store the generated vectors in a persistent, embedded Qdrant collection on disk. Each vector is linked to its item ID and retrieval metadata so the collection can support semantic search and filtered retrieval. Rerunning the deployment cell rebuilds the collection from the saved embeddings and records.

In [ ]:
import json
from uuid import NAMESPACE_URL, uuid5

import numpy as np
from qdrant_client import QdrantClient, models

QDRANT_PATH = released_items_path.parent / "qdrant"
QDRANT_COLLECTION = "timss11_g4_math_items"
QDRANT_BATCH_SIZE = 100

with retrieval_records_path.open(encoding="utf-8") as records_file:
    records = [json.loads(line) for line in records_file]
records_by_item_id = {record["item_id"]: record for record in records}
with np.load(embedding_path, allow_pickle=True) as embedding_data:
    item_ids = [str(item_id) for item_id in embedding_data["item_ids"]]
    embeddings = embedding_data["embeddings"]
if len(records) != len(records_by_item_id):
    raise ValueError("Retrieval records must have unique item IDs.")
if len(item_ids) != len(records_by_item_id) or embeddings.shape[0] != len(item_ids):
    raise ValueError("Embedding vectors, item IDs, and retrieval records must have matching lengths.")
if set(item_ids) != set(records_by_item_id):
    raise ValueError("Embedding item IDs do not match the retrieval record item IDs.")

if "qdrant_client" in globals():
    qdrant_client.close()
qdrant_client = QdrantClient(path=str(QDRANT_PATH))
if qdrant_client.collection_exists(QDRANT_COLLECTION):
    qdrant_client.delete_collection(QDRANT_COLLECTION)
qdrant_client.create_collection(
    collection_name=QDRANT_COLLECTION,
    vectors_config=models.VectorParams(
        size=int(embeddings.shape[1]),
        distance=models.Distance.COSINE,
    ),
)

for start in range(0, len(item_ids), QDRANT_BATCH_SIZE):
    batch_item_ids = item_ids[start : start + QDRANT_BATCH_SIZE]
    points = [
        models.PointStruct(
            id=str(uuid5(NAMESPACE_URL, f"timss11-g4-math:{item_id}")),
            vector=embeddings[index].tolist(),
            payload=records_by_item_id[item_id],
        )
        for index, item_id in enumerate(batch_item_ids, start=start)
    ]
    qdrant_client.upsert(
        collection_name=QDRANT_COLLECTION,
        points=points,
        wait=True,
    )

print(
    f"Deployed {len(item_ids)} item vectors to collection "
    f"'{QDRANT_COLLECTION}' at {QDRANT_PATH.resolve()}"
)


In [ ]:
# check the first point in the collection
first_point = qdrant_client.retrieve(
    collection_name=QDRANT_COLLECTION,
    ids=[str(uuid5(NAMESPACE_URL, f"timss11-g4-math:{item_ids[0]}"))],
)
print(first_point)

### Filter the collection by metadata

Every point's payload carries the item's `assessment`, `grade`, and `subject`, such as `"TIMSS 2011"`, `4`, and `"Mathematics"`.
`metadata_filter` builds a Qdrant filter on any combination of them, for use with `query_points`, `scroll`, and `count`, so searches can stay within one assessment, grade, and subject as the collection grows.
Local Qdrant applies filters by scanning the payloads, which is fast at this size.
On a Qdrant server, also create payload indexes on these fields: keyword for `assessment` and `subject`, and integer for `grade`.

In [ ]:
from qdrant_client import models


def metadata_filter(assessment=None, grade=None, subject=None):
    """Qdrant filter that keeps the items with every given metadata value. None leaves that field unfiltered."""
    metadata = {"assessment": assessment, "grade": grade, "subject": subject}
    return models.Filter(
        must=[
            models.FieldCondition(key=key, match=models.MatchValue(value=value))
            for key, value in metadata.items()
            if value is not None
        ]
    )


for metadata in (
    {"assessment": "TIMSS 2011", "grade": 4, "subject": "Mathematics"},
    {"assessment": "TIMSS 2011", "grade": 8, "subject": "Mathematics"},
    {"subject": "Science"},
):
    matches = qdrant_client.count(QDRANT_COLLECTION, count_filter=metadata_filter(**metadata), exact=True).count
    print(f"{metadata}: {matches} items")

## TIMSS 2011 grade 4 USA data

Download the SPSS version of the [TIMSS 2011 International Database](https://timssandpirls.bc.edu/timss2011/international-database.html) (`T11_G4_SPSSData_pt3.zip`, countries Poland to Yemen), extract the six USA files, and convert each one with `src/timss_1/timss2011.py`.
The zip is downloaded only once.

Each file becomes `data/timss11_g4_usa/<file>.parquet` plus `<file>_codebook.csv` (label, format, missing values, and value labels for every variable).
Missing-value codes such as 96 (NOT REACHED) and 99 (OMITTED) are kept as values, as in SPSS, because TIMSS analyses often treat them differently.
`load(path, missing_to_nan=True)` turns them into NaN.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from timss_1.timss2011 import FILE_TYPES, convert, download, extract_country, load, variable_info

TIMSS_DATA_DIR = Path.cwd() / "data"
TIMSS_SPSS_ZIP = "T11_G4_SPSSData_pt3.zip"  # countries Poland to Yemen
COUNTRY = "usa"
USA_DIR = TIMSS_DATA_DIR / f"timss11_g4_{COUNTRY}"

zip_path = download(TIMSS_SPSS_ZIP, TIMSS_DATA_DIR / "raw")
sav_paths = extract_country(zip_path, COUNTRY, TIMSS_DATA_DIR / "raw" / zip_path.stem)
converted = []
for sav_path in sav_paths:
    rows, columns = convert(sav_path, USA_DIR).shape
    converted.append(
        {"file": sav_path.stem, "contents": FILE_TYPES[sav_path.stem[1:3]], "rows": rows, "columns": columns}
    )
print(f"Wrote Parquet files and codebooks to {USA_DIR}")
display(pd.DataFrame(converted))

### Load the converted data

`load` returns a pandas DataFrame, and `variable_info` returns each variable's label, format, value labels, and missing values.
Achievement item columns (such as `M031346A`) use the same IDs as the released items above.
NaN in an item column means the item was not in the student's booklet.

In [ ]:
achievement_path = USA_DIR / "asausam5.parquet"
achievement = load(achievement_path)
achievement_info = variable_info(achievement_path)
print(f"{len(achievement):,} students, {achievement.shape[1]} variables")
display(pd.read_csv(USA_DIR / "asausam5_codebook.csv").head())

# Responses to one item, with missing-value codes kept and turned into NaN.
item = "M031346A"
display(
    pd.DataFrame(
        {
            "codes kept": achievement[item].value_counts(dropna=False),
            "missing_to_nan=True": load(achievement_path, missing_to_nan=True)[item].value_counts(dropna=False),
        }
    )
    .rename(index=achievement_info[item]["value_labels"])
    .rename_axis(f"{item}: {achievement_info[item]['label']}")
    .astype("Int64")
)

In [ ]:
print(achievement.columns)

## TIMSS 2011 grade 4 mathematics item parameters

Download `T11_ItemParameters.zip` from the [TIMSS 2011 International Database](https://timssandpirls.bc.edu/timss2011/international-database.html), unzip it to `data/raw/T11_ItemParameters/`, and read the `MAT` sheet of `T11_G4_ItemParameters.xlsx`.
The sheet holds the IRT item parameters from the concurrent calibration of the 2007 and 2011 fourth-grade mathematics data, one row per item, with the estimate and standard error of each parameter.
`item_parameters` has one row per item: the item ID (such as `M031286`), its block position (such as `M01_01`), the sheet's item group, the slope, location, guessing, step 1, and step 2 estimates, and their standard errors (`*_se`).

A blank parameter does not apply to the item, and the pattern of blanks gives the IRT model (`irt_model`).
Multiple-choice items use the 3PL model and have a guessing parameter.
Two-point constructed-response items use the generalized partial credit model (GPCM) and have two step parameters.
One-point constructed-response items use the 2PL model and have neither.

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from timss_1.timss2011 import download

TIMSS_DATA_DIR = Path.cwd() / "data"
ITEM_PARAMETERS_ZIP = "T11_ItemParameters.zip"
ITEM_PARAMETERS_XLSX = "T11_G4_ItemParameters.xlsx"
ITEM_PARAMETERS_SHEET = "MAT"
# Sheet heading -> column name. Each parameter spans two columns: the estimate and its standard error.
ITEM_PARAMETERS = {
    "Slope (aj)": "slope",
    "Location (bj)": "location",
    "Guessing (cj)": "guessing",
    "Step 1 (dj1)": "step1",
    "Step 2 (dj2)": "step2",
}

item_parameters_zip = download(ITEM_PARAMETERS_ZIP, TIMSS_DATA_DIR / "raw")
item_parameters_dir = item_parameters_zip.with_suffix("")
with zipfile.ZipFile(item_parameters_zip) as archive:
    archive.extractall(item_parameters_dir)

# Row 1 is the title and row 2 the headings. Item rows follow, under group rows such as
# "Items Introduced in 2011:" that have nothing in the item ID column.
sheet = pd.read_excel(item_parameters_dir / ITEM_PARAMETERS_XLSX, sheet_name=ITEM_PARAMETERS_SHEET, header=None)
headings = sheet.iloc[1].tolist()
if headings[0] != "Item" or headings[2::2] != list(ITEM_PARAMETERS) or sheet.shape[1] != 2 + 2 * len(ITEM_PARAMETERS):
    raise ValueError(f"Unexpected headings in the {ITEM_PARAMETERS_SHEET} sheet: {headings}")
body = sheet.iloc[2:].dropna(how="all")
is_group_row = body[1].isna()
item_rows = body[~is_group_row]
parameter_names = list(ITEM_PARAMETERS.values())

item_parameters = pd.concat(
    [
        pd.DataFrame(
            {
                "item_id": item_rows[1],
                "block_position": item_rows[0],
                "item_group": body[0].where(is_group_row).ffill()[~is_group_row].str.strip(" :"),
            }
        ),
        item_rows.iloc[:, 2::2].set_axis(parameter_names, axis=1).apply(pd.to_numeric),
        item_rows.iloc[:, 3::2].set_axis([f"{name}_se" for name in parameter_names], axis=1).apply(pd.to_numeric),
    ],
    axis=1,
).reset_index(drop=True)
item_parameters.insert(
    3,
    "irt_model",
    np.select(
        [item_parameters["guessing"].notna(), item_parameters["step1"].notna()], ["3PL", "GPCM"], default="2PL"
    ),
)

problems = []
if not item_parameters["item_id"].str.fullmatch(r"M\d{6}[A-Z]?").all():
    problems.append("item IDs not shaped like M031286")
duplicated_ids = item_parameters["item_id"][item_parameters["item_id"].duplicated()]
if not duplicated_ids.empty:
    problems.append(f"duplicate item IDs {duplicated_ids.tolist()}")
for name in parameter_names:
    if not item_parameters[name].notna().equals(item_parameters[f"{name}_se"].notna()):
        problems.append(f"{name} estimates and standard errors are not present for the same items")
if item_parameters[["slope", "location"]].isna().any(axis=None):
    problems.append("items without a slope or location")
if not item_parameters["step1"].notna().equals(item_parameters["step2"].notna()):
    problems.append("items with only one step parameter")
if (item_parameters["guessing"].notna() & item_parameters["step1"].notna()).any():
    problems.append("items with both guessing and step parameters")
assert not problems, "\n".join(problems)

print(f"{len(item_parameters):,} grade 4 mathematics items")
display(pd.crosstab(item_parameters["item_group"], item_parameters["irt_model"], margins=True))
display(item_parameters.head())

In [ ]:
item_parameters_path = TIMSS_DATA_DIR / "timss11_g4_math_item_parameters.xlsx"
item_parameters.to_excel(item_parameters_path, index=False)
print(f"Exported {len(item_parameters):,} rows to {item_parameters_path.resolve()}")

In [ ]:
item_parameters.columns

### Add item parameters to the Qdrant collection

Store each item's row of `item_parameters` as an `item_parameters` object in the payload of its point in the local Qdrant collection, so search results carry the IRT parameters and filters can use them (for example a range on `item_parameters.location`).
Only items already in the collection are updated, and each of them must have parameters.
Parameters that do not apply to an item are stored as null.
Rerunning the deployment cell above rebuilds the collection without `item_parameters`, so rerun this cell after it.

In [ ]:
from qdrant_client import QdrantClient, models

QDRANT_PATH = TIMSS_DATA_DIR / "qdrant"
QDRANT_COLLECTION = "timss11_g4_math_items"

if "qdrant_client" in globals():
    qdrant_client.close()
qdrant_client = QdrantClient(path=str(QDRANT_PATH))


def collection_points(payload_fields):
    """Every point in the collection, with only the given payload fields."""
    offset = None
    while True:
        points, offset = qdrant_client.scroll(
            collection_name=QDRANT_COLLECTION, with_payload=payload_fields, limit=256, offset=offset
        )
        yield from points
        if offset is None:
            return


point_ids = {point.payload["item_id"]: point.id for point in collection_points(["item_id"])}
if len(point_ids) != qdrant_client.count(QDRANT_COLLECTION).count:
    raise ValueError(f"Item IDs in {QDRANT_COLLECTION} are not unique")
missing_ids = sorted(point_ids.keys() - set(item_parameters["item_id"]))
if missing_ids:
    raise ValueError(f"No item parameters for these items in {QDRANT_COLLECTION}: {missing_ids}")

# JSON has no NaN, so parameters that do not apply to an item are stored as null.
collection_parameters = item_parameters.set_index("item_id").loc[list(point_ids)]
parameter_payloads = (
    collection_parameters.astype(object).where(collection_parameters.notna(), None).to_dict("index")
)
qdrant_client.batch_update_points(
    collection_name=QDRANT_COLLECTION,
    update_operations=[
        models.SetPayloadOperation(
            set_payload=models.SetPayload(payload={"item_parameters": parameter_payloads[item_id]}, points=[point_id])
        )
        for item_id, point_id in point_ids.items()
    ],
)

stored_parameters = {
    point.payload["item_id"]: point.payload.get("item_parameters")
    for point in collection_points(["item_id", "item_parameters"])
}
assert stored_parameters == parameter_payloads, "item parameters in the collection differ from item_parameters"
print(f"Added item parameters to all {len(stored_parameters)} items in {QDRANT_COLLECTION}")
display(pd.DataFrame.from_dict(stored_parameters, orient="index").rename_axis("item_id").sort_index().head())

## Predict IRT parameters for new items with an LLM

`predict_item_parameters(item)` predicts the parameters a new grade 4 mathematics item would receive in the TIMSS 2011 concurrent calibration, in two steps:

1. `find_similar_items` embeds the item's stem and options exactly like the stored item vectors and returns the most similar calibrated items in the Qdrant collection from the TIMSS 2011 calibration (`CALIBRATION_ASSESSMENT`) with the same grade and subject (see `metadata_filter`), optionally only those scored with a given IRT model.
2. An LLM (`PARAMETER_LLM_MODEL`, through an OpenAI-compatible client) receives the similar items with their full text, percent correct, and calibrated parameters, plus a summary of the calibrated parameters for the new item's model, and returns the parameters through a structured-output schema.

The system prompt casts the LLM as an expert in testing grade 4 mathematics who is very familiar with the item's content domain, main topic, and cognitive domain, and as a measurement expert in IRT models.
The context always includes the most similar items scored with the new item's model, so the LLM sees calibrated examples of every parameter it predicts.

The IRT model follows the TIMSS scaling: 3PL for multiple-choice items, 2PL for constructed-response items worth 1 point, and the generalized partial credit model (GPCM) for constructed-response items worth up to 2 points.
For a GPCM item the LLM gives step 1, and step 2 is its negative, a constraint every calibrated GPCM item meets.

An item is a `TimssItem` with the same fields as the released-items JSONL, including its `assessment`, `grade`, and `subject`.
Items from TIMSS 2007 or TIMSS 2011 are accepted: the 2011 calibration covers the data of both assessments, so either is predicted on its scale.
`exclude_item_ids` keeps items out of the context, both as reference items and from the calibration summary.
Run the cells above that load the released items, define `metadata_filter`, build `item_parameters`, and add it to the Qdrant collection first.

In [56]:
import functools
import json
from typing import Literal

import numpy as np
import pandas as pd
import requests
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, Field, model_validator
from qdrant_client import models

required_globals = [
    "gemini_api_key",  # first cells
    "released_items_path",  # Build semantic-search item chunks
    "make_embedding_text",  # Build semantic-search item chunks
    "metadata_filter",  # Filter the collection by metadata
    "item_parameters",  # TIMSS 2011 grade 4 mathematics item parameters
    "qdrant_client",  # Add item parameters to the Qdrant collection
    "QDRANT_COLLECTION",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise NameError(f"Run the cells above that define {missing_globals} first")

# PARAMETER_LLM_MODEL = "gemini-3.8-flash"
# PARAMETER_LLM_MODEL = "nvidia/nemotron-3.5-lightning:free"
PARAMETER_LLM_MODEL = "llama3.2"
# Caps each prediction's output, reasoning included. Small local models sometimes repeat themselves inside the JSON
# until the context is full; the cap turns that into a quick LengthFinishReasonError instead of minutes of output.
PARAMETER_MAX_TOKENS = 8192

# A new item is embedded exactly like the stored item vectors, so its similarities to them are comparable.
with np.load(released_items_path.with_name("timss11_g4_math_item_embeddings.npz")) as saved_embeddings:
    ITEM_EMBEDDING_MODEL = str(saved_embeddings["model"])
ITEM_EMBEDDING_DIMENSION = qdrant_client.get_collection(QDRANT_COLLECTION).config.params.vectors.size
ITEM_EMBEDDING_TASK_TYPE = "RETRIEVAL_DOCUMENT"
# item_parameters and the item_parameters payloads in the collection come from the TIMSS 2011 concurrent
# calibration of the 2007 and 2011 data, so predictions are on its scale, whichever assessment an item is from.
CALIBRATION_ASSESSMENT = "TIMSS 2011"
# TIMSS 2011 mathematics framework definitions of the cognitive domains.
COGNITIVE_DOMAIN_DESCRIPTIONS = {
    "Knowing": "recalling and recognizing the facts, concepts, and procedures students need to know",
    "Applying": "applying knowledge and conceptual understanding to solve problems or answer questions",
    "Reasoning": "going beyond routine problems to unfamiliar situations, complex contexts, and multistep problems",
}


class ItemOption(BaseModel):
    label: str
    text: str


class ScoringGuideCategory(BaseModel):
    category: str
    descriptors: list[str]


class TimssItem(BaseModel):
    """A grade 4 mathematics item, with the same fields as the released-items JSONL."""

    item_id: str | None = None
    # The TIMSS 2011 calibration covers the 2007 and 2011 data, so items from either assessment are predicted on its
    # scale. item_parameters holds only grade 4 mathematics, so predictions accept only that grade and subject.
    assessment: Literal["TIMSS 2007", "TIMSS 2011"] = "TIMSS 2011"
    grade: Literal[4] = 4
    subject: Literal["Mathematics"] = "Mathematics"
    content_domain: Literal["Number", "Geometric Shapes and Measures", "Data Display"]
    main_topic: str
    cognitive_domain: Literal["Knowing", "Applying", "Reasoning"]
    item_type: Literal["multiple_choice", "constructed_response"]
    max_points: Literal[1, 2] = 1
    stem: str
    options: list[ItemOption] | None = None
    correct_option: str | None = None
    scoring_guide: list[ScoringGuideCategory] | None = None
    figure_description: str | None = None

    @model_validator(mode="after")
    def check_format(self):
        if self.item_type == "multiple_choice":
            labels = [option.label for option in self.options or []]
            if len(labels) < 2 or self.correct_option not in labels or self.max_points != 1:
                raise ValueError("a multiple-choice item needs options, a correct option among them, and max_points=1")
        elif self.options or self.correct_option:
            raise ValueError("a constructed-response item has no options or correct option")
        return self

    @property
    def irt_model(self):
        """3PL for multiple choice, 2PL for 1-point and GPCM for 2-point constructed response."""
        if self.item_type == "multiple_choice":
            return "3PL"
        return "GPCM" if self.max_points == 2 else "2PL"


reference_items = {
    record["item_id"]: TimssItem.model_validate(record)
    for record in map(json.loads, released_items_path.read_text(encoding="utf-8").splitlines())
}


@functools.cache
def embed_text(text):
    """Embedding of one text, cached so repeated searches for the same item call the API once."""
    response = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/{ITEM_EMBEDDING_MODEL}:embedContent",
        headers={"x-goog-api-key": gemini_api_key},
        json={
            "content": {"parts": [{"text": text}]},
            "taskType": ITEM_EMBEDDING_TASK_TYPE,
            "outputDimensionality": ITEM_EMBEDDING_DIMENSION,
        },
        timeout=120,
    )
    response.raise_for_status()
    return tuple(response.json()["embedding"]["values"])


def find_similar_items(item, limit=10, irt_model=None, exclude_item_ids=()):
    """Collection items most similar to `item` by stem and options, most similar first.

    Only CALIBRATION_ASSESSMENT items with the same grade and subject as `item` are searched, because item
    parameters from different calibrations are not on the same scale. `irt_model` keeps only items scored with
    that model. `exclude_item_ids` leaves items out, such as the item itself when checking predictions against
    calibrated items.
    """
    must = [metadata_filter(CALIBRATION_ASSESSMENT, item.grade, item.subject)]
    if irt_model is not None:
        must.append(models.FieldCondition(key="item_parameters.irt_model", match=models.MatchValue(value=irt_model)))
    must_not = []
    if exclude_item_ids:
        must_not.append(models.FieldCondition(key="item_id", match=models.MatchAny(any=list(exclude_item_ids))))
    return qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=list(embed_text(make_embedding_text(item.model_dump()))),
        query_filter=models.Filter(must=must, must_not=must_not),
        limit=limit,
        with_payload=True,
    ).points


def similar_items_table(points):
    """One row per point from find_similar_items: similarity, classification, and calibrated parameters."""
    return pd.DataFrame(
        [
            {
                "item_id": point.payload["item_id"],
                "similarity": point.score,
                **{key: point.payload[key] for key in ("content_domain", "main_topic", "cognitive_domain")},
                "pct_correct_intl_avg": point.payload["pct_correct_intl_avg"],
                "irt_model": point.payload["item_parameters"]["irt_model"],
                **{name: point.payload["item_parameters"][name] for name in PARAMETER_SYMBOLS},
            }
            for point in points
        ]
    ).astype({name: float for name in PARAMETER_SYMBOLS})


class TwoPLParameters(BaseModel):
    rationale: str = Field(
        description="Brief reasoning: the reference items that informed the prediction and the adjustments made."
    )
    slope: float = Field(ge=0, description="Slope (a).")
    location: float = Field(description="Location (b).")


class ThreePLParameters(TwoPLParameters):
    guessing: float = Field(ge=0, le=1, description="Guessing (c), the lower asymptote.")


class GPCMParameters(TwoPLParameters):
    step1: float = Field(description="Step 1 (d1). Step 2 is -d1.")


PARAMETER_SCHEMAS = {"3PL": ThreePLParameters, "2PL": TwoPLParameters, "GPCM": GPCMParameters}
PARAMETER_SYMBOLS = {"slope": "a", "location": "b", "guessing": "c", "step1": "d1", "step2": "d2"}


def parameter_system_prompt(item):
    subject = item.subject.lower()
    return f"""\
You are an expert in testing grade {item.grade} {subject}, very familiar with assessing grade {item.grade} students \
in the {item.content_domain} content domain, especially {item.main_topic}, and in the {item.cognitive_domain} \
cognitive domain ({COGNITIVE_DOMAIN_DESCRIPTIONS[item.cognitive_domain]}). You have written, reviewed, and \
analyzed many grade {item.grade} {subject} test items in these areas, and you know from experience how grade \
{item.grade} students across the TIMSS countries perform on them. You are also a measurement expert in item \
response theory (IRT). You know the 3PL, 2PL, and generalized partial credit (GPCM) models, how TIMSS \
calibrates items with them, and how content, cognitive demand, number of steps, the numbers involved, reading \
load, figures, response format, distractor plausibility, and scoring rules shape item parameters.

Your task is to predict the IRT parameters that a new {item.assessment} grade {item.grade} {subject} item would \
receive in the {CALIBRATION_ASSESSMENT} concurrent calibration of the 2007 and 2011 data, on the same scale as \
the calibrated reference items you are given.

TIMSS scales items with these models, where theta is student ability and 1.7 is the scaling constant:
- 3PL, for multiple-choice items: P(correct | theta) = c + (1 - c) / (1 + exp(-1.7 a (theta - b))).
- 2PL, for constructed-response items worth 1 point: P(correct | theta) = 1 / (1 + exp(-1.7 a (theta - b))).
- GPCM, for constructed-response items worth up to 2 points: P(score = k | theta) is proportional to \
exp(sum over v = 0..k of 1.7 a (theta - b + d_v)), with d_0 = 0 and d_2 = -d_1.

The parameters:
- Slope (a): discrimination, how sharply the chance of success rises with ability. Always positive.
- Location (b): overall difficulty on the theta scale. Higher is harder.
- Guessing (c), 3PL only: the chance that a student of very low ability answers correctly. It is usually \
below 1 divided by the number of options, because plausible distractors attract low-ability students.
- Step 1 (d1), GPCM only: the 0-to-1-point boundary is at theta = b - d1 and the 1-to-2-point boundary at \
theta = b + d1. A positive d1 means partial credit is reached well before full credit. A negative d1 means \
the boundaries are reversed, so partial credit is rarely the most likely score. Step 2 is -d1, so give d1 only.

How to work:
1. Study the reference items. They are calibrated {CALIBRATION_ASSESSMENT} items chosen for their similarity to the \
new item. Their percent correct shows how hard each one was for real students, and their parameters anchor the scale.
2. Compare the new item with the most relevant reference items on content, cognitive demand, number and \
difficulty of steps, the numbers involved, reading load, and figures, and also on distractor plausibility \
for multiple-choice items or on how demanding the scoring guide is for constructed-response items. Decide \
whether the new item should be easier or harder, and more or less discriminating, than each of them.
3. Treat the calibration summary for the new item's model as a prior, and stay within its range unless the \
new item is clearly unusual.
4. Combine the evidence instead of copying the parameters of one reference item.

The new item is scored with the {item.irt_model} model. In the rationale, briefly name the reference items \
that informed you and the adjustments you made, then give the parameters."""


def describe_item(item):
    """Plain-text rendering of an item, the same for reference items and the new item."""
    item_format = item.item_type.replace("_", " ")
    lines = [
        f"Content domain: {item.content_domain}; main topic: {item.main_topic}; "
        f"cognitive domain: {item.cognitive_domain}",
        f"Format: {item_format}, worth {'up to 2 points' if item.max_points == 2 else '1 point'}, "
        f"scored with the {item.irt_model} model",
        f"Stem:\n{item.stem}",
    ]
    if item.figure_description:
        lines.append(f"Figure description: {item.figure_description}")
    if item.options:
        lines.append("Options:\n" + "\n".join(f"{option.label}. {option.text}" for option in item.options))
        lines.append(f"Correct option: {item.correct_option}")
    if item.scoring_guide:
        lines.append(
            "Scoring guide:\n"
            + "\n".join(f"- {category.category}: {' | '.join(category.descriptors)}" for category in item.scoring_guide)
        )
    return "\n".join(lines)


def calibration_summary(item, exclude_item_ids=()):
    """Distribution of the parameters of the calibrated items scored with the same model as `item`, except
    `exclude_item_ids`."""
    names = [name for name in PARAMETER_SCHEMAS[item.irt_model].model_fields if name != "rationale"]
    included = (item_parameters["irt_model"] == item.irt_model) & ~item_parameters["item_id"].isin(exclude_item_ids)
    calibrated = item_parameters.loc[included, names]
    summary = (
        calibrated.rename(columns=lambda name: f"{name} ({PARAMETER_SYMBOLS[name]})")
        .describe(percentiles=[0.1, 0.5, 0.9])
        .drop(index="count")
        .round(3)
    )
    return (
        f"Calibration summary of {len(calibrated)} {CALIBRATION_ASSESSMENT} grade {item.grade} "
        f"{item.subject.lower()} items scored with the {item.irt_model} model:\n{summary.to_string()}"
    )


def parameter_user_prompt(item, references, exclude_item_ids=()):
    blocks = [calibration_summary(item, exclude_item_ids), "Reference items, most similar first:"]
    for rank, point in enumerate(references, start=1):
        payload = point.payload
        if "item_parameters" not in payload:
            raise ValueError(f"{payload['item_id']} has no item parameters; run 'Add item parameters to the Qdrant collection'")
        parameters = ", ".join(
            f"{name} ({PARAMETER_SYMBOLS[name]}) = {value:.3f}"
            for name, value in payload["item_parameters"].items()
            if name in PARAMETER_SYMBOLS and value is not None
        )
        blocks.append(
            f"### Reference item {rank}: {payload['item_id']} (cosine similarity {point.score:.3f})\n"
            f"{describe_item(reference_items[payload['item_id']])}\n"
            f"Percent correct in 2011: international average {payload['pct_correct_intl_avg']}, "
            f"United States {payload['pct_correct_usa']}\n"
            f"Calibrated parameters: {parameters}"
        )
    blocks.append(f"### New item\n{describe_item(item)}")
    return "\n\n".join(blocks)


gemini_client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=gemini_api_key, max_retries=5, timeout=600
)

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key, max_retries=5, timeout=600
)

ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def predict_item_parameters(
    item, n_similar=10, n_same_model=5, exclude_item_ids=(), model=PARAMETER_LLM_MODEL, client=ollama_client
):
    """Predict the IRT parameters of `item` with an LLM, using similar calibrated items as context.

    The context holds the `n_similar` most similar items of any format plus the `n_same_model` most similar
    items scored with the same model as `item`, so the LLM always sees calibrated examples of the
    parameters it predicts. `exclude_item_ids` are left out of both the reference items and the calibration
    summary. `model` is requested from `client`, an OpenAI-compatible client such as `ollama_client`,
    `gemini_client`, or `openrouter_client`. Returns one row for a predictions dataframe.
    """
    nearest = find_similar_items(item, n_similar, exclude_item_ids=exclude_item_ids)
    nearest_same_model = find_similar_items(
        item, n_same_model, irt_model=item.irt_model, exclude_item_ids=exclude_item_ids
    )
    references = sorted(
        {point.id: point for point in [*nearest, *nearest_same_model]}.values(),
        key=lambda point: point.score,
        reverse=True,
    )
    if not references:
        raise ValueError(f"No {CALIBRATION_ASSESSMENT} grade {item.grade} {item.subject} items in {QDRANT_COLLECTION}")
    completion = client.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": parameter_system_prompt(item)},
            {"role": "user", "content": parameter_user_prompt(item, references, exclude_item_ids)},
        ],
        response_format=PARAMETER_SCHEMAS[item.irt_model],
        max_tokens=PARAMETER_MAX_TOKENS,
    )
    prediction = completion.choices[0].message.parsed
    if prediction is None:
        raise RuntimeError(f"No parsed prediction (finish_reason={completion.choices[0].finish_reason})")
    parameters = dict.fromkeys(PARAMETER_SYMBOLS) | prediction.model_dump(exclude={"rationale"})
    if item.irt_model == "GPCM":
        parameters["step2"] = -parameters["step1"]
    return {
        "item_id": item.item_id,
        "irt_model": item.irt_model,
        **parameters,
        "rationale": prediction.rationale,
        "reference_item_ids": [point.payload["item_id"] for point in references],
    }

### Predict a new item

A made-up multiple-choice item, scored with the 3PL model.

In [57]:
new_item = TimssItem(
    assessment="TIMSS 2011",
    grade=4,
    subject="Mathematics",
    content_domain="Number",
    main_topic="Whole Numbers",
    cognitive_domain="Applying",
    item_type="multiple_choice",
    stem="Maria had 52 stickers. She gave 18 stickers to her brother. How many stickers does Maria have now?",
    options=[
        {"label": "A", "text": "34"},
        {"label": "B", "text": "44"},
        {"label": "C", "text": "46"},
        {"label": "D", "text": "70"},
    ],
    correct_option="A",
)
print(f"Most similar calibrated items ({new_item.irt_model} model):")
display(similar_items_table(find_similar_items(new_item, limit=5)).round(3))

new_item_prediction = predict_item_parameters(new_item)
display(pd.DataFrame([new_item_prediction]).drop(columns=["item_id", "rationale"]).round(3))
print(new_item_prediction["rationale"])

Most similar calibrated items (3PL model):


,item_id,similarity,content_domain,main_topic,cognitive_domain,pct_correct_intl_avg,irt_model,slope,location,guessing,step1,step2
0,M031317,0.862,Number,Number Sentences with Whole Numbers,Knowing,39,3PL,1.278,0.641,0.147,NaN,NaN
1,M041107,0.857,Number,Number Sentences with Whole Numbers,Applying,78,3PL,0.849,-1.328,0.171,NaN,NaN
2,M041011,0.829,Number,Whole Numbers,Knowing,66,3PL,1.100,-0.402,0.212,NaN,NaN
3,M031187,0.823,Number,Number Sentences with Whole Numbers,Applying,73,3PL,0.679,-0.897,0.198,NaN,NaN
4,M031009,0.818,Number,Whole Numbers,Applying,34,2PL,0.796,0.484,NaN,NaN,NaN


,irt_model,slope,location,guessing,step1,step2,reference_item_ids
0,3PL,1.343,-0.651,0.136,None,None,"[M031317, M041107, M041011, M031187, M031009, ..."


The new item is similar to reference item 4 (cosine similarity 0.823), M031187.  Since the stem is a subtraction question, the new item is harder than M031187. The number of stickers Maria had is lower than the number she gave to her brother.  The new item should have a higher slope (a) parameter than M031187.  The correct option for the new item should be lower than the correct option for M031187.


### Check predictions against calibrated items

Predict one calibrated item per IRT model as if it were new and compare the prediction with its calibrated parameters.
Each item is left out of its own context, along with the other parts of the same item, which share its stimulus.

In [58]:
# The first calibrated item of each model, predicted with it and the other parts of the same item
# (such as M031346A and M031346B for M031346C) left out of the context.
check_item_ids = (
    item_parameters[item_parameters["item_id"].isin(reference_items)].groupby("irt_model")["item_id"].first().tolist()
)
calibrated_parameters = item_parameters.set_index("item_id")
comparison = []
for item_id in check_item_ids:
    same_stimulus = [other_id for other_id in reference_items if other_id[:7] == item_id[:7]]
    prediction = predict_item_parameters(reference_items[item_id], exclude_item_ids=same_stimulus)
    comparison.extend(
        {
            "item_id": item_id,
            "irt_model": prediction["irt_model"],
            "parameter": name,
            "predicted": prediction[name],
            "calibrated": calibrated_parameters.loc[item_id, name],
        }
        for name in PARAMETER_SYMBOLS
        if prediction[name] is not None
    )
comparison = pd.DataFrame(comparison)
comparison["difference"] = comparison["predicted"] - comparison["calibrated"]
display(comparison.round(3))

,item_id,irt_model,parameter,predicted,calibrated,difference
0,M031346A,2PL,slope,1.062,1.540,-0.478
1,M031346A,2PL,location,-0.101,-0.516,0.415
2,M031083,3PL,slope,0.944,0.884,0.060
3,M031083,3PL,location,0.235,-0.566,0.801
4,M031083,3PL,guessing,0.101,0.212,-0.111
5,M031346C,GPCM,slope,1.051,1.313,-0.262
6,M031346C,GPCM,location,-0.341,0.147,-0.488
7,M031346C,GPCM,step1,-0.734,0.474,-1.208
8,M031346C,GPCM,step2,0.734,-0.474,1.208


## TIMSS 2007 grade 4 released mathematics items

Build `data/timss07_g4_math_released_items.jsonl` (one item per line, with the same fields as `data/timss11_g4_math_released_items.jsonl`) from the zip linked as "Mathematics" under "Grade 4 Released Items" on the [TIMSS 2007 Released Items](https://timssandpirls.bc.edu/timss2007/items.html) page.

The PDF in the zip is laid out differently from the 2011 NCES booklet.
Each item page carries the item documentation as text (item ID, block, block sequence, content domain, cognitive domain, maximum points, and key), but the item itself is a scanned image under a red copyright watermark.
Each constructed-response item is followed by a page whose only image is its scoring guide, a table of score codes.
The PDF has no accessibility transcription (`accessibility_text` is null), item label, topic area, or percent correct.
"Write the JSONL" fills `item_label` and `main_topic` from the TIMSS 2007 item information file and `pct_correct_intl_avg`, `pct_correct_usa`, and `pct_correct` from NCES's item statistics workbook.

Extraction is hybrid:

- **Deterministic, from the PDF:** the item documentation, cross-checked against the item table on PDF pages 4-5, and the item and scoring-guide images, taken from the PDF as embedded, without the watermark.
- **`gemini-3.8-flash` through OpenRouter (or the Gemini API, in the alternative cell at the end of the section):** `stem`, `options`, `scoring_guide`, `scoring_notes`, `has_figure`, and `figure_description`, read from those images and checked against the documentation before they are kept.

`figure_description` is model-generated.

In [ ]:
import base64
import json
import re
import time
import zipfile
from bisect import bisect_right
from collections import defaultdict
from itertools import groupby
from pathlib import Path

import numpy as np
import openai
import pandas as pd
import pydantic
import pymupdf
import requests
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, Field

from timss_1 import released_items as ri

# The "Mathematics" link under "Grade 4 Released Items" on https://timssandpirls.bc.edu/timss2007/items.html
T07_ZIP_URL = "https://timssandpirls.bc.edu/timss2007/PDF/T07_G4_Released_Items_MAT.zip"
T07_DATA_DIR = Path.cwd() / "data"
T07_ZIP_PATH = T07_DATA_DIR / "raw" / "T07_G4_Released_Items_MAT.zip"
T07_PDF_PATH = T07_ZIP_PATH.with_suffix("") / "T07_G4_Released_Items_MAT.pdf"
T07_OUTPUT_PATH = T07_DATA_DIR / "timss07_g4_math_released_items.jsonl"

if not T07_ZIP_PATH.exists():
    response = requests.get(T07_ZIP_URL, timeout=120)
    response.raise_for_status()
    if not response.content.startswith(b"PK"):
        raise ValueError(f"{T07_ZIP_URL} did not return a zip file")
    T07_ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    T07_ZIP_PATH.write_bytes(response.content)
if not T07_PDF_PATH.exists():
    with zipfile.ZipFile(T07_ZIP_PATH) as archive:
        archive.extract(T07_PDF_PATH.name, T07_PDF_PATH.parent)

t07_pdf = pymupdf.open(T07_PDF_PATH)
print(f"{T07_PDF_PATH.name}: {t07_pdf.page_count} pages")

### Deterministic parse of the item documentation and images

The item table on PDF pages 4-5 lists every item with its subject, grade, block, block sequence, content domain, cognitive domain, maximum points, and key ("See scoring guide" for constructed-response items).
Each item page repeats that documentation in its header and sidebar, and each scoring-guide page names its item in the header.
MP31350 is not an item: it is the map and instructions shared by the parts of the Marytown item M031350A-C, so its documentation is blank apart from the block and block sequence.
The parts of a multi-part item share a block and block sequence, and their pages show the same booklet pages (the shared introduction and every part), so each part is transcribed from all of them.

In [ ]:
T07_VISIBLE_TEXT = pymupdf.TEXTFLAGS_TEXT | pymupdf.TEXT_IGNORE_ACTUALTEXT
T07_VISIBLE_WORDS = pymupdf.TEXTFLAGS_WORDS | pymupdf.TEXT_IGNORE_ACTUALTEXT
T07_ITEM_ID = re.compile(r"M(?:P\d{5}|\d{6})[A-Z]?")
T07_TABLE_PAGES = range(3, 5)  # PDF pages 4-5
T07_FOOTER_Y = 740
T07_FIELDS = [
    "Item ID",
    "Subject",
    "Grade",
    "Block",
    "Block Seq",
    "Content Domain",
    "Cognitive Domain",
    "Maximum Points",
    "Key",
]
# First word of each item-table column heading ("Block" starts both "Block" and "Block Seq").
T07_COLUMN_HEADINGS = {"Item", "Subject", "Grade", "Block", "Content", "Cognitive", "Maximum", "Key"}
T07_CONTENT_DOMAINS = {"Number", "Geometric Shapes and Measures", "Data Display"}
T07_COGNITIVE_DOMAINS = {"Knowing", "Applying", "Reasoning"}
T07_CONSTRUCTED_RESPONSE_KEY = "See scoring guide"


def t07_lines(words, tolerance=4):
    """Group words into lines by vertical center, each line left to right."""
    lines = []
    for w in sorted(words, key=lambda w: (w[1] + w[3]) / 2):
        center = (w[1] + w[3]) / 2
        if lines and center - lines[-1][0] < tolerance:
            lines[-1][1].append(w)
        else:
            lines.append((center, [w]))
    return [sorted(line, key=lambda w: w[0]) for _, line in lines]


def t07_table_rows(page):
    """Item-table rows as {field: text}; each word belongs to the column whose heading starts left of it."""
    words = page.get_text("words", flags=T07_VISIBLE_WORDS)
    first_row_y = min(w[1] for w in words if T07_ITEM_ID.fullmatch(w[4]))
    column_x = sorted(w[0] for w in words if w[1] < first_row_y - 5 and w[4] in T07_COLUMN_HEADINGS)
    assert len(column_x) == len(T07_FIELDS), f"item-table columns on page {page.number + 1}: {column_x}"
    rows = []
    for line in t07_lines(w for w in words if first_row_y - 5 <= w[1] < T07_FOOTER_Y):
        row = defaultdict(list)
        for w in line:
            row[T07_FIELDS[bisect_right(column_x, w[0] + 1) - 1]].append(w[4])
        rows.append({field: " ".join(texts) for field, texts in row.items()})
    return rows


def t07_page_fields(page):
    """Documentation on an item or scoring-guide page: the header's label/value pairs and the sidebar's labeled boxes."""
    fields = {}
    for block in page.get_text("blocks", flags=T07_VISIBLE_TEXT):
        lines = [line.strip() for line in block[4].splitlines() if line.strip()]
        if lines[:1] == ["Item ID"]:
            fields.update(zip(lines[::2], lines[1::2]))
        elif lines and lines[0] in T07_FIELDS:
            fields[lines[0]] = " ".join(lines[1:])
    return fields


def t07_booklet_page(page):
    """The printed page number: the first word of the footer on left-hand pages, the last on right-hand ones."""
    footer = sorted((w for w in page.get_text("words", flags=T07_VISIBLE_WORDS) if w[1] >= T07_FOOTER_Y), key=lambda w: w[0])
    return int(next(w[4] for w in (footer[0], footer[-1]) if w[4].isdigit()))


def t07_main_image(page):
    """xref of the largest image on the page, the item or the scoring guide; the others are logos."""
    return max(page.get_image_info(xrefs=True), key=lambda info: abs(pymupdf.Rect(info["bbox"])))["xref"]


t07_table = [row for number in T07_TABLE_PAGES for row in t07_table_rows(t07_pdf[number])]

# Item pages (including the shared introduction MP31350) and scoring-guide pages, after the item table.
t07_pages = {}
t07_scoring_images = defaultdict(list)
for page in t07_pdf.pages(T07_TABLE_PAGES[-1] + 1):
    fields = t07_page_fields(page)
    if "Item ID" not in fields:
        continue
    if "Key" in fields:
        assert fields["Item ID"] not in t07_pages, f"second item page for {fields['Item ID']}"
        t07_pages[fields["Item ID"]] = {
            **fields,
            "pdf_page": page.number + 1,
            "booklet_page": t07_booklet_page(page),
            "image": t07_main_image(page),
        }
    else:
        t07_scoring_images[fields["Item ID"]].append(t07_main_image(page))

assert [row["Item ID"] for row in t07_table] == list(t07_pages), "item table and item pages list different items"
for row in t07_table:
    page_fields = t07_pages[row["Item ID"]]
    mismatches = [f for f in T07_FIELDS if row.get(f, "") != page_fields.get(f, "")]
    assert not mismatches, f"{row['Item ID']}: item table and item page differ in {mismatches}"

# Booklet page images of each multi-part item (by block and block sequence), in page order.
t07_unit_images = defaultdict(list)
for doc in t07_pages.values():
    images = t07_unit_images[doc["Block"], doc["Block Seq"]]
    if doc["image"] not in images:
        images.append(doc["image"])

t07_items = []
for item_id, doc in t07_pages.items():
    if not doc["Key"]:
        assert item_id.startswith("MP") and not (doc["Content Domain"] or doc["Maximum Points"]), item_id
        continue  # shared introduction of a multi-part item
    unit = [i for i, d in t07_pages.items() if (d["Block"], d["Block Seq"]) == (doc["Block"], doc["Block Seq"]) and d["Key"]]
    is_multiple_choice = doc["Key"] != T07_CONSTRUCTED_RESPONSE_KEY
    assert (doc["Subject"], doc["Grade"]) == ("M", "4"), item_id
    assert doc["Content Domain"] in T07_CONTENT_DOMAINS and doc["Cognitive Domain"] in T07_COGNITIVE_DOMAINS, item_id
    assert doc["Maximum Points"] in {"1", "2"}, item_id
    assert not is_multiple_choice or (doc["Key"] in set("ABCD") and doc["Maximum Points"] == "1"), item_id
    assert len(t07_scoring_images[item_id]) == (0 if is_multiple_choice else 1), f"{item_id}: scoring-guide pages"
    assert len(unit) == 1 or item_id[-1].isalpha(), f"{item_id}: part of a multi-part item without a part letter"
    t07_items.append(
        {
            "item_id": item_id,
            "block": doc["Block"],
            "block_seq": doc["Block Seq"],
            "parts": unit if len(unit) > 1 else None,
            "part": item_id[-1] if len(unit) > 1 else None,
            "content_domain": doc["Content Domain"],
            "cognitive_domain": doc["Cognitive Domain"],
            "item_type": "multiple_choice" if is_multiple_choice else "constructed_response",
            "max_points": int(doc["Maximum Points"]),
            "correct_option": doc["Key"] if is_multiple_choice else None,
            "booklet_page": doc["booklet_page"],
            "pdf_page": doc["pdf_page"],
            "page_images": t07_unit_images[doc["Block"], doc["Block Seq"]],
            "scoring_image": t07_scoring_images[item_id][0] if not is_multiple_choice else None,
        }
    )

t07_items_df = pd.DataFrame(t07_items)
print(f"Parsed {len(t07_items)} items")
display(t07_items_df.groupby(["content_domain", "item_type"]).size().unstack(fill_value=0))
print("Parts of multi-part items, with the number of booklet page images each is transcribed from:")
display(
    t07_items_df[t07_items_df["part"].notna()]
    .assign(page_images=lambda df: df["page_images"].map(len))[
        ["item_id", "block", "block_seq", "item_type", "max_points", "page_images", "pdf_page"]
    ]
)

### Transcribe stems, options, and scoring guides with Gemini

`gemini-3.8-flash` is called through OpenRouter (`google/gemini-3.8-flash`) with `openrouter_api_key`, by `ri.Transcriber` (in `src/timss_1/released_items.py`), which every section that transcribes through OpenRouter uses.
That avoids the Gemini API free tier, which allows only 20 requests per day and on 2026-10-02 rejected every image request with 503 "high demand".
OpenRouter bills per token (about $0.03 per request of 4 items, so about $0.50 for the whole set), and the cell prints each request's cost.

Each request covers up to `T07_ITEMS_PER_REQUEST` items in booklet order, keeping the parts of a multi-part item together, which makes 20 requests for the whole set.
It sends every item's booklet page image (all of them for a part of a multi-part item, once per request) and, for a constructed-response item, its scoring guide.
Requests go only to providers that support structured output (`require_parameters`), one at a time, and nothing is retried: the first failed request (an error status, or OpenRouter's HTTP 200 with `finish_reason="error"`) stops the cell, and rerunning it requests only the items without a usable transcription.
Cached transcriptions are checked again on every run, so checks added later apply to them too.

A transcription is kept only if it passes the checks against the documentation in `t07_problems`: options A-D exactly for multiple-choice items, the item ID printed on the scoring guide, the scoring categories and code ranges implied by the maximum points, and only the item's own part label in the stem of a part.
Kept transcriptions are cached per item under `data/cache/item_transcriptions/timss07_g4_math/gemini-3.8-flash-v1/`, shared with the Gemini API cell at the end of this section, so either route reuses what the other has done.
Bump `T07_PROMPT_VERSION` (in both cells) after editing the prompt or schema to re-extract everything, or delete one item's file to re-extract only that item.

In [ ]:
LLM_MODEL = "google/gemini-3.8-flash"
T07_PROMPT_VERSION = 1
# Named after the model without OpenRouter's "google/" prefix, so this cell and the Gemini API cell at the end of the
# section share one cache: same model, prompt, and schema.
T07_LLM_CACHE_DIR = (
    T07_DATA_DIR
    / "cache"
    / "item_transcriptions"
    / "timss07_g4_math"
    / f"{LLM_MODEL.split('/')[-1]}-v{T07_PROMPT_VERSION}"
)
T07_ITEMS_PER_REQUEST = 4
# Codes each printed scoring-guide category may hold, by the item's maximum points: the first digit of a TIMSS
# score code is the score, except 7 (incorrect) and 9 (nonresponse).
T07_CATEGORY_CODES = {
    "Correct Response": {1: range(10, 20), 2: range(20, 30)},
    "Partial Response": {2: range(10, 20)},
    "Incorrect Response": {1: range(70, 80), 2: range(70, 80)},
    "Nonresponse": {1: range(90, 100), 2: range(90, 100)},
}


class T07Option(BaseModel):
    label: str = Field(description="Option letter as printed, e.g. 'A'.")
    text: str = Field(description="Option content in the same notation as the stem.")


class T07ScoringCode(BaseModel):
    code: int = Field(description="Score code as printed, e.g. 10.")
    response: str = Field(description="The code's response description, verbatim.")


class T07ScoringCategory(BaseModel):
    category: str = Field(description="Category heading as printed, e.g. 'Correct Response'.")
    codes: list[T07ScoringCode] = Field(description="Each code row under the heading, in printed order.")


class T07Transcription(BaseModel):
    item_id: str
    stem: str
    options: list[T07Option]
    scoring_guide_item_id: str | None
    scoring_guide: list[T07ScoringCategory]
    scoring_notes: str | None
    has_figure: bool
    figure_description: str | None


T07_TRANSCRIPTION_INSTRUCTIONS = """\
You transcribe items from the TIMSS 2007 grade 4 released mathematics items into structured data.

Each request covers several items, listed by ID. For each item you receive:
1. The booklet page the item appears on, as an image, exactly as students saw it. The parts of a multi-part item \
(IDs ending in A, B, C, ...) share their booklet pages, which are shown once, before the parts. Those pages hold an \
introduction shared by all parts, which may fill a page of its own, followed by every part, each starting with its \
part label (e.g. "B.").
2. For a constructed-response item, its scoring guide as an image: a table of score codes headed by the item ID.

The images are the only source. Transcribe what is printed, verbatim; never add, correct, or solve anything.

Return one entry per item, in the order given:
- item_id: The item ID as given.
- stem: Everything the student sees for this item except the multiple-choice response options, in reading order. \
For a part of a multi-part item, the shared introduction (everything before the first part label) followed by this \
part only, starting with its part label; leave out the other parts. Leave out page furniture: the item ID printed \
sideways in the margin, and navigation lines such as "This ... question continues on the next page." or "End of ... \
section.". Write math in plain-text notation: fractions as 1/2, mixed numbers as 1 1/2, and the symbols ×, ÷, −, =, \
°. Write an empty box in a number sentence as □ and an answer blank as ____, keeping labels such as "Answer:". Write \
tables as Markdown tables and bulleted lists as "- " lines. Replace each picture, diagram, graph, grid, or map with \
"[Image: <short description>]". Separate paragraphs with a newline.
- options: For a multiple-choice item, one entry per response option in printed order, labeled with its circled \
letter, in the same notation as the stem; a pictured option is "[Image: <description>]". Empty for \
constructed-response items.
- scoring_guide_item_id: The item ID printed in the header of the scoring guide ("Item: ..."); null for \
multiple-choice items.
- scoring_guide: For a constructed-response item, each category of the scoring guide in printed order (e.g. \
"Correct Response", "Partial Response", "Incorrect Response", "Nonresponse"), with its heading verbatim, except that \
"Partially Correct Response" is written "Partial Response", and each code row under it: the score code and its \
response description, verbatim, in the same notation as the stem. A picture that illustrates a code, such as a \
sample answer in the row below it, belongs to that code's response, as "[Image: <description>]". Empty for \
multiple-choice items.
- scoring_notes: Any scoring-guide text outside the code table, such as a note; null if none.
- has_figure: Whether this item's stem or options contain any picture, diagram, graph, grid, map, or drawn shape. A \
table of text or numbers alone does not count.
- figure_description: If has_figure, a precise, objective description of every visual element in this item's stem \
and options, giving all the information a student would need to answer without seeing it: counts, labels, \
measurements, positions, grid coordinates, bar heights, shading, and orientation. Describe what is drawn; do not \
solve the item or reveal the answer. Null if has_figure is false.
"""


def t07_image_part(xref):
    return ri.embedded_image_part(t07_pdf, xref)


def t07_request_content(batch):
    """The user message for a batch: each item's booklet page images (once per multi-part item) and scoring guide."""
    content = [
        {"type": "text", "text": f"Transcribe these {len(batch)} items, in this order: {', '.join(i['item_id'] for i in batch)}."}
    ]
    shown_units = set()
    for item in batch:
        points = f"{item['max_points']} point{'s' if item['max_points'] > 1 else ''}"
        description = f"{item['item_type'].replace('_', '-')}, worth {points}"
        if item["parts"]:
            if item["parts"][0] not in shown_units:
                shown_units.add(item["parts"][0])
                content.append(
                    {
                        "type": "text",
                        "text": f"Booklet pages of the multi-part item with parts {', '.join(item['parts'])}, in order:",
                    }
                )
                content += [t07_image_part(xref) for xref in item["page_images"]]
            content.append(
                {"type": "text", "text": f"Item {item['item_id']}: part {item['part']} of that multi-part item, {description}."}
            )
        else:
            content.append({"type": "text", "text": f"Item {item['item_id']}: {description}. Booklet page:"})
            content.append(t07_image_part(item["page_images"][0]))
        if item["scoring_image"]:
            content.append({"type": "text", "text": f"Scoring guide of item {item['item_id']}:"})
            content.append(t07_image_part(item["scoring_image"]))
    return content


def t07_problems(item, t, returned):
    """Disagreements between a transcription and the item's documentation; empty if the transcription is usable."""
    problems = []
    if not t.stem.strip():
        problems.append("empty stem")
    if item["item_type"] == "multiple_choice":
        if [o.label for o in t.options] != ["A", "B", "C", "D"]:
            problems.append(f"option labels {[o.label for o in t.options]}")
        if t.scoring_guide or t.scoring_guide_item_id:
            problems.append("multiple-choice item has a scoring guide")
    else:
        if t.options:
            problems.append("constructed-response item has options")
        if t.scoring_guide_item_id != item["item_id"]:
            problems.append(f"scoring guide headed {t.scoring_guide_item_id!r}")
        expected = [c for c, codes in T07_CATEGORY_CODES.items() if item["max_points"] in codes]
        categories = [c.category for c in t.scoring_guide]
        if categories != expected:
            problems.append(f"scoring categories {categories}, expected {expected} for {item['max_points']} points")
        for c in t.scoring_guide:
            allowed = T07_CATEGORY_CODES.get(c.category, {}).get(item["max_points"], range(0))
            codes = [code.code for code in c.codes]
            if not codes or any(code not in allowed for code in codes):
                problems.append(f"{c.category} codes {codes}")
    if item["part"]:
        labels = re.findall(r"(?m)^([A-Z])\.\s", t.stem)
        if labels != [item["part"]]:
            problems.append(f"part labels {labels} in the stem")
    if t.has_figure != bool(t.figure_description):
        problems.append(f"has_figure={t.has_figure} but figure_description={t.figure_description!r}")
    return problems


t07_transcriber = ri.Transcriber(
    api_key=globals().get("openrouter_api_key"),
    model=LLM_MODEL,
    cache_dir=T07_LLM_CACHE_DIR,
    schema=T07Transcription,
    instructions=T07_TRANSCRIPTION_INSTRUCTIONS,
)
t07_transcriptions = t07_transcriber.run(
    t07_items,
    request_content=t07_request_content,
    problems=t07_problems,
    items_per_request=T07_ITEMS_PER_REQUEST,
    unit=lambda item: (item["block"], item["block_seq"]),
)
print(f"All {len(t07_transcriptions)} items are transcribed")

### Write the JSONL

`item_label` and `main_topic` (the 2007 framework's topic area) come from `T07_G4_ItemInformation.xls` in `T07_Items.zip` (the "Items" link on the [TIMSS 2007 International Database and User Guide](https://timssandpirls.bc.edu/TIMSS2007/idb_ug.html) page), read by `ri.timss_2007_math_item_information`, which gives the topic areas their framework names; the file must agree with the booklet on each item's content domain, cognitive domain, maximum points, and key.
`pct_correct_intl_avg`, `pct_correct_usa`, and `pct_correct` come from NCES's "TIMSS Grade 4 mathematics item statistics" workbook (TIMSS 2007 on the [NCES released-questions page](https://nces.ed.gov/timss/released-questions.asp)), read by `ri.nces_2007_percent_correct`; the workbook compares each result with the U.S. average by cell color, not with the international average, so `vs_intl_avg` is null.
`scoring_guide` keeps each printed category with its response descriptions in printed order; the two-digit score codes (such as 10, 70, and 99) are used only by the checks above, matching the 2011 file, whose scoring guides have none.
`ri.write_jsonl` checks that every record has the fields of the 2011 file, in the same order.

In [ ]:
from timss_1 import released_items as ri

# Label and topic area from the item information file of the TIMSS 2007 International Database ("Items" on
# https://timssandpirls.bc.edu/TIMSS2007/idb_ug.html), which must agree with the booklet's documentation.
T07_ITEMS_ZIP_URL = "https://timssandpirls.bc.edu/TIMSS2007/PDF/T07_Items.zip"
# Percent correct from the NCES statistics workbook for the TIMSS 2007 grade 4 released items ("TIMSS Grade 4
# mathematics item statistics" under TIMSS 2007 on https://nces.ed.gov/timss/released-questions.asp).
T07_STATISTICS_URL = "https://nces.ed.gov/timss/xls/TIMSS_Grade4_Math_Released_Item_Statistics.xlsx"

t07_information = ri.timss_2007_math_item_information(
    ri.download(T07_ITEMS_ZIP_URL, T07_DATA_DIR / "raw" / "T07_Items.zip"), 4
)
t07_statistics = ri.nces_2007_percent_correct(
    ri.download(T07_STATISTICS_URL, T07_DATA_DIR / "raw" / "TIMSS_Grade4_Math_Released_Item_Statistics.xlsx"),
    "Gr. 4 Mathematics",
)
t07_ids = [item["item_id"] for item in t07_items]
assert set(t07_ids) == set(t07_statistics), "the statistics workbook and the booklet have different items"
for item in t07_items:
    info = t07_information.loc[item["item_id"]]
    assert info["Released"] == "Yes", f"{item['item_id']}: not released in the item information"
    assert (info["Content Domain"], info["Cognitive Domain"], int(info["Maximum Points"])) == (
        item["content_domain"],
        item["cognitive_domain"],
        item["max_points"],
    ), f"{item['item_id']}: item information differs from the booklet"
    assert (info["Key"] if info["Key"] != "-" else None) == item["correct_option"], f"{item['item_id']}: key differs"


def tidy(text):
    """Even out the model's formatting noise: answer blanks become ____ and runs of spaces one space."""
    return re.sub(r"_{2,}", "____", re.sub(r" {2,}", " ", text))


t07_records = []
for item in t07_items:
    t = t07_transcriptions[item["item_id"]]
    info = t07_information.loc[item["item_id"]]
    t07_records.append(
        {
            "item_id": item["item_id"],
            "item_label": info["Item Label"],
            "assessment": "TIMSS 2007",
            "grade": 4,
            "subject": "Mathematics",
            "content_domain": item["content_domain"],
            "main_topic": info["Topic Area"],
            "cognitive_domain": item["cognitive_domain"],
            "item_type": item["item_type"],
            "max_points": item["max_points"],
            "stem": tidy(t.stem),
            "options": [{"label": o.label, "text": tidy(o.text)} for o in t.options] or None,
            "correct_option": item["correct_option"],
            "scoring_guide": [
                {"category": c.category, "descriptors": [tidy(code.response) for code in c.codes]}
                for c in t.scoring_guide
            ]
            or None,
            "scoring_notes": t.scoring_notes,
            "has_figure": t.has_figure,
            "figure_description": t.figure_description,
            "accessibility_text": None,
            **t07_statistics[item["item_id"]],
            "booklet_page": item["booklet_page"],
            "pdf_page": item["pdf_page"],
            "source_url": T07_ZIP_URL,
            "transcription_model": LLM_MODEL,
        }
    )

t07_released_items = ri.write_jsonl(t07_records, T07_OUTPUT_PATH)
print(f"Wrote {len(t07_released_items)} items to {T07_OUTPUT_PATH.resolve()}")
t07_preview_columns = ["item_id", "item_label", "main_topic", "item_type", "pct_correct_intl_avg", "pct_correct_usa", "stem"]
display(t07_released_items[t07_preview_columns].head())

### Alternative: transcribe through the Gemini API

The cell below is the earlier, unchanged version of the transcription cell, which calls `gemini-3.8-flash` through the Gemini API's OpenAI-compatible endpoint with `gemini_api_key`.
The free tier allows only 20 requests per day per project, resetting at midnight Pacific time, so it sends up to 8 items per request, at least 20 seconds apart, which makes 10 requests for the whole set.
It uses the same model, prompt, and cache as the OpenRouter cell, so after a complete OpenRouter run it sends no requests, and after a partial one it requests only the missing items.
To use it instead of the OpenRouter cell, run it and then rerun "Write the JSONL"; `transcription_model` then reads `gemini-3.8-flash`, because this cell sets `LLM_MODEL`.

In [ ]:
LLM_MODEL = "gemini-3.8-flash"
T07_PROMPT_VERSION = 1
T07_LLM_CACHE_DIR = (
    T07_DATA_DIR / "cache" / "item_transcriptions" / "timss07_g4_math" / f"{LLM_MODEL}-v{T07_PROMPT_VERSION}"
)
T07_ITEMS_PER_REQUEST = 8
T07_SECONDS_BETWEEN_REQUESTS = 20
T07_INK_LEVEL = 200  # gray levels darker than this are printed content when trimming blank image margins
T07_TRIM_MARGIN = 10  # pixels of blank margin kept around the content
# Codes each printed scoring-guide category may hold, by the item's maximum points: the first digit of a TIMSS
# score code is the score, except 7 (incorrect) and 9 (nonresponse).
T07_CATEGORY_CODES = {
    "Correct Response": {1: range(10, 20), 2: range(20, 30)},
    "Partial Response": {2: range(10, 20)},
    "Incorrect Response": {1: range(70, 80), 2: range(70, 80)},
    "Nonresponse": {1: range(90, 100), 2: range(90, 100)},
}


class T07Option(BaseModel):
    label: str = Field(description="Option letter as printed, e.g. 'A'.")
    text: str = Field(description="Option content in the same notation as the stem.")


class T07ScoringCode(BaseModel):
    code: int = Field(description="Score code as printed, e.g. 10.")
    response: str = Field(description="The code's response description, verbatim.")


class T07ScoringCategory(BaseModel):
    category: str = Field(description="Category heading as printed, e.g. 'Correct Response'.")
    codes: list[T07ScoringCode] = Field(description="Each code row under the heading, in printed order.")


class T07Transcription(BaseModel):
    item_id: str
    stem: str
    options: list[T07Option]
    scoring_guide_item_id: str | None
    scoring_guide: list[T07ScoringCategory]
    scoring_notes: str | None
    has_figure: bool
    figure_description: str | None


class T07TranscriptionBatch(BaseModel):
    items: list[T07Transcription]


T07_TRANSCRIPTION_INSTRUCTIONS = """\
You transcribe items from the TIMSS 2007 grade 4 released mathematics items into structured data.

Each request covers several items, listed by ID. For each item you receive:
1. The booklet page the item appears on, as an image, exactly as students saw it. The parts of a multi-part item \
(IDs ending in A, B, C, ...) share their booklet pages, which are shown once, before the parts. Those pages hold an \
introduction shared by all parts, which may fill a page of its own, followed by every part, each starting with its \
part label (e.g. "B.").
2. For a constructed-response item, its scoring guide as an image: a table of score codes headed by the item ID.

The images are the only source. Transcribe what is printed, verbatim; never add, correct, or solve anything.

Return one entry per item, in the order given:
- item_id: The item ID as given.
- stem: Everything the student sees for this item except the multiple-choice response options, in reading order. \
For a part of a multi-part item, the shared introduction (everything before the first part label) followed by this \
part only, starting with its part label; leave out the other parts. Leave out page furniture: the item ID printed \
sideways in the margin, and navigation lines such as "This ... question continues on the next page." or "End of ... \
section.". Write math in plain-text notation: fractions as 1/2, mixed numbers as 1 1/2, and the symbols ×, ÷, −, =, \
°. Write an empty box in a number sentence as □ and an answer blank as ____, keeping labels such as "Answer:". Write \
tables as Markdown tables and bulleted lists as "- " lines. Replace each picture, diagram, graph, grid, or map with \
"[Image: <short description>]". Separate paragraphs with a newline.
- options: For a multiple-choice item, one entry per response option in printed order, labeled with its circled \
letter, in the same notation as the stem; a pictured option is "[Image: <description>]". Empty for \
constructed-response items.
- scoring_guide_item_id: The item ID printed in the header of the scoring guide ("Item: ..."); null for \
multiple-choice items.
- scoring_guide: For a constructed-response item, each category of the scoring guide in printed order (e.g. \
"Correct Response", "Partial Response", "Incorrect Response", "Nonresponse"), with its heading verbatim, except that \
"Partially Correct Response" is written "Partial Response", and each code row under it: the score code and its \
response description, verbatim, in the same notation as the stem. A picture that illustrates a code, such as a \
sample answer in the row below it, belongs to that code's response, as "[Image: <description>]". Empty for \
multiple-choice items.
- scoring_notes: Any scoring-guide text outside the code table, such as a note; null if none.
- has_figure: Whether this item's stem or options contain any picture, diagram, graph, grid, map, or drawn shape. A \
table of text or numbers alone does not count.
- figure_description: If has_figure, a precise, objective description of every visual element in this item's stem \
and options, giving all the information a student would need to answer without seeing it: counts, labels, \
measurements, positions, grid coordinates, bar heights, shading, and orientation. Describe what is drawn; do not \
solve the item or reveal the answer. Null if has_figure is false.
"""


def t07_image_part(xref):
    """An embedded image as a PNG message part, trimmed to its printed content so the model sees it at full size."""
    pix = pymupdf.Pixmap(t07_pdf, xref)
    if pix.colorspace.n == 4:  # one item image is CMYK
        pix = pymupdf.Pixmap(pymupdf.csRGB, pix)
    gray = pix if pix.n == 1 else pymupdf.Pixmap(pymupdf.csGRAY, pix)
    ink = np.frombuffer(gray.samples, np.uint8).reshape(gray.height, gray.width) < T07_INK_LEVEL
    rows, columns = np.flatnonzero(ink.any(axis=1)), np.flatnonzero(ink.any(axis=0))
    clip = pymupdf.IRect(columns[0], rows[0], columns[-1] + 1, rows[-1] + 1) + (
        -T07_TRIM_MARGIN,
        -T07_TRIM_MARGIN,
        T07_TRIM_MARGIN,
        T07_TRIM_MARGIN,
    )
    clip &= pymupdf.IRect(0, 0, pix.width, pix.height)
    trimmed = pymupdf.Pixmap(pix.colorspace, clip, False)
    trimmed.copy(pix, clip)
    png = base64.b64encode(trimmed.tobytes("png")).decode()
    return {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{png}"}}


def t07_request_content(batch):
    """The user message for a batch: each item's booklet page images (once per multi-part item) and scoring guide."""
    content = [
        {"type": "text", "text": f"Transcribe these {len(batch)} items, in this order: {', '.join(i['item_id'] for i in batch)}."}
    ]
    shown_units = set()
    for item in batch:
        points = f"{item['max_points']} point{'s' if item['max_points'] > 1 else ''}"
        description = f"{item['item_type'].replace('_', '-')}, worth {points}"
        if item["parts"]:
            if item["parts"][0] not in shown_units:
                shown_units.add(item["parts"][0])
                content.append(
                    {
                        "type": "text",
                        "text": f"Booklet pages of the multi-part item with parts {', '.join(item['parts'])}, in order:",
                    }
                )
                content += [t07_image_part(xref) for xref in item["page_images"]]
            content.append(
                {"type": "text", "text": f"Item {item['item_id']}: part {item['part']} of that multi-part item, {description}."}
            )
        else:
            content.append({"type": "text", "text": f"Item {item['item_id']}: {description}. Booklet page:"})
            content.append(t07_image_part(item["page_images"][0]))
        if item["scoring_image"]:
            content.append({"type": "text", "text": f"Scoring guide of item {item['item_id']}:"})
            content.append(t07_image_part(item["scoring_image"]))
    return content


def t07_problems(item, t):
    """Disagreements between a transcription and the item's documentation; empty if the transcription is usable."""
    problems = []
    if not t.stem.strip():
        problems.append("empty stem")
    if item["item_type"] == "multiple_choice":
        if [o.label for o in t.options] != ["A", "B", "C", "D"]:
            problems.append(f"option labels {[o.label for o in t.options]}")
        if t.scoring_guide or t.scoring_guide_item_id:
            problems.append("multiple-choice item has a scoring guide")
    else:
        if t.options:
            problems.append("constructed-response item has options")
        if t.scoring_guide_item_id != item["item_id"]:
            problems.append(f"scoring guide headed {t.scoring_guide_item_id!r}")
        expected = [c for c, codes in T07_CATEGORY_CODES.items() if item["max_points"] in codes]
        categories = [c.category for c in t.scoring_guide]
        if categories != expected:
            problems.append(f"scoring categories {categories}, expected {expected} for {item['max_points']} points")
        for c in t.scoring_guide:
            allowed = T07_CATEGORY_CODES.get(c.category, {}).get(item["max_points"], range(0))
            codes = [code.code for code in c.codes]
            if not codes or any(code not in allowed for code in codes):
                problems.append(f"{c.category} codes {codes}")
    if item["part"]:
        labels = re.findall(r"(?m)^([A-Z])\.\s", t.stem)
        if labels != [item["part"]]:
            problems.append(f"part labels {labels} in the stem")
    if t.has_figure != bool(t.figure_description):
        problems.append(f"has_figure={t.has_figure} but figure_description={t.figure_description!r}")
    return problems


def t07_batches(items, size):
    """Items in booklet order, in batches of at most `size` that never split a multi-part item."""
    batches = []
    for _, unit in groupby(items, key=lambda item: (item["block"], item["block_seq"])):
        unit = list(unit)
        if batches and len(batches[-1]) + len(unit) <= size:
            batches[-1] += unit
        else:
            batches.append(unit)
    return batches


def t07_transcription_path(item_id):
    return T07_LLM_CACHE_DIR / f"{item_id}.json"


def t07_transcribe(batch):
    """One request for a batch; returns the transcriptions by item ID, empty if the response could not be parsed."""
    choice = t07_llm_client.chat.completions.parse(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": T07_TRANSCRIPTION_INSTRUCTIONS},
            {"role": "user", "content": t07_request_content(batch)},
        ],
        response_format=T07TranscriptionBatch,
    ).choices[0]
    if choice.message.parsed is None:
        print(f"  No parsed response (finish_reason={choice.finish_reason})")
        return {}
    return {t.item_id: t for t in choice.message.parsed.items}


if not globals().get("gemini_api_key"):
    raise RuntimeError("Run the first cells to load GEMINI_API_KEY from .env")
# No client retries: every attempt counts against the daily quota.
t07_llm_client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=gemini_api_key,
    max_retries=0,
    timeout=600,
)
T07_LLM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

t07_pending = [item for item in t07_items if not t07_transcription_path(item["item_id"]).exists()]
t07_requests = t07_batches(t07_pending, T07_ITEMS_PER_REQUEST)
t07_rejected = {}  # item ID -> (transcription or None, problems), for inspection
print(f"Transcribing {len(t07_pending)} of {len(t07_items)} items with {LLM_MODEL} in {len(t07_requests)} requests")
next_request_at = 0.0
for number, batch in enumerate(t07_requests, 1):
    time.sleep(max(0.0, next_request_at - time.monotonic()))
    next_request_at = time.monotonic() + T07_SECONDS_BETWEEN_REQUESTS
    request = f"Request {number}/{len(t07_requests)} ({batch[0]['item_id']} to {batch[-1]['item_id']})"
    try:
        returned = t07_transcribe(batch)
    except (openai.OpenAIError, pydantic.ValidationError) as error:
        print(f"{request} failed, so no further requests were sent:\n{error}")
        break
    kept = 0
    for item in batch:
        transcription = returned.get(item["item_id"])
        problems = t07_problems(item, transcription) if transcription else ["missing from the response"]
        if problems:
            t07_rejected[item["item_id"]] = (transcription, problems)
            print(f"  Rejected {item['item_id']}: {'; '.join(problems)}")
        else:
            t07_transcription_path(item["item_id"]).write_text(transcription.model_dump_json(indent=2))
            kept += 1
    print(f"{request}: kept {kept} of {len(batch)} transcriptions")
    if not kept:
        print("No transcription in this request was usable, so no further requests were sent.")
        break

t07_missing = [item["item_id"] for item in t07_items if not t07_transcription_path(item["item_id"]).exists()]
if t07_missing:
    raise RuntimeError(
        f"{len(t07_missing)} items are not transcribed yet: {', '.join(t07_missing)}. Rerun this cell to request "
        "only those; after a 429, wait until the daily quota resets at midnight Pacific time."
    )
t07_transcriptions = {
    item["item_id"]: T07Transcription.model_validate_json(t07_transcription_path(item["item_id"]).read_text())
    for item in t07_items
}
print(f"All {len(t07_transcriptions)} items are transcribed")

## TIMSS 2007 grade 4 mathematics item information and parameters

Download `T07_Items.zip` (the "Items" link on the [TIMSS 2007 International Database and User Guide](https://timssandpirls.bc.edu/TIMSS2007/idb_ug.html) page, 56 MB), unzip it to `data/raw/T07_Items/`, and combine two of its workbooks into `t07_item_information`, one row per grade 4 mathematics item, exported to `data/timss07_g4_math_item_information.xlsx`:

- `T07_G4_ItemInformation.xls` documents every grade 4 item in the 2007 assessment: its block and position in the block, cycle (3 for a trend item first used in TIMSS 2003, 4 for an item new in 2007), content domain, topic area and objective, cognitive domain, item type, number of options, key, maximum points, label, and whether it was released and used in IRT scaling. Its 179 mathematics rows are kept.
- The `G4-MAT` sheet of `T07_ItemParameters_Overall.xls` holds the IRT item parameters from the joint 2003-2007 calibration of overall mathematics, with the estimate and standard error of each parameter.

The calibration combined three sets of data (TIMSS 2007 Technical Report, chapter 11), and the `G4-MAT` sheet lists items from each.
The 296 rows with IDs starting `M0` or `MF` come from the TIMSS 2003 booklets and the 2003 booklets readministered in the 2007 bridging study.
The 177 rows with IDs starting `MZ` are the items in the TIMSS 2007 assessment booklets, which the calibration gave their own parameters; their positions are the 2007 block positions in the item information file.
`t07_item_information` takes its parameters from the `MZ` rows only.
Two items, M031223 and M031002, were not used in scaling and have no parameters.

Cleaning:

- Values are stripped of stray spaces; "Yes"/"No" become booleans; "-" (no options or key for a constructed-response item) becomes empty.
- Subject and item type use the words in the released-items JSONL ("Mathematics", "multiple_choice", "constructed_response").
- The file spells most topic areas several ways (such as "2-and 3-dimensional shapes" and "2- & 3- Dimensional"); `topic_area` uses the names in the TIMSS 2007 Assessment Frameworks.
- The file gives M031350B the position of part A ("08A"); the parameter sheet has "08B".
- `irt_model` follows from which parameters are present, as for 2011: 3PL for multiple-choice items, GPCM for 2-point and 2PL for 1-point constructed-response items.

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display

# The "Items" link on https://timssandpirls.bc.edu/TIMSS2007/idb_ug.html
T07_ITEMS_ZIP_URL = "https://timssandpirls.bc.edu/TIMSS2007/PDF/T07_Items.zip"
T07_DATA_DIR = Path.cwd() / "data"
T07_ITEMS_ZIP_PATH = T07_DATA_DIR / "raw" / "T07_Items.zip"
T07_ITEMS_DIR = T07_ITEMS_ZIP_PATH.with_suffix("")  # the zip holds a T07_Items/ folder
T07_ITEM_INFORMATION_XLS = T07_ITEMS_DIR / "T07_G4_ItemInformation.xls"
T07_ITEM_PARAMETERS_XLS = T07_ITEMS_DIR / "T07_ItemParameters_Overall.xls"
T07_ITEM_INFORMATION_PATH = T07_DATA_DIR / "timss07_g4_math_item_information.xlsx"

if not T07_ITEMS_ZIP_PATH.exists():
    T07_ITEMS_ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    partial_path = T07_ITEMS_ZIP_PATH.with_suffix(".zip.part")
    with requests.get(T07_ITEMS_ZIP_URL, stream=True, timeout=120) as response:
        response.raise_for_status()
        with partial_path.open("wb") as f:
            for chunk in response.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    if not zipfile.is_zipfile(partial_path):
        raise ValueError(f"{T07_ITEMS_ZIP_URL} did not return a zip file")
    partial_path.rename(T07_ITEMS_ZIP_PATH)
if not (T07_ITEM_INFORMATION_XLS.exists() and T07_ITEM_PARAMETERS_XLS.exists()):
    with zipfile.ZipFile(T07_ITEMS_ZIP_PATH) as archive:
        archive.extractall(T07_ITEMS_ZIP_PATH.parent)

print(f"{T07_ITEMS_DIR.relative_to(Path.cwd())}:")
for path in sorted(T07_ITEMS_DIR.rglob("*.*")):
    print(f"  {path.relative_to(T07_ITEMS_DIR)} ({path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
T07_INFORMATION_COLUMNS = {  # item information heading (line breaks as spaces) -> column name
    "Item ID": "item_id",
    "Grade": "grade",
    "Subject": "subject",
    "Cycle": "cycle",
    "Released": "released",
    "In Scaling": "in_scaling",
    "TIMSS 2007 Block": "block",
    "TIMSS 2007 Block Seq": "block_seq",
    "Content Domain": "content_domain",
    "Topic Area": "topic_area",
    "Topic Area Bullet (Objective)": "topic_area_objective",
    "Cognitive Domain": "cognitive_domain",
    "Scientific Inquiry": "scientific_inquiry",  # science only
    "Item Type": "item_type",
    "Options": "option_count",
    "Key": "key",
    "Maximum Points": "max_points",
    "Item Label": "item_label",
}
T07_PARAMETERS_SHEET = "G4-MAT"
# Sheet heading -> column name. Each parameter spans two columns: the estimate and its standard error.
T07_PARAMETERS = {
    "Slope (aj)": "slope",
    "Location (bj)": "location",
    "Guessing (cj)": "guessing",
    "Step 1 (dj1)": "step1",
    "Step 2 (dj2)": "step2",
}
# TIMSS 2007 grade 4 mathematics topic areas by content domain, with the item information file's spellings of each.
T07_TOPIC_AREAS = {
    "Number": {
        "Whole Numbers": ["Whole Numbers"],
        "Fractions and Decimals": ["Fractions and Decimals", "Fraction and Decimal"],
        "Number Sentences with Whole Numbers": ["Number Sentences", "Number Sentence"],
        "Patterns and Relationships": ["Patterns and Relationships", "Pattern & Relationships"],
    },
    "Geometric Shapes and Measures": {
        "Lines and Angles": ["Lines and Angles"],
        "Two- and Three-dimensional Shapes": ["2-and 3-dimensional shapes", "2- & 3- Dimensional"],
        "Location and Movement": ["Location and Movements"],
    },
    "Data Display": {
        "Reading and Interpreting": ["Reading and Interpreting"],
        "Organizing and Representing": ["Organizing and Representing", "Organizing & Representing"],
    },
}
T07_BLOCK_SEQ_FIXES = {"M031350B": "08B"}  # the item information file repeats part A's "08A"
T07_ITEM_TYPES = {"MC": "multiple_choice", "CR": "constructed_response"}
T07_YES_NO = {"Yes": True, "No": False}

# Item information: one row per grade 4 item, mathematics and science.
information = pd.read_excel(T07_ITEM_INFORMATION_XLS, dtype=str)
information.columns = [" ".join(str(column).split()) for column in information.columns]
if list(information.columns) != list(T07_INFORMATION_COLUMNS):
    raise ValueError(f"Unexpected item information headings: {list(information.columns)}")
information = information.rename(columns=T07_INFORMATION_COLUMNS).apply(lambda column: column.str.strip())
information = information[information["subject"] == "M"].copy()
assert information["scientific_inquiry"].isna().all() and (information["grade"] == "4").all()
assert information["item_id"].is_unique, "duplicate item IDs in the item information file"

topic_names = {
    (domain, spelling): topic
    for domain, topics in T07_TOPIC_AREAS.items()
    for topic, spellings in topics.items()
    for spelling in spellings
}
unknown_topics = set(zip(information["content_domain"], information["topic_area"])) - set(topic_names)
assert not unknown_topics, f"topic areas not in T07_TOPIC_AREAS: {unknown_topics}"
assert set(information["released"]) | set(information["in_scaling"]) <= set(T07_YES_NO)
assert set(information["item_type"]) <= set(T07_ITEM_TYPES)

information["block_seq"] = information["item_id"].map(T07_BLOCK_SEQ_FIXES).fillna(information["block_seq"])
information = information.assign(
    grade=4,
    subject="Mathematics",
    cycle=information["cycle"].astype(int),
    released=information["released"].map(T07_YES_NO),
    in_scaling=information["in_scaling"].map(T07_YES_NO),
    block_position=information["block"] + "_" + information["block_seq"],
    topic_area=[topic_names[pair] for pair in zip(information["content_domain"], information["topic_area"])],
    topic_area_objective=information["topic_area_objective"].astype(int),
    item_type=information["item_type"].map(T07_ITEM_TYPES),
    option_count=pd.to_numeric(information["option_count"].mask(information["option_count"] == "-")).astype("Int64"),
    key=information["key"].mask(information["key"] == "-"),
    max_points=information["max_points"].astype(int),
).drop(columns="scientific_inquiry")

# Parameters: row 1 is the title and row 2 the headings; item rows follow, with IDs and positions padded by a space.
sheet = pd.read_excel(T07_ITEM_PARAMETERS_XLS, sheet_name=T07_PARAMETERS_SHEET, header=None)
headings = sheet.iloc[1].tolist()
if headings[0] != "Item" or headings[2::2] != list(T07_PARAMETERS) or sheet.shape[1] != 2 + 2 * len(T07_PARAMETERS):
    raise ValueError(f"Unexpected headings in the {T07_PARAMETERS_SHEET} sheet: {headings}")
body = sheet.iloc[2:].dropna(how="all")
parameter_names = list(T07_PARAMETERS.values())
parameters = pd.concat(
    [
        pd.DataFrame({"sheet_id": body[0].str.strip(), "parameter_position": body[1].str.strip()}),
        body.iloc[:, 2::2].set_axis(parameter_names, axis=1).apply(pd.to_numeric),
        body.iloc[:, 3::2].set_axis([f"{name}_se" for name in parameter_names], axis=1).apply(pd.to_numeric),
    ],
    axis=1,
)
assert parameters["sheet_id"].str.fullmatch(r"M[0FZ]\d{5}[A-Z]?").all(), "unexpected IDs in the parameter sheet"
print("Rows in the G4-MAT sheet by ID prefix:", parameters["sheet_id"].str[:2].value_counts().to_dict())
# MZ31286 is M031286 in the TIMSS 2007 assessment booklets.
parameters = parameters[parameters["sheet_id"].str.startswith("MZ")]
parameters.insert(0, "item_id", "M0" + parameters.pop("sheet_id").str[2:])
parameters.insert(
    2,
    "irt_model",
    np.select(
        [parameters["guessing"].notna(), parameters["step1"].notna()], ["3PL", "GPCM"], default="2PL"
    ),
)

t07_item_information = information.merge(parameters, on="item_id", how="left", validate="one_to_one", indicator=True)

problems = []
if not t07_item_information["item_id"].str.fullmatch(r"M\d{6}[A-Z]?").all():
    problems.append("item IDs not shaped like M031286")
not_in_information = set(parameters["item_id"]) - set(information["item_id"])
if not_in_information:
    problems.append(f"parameters for items not in the item information file: {sorted(not_in_information)}")
has_parameters = t07_item_information.pop("_merge").eq("both")
if not has_parameters.equals(t07_item_information["in_scaling"]):
    problems.append("the items with parameters are not the items used in scaling")
scaled = t07_item_information[has_parameters]
moved = scaled[scaled["block_position"] != scaled["parameter_position"]]
if not moved.empty:
    problems.append(f"block positions differ between the two files: {moved['item_id'].tolist()}")
for name in parameter_names:
    if not scaled[name].notna().equals(scaled[f"{name}_se"].notna()):
        problems.append(f"{name} estimates and standard errors are not present for the same items")
if scaled[["slope", "location"]].isna().any(axis=None):
    problems.append("items without a slope or location")
if not scaled["step1"].notna().equals(scaled["step2"].notna()):
    problems.append("items with only one step parameter")
expected_model = np.select(
    [scaled["item_type"] == "multiple_choice", scaled["max_points"] == 2], ["3PL", "GPCM"], default="2PL"
)
if (scaled["irt_model"] != expected_model).any():
    problems.append("IRT models that do not match the item type and maximum points")
multiple_choice = t07_item_information["item_type"] == "multiple_choice"
if not (multiple_choice == t07_item_information["key"].isin(list("ABCD"))).all():
    problems.append("keys that do not match the item type")
if not (multiple_choice == t07_item_information["option_count"].notna()).all():
    problems.append("option counts that do not match the item type")
assert not problems, "\n".join(problems)
t07_item_information.loc[~has_parameters, "irt_model"] = None

t07_item_information = t07_item_information[
    [
        "item_id",
        "block_position",
        "block",
        "block_seq",
        "cycle",
        "released",
        "in_scaling",
        "grade",
        "subject",
        "content_domain",
        "topic_area",
        "topic_area_objective",
        "cognitive_domain",
        "item_type",
        "option_count",
        "key",
        "max_points",
        "item_label",
        "irt_model",
        *parameter_names,
        *[f"{name}_se" for name in parameter_names],
    ]
].reset_index(drop=True)

print(f"{len(t07_item_information)} grade 4 mathematics items")
display(
    pd.crosstab(
        [t07_item_information["content_domain"], t07_item_information["topic_area"]],
        t07_item_information["released"].map({True: "released", False: "secure"}),
        margins=True,
    )
)
display(
    pd.crosstab(
        [t07_item_information["item_type"], t07_item_information["max_points"]],
        t07_item_information["irt_model"].fillna("not scaled"),
        margins=True,
    )
)
display(t07_item_information.head())

In [ ]:
t07_item_information.to_excel(T07_ITEM_INFORMATION_PATH, index=False)
print(f"Exported {len(t07_item_information):,} rows to {T07_ITEM_INFORMATION_PATH.resolve()}")

## TIMSS 2007 grade 4 released mathematics items with item information

Join `data/timss07_g4_math_released_items.jsonl` with `data/timss07_g4_math_item_information.xlsx` on the item ID, keeping only the 74 items in the JSONL, and export the result to `data/timss07_g4_math_released_items_analysis.xlsx`.

`t07_released_analysis` has one row per item and every column of the JSONL, in the same order.
`item_label` and `main_topic` (the 2007 framework's topic area), which the JSONL takes from the same item information file, must agree with it.
Then come the item information columns the JSONL does not have: block position, block, block sequence, cycle, released, in scaling, topic area objective, number of options, and the IRT model and parameters.
Grade, subject, content domain, cognitive domain, item type, maximum points, and the multiple-choice key appear in both sources, which come from different documents (the released-items PDF and the item information file), so they are checked for agreement and kept once.

Added columns, as in the 2011 analysis:

- `correct_answer`: the key (such as "A") for a multiple-choice item; the descriptors under "Correct Response" in the scoring guide, joined with "; ", for a constructed-response item.
- `stem_word_count` and `option_A_word_count` to `option_D_word_count`: words in the stem and in each option. Words are word-like tokens, so answer blanks and symbols such as □ do not count, but the words of `[Image: ...]` descriptions do.
- `correct_option_word_count` and `distractors_word_count`: words in the correct option and in all incorrect options together, for multiple-choice items.

The list columns `options` and `scoring_guide` are written to Excel as JSON text.

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

T07_DATA_DIR = Path.cwd() / "data"
T07_RELEASED_ITEMS_PATH = T07_DATA_DIR / "timss07_g4_math_released_items.jsonl"
T07_ITEM_INFORMATION_PATH = T07_DATA_DIR / "timss07_g4_math_item_information.xlsx"
T07_ANALYSIS_PATH = T07_DATA_DIR / "timss07_g4_math_released_items_analysis.xlsx"
# Fields in both sources under the same name, which must agree.
T07_SHARED_FIELDS = ["grade", "subject", "content_domain", "cognitive_domain", "item_type", "max_points"]
# JSONL column -> merged item information column with the same content (the JSONL takes it from the item information).
T07_MATCHING_FIELDS = {"item_label": "item_label_information", "main_topic": "topic_area"}
T07_OPTION_LABELS = ["A", "B", "C", "D"]


def word_count(text):
    """Count word-like tokens while ignoring underscores used for answer blanks."""
    return len(re.findall(r"[^\W_]+(?:['’][^\W_]+)*", text or "", flags=re.UNICODE))


def as_list(value):
    return value if isinstance(value, list) else []


def answer_and_word_counts(item):
    """The correct answer and the word counts of the stem and options of one item."""
    option_words = {option["label"]: word_count(option["text"]) for option in as_list(item["options"])}
    if item["item_type"] == "multiple_choice":
        correct_answer = item["correct_option"]
        correct_option_words = option_words[correct_answer]
        distractor_words = sum(words for label, words in option_words.items() if label != correct_answer)
    else:
        correct = next(c for c in as_list(item["scoring_guide"]) if c["category"] == "Correct Response")
        correct_answer = "; ".join(correct["descriptors"])
        correct_option_words = distractor_words = None
    return pd.Series(
        {
            "correct_answer": correct_answer,
            "stem_word_count": word_count(item["stem"]),
            **{f"option_{label}_word_count": option_words.get(label) for label in T07_OPTION_LABELS},
            "correct_option_word_count": correct_option_words,
            "distractors_word_count": distractor_words,
        }
    )


with T07_RELEASED_ITEMS_PATH.open(encoding="utf-8") as f:
    t07_released = pd.DataFrame([json.loads(line) for line in f])
t07_information = pd.read_excel(T07_ITEM_INFORMATION_PATH, dtype={"block_seq": str})
t07_information["option_count"] = t07_information["option_count"].astype("Int64")

merged = t07_released.merge(
    t07_information, on="item_id", how="left", suffixes=("", "_information"), validate="one_to_one", indicator=True
)
problems = []
missing = merged.loc[merged.pop("_merge") != "both", "item_id"].tolist()
if missing:
    problems.append(f"items not in the item information: {missing}")
for field in T07_SHARED_FIELDS:
    differ = merged[field] != merged.pop(f"{field}_information")
    if differ.any():
        problems.append(f"{field} differs between the sources for {merged.loc[differ, 'item_id'].tolist()}")
differ = merged["correct_option"].fillna("") != merged.pop("key").fillna("")
if differ.any():
    problems.append(f"the multiple-choice key differs between the sources for {merged.loc[differ, 'item_id'].tolist()}")
for target, source in T07_MATCHING_FIELDS.items():
    differ = merged[target] != merged.pop(source)
    if differ.any():
        problems.append(f"{target} differs between the sources for {merged.loc[differ, 'item_id'].tolist()}")
if not merged["released"].all():
    problems.append("released items that the item information does not mark as released")
assert not problems, "\n".join(problems)

added = merged.apply(answer_and_word_counts, axis=1)
count_columns = [column for column in added.columns if column.endswith("_word_count")]
added[count_columns] = added[count_columns].astype("Int64")
t07_released_analysis = pd.concat([merged, added], axis=1)

print(
    f"{len(t07_released_analysis)} items: {len(t07_released.columns)} JSONL columns, "
    f"{len(merged.columns) - len(t07_released.columns)} item information columns, {len(added.columns)} added columns"
)
display(
    t07_released_analysis.groupby("item_type")[["stem_word_count", "correct_option_word_count", "distractors_word_count"]]
    .describe()
    .T.round(1)
)
display(
    t07_released_analysis[
        [
            "item_id",
            "item_label",
            "main_topic",
            "item_type",
            "correct_answer",
            "irt_model",
            "location",
            "stem_word_count",
            *[f"option_{label}_word_count" for label in T07_OPTION_LABELS],
            "correct_option_word_count",
            "distractors_word_count",
        ]
    ].head(10)
)

In [ ]:
def excel_value(value):
    """Lists and dicts as JSON text, since an Excel cell holds a single value."""
    return json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value


t07_released_analysis.map(excel_value).to_excel(T07_ANALYSIS_PATH, index=False)
print(f"Exported {len(t07_released_analysis):,} rows to {T07_ANALYSIS_PATH.resolve()}")

## Predict IRT parameters for the TIMSS 2007 released items

Predict the parameters of the 74 items in `data/timss07_g4_math_released_items.jsonl` with `predict_item_parameters` from "Predict IRT parameters for new items with an LLM", so run that section, and the cells it needs, first.
For each item, `find_similar_items` finds the most similar items in the Qdrant collection, which holds the 73 TIMSS 2011 released items with their calibrated parameters, and the LLM predicts the item's parameters with those items as context.

- The IRT model follows the item's format and points (`TimssItem.irt_model`): 3PL for multiple-choice items, 2PL for constructed-response items worth 1 point, and GPCM for constructed-response items worth up to 2 points.
- The system prompt (`parameter_system_prompt`, printed below for the first item) makes the LLM an expert in testing grade 4 mathematics who is very familiar with the item's content domain, topic, and cognitive domain, and a measurement expert in IRT models.
- The released-items PDF has no topic areas, so `main_topic` comes from the item information (`data/timss07_g4_math_item_information.xlsx`), in the TIMSS 2007 framework's terms.

The predictions are on the scale of the TIMSS 2011 concurrent calibration of the 2007 and 2011 data, which also calibrated these 74 items (the "Items Released in 2007" group of `item_parameters`).
Their calibrated parameters are therefore a held-out check: the 74 items are not in the Qdrant collection, and they are left out of the calibration summary in every prompt.
The parameters in the 2007 item information come from another calibration (the joint 2003-2007 scaling), on another scale, so they are not compared.

Predictions are kept in `t07_predictions` as they arrive.
An item whose LLM output is unusable (outside the schema, or cut off at `PARAMETER_MAX_TOKENS`) is skipped and reported, any other error stops the cell, and rerunning the prediction cell requests only the items still missing; rerun the cell before it to start over.
The results, with the calibrated parameters next to the predictions, are exported to `data/timss07_g4_math_predicted_item_parameters.xlsx`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

required_globals = [
    "TimssItem",  # Predict IRT parameters for new items with an LLM
    "predict_item_parameters",
    "parameter_system_prompt",
    "PARAMETER_LLM_MODEL",
    "PARAMETER_SYMBOLS",
    "item_parameters",  # TIMSS 2011 grade 4 mathematics item parameters
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise NameError(f"Run 'Predict IRT parameters for new items with an LLM' and the cells it needs first: {missing_globals}")

T07_DATA_DIR = Path.cwd() / "data"
T07_RELEASED_ITEMS_PATH = T07_DATA_DIR / "timss07_g4_math_released_items.jsonl"
T07_ITEM_INFORMATION_PATH = T07_DATA_DIR / "timss07_g4_math_item_information.xlsx"
T07_PREDICTIONS_PATH = T07_DATA_DIR / "timss07_g4_math_predicted_item_parameters.xlsx"

t07_topic_areas = pd.read_excel(T07_ITEM_INFORMATION_PATH).set_index("item_id")["topic_area"]
with T07_RELEASED_ITEMS_PATH.open(encoding="utf-8") as f:
    t07_released_records = [json.loads(line) for line in f]
missing_topics = [record["item_id"] for record in t07_released_records if record["item_id"] not in t07_topic_areas]
assert not missing_topics, f"items without a topic area in the item information: {missing_topics}"
t07_items_to_predict = {
    record["item_id"]: TimssItem.model_validate(record | {"main_topic": t07_topic_areas[record["item_id"]]})
    for record in t07_released_records
}
t07_predictions = {}  # item ID -> prediction; the next cell fills it

print(f"{len(t07_items_to_predict)} items to predict with {PARAMETER_LLM_MODEL}")
display(
    pd.crosstab(
        pd.Series([item.item_type for item in t07_items_to_predict.values()], name="item_type"),
        pd.Series([item.irt_model for item in t07_items_to_predict.values()], name="irt_model"),
        margins=True,
    )
)
first_item = next(iter(t07_items_to_predict.values()))
print(f"System prompt for {first_item.item_id}:\n\n{parameter_system_prompt(first_item)}")

In [ ]:
import time

import openai
import pydantic

# None of the 74 items may inform another's prediction: they are not in the Qdrant collection, and this keeps them
# out of the calibration summary.
t07_held_out_ids = list(t07_items_to_predict)
t07_unusable = {}  # item ID -> error, for items whose LLM output could not be used
pending = [item for item_id, item in t07_items_to_predict.items() if item_id not in t07_predictions]
print(f"Predicting {len(pending)} of {len(t07_items_to_predict)} items with {PARAMETER_LLM_MODEL}")
for number, item in enumerate(pending, 1):
    started = time.monotonic()
    try:
        prediction = predict_item_parameters(item, exclude_item_ids=t07_held_out_ids, model=PARAMETER_LLM_MODEL)
    except (pydantic.ValidationError, openai.LengthFinishReasonError, RuntimeError) as error:
        # Output that does not fit the schema or its bounds, or no parsed output at all.
        t07_unusable[item.item_id] = error
        print(f"{number}/{len(pending)} {item.item_id}: unusable output, skipped ({type(error).__name__})")
        continue
    t07_predictions[item.item_id] = prediction | {"llm_model": PARAMETER_LLM_MODEL}
    print(f"{number}/{len(pending)} {item.item_id} ({item.irt_model}) in {time.monotonic() - started:.0f} s")

if t07_unusable:
    raise RuntimeError(
        f"Unusable LLM output for {len(t07_unusable)} items ({', '.join(t07_unusable)}); "
        "rerun this cell to request only the items still missing"
    )
print(f"All {len(t07_predictions)} items are predicted")

In [77]:
t07_parameter_names = list(PARAMETER_SYMBOLS)
calibrated = item_parameters.set_index("item_id")
t07_predicted_parameters = pd.DataFrame(t07_predictions.values())
t07_predicted_parameters[t07_parameter_names] = t07_predicted_parameters[t07_parameter_names].astype(float)

not_calibrated = sorted(set(t07_predicted_parameters["item_id"]) - set(calibrated.index))
assert not not_calibrated, f"items without calibrated parameters: {not_calibrated}"
calibrated_rows = calibrated.loc[t07_predicted_parameters["item_id"]]
assert (calibrated_rows["irt_model"].to_numpy() == t07_predicted_parameters["irt_model"].to_numpy()).all(), (
    "an item was predicted with another IRT model than it was calibrated with"
)
for name in t07_parameter_names:
    t07_predicted_parameters[f"{name}_calibrated"] = calibrated_rows[name].to_numpy()
t07_predicted_parameters = t07_predicted_parameters[
    [
        "item_id",
        "irt_model",
        "llm_model",
        *t07_parameter_names,
        *[f"{name}_calibrated" for name in t07_parameter_names],
        "rationale",
        "reference_item_ids",
    ]
]

# Baseline: the mean of each parameter over the calibrated items with the same model, without the 74 items,
# which is the mean in each prompt's calibration summary. A useful prediction beats it.
prior_means = (
    item_parameters[~item_parameters["item_id"].isin(t07_held_out_ids)].groupby("irt_model")[t07_parameter_names].mean()
)
accuracy = []
for name in ["slope", "location", "guessing", "step1"]:  # step 2 is -step 1
    rows = t07_predicted_parameters[t07_predicted_parameters[name].notna()]
    predicted, actual = rows[name], rows[f"{name}_calibrated"]
    baseline = prior_means.loc[rows["irt_model"], name].to_numpy()
    accuracy.append(
        {
            "parameter": f"{name} ({PARAMETER_SYMBOLS[name]})",
            "items": len(rows),
            "correlation": predicted.corr(actual) if len(rows) > 2 else float("nan"),
            "mean_difference": (predicted - actual).mean(),
            "mean_absolute_error": (predicted - actual).abs().mean(),
            "root_mean_squared_error": ((predicted - actual) ** 2).mean() ** 0.5,
            "baseline_mean_absolute_error": (baseline - actual).abs().mean(),
        }
    )
print(f"Predicted ({t07_predicted_parameters['llm_model'].iloc[0]}) vs. calibrated parameters of the TIMSS 2007 released items:")
display(pd.DataFrame(accuracy).set_index("parameter").round(3))
print("Largest location errors:")
display(
    t07_predicted_parameters.assign(location_error=lambda df: df["location"] - df["location_calibrated"])
    .sort_values("location_error", key=abs, ascending=False)[
        ["item_id", "irt_model", "location", "location_calibrated", "location_error", "rationale"]
    ]
    .head(10)
    .round(3)
)

NameError: name 't07_predictions' is not defined

In [76]:
for name in t07_parameter_names:
    t07_predicted_parameters[f"{name}_difference"] = (
        t07_predicted_parameters[name] - t07_predicted_parameters[f"{name}_calibrated"]
    )

difference_columns = [f"{name}_difference" for name in t07_parameter_names]
difference_summary = t07_predicted_parameters[difference_columns].describe().T
difference_summary.index = difference_summary.index.str.removesuffix("_difference")
difference_summary.index.name = "parameter"
difference_summary = difference_summary.rename(columns={"count": "items", "50%": "median"})
print("Predicted minus calibrated parameter differences:")
display(difference_summary.round(3))

NameError: name 't07_parameter_names' is not defined

In [ ]:
t07_predicted_parameters.assign(reference_item_ids=t07_predicted_parameters["reference_item_ids"].map(json.dumps)).to_excel(
    T07_PREDICTIONS_PATH, index=False
)
print(f"Exported {len(t07_predicted_parameters):,} rows to {T07_PREDICTIONS_PATH.resolve()}")

In [ ]:
# from pathlib import Path

# import pandas as pd
# from IPython.display import display

# T07_PREDICTIONS_PATH = Path.cwd() / "data" / "timss07_g4_math_predicted_item_parameters.xlsx"
# if not T07_PREDICTIONS_PATH.is_file():
#     raise FileNotFoundError(f"Run the TIMSS 2007 prediction export first: {T07_PREDICTIONS_PATH}")

# t07_predicted_parameters = pd.read_excel(T07_PREDICTIONS_PATH)
# calibrated_columns = [column for column in t07_predicted_parameters if column.endswith("_calibrated")]
# if not calibrated_columns:
#     raise ValueError(f"No calibrated parameter columns found in {T07_PREDICTIONS_PATH.name}")

# for calibrated_column in calibrated_columns:
#     parameter = calibrated_column.removesuffix("_calibrated")
#     if parameter not in t07_predicted_parameters:
#         raise ValueError(f"Missing predicted parameter column {parameter!r}")
#     t07_predicted_parameters[f"{parameter}_difference"] = (
#         pd.to_numeric(t07_predicted_parameters[parameter], errors="raise")
#         - pd.to_numeric(t07_predicted_parameters[calibrated_column], errors="raise")
#     )

# difference_columns = [f"{column.removesuffix('_calibrated')}_difference" for column in calibrated_columns]
# difference_summary = t07_predicted_parameters[difference_columns].describe().T
# difference_summary.index = difference_summary.index.str.removesuffix("_difference")
# difference_summary.index.name = "parameter"
# difference_summary = difference_summary.rename(columns={"count": "items", "50%": "median"})

# t07_predicted_parameters.to_excel(T07_PREDICTIONS_PATH, index=False)
# print(f"Updated {len(t07_predicted_parameters):,} rows in {T07_PREDICTIONS_PATH.resolve()}")
# print("Predicted minus calibrated differences:")
# display(difference_summary.round(3))

Updated 74 rows in /Users/ze/Documents/projects/timss-1/notebooks/data/timss07_g4_math_predicted_item_parameters.xlsx
Predicted minus calibrated differences:


,items,mean,std,min,25%,median,75%,max
parameter,,,,,,,,
slope,74.0,0.079,0.383,-0.517,-0.132,0.058,0.259,2.444
location,74.0,-0.149,0.688,-1.502,-0.564,-0.162,0.224,1.902
guessing,36.0,-0.020,0.065,-0.223,-0.063,-0.013,0.021,0.092
step1,3.0,0.028,0.373,-0.371,-0.143,0.085,0.227,0.369
step2,3.0,-0.028,0.373,-0.369,-0.227,-0.085,0.143,0.371


## Predict the TIMSS 2007 item parameters with gemini-3.8-flash and compare the LLMs

Rerun the predictions of the previous section with `gemini-3.8-flash` through the Gemini API free tier (its OpenAI-compatible endpoint, with `gemini_api_key`).
The function, prompts, reference items, and held-out items are the same as for the `llama3.2` run, so the two runs differ only in the LLM.

The free tier allows 20 `gemini-3.8-flash` requests per day per project, and the count resets at midnight Pacific time (3:00 a.m. Eastern).
Each item takes one request, so the 74 items take four days, fewer if the day's quota is otherwise unused.
The client makes no retries, requests start at least `T07_GEMINI_SECONDS_BETWEEN_REQUESTS` apart, and the first failure stops the cell: a 429 once the day's quota is used up, a 503 when the model is overloaded, or output that cannot be used.
Each prediction is saved under `data/cache/parameter_predictions/timss07_g4_math/gemini-3.8-flash/`, so rerunning the prediction cell on a later day, even after a kernel restart, requests only the items still missing.

The comparison cells combine those predictions with the `llama3.2` predictions in `data/timss07_g4_math_predicted_item_parameters.xlsx`, written by the previous section.
For each item, LLM, and parameter, the difference is the predicted value minus the calibrated value in the TIMSS 2011 calibration (`item_parameters`).
Step 2 is left out, because it is minus step 1 for every GPCM item, both predicted and calibrated, so its differences mirror those of step 1.
The summary by parameter and LLM uses only the items that every LLM has predicted, so the LLMs are compared on the same items while the `gemini-3.8-flash` run is incomplete.
It reports the number of items, the mean difference (bias), its standard deviation, the mean absolute and root mean squared differences, the minimum, median, and maximum difference, and the correlation of predicted with calibrated values.
The results are exported to `data/timss07_g4_math_predicted_item_parameters_by_llm.xlsx`, with sheets `summary`, `by_item` (one row per item and parameter, with each LLM's prediction and difference side by side), and `differences` (one row per item, parameter, and LLM).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

T07_DATA_DIR = Path.cwd() / "data"
T07_GEMINI_MODEL = "gemini-3.8-flash"
T07_GEMINI_SECONDS_BETWEEN_REQUESTS = 15  # at most 4 requests per minute
T07_GEMINI_CACHE_DIR = T07_DATA_DIR / "cache" / "parameter_predictions" / "timss07_g4_math" / T07_GEMINI_MODEL
T07_LLAMA_PREDICTIONS_PATH = T07_DATA_DIR / "timss07_g4_math_predicted_item_parameters.xlsx"  # previous section
T07_LLM_COMPARISON_PATH = T07_DATA_DIR / "timss07_g4_math_predicted_item_parameters_by_llm.xlsx"
# Step 2 is minus step 1 for every GPCM item, predicted and calibrated, so it is not compared separately.
T07_COMPARED_PARAMETERS = ["slope", "location", "guessing", "step1"]


def t07_gemini_prediction_path(item_id):
    return T07_GEMINI_CACHE_DIR / f"{item_id}.json"


def t07_cached_gemini_predictions():
    """The saved gemini-3.8-flash predictions, by item ID."""
    return {path.stem: json.loads(path.read_text()) for path in sorted(T07_GEMINI_CACHE_DIR.glob("*.json"))}


T07_GEMINI_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(t07_cached_gemini_predictions())} items already predicted with {T07_GEMINI_MODEL}")

In [ ]:
import time

import openai
import pydantic
import requests
from openai import OpenAI

required_globals = [
    "gemini_api_key",  # first cells
    "predict_item_parameters",  # Predict IRT parameters for new items with an LLM
    "t07_items_to_predict",  # first cell of Predict IRT parameters for the TIMSS 2007 released items
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise NameError(f"Run the cells that define {missing_globals} first")

# No client retries: every attempt counts against the free tier's daily quota.
t07_gemini_client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=gemini_api_key,
    max_retries=0,
    timeout=600,
)
# As in the llama3.2 run, none of the 74 items informs another's prediction.
held_out_ids = list(t07_items_to_predict)
pending = [item for item_id, item in t07_items_to_predict.items() if not t07_gemini_prediction_path(item_id).exists()]
print(f"Predicting {len(pending)} of {len(t07_items_to_predict)} items with {T07_GEMINI_MODEL} (free tier: 20 requests per day)")
next_request_at = 0.0
for number, item in enumerate(pending, 1):
    time.sleep(max(0.0, next_request_at - time.monotonic()))
    next_request_at = time.monotonic() + T07_GEMINI_SECONDS_BETWEEN_REQUESTS
    try:
        prediction = predict_item_parameters(
            item, exclude_item_ids=held_out_ids, model=T07_GEMINI_MODEL, client=t07_gemini_client
        )
    except (openai.OpenAIError, pydantic.ValidationError, RuntimeError, requests.RequestException) as error:
        print(f"{number}/{len(pending)} {item.item_id} failed, so no further requests were sent:\n{error}")
        break
    t07_gemini_prediction_path(item.item_id).write_text(
        json.dumps(prediction | {"llm_model": T07_GEMINI_MODEL}, ensure_ascii=False, indent=2)
    )
    print(f"{number}/{len(pending)} {item.item_id} ({item.irt_model})")

still_missing = len(t07_items_to_predict) - len(t07_cached_gemini_predictions())
print(f"{len(t07_items_to_predict) - still_missing} of {len(t07_items_to_predict)} items are predicted with {T07_GEMINI_MODEL}")
if still_missing:
    print(
        f"Rerun this cell later to request the other {still_missing}: after a 503, once the model is less busy; "
        "after a 429, once the daily quota resets at midnight Pacific time."
    )

In [ ]:
if "item_parameters" not in globals():
    raise NameError("Run 'TIMSS 2011 grade 4 mathematics item parameters' first")

prediction_columns = ["item_id", "irt_model", "llm_model", *T07_COMPARED_PARAMETERS]
predictions = pd.concat(
    [
        pd.read_excel(T07_LLAMA_PREDICTIONS_PATH)[prediction_columns],
        pd.DataFrame(list(t07_cached_gemini_predictions().values()), columns=prediction_columns),
    ],
    ignore_index=True,
)
assert not predictions.duplicated(["item_id", "llm_model"]).any(), "an item is predicted twice by one LLM"
calibrated = item_parameters.set_index("item_id")
assert (calibrated.loc[predictions["item_id"], "irt_model"].to_numpy() == predictions["irt_model"].to_numpy()).all(), (
    "an item was predicted with another IRT model than it was calibrated with"
)

# One row per item, parameter, and LLM, for the parameters of the item's IRT model.
t07_llm_differences = predictions.melt(
    id_vars=["item_id", "irt_model", "llm_model"],
    value_vars=T07_COMPARED_PARAMETERS,
    var_name="parameter",
    value_name="predicted",
).dropna(subset=["predicted"])
t07_llm_differences["predicted"] = t07_llm_differences["predicted"].astype(float)
t07_llm_differences["calibrated"] = [
    calibrated.at[item_id, parameter]
    for item_id, parameter in zip(t07_llm_differences["item_id"], t07_llm_differences["parameter"])
]
assert t07_llm_differences["calibrated"].notna().all(), "a predicted parameter that the calibration does not have"
t07_llm_differences["difference"] = t07_llm_differences["predicted"] - t07_llm_differences["calibrated"]

items_by_llm = predictions.groupby("llm_model")["item_id"].agg(set)
common_items = set.intersection(*items_by_llm)
t07_llm_differences["predicted_by_every_llm"] = t07_llm_differences["item_id"].isin(common_items)
# Items in booklet order, as in the predictions file, then parameters in T07_COMPARED_PARAMETERS order.
item_order = {item_id: position for position, item_id in enumerate(dict.fromkeys(predictions["item_id"]))}
t07_llm_differences = (
    t07_llm_differences.assign(
        parameter=pd.Categorical(t07_llm_differences["parameter"], categories=T07_COMPARED_PARAMETERS, ordered=True),
        item_order=t07_llm_differences["item_id"].map(item_order),
    )
    .sort_values(["item_order", "parameter", "llm_model"])
    .drop(columns="item_order")
    .reset_index(drop=True)
)


def summarize_differences(rows):
    difference = rows["difference"]
    return pd.Series(
        {
            "items": len(rows),
            "mean_difference": difference.mean(),
            "sd_difference": difference.std(),
            "mean_absolute_difference": difference.abs().mean(),
            "root_mean_squared_difference": np.sqrt((difference**2).mean()),
            "min_difference": difference.min(),
            "median_difference": difference.median(),
            "max_difference": difference.max(),
            # Undefined for fewer than 3 items or a constant column.
            "correlation": (
                rows["predicted"].corr(rows["calibrated"])
                if len(rows) > 2 and rows["predicted"].nunique() > 1 and rows["calibrated"].nunique() > 1
                else np.nan
            ),
        }
    )


t07_llm_summary = (
    t07_llm_differences[t07_llm_differences["predicted_by_every_llm"]]
    .groupby(["parameter", "llm_model"], observed=True)[["predicted", "calibrated", "difference"]]
    .apply(summarize_differences)
    .astype({"items": int})
)

# One row per item and parameter, with each LLM's prediction and difference side by side.
t07_llm_by_item = t07_llm_differences.pivot(
    index=["item_id", "irt_model", "parameter", "calibrated"], columns="llm_model", values=["predicted", "difference"]
)
t07_llm_by_item.columns = [f"{value}_{llm}" for value, llm in t07_llm_by_item.columns]
t07_llm_by_item = (
    t07_llm_by_item.reset_index()
    .assign(item_order=lambda df: df["item_id"].map(item_order))
    .sort_values(["item_order", "parameter"])
    .drop(columns="item_order")
    .reset_index(drop=True)
)

print(
    "Items predicted: "
    + ", ".join(f"{llm} {len(items)}" for llm, items in items_by_llm.items())
    + f"; compared on the {len(common_items)} items every LLM has predicted"
)
print("Predicted minus calibrated parameters, by parameter and LLM:")
display(t07_llm_summary.round(3))
display(t07_llm_by_item[t07_llm_by_item["item_id"].isin(common_items)].head(12).round(3))

In [ ]:
with pd.ExcelWriter(T07_LLM_COMPARISON_PATH) as writer:
    t07_llm_summary.reset_index().to_excel(writer, sheet_name="summary", index=False)
    t07_llm_by_item.to_excel(writer, sheet_name="by_item", index=False)
    t07_llm_differences.to_excel(writer, sheet_name="differences", index=False)
print(f"Exported the comparison to {T07_LLM_COMPARISON_PATH.resolve()}")

## TIMSS 2003 grade 4 released mathematics items

Build `data/timss03_g4_math_released_items.jsonl` (one item per line, with the same fields as `data/timss11_g4_math_released_items.jsonl`) from NCES's [Complete 2003 TIMSS 4 mathematics set](https://nces.ed.gov/timss/released-questions.asp).

The booklet is laid out like the 2011 one: each item page has a classification header, the item label, the boxed item, the item number with the answer key or scoring guide below it, and the percent-correct table in the right column.
It differs in a few ways:

- The classification follows the TIMSS 2003 framework, so `content_domain` is one of Patterns and Relationships, Data, Geometry, Measurement, and Number, and `cognitive_domain` is one of Knowing Facts and Procedures, Using Concepts, Solving Routine Problems, and Reasoning.
- The PDF has no accessibility transcription, so `accessibility_text` is null.
- Some fonts have no Unicode map, so parts of the text layer are garbled (such as the stems of M031249 and M031344A-C), and the labels on the item pages are miscapitalized or garbled, so `item_label` comes from the item index on the last page.
- Some multiple-choice items have five options (A-E).
- M011013 is printed without its response options, so its item box comes from IEA's [TIMSS 2003 Released Items](https://timssandpirls.bc.edu/timss2003i/released.html) (`T03_RELEASED_M4.pdf`) instead, whose page shows the whole item as images and must classify it as the booklet does.
- 2-point items have a "Partially Correct Response" category, after "Correct Response" or, for M031269, "Fully Correct Response".

Extraction is hybrid, as for 2011:

- **Deterministic, from the PDF:** item ID, content domain, main topic, cognitive domain, answer key, response-option letters, scoring-guide category headings (and from them the maximum points), percent correct by education system with the significance markers, and the item label from the index; the item type, key, maximum points, and number of options are checked against the 2003 item information (`Aitinfm3.xls` in `t03_items.zip`), which also gives the number of options of M011013.
- **`gemini-3.8-flash` through OpenRouter:** `stem`, `options`, `scoring_guide`, `scoring_notes`, `has_figure`, and `figure_description`, read from rendered images of the item box and of the answer key or scoring guide, then checked against the deterministic fields.

`figure_description` is model-generated.

In [ ]:
import json
import re
from itertools import groupby
from pathlib import Path

import pandas as pd
import pymupdf
import requests
from IPython.display import display
from pydantic import BaseModel, Field

from timss_1 import released_items as ri

# The "Complete 2003 TIMSS 4 mathematics set" link on https://nces.ed.gov/timss/released-questions.asp
T03_PDF_URL = "https://nces.ed.gov/timss/pdf/TIMSS2003_G4_Math.pdf"
T03_DATA_DIR = Path.cwd() / "data"
T03_PDF_PATH = T03_DATA_DIR / "raw" / "TIMSS2003_G4_Math.pdf"
T03_OUTPUT_PATH = T03_DATA_DIR / "timss03_g4_math_released_items.jsonl"
# The item information of the TIMSS 2003 International Database ("Items" on
# https://timssandpirls.bc.edu/timss2003i/userguide.html), which must agree with the booklet.
T03_ITEMS_ZIP_URL = "https://timssandpirls.bc.edu/timss2003i/PDF/t03_items.zip"
# IEA's TIMSS 2003 Released Items, "Fourth Grade Mathematics" on https://timssandpirls.bc.edu/timss2003i/released.html
T03_IEA_PDF_URL = "https://timssandpirls.bc.edu/PDF/T03_RELEASED_M4.pdf"
# Items whose box comes from IEA's released items: the NCES page of M011013 leaves out its response options.
T03_IEA_ITEMS = ["M011013"]

if not T03_PDF_PATH.exists():
    response = requests.get(T03_PDF_URL, timeout=120)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"{T03_PDF_URL} did not return a PDF")
    T03_PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
    T03_PDF_PATH.write_bytes(response.content)

t03_pdf = pymupdf.open(T03_PDF_PATH)
print(f"{T03_PDF_PATH.name}: {t03_pdf.page_count} pages")

### Deterministic parse of the item pages

Every item starts on a page with the same layout (coordinates in points, origin top-left): the classification header (y 70-115), the item label, the item box from y 148 down to the "Item Number:" line, the answer key or scoring guide below it, and the "Overall Percent Correct" table with its legend in the right column (x > 420).
The scoring guides of M031345A-C and M031344A-C continue on the next page; the other continuation pages show sample student responses and are skipped.
The header's small capitals come out of the text layer in mixed case ("PaTTeRns anD RelaTIOnshIPs"), so classifications are matched ignoring case.
The pages of the multi-part items M031347 and M031348 classify every part, one header row per part, and the row of the page's own part is used.
The page template leaves placeholders such as "Item Number: M022002A" under white boxes, which `t03_words` leaves out.

In [ ]:
T03_VISIBLE_WORDS = pymupdf.TEXTFLAGS_WORDS | pymupdf.TEXT_IGNORE_ACTUALTEXT
T03_VISIBLE_DICT = pymupdf.TEXTFLAGS_DICT | pymupdf.TEXT_IGNORE_ACTUALTEXT
T03_ITEM_ID = re.compile(r"M\d{6}[A-Z]?")
T03_INDEX_PAGE = 118  # PDF page 119, "Item Index 2003"
T03_INDEX_RIGHT_COLUMN_X = 260  # between the left column's page numbers and the right column's item IDs
# Item page layout, in points from the top-left corner.
T03_HEADER_Y = (70, 115)  # classification rows under "Content Domain", "Main Topic", and "Cognitive Domain"
T03_TOPIC_X = 220  # left edge of the "Main Topic" column
T03_COGNITIVE_X = 391  # left edge of the "Cognitive Domain" column
T03_ITEM_BOX_TOP = 148  # just below the item label
T03_RIGHT_COLUMN_X = 420  # the percent-correct table and its legend lie to the right
T03_PERCENT_X = 520  # left edge of the table's percent column
T03_MARKER_X = 540  # left edge of the table's significance-marker column
T03_FOOTER_Y = 745

T03_CONTENT_DOMAINS = ["Patterns and Relationships", "Data", "Geometry", "Measurement", "Number"]
# Main topics printed in this booklet, by content domain, used to validate the parsed header.
T03_MAIN_TOPICS = {
    "Patterns and Relationships": ["Patterns", "Equations and Formulas", "Relationships"],
    "Data": ["Data Representation", "Data Interpretation"],
    "Geometry": [
        "Lines and Angles",
        "Two- and Three-Dimensional Shapes",
        "Congruence and Similarity",
        "Locations and Spatial Relationships",
    ],
    "Measurement": ["Attributes and Units", "Tools, Techniques, and Formulas"],
    "Number": ["Whole Numbers", "Fractions and Decimals", "Ratio, Proportions, and Percent"],
}
T03_COGNITIVE_DOMAINS = ["Knowing Facts and Procedures", "Using Concepts", "Solving Routine Problems", "Reasoning"]
# Category headings of a scoring guide, in printed order: 1-point items, then 2-point items.
T03_SCORING_GUIDES = [
    ["Correct Response", "Incorrect Response"],
    ["Correct Response", "Partially Correct Response", "Incorrect Response"],
    ["Fully Correct Response", "Partially Correct Response", "Incorrect Response"],
]
T03_SCORING_CATEGORIES = {category for guide in T03_SCORING_GUIDES for category in guide}
T03_SIGNIFICANCE = {"Higher": "higher", "Not different": None, "Lower": "lower"}  # legend label -> vs_intl_avg
T03_SUPERSCRIPTS = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")


def t03_word_center(w):
    return pymupdf.Point((w[0] + w[2]) / 2, (w[1] + w[3]) / 2)


def t03_words(page):
    """Visible words, each once.

    The page template leaves placeholders such as "Title Name Pulled from Item" and "Item Number: M022002A" under
    white boxes, and the first item page draws its percent-correct table twice, the first copy of the "International
    average" row under the bar that highlights it. get_text returns all of that text, so a word inside a text span
    that an opaque fill drawn later covers is kept only if visible characters start and end it."""
    fills = [(d["seqno"], d["rect"]) for d in page.get_drawings() if d["fill"] is not None and d["fill_opacity"] == 1]
    covered, visible_chars = [], []
    for span in page.get_texttrace():
        if any(seqno > span["seqno"] and rect.contains(span["bbox"]) for seqno, rect in fills):
            covered.append(pymupdf.Rect(span["bbox"]))
        else:
            visible_chars += [(chr(c[0]), pymupdf.Rect(c[3])) for c in span["chars"]]

    def visible_char_at(char, x, w, edge):
        return any(
            c == char and abs(getattr(rect, edge) - x) < 0.5 and rect.y0 < w[3] and rect.y1 > w[1]
            for c, rect in visible_chars
        )

    def is_visible(w):
        if not any(t03_word_center(w) in rect for rect in covered):
            return True
        return visible_char_at(w[4][0], w[0], w, "x0") and visible_char_at(w[4][-1], w[2], w, "x1")

    unique = {}  # copies drawn on top of each other count once
    for w in page.get_text("words", flags=T03_VISIBLE_WORDS):
        if is_visible(w):
            unique.setdefault((round(w[0]), round(w[1]), w[4]), w)
    return list(unique.values())


def t03_lines(words, tolerance=4):
    """Group words (or spans) into lines by vertical center, each line left to right."""
    lines = []
    for w in sorted(words, key=lambda w: (w[1] + w[3]) / 2):
        center = (w[1] + w[3]) / 2
        if lines and center - lines[-1][0] < tolerance:
            lines[-1][1].append(w)
        else:
            lines.append((center, [w]))
    return [sorted(line, key=lambda w: w[0]) for _, line in lines]


def t03_text(words):
    return " ".join(w[4] for w in words)


def t03_canonical(text, names):
    """The name in `names` that `text` spells. The header's small capitals come out of the PDF in mixed case
    ("PaTTeRns anD RelaTIOnshIPs"), so case is ignored."""
    match = next((name for name in names if name.casefold() == text.casefold()), None)
    if match is None:
        raise ValueError(f"unexpected classification {text!r}, expected one of {names}")
    return match


def t03_classification(words, part):
    """(content domain, main topic, cognitive domain) from the page header.

    Most pages print one row, whose cells may wrap onto a second line. The pages of some multi-part items print a row
    per part, each starting with its part label (such as "B."), and the row of the page's own part is used."""
    lines = t03_lines(w for w in words if T03_HEADER_Y[0] <= w[1] < T03_HEADER_Y[1])
    if len(lines) > 1 and all(re.fullmatch(r"[A-Za-z]\.", line[0][4]) for line in lines):
        lines = [{line[0][4][0].upper(): line[1:] for line in lines}[part]]
    cells = []
    for x0, x1 in ((0, T03_TOPIC_X), (T03_TOPIC_X, T03_COGNITIVE_X), (T03_COGNITIVE_X, float("inf"))):
        cell = ""
        for line in lines:
            text = t03_text(w for w in line if x0 <= w[0] < x1)
            if text:  # a line ending in a hyphen ("Two- and Three-") continues the word on the next line
                cell += text if not cell or cell.endswith("-") else f" {text}"
        cells.append(cell)
    content_domain = t03_canonical(cells[0], T03_CONTENT_DOMAINS)
    main_topic = t03_canonical(cells[1], T03_MAIN_TOPICS[content_domain])
    return content_domain, main_topic, t03_canonical(cells[2], T03_COGNITIVE_DOMAINS)


def t03_pct_correct(words):
    """Rows of the "Overall Percent Correct" table, each row's marker translated with the legend below the table."""
    right = [w for w in words if w[0] >= T03_RIGHT_COLUMN_X]
    table_top = next(w[3] for w in right if w[4] == "Overall")
    legend_top = next(w[1] for w in right if w[4] == "Country")  # "Country average vs. International average:"
    legend = {}  # marker glyph -> vs_intl_avg, from legend lines such as "Higher ▲"
    for line in t03_lines(w for w in right if w[1] > legend_top):
        if t03_text(line[:-1]) in T03_SIGNIFICANCE:
            legend[line[-1][4]] = T03_SIGNIFICANCE[t03_text(line[:-1])]
    assert len(legend) == len(T03_SIGNIFICANCE), f"legend {legend}"

    rows = []
    for line in t03_lines(w for w in right if table_top < w[1] < legend_top):
        name = t03_text(w for w in line if w[0] < T03_PERCENT_X)
        values = [w[4] for w in line if T03_PERCENT_X <= w[0] < T03_MARKER_X]
        markers = [w[4] for w in line if w[0] >= T03_MARKER_X]
        assert len(values) == 1 and values[0].isdigit(), f"percent-correct row {t03_text(line)!r}"
        # The international average row is compared with nothing, whatever its marker cell shows.
        if name != "International average":
            assert len(markers) == 1 and markers[0] in legend, f"percent-correct row {t03_text(line)!r}"
        rows.append(
            {
                "education_system": name,
                "pct_correct": int(values[0]),
                "vs_intl_avg": legend.get(markers[0]) if markers else None,
                "benchmarking": False,  # the 2003 booklet lists no benchmarking participants
            }
        )
    return rows


def t03_option_labels(page, box):
    """Letters of the circled response options in the item box, which the booklet draws as a, b, c, ... in its
    T03Options font."""
    return sorted(
        {
            span["text"].strip().upper()
            for block in page.get_text("dict", flags=T03_VISIBLE_DICT, clip=box)["blocks"]
            for line in block.get("lines", [])
            for span in line["spans"]
            if span["font"] == "T03Options" and span["text"].strip() in set("abcde")
        }
    )


def t03_scoring_area(page, words, top, right):
    """The answer key or scoring guide on a page, from `top` down to any sample student responses or the footer:
    its clip, as (page index, rectangle), and the scoring-guide category headings in it."""
    lines = t03_lines(w for w in words if top <= w[1] < T03_FOOTER_Y and w[0] < right)
    bottom = next(
        (line[0][1] for line in lines if t03_text(line).casefold().startswith("student responses")), T03_FOOTER_Y
    )
    headings = [t03_text(line) for line in lines if line[0][1] < bottom and t03_text(line) in T03_SCORING_CATEGORIES]
    return (page.number, pymupdf.Rect(40, top, right, bottom)), headings


def t03_item_index(page):
    """The item index on the last page as {item ID: (content domain, item label, booklet page)}, column by column."""
    spans = [
        (*span["bbox"], span["text"], span["size"])
        for block in page.get_text("dict", flags=T03_VISIBLE_DICT)["blocks"]
        for line in block.get("lines", [])
        for span in line["spans"]
    ]
    index = {}
    for column in (
        [s for s in spans if s[0] < T03_INDEX_RIGHT_COLUMN_X],
        [s for s in spans if s[0] >= T03_INDEX_RIGHT_COLUMN_X],
    ):
        domain = None
        for line in t03_lines(column):
            size = max(s[5] for s in line)
            text, previous_x1 = "", None
            for s in line:
                if previous_x1 is not None and s[0] - previous_x1 > 2:
                    text += " "  # the page number, far right of the label
                text += s[4].translate(T03_SUPERSCRIPTS) if s[5] < 0.8 * size else s[4]  # "cm2" -> "cm²"
                previous_x1 = s[2]
            text = " ".join(text.split())
            if text in T03_CONTENT_DOMAINS:
                domain = text
            elif match := re.fullmatch(rf"({T03_ITEM_ID.pattern}) (.+) (\d+)", text):
                index[match[1]] = (domain, match[2], int(match[3]))
    return index


def t03_item_number(lines):
    """The item ID printed after "Item Number:" and the top of its line, or (None, None) if the page has none."""
    for line in lines:
        if t03_text(line[:2]) == "Item Number:":
            assert len(line) == 3 and T03_ITEM_ID.fullmatch(line[2][4]), t03_text(line)
            return line[2][4], line[0][1]
    return None, None


def t03_garbled_areas(page):
    """Areas of the text spans in fonts without a Unicode map, whose text comes out of the PDF as shifted character
    codes ("3`eiWd,\\x12QQQQ" for "Answer: ____"), recognized by the "\\x12" or other control character."""
    return [
        pymupdf.Rect(span["bbox"])
        for block in page.get_text("dict", flags=T03_VISIBLE_DICT)["blocks"]
        for line in block.get("lines", [])
        for span in line["spans"]
        if re.search(r"[\x00-\x1f]", span["text"])
    ]


def t03_scoring_top(words):
    """Top of the "SCORING" heading (printed in small capitals as "sCORInG" on some pages), or None."""
    return next((w[1] for w in words if w[4].casefold() == "scoring" and w[0] < 60), None)


# Item pages, each followed by any continuation pages: the rest of a scoring guide, or sample student responses.
t03_items = []
for page in t03_pdf:
    words = t03_words(page)
    lines = t03_lines(words)
    item_id, number_y = t03_item_number(lines)
    if item_id is None:
        continue
    scoring_top = t03_scoring_top(words)
    garbled = t03_garbled_areas(page)
    if not any(w[4] == "Overall" for w in words):  # a continuation page, without the "Overall Percent Correct" table
        assert t03_items and t03_items[-1]["item_id"] == item_id, f"page {page.number + 1} continues {item_id}"
        if scoring_top is not None:
            title_top = next(line[0][1] for line in lines if t03_text(line).endswith("(continued)"))
            clip, headings = t03_scoring_area(page, words, title_top - 2, page.rect.width - 40)
            t03_items[-1]["scoring_clips"].append(clip)
            t03_items[-1]["scoring_categories"] += headings
        continue

    content_domain, main_topic, cognitive_domain = t03_classification(words, item_id[7:] or None)
    key_line = next((line for line in lines if t03_text(line[:2]) == "Correct Response:"), None)
    box = pymupdf.Rect(40, T03_ITEM_BOX_TOP, T03_RIGHT_COLUMN_X, number_y - 1)
    pct_rows = t03_pct_correct(words)
    intl_avg = next(r for r in pct_rows if r["education_system"] == "International average")
    usa = next(r for r in pct_rows if r["education_system"] == "United States")
    item = {
        "item_id": item_id,
        "content_domain": content_domain,
        "main_topic": main_topic,
        "cognitive_domain": cognitive_domain,
        "item_type": "multiple_choice" if key_line else "constructed_response",
        "option_labels": t03_option_labels(page, box),
        "correct_option": key_line[2][4].upper() if key_line else None,  # small capitals, as for the header
        "pct_correct_intl_avg": intl_avg["pct_correct"],
        "pct_correct_usa": usa["pct_correct"],
        "pct_correct": [r for r in pct_rows if r is not intl_avg],
        "booklet_page": int(max((w for w in words if w[1] >= T03_FOOTER_Y), key=lambda w: w[0])[4]),
        "pdf_page": page.number + 1,
        "item_box": (page.number, box),
        "scoring_clips": [],
        "scoring_categories": [],
        # Legible words in the item box, used later to check the model's transcription.
        "_item_box_text": t03_text(
            w
            for w in sorted(words, key=lambda w: (w[1], w[0]))
            if t03_word_center(w) in box and not any(t03_word_center(w) in area for area in garbled)
        ),
    }
    if key_line or scoring_top is not None:  # the answer key or scoring guide starts on the item page
        clip, headings = t03_scoring_area(page, words, number_y - 1, T03_RIGHT_COLUMN_X)
        item["scoring_clips"].append(clip)
        item["scoring_categories"] += headings
    t03_items.append(item)

# The parts of a multi-part item (IDs ending in A, B, C, ...) share one item box, printed on every part's page.
for _, unit in groupby(t03_items, key=lambda item: item["item_id"][:7]):
    unit = list(unit)
    for item in unit:
        item["parts"] = [i["item_id"] for i in unit] if len(unit) > 1 else None
        item["part"] = item["item_id"][7:] if len(unit) > 1 else None
    assert len(unit) == 1 or all(item["part"] for item in unit), f"{unit[0]['item_id']}: parts without part letters"
    assert len({item["_item_box_text"] for item in unit}) == 1, f"{unit[0]['item_id']}: parts with different item boxes"

# The item index gives each item's label and checks the item pages against it. It lists every part of a multi-part
# item under the content domain of the first part, even M031347C, whose own page classifies it as Number.
t03_index = t03_item_index(t03_pdf[T03_INDEX_PAGE])
assert [item["item_id"] for item in t03_items] == list(t03_index), "item pages and item index list different items"
t03_domains = {item["item_id"]: item["content_domain"] for item in t03_items}
for item in t03_items:
    domain, label, booklet_page = t03_index[item["item_id"]]
    unit_domain = t03_domains[(item["parts"] or [item["item_id"]])[0]]
    assert (domain, booklet_page) == (unit_domain, item["booklet_page"]), f"{item['item_id']}: index differs"
    item["item_label"] = label

    if item["item_type"] == "multiple_choice":
        assert item["option_labels"] in ([], list("ABCD"), list("ABCDE")), f"{item['item_id']}: {item['option_labels']}"
        assert not item["option_labels"] or item["correct_option"] in item["option_labels"], item["item_id"]
        assert not item["scoring_categories"], f"{item['item_id']}: multiple-choice item with a scoring guide"
    else:
        assert item["scoring_categories"] in T03_SCORING_GUIDES, f"{item['item_id']}: {item['scoring_categories']}"
    item["max_points"] = 2 if "Partially Correct Response" in item["scoring_categories"] else 1

# The item information must agree on each item's type, key, maximum points, and number of printed options. It also
# gives the number of options of the items that come from IEA's released items, whose page must classify them as the
# booklet does.
t03_information = ri.timss_2003_math_item_information(
    ri.download(T03_ITEMS_ZIP_URL, T03_DATA_DIR / "raw" / "t03_items.zip"), 4
).set_index("Unique ID")
t03_iea_pdf = pymupdf.open(ri.download(T03_IEA_PDF_URL, T03_DATA_DIR / "raw" / "T03_RELEASED_M4.pdf"))
for item in t03_items:
    info = t03_information.loc[item["item_id"]]
    assert info["Released Item"] == "Yes", f"{item['item_id']}: not released in the item information"
    assert (info["ItemType"], info["Key"], int(info["Maximum Points"])) == (
        "MC" if item["item_type"] == "multiple_choice" else "CR",
        item["correct_option"] or "X",
        item["max_points"],
    ), f"{item['item_id']}: item information differs from the booklet"
    option_count = int(info["Response Options"])
    assert len(item["option_labels"]) in (0, option_count), f"{item['item_id']}: {option_count} options in the item information"
    if item["item_id"] in T03_IEA_ITEMS:
        fields, item["iea_images"] = ri.iea_2003_item(t03_iea_pdf, item["item_id"])
        booklet = [item["content_domain"], item["main_topic"], item["cognitive_domain"], item["correct_option"]]
        assert [fields[name].casefold() for name in ri.IEA_2003_FIELDS] == [value.casefold() for value in booklet], fields
        item["option_labels"] = list("ABCDE"[:option_count])

# Not every table lists every education system (Slovenia has no percent correct for M011001).
t03_systems = {r["education_system"] for item in t03_items for r in item["pct_correct"]}
t03_missing_systems = {
    item["item_id"]: sorted(t03_systems - {r["education_system"] for r in item["pct_correct"]})
    for item in t03_items
    if len(item["pct_correct"]) < len(t03_systems)
}

t03_items_df = pd.DataFrame(t03_items)
print(f"Parsed {len(t03_items)} items")
display(t03_items_df.groupby(["content_domain", "item_type"]).size().unstack(fill_value=0))
display(t03_items_df.groupby(["item_type", "max_points"]).size())
print("Parts of multi-part items:")
display(
    t03_items_df[t03_items_df["part"].notna()][
        ["item_id", "item_label", "content_domain", "main_topic", "cognitive_domain", "item_type", "max_points"]
    ]
)
print("Items whose box comes from IEA's released items:", T03_IEA_ITEMS)
print(
    "Multiple-choice items whose response options are not printed in the booklet:",
    [item["item_id"] for item in t03_items if item["item_type"] == "multiple_choice" and not item["option_labels"]],
)
print(f"{len(t03_systems)} education systems; tables without some of them: {t03_missing_systems}")

### Transcribe stems, options, and scoring guides with Gemini

`gemini-3.8-flash` is called through OpenRouter (`google/gemini-3.8-flash`) by `ri.Transcriber`, as in the 2007 section, and the cell prints each request's cost (about $0.02 per request of 4 items, and $0.48 for the whole set on 2026-10-02).
Each request covers up to `T03_ITEMS_PER_REQUEST` items in booklet order, keeping the parts of a multi-part item together, which makes 21 requests for the whole set.
For each item it sends the item box (once per multi-part item) and the answer key or scoring guide, rendered at 300 DPI and trimmed to the printed content; for M011013 it sends the item's images from IEA's released items instead of the booklet's box.
Requests go only to providers that support structured output (`require_parameters`), one at a time, and nothing is retried: the first failed request stops the cell, and rerunning it requests only the items without a usable transcription.
Cached transcriptions are checked again on every run, so M011013's first transcription, made without its options, fails the option check and is requested again.
Set `T03_MAX_REQUESTS` to send at most that many requests in one run.

A transcription is kept only if it passes the checks against the deterministic parse in `t03_problems`: the option letters and answer key of a multiple-choice item, the scoring-guide category headings of a constructed-response item, and only the item's own part label in the stem of a part.
Kept transcriptions are cached per item under `data/cache/item_transcriptions/timss03_g4_math/gemini-3.8-flash-v1/`.
Bump `T03_PROMPT_VERSION` after editing the prompt or schema to re-extract everything, or delete one item's file to re-extract only that item.

In [ ]:
T03_LLM_MODEL = "google/gemini-3.8-flash"
T03_PROMPT_VERSION = 1
T03_LLM_CACHE_DIR = (
    T03_DATA_DIR
    / "cache"
    / "item_transcriptions"
    / "timss03_g4_math"
    / f"{T03_LLM_MODEL.split('/')[-1]}-v{T03_PROMPT_VERSION}"
)
T03_ITEMS_PER_REQUEST = 4
T03_MAX_REQUESTS = None  # a number sends at most that many requests in one run, such as 1 for a test


class T03Option(BaseModel):
    label: str = Field(description="Option letter as printed, e.g. 'A'.")
    text: str = Field(description="Option content in the same notation as the stem.")


class T03ScoringCategory(BaseModel):
    category: str = Field(description="Category heading as printed, e.g. 'Correct Response'.")
    descriptors: list[str] = Field(description="Each bullet under the heading, in printed order.")


class T03Transcription(BaseModel):
    item_id: str
    stem: str
    options: list[T03Option]
    answer_key: str | None
    scoring_guide: list[T03ScoringCategory]
    scoring_notes: str | None
    has_figure: bool
    figure_description: str | None


T03_TRANSCRIPTION_INSTRUCTIONS = """\
You transcribe items from the TIMSS 2003 grade 4 released mathematics booklet into structured data.

Each request covers several items, listed by ID. For each item you receive:
1. An image of the boxed item exactly as students saw it. The parts of a multi-part item (IDs ending in A, B, C, ...) \
share one item box, which is shown once, before the parts: an introduction shared by all parts, followed by every \
part, each starting with its part label (e.g. "B.").
2. One or more images of what the booklet prints below the item box: the item number and either the multiple-choice \
answer key ("Correct Response: X") or the scoring guide under "SCORING", which may continue on the next page.

The images are the only source. Transcribe what is printed, verbatim; never add, correct, or solve anything.

Return one entry per item, in the order given:
- item_id: The item ID as given.
- stem: Everything in the item box for this item except multiple-choice response options, in reading order, with the \
wording verbatim. For a part of a multi-part item, the shared introduction (everything before the first part label) \
followed by this part only, starting with its part label; leave out the other parts. Write math in plain-text \
notation: fractions as 1/2, mixed numbers as 1 1/2, and the symbols ×, ÷, −, =, °. Write an empty box in a number \
sentence as □, a filled box as ■, a number tile (a digit printed in a small square) as the digit in square brackets, \
e.g. [5], and an answer blank as ____, keeping labels such as "Answer:". Write tables as Markdown tables and bulleted \
lists as "- " lines. Replace each picture, diagram, graph, grid, or map with "[Image: <short description>]". Separate \
paragraphs with a newline.
- options: For a multiple-choice item, one entry per response option in printed order, labeled with its circled \
letter (A, B, C, ...), in the same notation as the stem; a pictured option is "[Image: <description>]". Empty for \
constructed-response items, and for a multiple-choice item whose options are not printed.
- answer_key: The letter printed in the "Correct Response:" box of a multiple-choice item; null for \
constructed-response items.
- scoring_guide: For a constructed-response item, each category under "SCORING" in printed order, with its heading \
verbatim (e.g. "Correct Response", "Fully Correct Response", "Partially Correct Response", "Incorrect Response") and \
each bullet verbatim as a descriptor, in the same notation as the stem. Write a picture of a sample answer, whether it \
is a bullet of its own or follows a bullet's text, as "[Image: <description>]". Empty for multiple-choice items.
- scoring_notes: Any scoring-guide text that is not a heading or bullet; null if none.
- has_figure: Whether this item's stem or options contain any picture, diagram, graph, grid, map, or drawn shape. A \
table of text or numbers alone does not count.
- figure_description: If has_figure, a precise, objective description of every visual element in this item's stem \
and options, giving all the information a student would need to answer without seeing it: counts, labels, \
measurements, positions, grid coordinates, bar heights, shading, and orientation. Describe what is drawn; do not \
solve the item or reveal the answer. Null if has_figure is false.
"""


def t03_item_box(item):
    """The item box as image parts: the booklet's, or the item's images in IEA's released items."""
    if item.get("iea_images"):
        return [ri.embedded_image_part(t03_iea_pdf, xref) for xref in item["iea_images"]]
    page_index, rect = item["item_box"]
    return [ri.page_area_part(t03_pdf[page_index], rect)]


def t03_description(item):
    if item["item_type"] == "constructed_response":
        return f"constructed-response, worth {item['max_points']} point{'s' if item['max_points'] > 1 else ''}"
    labels = item["option_labels"]
    options = f"options {labels[0]} to {labels[-1]}" if labels else "its response options are not printed"
    return f"multiple-choice ({options}), worth 1 point"


def t03_request_content(batch):
    """The user message for a batch: each item's box (once per multi-part item) and what is printed below it."""
    content = [
        {"type": "text", "text": f"Transcribe these {len(batch)} items, in this order: {', '.join(i['item_id'] for i in batch)}."}
    ]
    shown_units = set()
    for item in batch:
        if item["parts"]:
            if item["parts"][0] not in shown_units:
                shown_units.add(item["parts"][0])
                content.append({"type": "text", "text": f"Item box of the multi-part item with parts {', '.join(item['parts'])}:"})
                content += t03_item_box(item)
            content.append(
                {
                    "type": "text",
                    "text": f"Item {item['item_id']}: part {item['part']} of that multi-part item, {t03_description(item)}. "
                    "Printed below its item box:",
                }
            )
        else:
            content.append({"type": "text", "text": f"Item {item['item_id']}: {t03_description(item)}. Item box:"})
            content += t03_item_box(item)
            content.append({"type": "text", "text": "Printed below the item box:"})
        content += [ri.page_area_part(t03_pdf[page_index], rect) for page_index, rect in item["scoring_clips"]]
    return content


def t03_problems(item, t, returned):
    """Disagreements between a transcription and the deterministic parse; empty if the transcription is usable."""
    problems = []
    if not t.stem.strip():
        problems.append("empty stem")
    if item["item_type"] == "multiple_choice":
        if [o.label for o in t.options] != item["option_labels"]:
            problems.append(f"option labels {[o.label for o in t.options]}, expected {item['option_labels']}")
        if t.answer_key != item["correct_option"]:
            problems.append(f"answer key {t.answer_key!r}, expected {item['correct_option']!r}")
        if t.scoring_guide:
            problems.append("multiple-choice item has a scoring guide")
    else:
        if t.options or t.answer_key:
            problems.append("constructed-response item has options or an answer key")
        categories = [c.category for c in t.scoring_guide]
        if categories != item["scoring_categories"]:
            problems.append(f"scoring categories {categories}, expected {item['scoring_categories']}")
        if any(not c.descriptors for c in t.scoring_guide):
            problems.append("scoring category without descriptors")
    if item["part"]:
        labels = re.findall(r"(?m)^([A-Z])\.\s", t.stem)
        if labels != [item["part"]]:
            problems.append(f"part labels {labels} in the stem")
    if t.has_figure != bool(t.figure_description):
        problems.append(f"has_figure={t.has_figure} but figure_description={t.figure_description!r}")
    return problems


t03_transcriber = ri.Transcriber(
    api_key=globals().get("openrouter_api_key"),
    model=T03_LLM_MODEL,
    cache_dir=T03_LLM_CACHE_DIR,
    schema=T03Transcription,
    instructions=T03_TRANSCRIPTION_INSTRUCTIONS,
)
t03_transcriptions = t03_transcriber.run(
    t03_items,
    request_content=t03_request_content,
    problems=t03_problems,
    items_per_request=T03_ITEMS_PER_REQUEST,
    unit=lambda item: item["item_id"][:7],
    max_requests=T03_MAX_REQUESTS,
)
print(f"All {len(t03_transcriptions)} items are transcribed")

### Validate and write the JSONL

The coverage report lists words drawn in each item box that are missing from the transcribed stems and options of the item (of all its parts, for a multi-part item).
Misses are usually figure labels, but anything else deserves a look at the page.
Words in fonts without a Unicode map are left out of the report, since the text layer garbles them.
Each record has the fields of the 2011 file, in the same order, which the cell checks against `timss11_g4_math_released_items.jsonl`.

In [ ]:
def t03_normalized(text):
    return re.sub(r"[^0-9a-z]", "", text.lower())


def t03_tidy(text):
    """Even out the model's answer blanks as ____. Unlike for 2011, runs of spaces are kept: in this booklet they align
    the digits of the calculations written vertically (M011015 and M031344A-C)."""
    return re.sub(r"_{2,}", "____", text)


t03_coverage_rows = []
for _, unit in groupby(t03_items, key=lambda item: item["item_id"][:7]):
    unit = list(unit)
    transcribed = t03_normalized(
        " ".join(
            text
            for t in (t03_transcriptions[item["item_id"]] for item in unit)
            for text in [t.stem, *(o.text for o in t.options)]
        )
    )
    drawn = [w for w in map(t03_normalized, unit[0]["_item_box_text"].split()) if len(w) >= 3]
    missing = [w for w in drawn if w not in transcribed]
    if missing:
        t03_coverage_rows.append(
            {
                "item_id": ", ".join(item["item_id"] for item in unit),
                "coverage": 1 - len(missing) / len(drawn),
                "missing_words": " ".join(missing),
            }
        )
print(f"{len(t03_coverage_rows)} item boxes have drawn words missing from the transcription")
if t03_coverage_rows:
    display(pd.DataFrame(t03_coverage_rows).sort_values("coverage"))

t03_records = []
for item in t03_items:
    t = t03_transcriptions[item["item_id"]]
    t03_records.append(
        {
            "item_id": item["item_id"],
            "item_label": item["item_label"],
            "assessment": "TIMSS 2003",
            "grade": 4,
            "subject": "Mathematics",
            "content_domain": item["content_domain"],
            "main_topic": item["main_topic"],
            "cognitive_domain": item["cognitive_domain"],
            "item_type": item["item_type"],
            "max_points": item["max_points"],
            "stem": t03_tidy(t.stem),
            "options": [{"label": o.label, "text": t03_tidy(o.text)} for o in t.options] or None,
            "correct_option": item["correct_option"],
            "scoring_guide": [
                {"category": c.category, "descriptors": [t03_tidy(d) for d in c.descriptors]} for c in t.scoring_guide
            ]
            or None,
            "scoring_notes": t.scoring_notes,
            "has_figure": t.has_figure,
            "figure_description": t.figure_description,
            "accessibility_text": None,
            "pct_correct_intl_avg": item["pct_correct_intl_avg"],
            "pct_correct_usa": item["pct_correct_usa"],
            "pct_correct": item["pct_correct"],
            "booklet_page": item["booklet_page"],
            "pdf_page": item["pdf_page"],
            "source_url": T03_PDF_URL,
            "transcription_model": T03_LLM_MODEL,
        }
    )

t03_schema_path = T03_DATA_DIR / "timss11_g4_math_released_items.jsonl"
if t03_schema_path.exists():
    with t03_schema_path.open(encoding="utf-8") as f:
        t03_schema_fields = list(json.loads(f.readline()))
    assert list(t03_records[0]) == t03_schema_fields, f"fields differ from {t03_schema_path.name}"

with T03_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for record in t03_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

t03_released_items = pd.read_json(T03_OUTPUT_PATH, lines=True)
print(f"Wrote {len(t03_released_items)} items to {T03_OUTPUT_PATH.resolve()}")
t03_preview_columns = ["item_id", "content_domain", "main_topic", "cognitive_domain", "item_type", "stem", "options"]
display(t03_released_items[t03_preview_columns].head())

## TIMSS 1995 grade 4 released mathematics items

Build `data/timss95_g4_math_released_items.jsonl` (one item per line, with the same fields as `data/timss11_g4_math_released_items.jsonl`) from NCES's [Complete 1995 TIMSS 4 mathematics set](https://nces.ed.gov/timss/released-questions.asp).

The booklet is laid out like the 2003 one, one page per item, with these differences:

- The classification follows the TIMSS 1995 framework, which has no topic areas, so `main_topic` is null, `content_domain` is one of Whole Numbers, Fractions and Proportionality, Measurement, Estimation, and Number Sense, Data Representation, Analysis, and Probability, Geometry, and Patterns, Relations, and Functions, and `cognitive_domain` is one of Knowing, Performing Routine Procedures, Using Complex Procedures, and Solving Problems.
- Item IDs are the short 1995 ones, such as I3 and V4A.
- The PDF has no accessibility transcription, so `accessibility_text` is null.
- Response options are printed as text ("A. 823"), except in M1, I6, J2, and K8, whose option letters are part of the pictures; some items have five options (A-E).
- The parts of a multi-part item (V4, T4, U3, T1) are not labeled in the item box; each part's scoring guide starts with "Scoring for Part a", "Scoring for Part b", ...
- 2-point items have a "Partially Correct" or, for U1, "Partial Response" category.
- The PDF lost the empty box of S2's number sentence ("2,000 + □ + 30 + 9 = 2,739"), so the prompt asks for □ where the wording refers to a missing box.
- U3C's copy of the item box drops "time" from "at the same time", but its transcription, like those of U3A and U3B, reads "at the same time".

Extraction is hybrid, as for 2003:

- **Deterministic, from the PDF:** item ID, item label, content domain, cognitive domain, answer key, response-option letters, scoring-guide category headings (and from them the maximum points), and percent correct by education system with the significance markers; the label, content domain, and page of every item are checked against the item index on the last page.
- **`gemini-3.8-flash` through OpenRouter:** `stem`, `options`, `scoring_guide`, `scoring_notes`, `has_figure`, and `figure_description`, read from rendered images of the item box and of the answer key or scoring guide, then checked against the deterministic fields.

`figure_description` is model-generated.

In [ ]:
import json
import re
from itertools import groupby
from pathlib import Path

import pandas as pd
import pymupdf
import requests
from IPython.display import display
from pydantic import BaseModel, Field

from timss_1 import released_items as ri

# The "Complete 1995 TIMSS 4 mathematics set" link on https://nces.ed.gov/timss/released-questions.asp
T95_PDF_URL = "https://nces.ed.gov/timss/pdf/TIMSS1995_G4_Math.pdf"
T95_DATA_DIR = Path.cwd() / "data"
T95_PDF_PATH = T95_DATA_DIR / "raw" / "TIMSS1995_G4_Math.pdf"
T95_OUTPUT_PATH = T95_DATA_DIR / "timss95_g4_math_released_items.jsonl"

if not T95_PDF_PATH.exists():
    response = requests.get(T95_PDF_URL, timeout=120)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"{T95_PDF_URL} did not return a PDF")
    T95_PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
    T95_PDF_PATH.write_bytes(response.content)

t95_pdf = pymupdf.open(T95_PDF_PATH)
print(f"{T95_PDF_PATH.name}: {t95_pdf.page_count} pages")

### Deterministic parse of the item pages

Every item page has the same layout (coordinates in points, origin top-left): the classification header (y 75-115) with the content domain left of x 224 and the cognitive domain right of it, the item label (y 120-150), the item box from y 150 down to the "Item Number:" line, the answer key or scoring guide below it, and the "Overall Percent Correct" table with its legend in the right column (x > 420).
Words get accurate bounding boxes (`TEXT_ACCURATE_BBOXES`), because this PDF's fonts declare ascenders and descenders so tall that the default boxes overlap neighboring lines.
The item pages spell "Data Representation, Analysis and Probability" without the index's serial comma, so classifications are matched ignoring case and punctuation.
Greece's significance marker for M7 is misprinted as "a" and read as "O" (not different), and some tables leave out an education system.

In [ ]:
# The booklet's fonts declare very tall ascenders and descenders, so default word boxes reach far above and below the
# glyphs (the last line of a long scoring guide would overlap the footer); accurate boxes fit the glyphs.
T95_WORDS = pymupdf.TEXTFLAGS_WORDS | pymupdf.TEXT_IGNORE_ACTUALTEXT | pymupdf.TEXT_ACCURATE_BBOXES
T95_ITEM_ID = re.compile(r"[A-Z]\d[A-Z]?")  # 1995 item IDs: block letter, position, and any part letter
T95_INDEX_PAGE = 72  # PDF page 73, "Item Index 1995"
T95_INDEX_RIGHT_COLUMN_X = 260  # between the left column's page numbers and the right column's item IDs
# Item page layout, in points from the top-left corner.
T95_HEADER_Y = (75, 115)  # classification row under "Content Domain" and "Cognitive Domain"
T95_COGNITIVE_X = 224  # the line between the two columns
T95_LABEL_Y = (120, 150)  # item label
T95_ITEM_BOX_TOP = 150  # just below the item label
T95_RIGHT_COLUMN_X = 420  # the percent-correct table and its legend lie to the right
T95_PERCENT_X = 510  # left edge of the table's percent column
T95_MARKER_X = 535  # left edge of the table's significance-marker column
T95_FOOTER_Y = 750

# Spelled as in the item index; the item pages leave out the serial comma in "Analysis, and Probability".
T95_CONTENT_DOMAINS = [
    "Whole Numbers",
    "Fractions and Proportionality",
    "Measurement, Estimation, and Number Sense",
    "Data Representation, Analysis, and Probability",
    "Geometry",
    "Patterns, Relations, and Functions",
]
T95_COGNITIVE_DOMAINS = ["Knowing", "Performing Routine Procedures", "Using Complex Procedures", "Solving Problems"]
# Category headings of a scoring guide, in printed order: 1-point items, then 2-point items.
T95_SCORING_GUIDES = [
    ["Correct Response", "Incorrect Response"],
    ["Correct Response", "Partially Correct", "Incorrect Response"],
    ["Correct Response", "Partial Response", "Incorrect Response"],
]
T95_SCORING_CATEGORIES = {category for guide in T95_SCORING_GUIDES for category in guide}
T95_SIGNIFICANCE = {"Higher": "higher", "Not different": None, "Lower": "lower"}  # legend label -> vs_intl_avg
# Greece's marker for M7 is printed as "a" in the font of the "O" (not different) markers, not in the font of the
# arrows, so it is read as "O".
T95_MARKER_MISPRINTS = {("M7", "Greece"): "O"}


def t95_word_center(w):
    return pymupdf.Point((w[0] + w[2]) / 2, (w[1] + w[3]) / 2)


def t95_lines(words, tolerance=4):
    """Group words into lines by vertical center, each line left to right."""
    lines = []
    for w in sorted(words, key=lambda w: (w[1] + w[3]) / 2):
        center = (w[1] + w[3]) / 2
        if lines and center - lines[-1][0] < tolerance:
            lines[-1][1].append(w)
        else:
            lines.append((center, [w]))
    return [sorted(line, key=lambda w: w[0]) for _, line in lines]


def t95_text(words):
    return " ".join(w[4] for w in words)


def t95_canonical(text, names):
    """The name in `names` that `text` spells, ignoring case and punctuation (the pages and the index differ in commas)."""
    letters = re.sub(r"[^a-z]", "", text.casefold())
    match = next((name for name in names if re.sub(r"[^a-z]", "", name.casefold()) == letters), None)
    if match is None:
        raise ValueError(f"unexpected classification {text!r}, expected one of {names}")
    return match


def t95_classification(words):
    """(content domain, cognitive domain) from the page header, whose cells may wrap onto a second line."""
    lines = t95_lines(w for w in words if T95_HEADER_Y[0] <= w[1] < T95_HEADER_Y[1])
    content = " ".join(t95_text(w for w in line if w[0] < T95_COGNITIVE_X) for line in lines)
    cognitive = " ".join(t95_text(w for w in line if T95_COGNITIVE_X <= w[0] < T95_RIGHT_COLUMN_X) for line in lines)
    return t95_canonical(content, T95_CONTENT_DOMAINS), t95_canonical(cognitive, T95_COGNITIVE_DOMAINS)


def t95_pct_correct(words, item_id):
    """Rows of the "Overall Percent Correct" table, each row's marker translated with the legend below the table."""
    right = [w for w in words if w[0] >= T95_RIGHT_COLUMN_X]
    table_top = next(w[3] for w in right if w[4] == "Overall")
    legend_top = next(w[1] for w in right if w[4] == "Country")  # "Country average vs. International average:"
    legend = {}  # marker glyph -> vs_intl_avg, from legend lines such as "Higher ▲"
    for line in t95_lines(w for w in right if w[1] > legend_top):
        if t95_text(line[:-1]) in T95_SIGNIFICANCE:
            legend[line[-1][4]] = T95_SIGNIFICANCE[t95_text(line[:-1])]
    assert len(legend) == len(T95_SIGNIFICANCE), f"legend {legend}"

    rows = []
    for line in t95_lines(w for w in right if table_top < w[1] < legend_top):
        name = t95_text(w for w in line if w[0] < T95_PERCENT_X)
        values = [w[4] for w in line if T95_PERCENT_X <= w[0] < T95_MARKER_X]
        markers = [T95_MARKER_MISPRINTS.get((item_id, name), w[4]) for w in line if w[0] >= T95_MARKER_X]
        assert len(values) == 1 and values[0].isdigit(), f"{item_id}: percent-correct row {t95_text(line)!r}"
        # The international average row is compared with nothing and has no marker.
        if name != "International average":
            assert len(markers) == 1 and markers[0] in legend, f"{item_id}: percent-correct row {t95_text(line)!r}"
        rows.append(
            {
                "education_system": name,
                "pct_correct": int(values[0]),
                "vs_intl_avg": legend[markers[0]] if markers else None,
                "benchmarking": False,  # the 1995 booklet lists no benchmarking participants
            }
        )
    return rows


def t95_item_number(lines):
    """The item ID printed after "Item Number:" and the top of its line, or (None, None) if the page has none."""
    for line in lines:
        if t95_text(line[:2]) == "Item Number:":
            assert len(line) == 3 and T95_ITEM_ID.fullmatch(line[2][4]), t95_text(line)
            return line[2][4], line[0][1]
    return None, None


def t95_item_index(page):
    """The item index on the last page as {item ID: (content domain, item label, booklet page)}, column by column."""
    words = page.get_text("words", flags=T95_WORDS)
    index = {}
    for column in (
        [w for w in words if w[0] < T95_INDEX_RIGHT_COLUMN_X],
        [w for w in words if w[0] >= T95_INDEX_RIGHT_COLUMN_X],
    ):
        domain = None
        for line in t95_lines(column):
            text = t95_text(line)
            if text in T95_CONTENT_DOMAINS:
                domain = text
            elif match := re.fullmatch(rf"({T95_ITEM_ID.pattern}) (.+) (\d+)", text):
                index[match[1]] = (domain, match[2], int(match[3]))
    return index


# Every item has one page, with its answer key or scoring guide below the item box.
t95_items = []
for page in t95_pdf:
    words = page.get_text("words", flags=T95_WORDS)
    lines = t95_lines(w for w in words if w[0] < T95_RIGHT_COLUMN_X)  # the left column, item box and below
    item_id, number_y = t95_item_number(lines)
    if item_id is None:
        continue
    content_domain, cognitive_domain = t95_classification(words)
    key_line = next((line for line in lines if t95_text(line[:2]) == "Correct Response:"), None)
    box = pymupdf.Rect(40, T95_ITEM_BOX_TOP, T95_RIGHT_COLUMN_X, number_y - 1)
    box_words = [w for w in words if t95_word_center(w) in box]
    # The answer key or scoring guide, whose last line may reach just below the top of the page number.
    scoring_words = [w for w in words if w[0] < T95_RIGHT_COLUMN_X and number_y - 1 <= w[1] < T95_FOOTER_Y]
    scoring_bottom = max(T95_FOOTER_Y, max(w[3] for w in scoring_words) + 0.5)
    scoring = pymupdf.Rect(40, number_y - 1, T95_RIGHT_COLUMN_X, scoring_bottom)
    scoring_lines = [t95_text(line) for line in t95_lines(scoring_words)]
    scored_parts = [m[1] for text in scoring_lines if (m := re.fullmatch(r"Scoring for Part ([a-z])", text))]
    assert key_line or "SCORING" in scoring_lines, f"{item_id}: neither an answer key nor a scoring guide"
    pct_rows = t95_pct_correct(words, item_id)
    intl_avg = next(r for r in pct_rows if r["education_system"] == "International average")
    usa = next(r for r in pct_rows if r["education_system"] == "United States")
    t95_items.append(
        {
            "item_id": item_id,
            "page_label": t95_text(w for w in words if T95_LABEL_Y[0] <= w[1] < T95_LABEL_Y[1] and w[0] < 420),
            "content_domain": content_domain,
            "cognitive_domain": cognitive_domain,
            "item_type": "multiple_choice" if key_line else "constructed_response",
            # Letters of the response options printed as text ("A. 823"); some items print them inside the pictures.
            "option_labels": [line[0][4][0] for line in t95_lines(box_words) if re.fullmatch(r"[A-E]\.", line[0][4])],
            "correct_option": key_line[2][4] if key_line else None,
            "pct_correct_intl_avg": intl_avg["pct_correct"],
            "pct_correct_usa": usa["pct_correct"],
            "pct_correct": [r for r in pct_rows if r is not intl_avg],
            "booklet_page": int(max((w for w in words if w[1] >= T95_FOOTER_Y), key=lambda w: w[0])[4]),
            "pdf_page": page.number + 1,
            "item_box": (page.number, box),
            "scoring_clip": (page.number, scoring),
            "scoring_categories": [text for text in scoring_lines if text in T95_SCORING_CATEGORIES],
            "scored_part": scored_parts[0].upper() if scored_parts else None,  # "Scoring for Part a"
            # Words in the item box, used later to check the model's transcription.
            "_item_box_text": t95_text(w for line in t95_lines(box_words) for w in line),
        }
    )

# The parts of a multi-part item (IDs ending in A, B, C, ...) share one item box, printed on every part's page, and
# each part's scoring guide starts with "Scoring for Part a", "Scoring for Part b", ... The copies of the item box
# are not always identical (U3C's page drops "time" from "at the same time"), so each part keeps its own.
t95_box_differences = {}
for _, unit in groupby(t95_items, key=lambda item: item["item_id"][:2]):
    unit = list(unit)
    for item in unit:
        item["parts"] = [i["item_id"] for i in unit] if len(unit) > 1 else None
        item["part"] = item["item_id"][2:] if len(unit) > 1 else None
        assert item["scored_part"] == item["part"], f"{item['item_id']}: scoring guide for part {item['scored_part']}"
        words, first_words = item["_item_box_text"].split(), unit[0]["_item_box_text"].split()
        if words != first_words:
            t95_box_differences[item["item_id"]] = {
                "missing": [w for w in first_words if w not in words],
                "extra": [w for w in words if w not in first_words],
            }

# The item index must agree with the item pages (its label of V1 ends in a period that the page lacks).
t95_index = t95_item_index(t95_pdf[T95_INDEX_PAGE])
assert [item["item_id"] for item in t95_items] == list(t95_index), "item pages and item index list different items"
for item in t95_items:
    domain, label, booklet_page = t95_index[item["item_id"]]
    assert (domain, label.rstrip("."), booklet_page) == (
        item["content_domain"],
        item["page_label"],
        item["booklet_page"],
    ), f"{item['item_id']}: index differs"
    item["item_label"] = item["page_label"]

    if item["item_type"] == "multiple_choice":
        assert item["option_labels"] in ([], list("ABCD"), list("ABCDE")), f"{item['item_id']}: {item['option_labels']}"
        assert not item["option_labels"] or item["correct_option"] in item["option_labels"], item["item_id"]
        assert not item["scoring_categories"], f"{item['item_id']}: multiple-choice item with a scoring guide"
        item["option_labels"] = item["option_labels"] or None  # None: the letters are printed inside the pictures
    else:
        assert not item["option_labels"], f"{item['item_id']}: constructed-response item with options"
        assert item["scoring_categories"] in T95_SCORING_GUIDES, f"{item['item_id']}: {item['scoring_categories']}"
    item["max_points"] = 2 if len(item["scoring_categories"]) == 3 else 1  # a partial-credit category

# Not every table need list every education system.
t95_systems = {r["education_system"] for item in t95_items for r in item["pct_correct"]}
t95_missing_systems = {
    item["item_id"]: sorted(t95_systems - {r["education_system"] for r in item["pct_correct"]})
    for item in t95_items
    if len(item["pct_correct"]) < len(t95_systems)
}

t95_items_df = pd.DataFrame(t95_items)
print(f"Parsed {len(t95_items)} items")
display(t95_items_df.groupby(["content_domain", "item_type"]).size().unstack(fill_value=0))
display(t95_items_df.groupby(["item_type", "max_points"]).size())
print("Parts of multi-part items:")
display(
    t95_items_df[t95_items_df["part"].notna()][
        ["item_id", "item_label", "content_domain", "cognitive_domain", "item_type", "max_points"]
    ]
)
print(
    "Multiple-choice items whose option letters are printed inside the pictures:",
    [item["item_id"] for item in t95_items if item["item_type"] == "multiple_choice" and not item["option_labels"]],
)
print(f"Parts whose item box differs from the first part's: {t95_box_differences}")
print(f"{len(t95_systems)} education systems; tables without some of them: {t95_missing_systems}")

### Transcribe stems, options, and scoring guides with Gemini

`gemini-3.8-flash` is called through OpenRouter (`google/gemini-3.8-flash`), as in the 2003 section, and the cell prints each request's cost (about $0.03 per request of 4 items, and $0.61 for the whole set on 2026-10-02).
Each request covers up to `T95_ITEMS_PER_REQUEST` items in booklet order, keeping the parts of a multi-part item together, which makes 18 requests for the whole set.
For each item it sends the item's label, the item box, and the answer key or scoring guide, rendered at 300 DPI and trimmed to the printed content.
Each part of a multi-part item gets the item box from its own page, since the copies can differ.
Requests go only to providers that support structured output (`require_parameters`), one at a time, and nothing is retried: the first failed request stops the cell, and rerunning it requests only the items without a usable transcription.
Cached transcriptions are checked again on every run, so checks added later apply to them too.
Set `T95_MAX_REQUESTS` to send at most that many requests in one run.

A transcription is kept only if it passes the checks in `t95_problems`: the option letters and answer key of a multiple-choice item (A-D or A-E when the letters are inside the pictures), the scoring-guide category headings of a constructed-response item, and a stem that differs from those of the item's other parts.
Kept transcriptions are cached per item under `data/cache/item_transcriptions/timss95_g4_math/gemini-3.8-flash-v1/`.
Bump `T95_PROMPT_VERSION` after editing the prompt or schema to re-extract everything, or delete one item's file to re-extract only that item.

In [ ]:
T95_LLM_MODEL = "google/gemini-3.8-flash"
T95_PROMPT_VERSION = 1
T95_LLM_CACHE_DIR = (
    T95_DATA_DIR
    / "cache"
    / "item_transcriptions"
    / "timss95_g4_math"
    / f"{T95_LLM_MODEL.split('/')[-1]}-v{T95_PROMPT_VERSION}"
)
T95_ITEMS_PER_REQUEST = 4
T95_MAX_REQUESTS = None  # a number sends at most that many requests in one run, such as 1 for a test


class T95Option(BaseModel):
    label: str = Field(description="Option letter as printed, without the period, e.g. 'A'.")
    text: str = Field(description="Option content in the same notation as the stem.")


class T95ScoringCategory(BaseModel):
    category: str = Field(description="Category heading as printed, e.g. 'Correct Response'.")
    descriptors: list[str] = Field(description="Each bullet under the heading, in printed order.")


class T95Transcription(BaseModel):
    item_id: str
    stem: str
    options: list[T95Option]
    answer_key: str | None
    scoring_guide: list[T95ScoringCategory]
    scoring_notes: str | None
    has_figure: bool
    figure_description: str | None


T95_TRANSCRIPTION_INSTRUCTIONS = """\
You transcribe items from the TIMSS 1995 grade 4 released mathematics booklet into structured data.

Each request covers several items, listed by ID with their labels. For each item you receive:
1. An image of the boxed item exactly as students saw it.
2. An image of what the booklet prints below the item box: the item number and either the multiple-choice answer key \
("Correct Response: X") or the scoring guide under "SCORING".

The parts of a multi-part item (IDs ending in A, B, C, ...) share one item box, whose image is given with each part. \
The parts are not labeled in the box: a part's scoring guide, which starts with "Scoring for Part a", "Scoring for \
Part b", ..., and its label show which question in the box the part is.

The images are the only source. Transcribe what is printed, verbatim; never add, correct, or solve anything. One \
exception: this PDF lost some of the empty boxes that number sentences refer to, leaving a blank gap (as in \
"2,000 +        + 30 + 9 = 2,739"); write □ in such a gap when the wording refers to a box.

Return one entry per item, in the order given:
- item_id: The item ID as given.
- stem: Everything in the item box for this item except multiple-choice response options, in reading order, with the \
wording verbatim. For a part of a multi-part item, leave out the questions that the other parts score, with their \
answer lines and pictures; keep everything else. Write math in plain-text notation: fractions as 1/2, mixed numbers \
as 1 1/2, and the symbols ×, ÷, −, =, °. Write an empty box in a number sentence as □, a card or tile showing a digit \
in a small square as the digit in square brackets, e.g. [5], and an answer blank as ____, keeping labels such as \
"Answer:". Write tables as Markdown tables and bulleted lists as "- " lines. Replace each picture, diagram, graph, \
grid, or map with "[Image: <short description>]". Separate paragraphs with a newline.
- options: For a multiple-choice item, one entry per response option in printed order, labeled with its letter \
(A, B, C, ...) without the period, in the same notation as the stem; a pictured option is "[Image: <description>]", \
and its letter may be printed inside the picture. Empty for constructed-response items.
- answer_key: The letter printed in the "Correct Response:" box of a multiple-choice item; null for \
constructed-response items.
- scoring_guide: For a constructed-response item, each category of the scoring guide in printed order, with its \
heading verbatim (e.g. "Correct Response", "Partially Correct", "Partial Response", "Incorrect Response") and each \
bullet verbatim as a descriptor, in the same notation as the stem; leave out the "Scoring for Part ..." line. Write a \
picture of a sample answer, whether it is a bullet of its own or follows a bullet's text, as \
"[Image: <description>]". Empty for multiple-choice items.
- scoring_notes: Any scoring-guide text that is not a heading or bullet, such as a note; null if none.
- has_figure: Whether this item's stem or options contain any picture, diagram, graph, grid, map, or drawn shape. A \
table of text or numbers alone does not count.
- figure_description: If has_figure, a precise, objective description of every visual element in this item's stem \
and options, giving all the information a student would need to answer without seeing it: counts, labels, \
measurements, positions, grid coordinates, bar heights, shading, and orientation. Describe what is drawn; do not \
solve the item or reveal the answer. Null if has_figure is false.
"""


def t95_image_part(clip):
    page_index, rect = clip
    return ri.page_area_part(t95_pdf[page_index], rect)


def t95_description(item):
    if item["item_type"] == "constructed_response":
        kind = f"constructed-response, worth {item['max_points']} point{'s' if item['max_points'] > 1 else ''}"
    else:
        labels = item["option_labels"]
        options = f"options {labels[0]} to {labels[-1]}" if labels else "option letters printed inside the pictures"
        kind = f"multiple-choice ({options}), worth 1 point"
    if item["parts"]:
        kind = f"part {item['part']} of the multi-part item with parts {', '.join(item['parts'])}, {kind}"
    return f"Item {item['item_id']} (\"{item['item_label']}\"): {kind}."


def t95_request_content(batch):
    """The user message for a batch: each item's box and what is printed below it."""
    content = [
        {"type": "text", "text": f"Transcribe these {len(batch)} items, in this order: {', '.join(i['item_id'] for i in batch)}."}
    ]
    for item in batch:
        content.append({"type": "text", "text": f"{t95_description(item)} Item box:"})
        content.append(t95_image_part(item["item_box"]))
        content.append({"type": "text", "text": "Printed below the item box:"})
        content.append(t95_image_part(item["scoring_clip"]))
    return content


def t95_problems(item, t, returned):
    """Disagreements between a transcription and the deterministic parse, or with the transcriptions of the other
    parts of its multi-part item in `returned`; empty if the transcription is usable."""
    problems = []
    if not t.stem.strip():
        problems.append("empty stem")
    labels = [o.label for o in t.options]
    if item["item_type"] == "multiple_choice":
        if labels not in ([item["option_labels"]] if item["option_labels"] else [list("ABCD"), list("ABCDE")]):
            problems.append(f"option labels {labels}, expected {item['option_labels'] or 'A-D or A-E'}")
        if t.answer_key != item["correct_option"] or t.answer_key not in labels:
            problems.append(f"answer key {t.answer_key!r}, expected {item['correct_option']!r}")
        if t.scoring_guide:
            problems.append("multiple-choice item has a scoring guide")
    else:
        if t.options or t.answer_key:
            problems.append("constructed-response item has options or an answer key")
        categories = [c.category for c in t.scoring_guide]
        if categories != item["scoring_categories"]:
            problems.append(f"scoring categories {categories}, expected {item['scoring_categories']}")
        if any(not c.descriptors for c in t.scoring_guide):
            problems.append("scoring category without descriptors")
    # Each part is its own question, so no two parts share a stem.
    for other in item["parts"] or []:
        if other != item["item_id"] and other in returned and returned[other].stem == t.stem:
            problems.append(f"same stem as {other}")
    if t.has_figure != bool(t.figure_description):
        problems.append(f"has_figure={t.has_figure} but figure_description={t.figure_description!r}")
    return problems


t95_transcriber = ri.Transcriber(
    api_key=globals().get("openrouter_api_key"),
    model=T95_LLM_MODEL,
    cache_dir=T95_LLM_CACHE_DIR,
    schema=T95Transcription,
    instructions=T95_TRANSCRIPTION_INSTRUCTIONS,
)
t95_transcriptions = t95_transcriber.run(
    t95_items,
    request_content=t95_request_content,
    problems=t95_problems,
    items_per_request=T95_ITEMS_PER_REQUEST,
    unit=lambda item: item["item_id"][:2],  # V4A and V4B are parts of V4
    max_requests=T95_MAX_REQUESTS,
)
print(f"All {len(t95_transcriptions)} items are transcribed")

### Validate and write the JSONL

The coverage report lists words drawn in each item box that are missing from the transcribed stems and options of the item (of all its parts, for a multi-part item).
Misses are usually figure labels, but anything else deserves a look at the page.
Each record has the fields of the 2011 file, in the same order, which the cell checks against `timss11_g4_math_released_items.jsonl`.

In [ ]:
def t95_normalized(text):
    return re.sub(r"[^0-9a-z]", "", text.lower())


def t95_tidy(text):
    """Even out the model's answer blanks as ____. As for 2003, runs of spaces are kept, since they can align the digits
    of calculations written vertically."""
    return re.sub(r"_{2,}", "____", text)


t95_coverage_rows = []
for _, unit in groupby(t95_items, key=lambda item: item["item_id"][:2]):
    unit = list(unit)
    transcribed = t95_normalized(
        " ".join(
            text
            for t in (t95_transcriptions[item["item_id"]] for item in unit)
            for text in [t.stem, *(o.text for o in t.options)]
        )
    )
    drawn = [w for w in map(t95_normalized, unit[0]["_item_box_text"].split()) if len(w) >= 3]
    missing = [w for w in drawn if w not in transcribed]
    if missing:
        t95_coverage_rows.append(
            {
                "item_id": ", ".join(item["item_id"] for item in unit),
                "coverage": 1 - len(missing) / len(drawn),
                "missing_words": " ".join(missing),
            }
        )
print(f"{len(t95_coverage_rows)} item boxes have drawn words missing from the transcription")
if t95_coverage_rows:
    display(pd.DataFrame(t95_coverage_rows).sort_values("coverage"))

t95_records = []
for item in t95_items:
    t = t95_transcriptions[item["item_id"]]
    t95_records.append(
        {
            "item_id": item["item_id"],
            "item_label": item["item_label"],
            "assessment": "TIMSS 1995",
            "grade": 4,
            "subject": "Mathematics",
            "content_domain": item["content_domain"],
            "main_topic": None,  # the 1995 booklet classifies items by content and cognitive domain only
            "cognitive_domain": item["cognitive_domain"],
            "item_type": item["item_type"],
            "max_points": item["max_points"],
            "stem": t95_tidy(t.stem),
            "options": [{"label": o.label, "text": t95_tidy(o.text)} for o in t.options] or None,
            "correct_option": item["correct_option"],
            "scoring_guide": [
                {"category": c.category, "descriptors": [t95_tidy(d) for d in c.descriptors]} for c in t.scoring_guide
            ]
            or None,
            "scoring_notes": t.scoring_notes,
            "has_figure": t.has_figure,
            "figure_description": t.figure_description,
            "accessibility_text": None,
            "pct_correct_intl_avg": item["pct_correct_intl_avg"],
            "pct_correct_usa": item["pct_correct_usa"],
            "pct_correct": item["pct_correct"],
            "booklet_page": item["booklet_page"],
            "pdf_page": item["pdf_page"],
            "source_url": T95_PDF_URL,
            "transcription_model": T95_LLM_MODEL,
        }
    )

t95_schema_path = T95_DATA_DIR / "timss11_g4_math_released_items.jsonl"
if t95_schema_path.exists():
    with t95_schema_path.open(encoding="utf-8") as f:
        t95_schema_fields = list(json.loads(f.readline()))
    assert list(t95_records[0]) == t95_schema_fields, f"fields differ from {t95_schema_path.name}"

with T95_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for record in t95_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

t95_released_items = pd.read_json(T95_OUTPUT_PATH, lines=True)
print(f"Wrote {len(t95_released_items)} items to {T95_OUTPUT_PATH.resolve()}")
t95_preview_columns = ["item_id", "content_domain", "main_topic", "cognitive_domain", "item_type", "stem", "options"]
display(t95_released_items[t95_preview_columns].head())

## TIMSS 2011 grade 8 released mathematics items

Build `data/timss11_g8_math_released_items.jsonl` (one item per line, with the fields of `data/timss11_g4_math_released_items.jsonl`, in the same order) from NCES's [Complete 2011 TIMSS 8 mathematics set](https://nces.ed.gov/timss/released-questions.asp).
This and the other new sections below share the helpers in `src/timss_1/released_items.py` (imported as `ri`): reading words off PDF pages, rendering page areas and embedded images for the model, the OpenRouter transcription loop with its per-item cache, and the record schema.

The booklet is laid out like the grade 4 one, including the official accessibility transcription of every item page (ActualText on tagged-PDF structure elements), with these differences:

- The classification follows the TIMSS 2011 grade 8 framework: `content_domain` is one of Number, Algebra, Geometry, and Data and Chance, and `cognitive_domain` one of Knowing, Applying, and Reasoning.
- M032047's page prints the content domain Number over the Algebra topic "Algebraic Expressions"; the item index lists it under Algebra, and so does the record.
- M042300Z is a derived item worth 2 points: its page shows parts A and B with a scoring guide for each (M042300A and M042300B), and its note gives 1 point for part A alone and 2 for both parts. Its scoring-guide categories start with the part ID, such as "M042300A Correct Response".
- `item_label` is the label on the item page; the item index spells a few labels differently (such as "What xy plus 1 mean").

Extraction is hybrid, as for grade 4:

- **Deterministic, from the PDF:** item ID, item label, content domain, main topic, cognitive domain, answer key, scoring-guide category headings (and from them the maximum points), accessibility text, and percent correct by education system with the significance markers; the item list and content domains are checked against the item index at the front of the booklet.
- **`gemini-3.8-flash` through OpenRouter:** `stem`, `options`, `scoring_guide`, `scoring_notes`, `has_figure`, and `figure_description`, read from rendered images of the item box and of the answer key or scoring guide, with the accessibility text as a guide, then checked against the deterministic fields.

`figure_description` is model-generated.

### Deterministic parse of the item pages

Every item page has the layout of the grade 4 booklet (coordinates in points, origin top-left): the classification row (y 75-112) with the content domain left of x 220, the main topic left of x 410, and the cognitive domain right of it; the item label (y 120-148); the item box (y 150-520); the "Item Number:" line with the answer key or scoring guide below it; and the "Overall Percent Correct" table in the right column (x > 415).
Pages without an "Item Number:" line (front matter and sample student responses) are skipped.
The table's significance markers are vector circles, told apart by their fill, and the legend's circle for "Percent higher than International average" tells which fill means higher; a dash (not applicable) becomes null and "#" (rounds to zero) 0.
The accessibility text of a page is split by position: the elements in the item box describe the item, and those below it, or without a bounding box, the pictures in the scoring guide.

In [ ]:
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
import pymupdf
from IPython.display import display

from timss_1 import released_items as ri

# The "Complete 2011 TIMSS 8 mathematics set" link on https://nces.ed.gov/timss/released-questions.asp
T11G8_PDF_URL = "https://nces.ed.gov/timss/pdf/TIMSS2011_G8_Math.pdf"
T11G8_DATA_DIR = Path.cwd() / "data"
T11G8_PDF_PATH = ri.download(T11G8_PDF_URL, T11G8_DATA_DIR / "raw" / "TIMSS2011_G8_Math.pdf")
T11G8_OUTPUT_PATH = T11G8_DATA_DIR / "timss11_g8_math_released_items.jsonl"
t11g8_pdf = pymupdf.open(T11G8_PDF_PATH)
T11G8_VISIBLE_TEXT = pymupdf.TEXTFLAGS_TEXT | pymupdf.TEXT_IGNORE_ACTUALTEXT

# Item page layout, in points from the top-left corner, as in the grade 4 booklet.
T11G8_RIGHT_COLUMN_X = 415  # the percent-correct table lies to the right
T11G8_HEADER_Y = (75, 112)  # classification row
T11G8_CONTENT_DOMAIN_X_MAX = 220
T11G8_MAIN_TOPIC_X_MAX = 410
T11G8_LABEL_Y = (120, 148)
T11G8_ITEM_BOX_Y = (150, 520)
T11G8_TABLE_LEGEND_Y = 728
T11G8_PERCENT_X_MIN = 525

T11G8_CONTENT_DOMAINS = {
    "NUMBER": "Number",
    "ALGEBRA": "Algebra",
    "GEOMETRY": "Geometry",
    "DATA AND CHANCE": "Data and Chance",
}
# Main topics of the TIMSS 2011 grade 8 mathematics framework, by content domain, spelled as printed in this booklet.
T11G8_MAIN_TOPICS = {
    "Number": ["Whole Numbers", "Fractions and Decimals", "Integers", "Ratio, Proportion and Percent"],
    "Algebra": ["Patterns", "Algebraic Expressions", "Equations/Formulas and Functions"],
    "Geometry": ["Geometric Shapes", "Geometric Measurement", "Location and Movement"],
    "Data and Chance": ["Data Organization and Representation", "Data Interpretation", "Chance"],
}
T11G8_TOPIC_DOMAINS = {topic: domain for domain, topics in T11G8_MAIN_TOPICS.items() for topic in topics}
T11G8_COGNITIVE_DOMAINS = {"Knowing", "Applying", "Reasoning"}
T11G8_PCT_SYMBOLS = {"—": None, "#": 0}  # "Not applicable" and "Rounds to zero"


def t11g8_words_in(page, x0, y0, x1, y1):
    """Visible words whose top-left corner falls inside the rectangle, in reading order."""
    return [
        w
        for w in page.get_text("words", flags=ri.VISIBLE_WORDS, sort=True)
        if x0 <= w[0] < x1 and y0 <= w[1] < y1
    ]


def t11g8_classification(page):
    header = t11g8_words_in(page, 0, T11G8_HEADER_Y[0], page.rect.width, T11G8_HEADER_Y[1])
    content = ri.text(w for w in header if w[0] < T11G8_CONTENT_DOMAIN_X_MAX)
    topic = ri.text(w for w in header if T11G8_CONTENT_DOMAIN_X_MAX <= w[0] < T11G8_MAIN_TOPIC_X_MAX)
    topic = topic.replace("/ ", "/")  # "Equations/ Formulas and Functions" wraps after the slash
    cognitive = ri.text(w for w in header if w[0] >= T11G8_MAIN_TOPIC_X_MAX)
    if content not in T11G8_CONTENT_DOMAINS or topic not in T11G8_TOPIC_DOMAINS or cognitive not in T11G8_COGNITIVE_DOMAINS:
        raise ValueError(f"Unexpected classification on page {page.number + 1}: {(content, topic, cognitive)}")
    return T11G8_CONTENT_DOMAINS[content], topic, cognitive


def t11g8_pct_correct(page):
    """Rows of the "Overall Percent Correct" table, with the significance marker of each row. The markers are
    vector circles, and the legend's circle for "Percent higher than International average" tells which fill means
    higher."""
    words = [
        w
        for w in t11g8_words_in(page, T11G8_RIGHT_COLUMN_X, 150, page.rect.width, T11G8_TABLE_LEGEND_Y)
        if w[3] - w[1] > 5  # drops 2pt-high hidden "Not applicable." text
    ]
    benchmark_y = next(w[1] for w in words if w[4] == "Benchmarking")
    circles = [
        d
        for d in page.get_drawings()
        if d["type"] == "fs" and d["rect"].x0 > T11G8_RIGHT_COLUMN_X and 5 < d["rect"].width < 7
    ]
    legend_higher_y = next(
        w[1] for w in page.get_text("words", flags=ri.VISIBLE_WORDS) if w[4] == "higher" and w[1] > T11G8_TABLE_LEGEND_Y
    )
    higher_fill = next(
        d["fill"] for d in circles if d["rect"].y0 > T11G8_TABLE_LEGEND_Y and abs(d["rect"].y0 - legend_higher_y) < 3
    )
    rows = []
    for line in ri.lines(words, tolerance=3):
        y_mid = (line[0][1] + line[0][3]) / 2
        name = ri.text(w for w in line if w[0] < T11G8_PERCENT_X_MIN)
        values = [w[4] for w in line if w[0] >= T11G8_PERCENT_X_MIN]
        if len(values) != 1 or not (values[0].isdigit() or values[0] in T11G8_PCT_SYMBOLS):
            continue  # headings such as "Education system  Percent correct"
        marker = next(
            (
                d
                for d in circles
                if d["rect"].x0 > T11G8_PERCENT_X_MIN and abs((d["rect"].y0 + d["rect"].y1) / 2 - y_mid) < 3
            ),
            None,
        )
        rows.append(
            {
                "education_system": name,
                "pct_correct": int(values[0]) if values[0].isdigit() else T11G8_PCT_SYMBOLS[values[0]],
                "vs_intl_avg": None if marker is None else ("higher" if marker["fill"] == higher_fill else "lower"),
                "benchmarking": y_mid > benchmark_y,
            }
        )
    return rows


def t11g8_accessibility_texts(doc):
    """Official ActualText of the tagged-PDF structure elements, as {page index: [(top y, text)]}. A few elements
    have no bounding box; on item pages that is only the description of a sample answer in a scoring guide, so they
    are placed at the bottom of the page, with the scoring guide."""
    page_index = {doc.page_xref(i): i for i in range(doc.page_count)}
    texts = defaultdict(list)
    for xref in range(1, doc.xref_length()):
        kind, actual_text = doc.xref_get_key(xref, "ActualText")
        page_kind, page_ref = doc.xref_get_key(xref, "Pg")
        if kind != "string" or page_kind != "xref":
            continue
        page = doc[page_index[int(page_ref.split()[0])]]
        bbox_kind, bbox = doc.xref_get_key(xref, "A/BBox")
        bbox = [float(v) for v in bbox.strip("[]").split()] if bbox_kind == "array" else [0, 0, 0, 0]
        actual_text = re.sub(r"[ \u00a0\u2009\u200a]+", " ", actual_text.replace("\r\n", "\n").replace("\r", "\n"))
        texts[page.number].append((page.rect.height - bbox[3], actual_text.strip()))
    return {number: sorted(entries) for number, entries in texts.items()}


def t11g8_index(doc):
    """The item index at the front of the booklet: {item ID: (content domain, label)}. The domain headings (such as
    "Algebra") come before their items, and the first section, Number, has none."""
    index, domain = {}, "Number"
    text_ = "".join(page.get_text(flags=T11G8_VISIBLE_TEXT) for page in doc.pages(0, 5))
    for match in re.finditer(r"^(Number|Algebra|Geometry|Data and Chance)\s*$|(M\d{6}[A-Z]?)\s*\n(.+?)\s*\.{2,}\s*\d+", text_, re.M):
        if match[1]:
            domain = match[1]
        else:
            index[match[2]] = (domain, " ".join(match[3].split()))
    return index


t11g8_alt_texts = t11g8_accessibility_texts(t11g8_pdf)
t11g8_items = []
for page in t11g8_pdf:
    page_text = page.get_text(flags=T11G8_VISIBLE_TEXT)
    if "Item Number:" not in page_text:
        continue  # front matter and pages of sample student responses
    item_id = re.search(r"Item Number:\s*(M\d{6}[A-Z]?)", page_text).group(1)
    content_domain, main_topic, cognitive_domain = t11g8_classification(page)
    mc_key = re.search(r"Correct Response:\s+([A-E])\b", page_text)
    pct_rows = t11g8_pct_correct(page)
    intl_avg = next(r for r in pct_rows if r["education_system"] == "International average")
    usa = next(r for r in pct_rows if r["education_system"] == "United States")
    entries = t11g8_alt_texts.get(page.number, [])
    t11g8_items.append(
        {
            "item_id": item_id,
            "item_label": ri.text(
                t11g8_words_in(page, 0, T11G8_LABEL_Y[0], T11G8_RIGHT_COLUMN_X, T11G8_LABEL_Y[1])
            ),
            "content_domain": content_domain,
            "main_topic": main_topic,
            "cognitive_domain": cognitive_domain,
            "item_type": "multiple_choice" if mc_key else "constructed_response",
            "correct_option": mc_key.group(1) if mc_key else None,
            "pct_correct_intl_avg": intl_avg["pct_correct"],
            "pct_correct_usa": usa["pct_correct"],
            "pct_correct": [r for r in pct_rows if r is not intl_avg],
            "booklet_page": int(
                next(w[4] for w in t11g8_words_in(page, 250, 760, 400, page.rect.height) if w[4].isdigit())
            ),
            "pdf_page": page.number + 1,
            "accessibility_text": "\n".join(t for y, t in entries if y < T11G8_ITEM_BOX_Y[1]),
            "_scoring_accessibility_text": "\n".join(t for y, t in entries if y >= T11G8_ITEM_BOX_Y[1]),
            # Visible text in the item box, used later to check the model's transcription.
            "_item_box_text": ri.text(
                t11g8_words_in(page, 0, T11G8_ITEM_BOX_Y[0], T11G8_RIGHT_COLUMN_X, T11G8_ITEM_BOX_Y[1])
            ),
        }
    )

t11g8_index_entries = t11g8_index(t11g8_pdf)
problems = []
ids = [item["item_id"] for item in t11g8_items]
if len(ids) != len(set(ids)):
    problems.append("duplicate item IDs")
if set(ids) != set(t11g8_index_entries):
    problems.append(f"item pages and index list different items: {set(ids) ^ set(t11g8_index_entries)}")
for item in t11g8_items:
    if not item["accessibility_text"]:
        problems.append(f"{item['item_id']}: no accessibility text")
    index_domain = t11g8_index_entries[item["item_id"]][0]
    if index_domain != item["content_domain"]:
        # M032047's page prints NUMBER over the Algebra topic "Algebraic Expressions"; the index lists it under
        # Algebra. The topic decides.
        if T11G8_TOPIC_DOMAINS[item["main_topic"]] != index_domain:
            problems.append(f"{item['item_id']}: content domain {item['content_domain']} on the page, {index_domain} in the index")
        print(f"{item['item_id']}: content domain {item['content_domain']} on the page, {index_domain} in the index and by topic")
        item["content_domain"] = index_domain
    elif T11G8_TOPIC_DOMAINS[item["main_topic"]] != item["content_domain"]:
        problems.append(f"{item['item_id']}: topic {item['main_topic']} is not in {item['content_domain']}")
assert not problems, "\n".join(problems)

# The labels on the item pages are used; the index spells a few of them differently.
t11g8_label_differences = {
    item["item_id"]: (item["item_label"], t11g8_index_entries[item["item_id"]][1])
    for item in t11g8_items
    if item["item_label"] != t11g8_index_entries[item["item_id"]][1]
}
print(f"Parsed {len(t11g8_items)} items; labels that differ in the index (page, index): {t11g8_label_differences}")
display(pd.DataFrame(t11g8_items).groupby(["content_domain", "item_type"]).size().unstack(fill_value=0))

### Transcribe stems, options, and scoring guides with Gemini

`gemini-3.8-flash` is called through OpenRouter (`google/gemini-3.8-flash`) with `openrouter_api_key`, by `ri.Transcriber`, which every new section uses.
Each request covers up to `T11G8_ITEMS_PER_REQUEST` items in booklet order, which makes 22 requests for the whole set (about $0.65 in total on 2026-10-03; the cell prints each request's cost).
For each item it sends the item's label and accessibility text and two images rendered at 300 DPI and trimmed to the printed content: the item label and box, and the answer key or scoring guide.
Requests go only to providers that support structured output (`require_parameters`), one at a time, and nothing is retried: the first failed request, or a request without a single usable transcription, stops the cell, and rerunning it requests only the items without a usable transcription.
Cached transcriptions are checked again on every run, so checks added later apply to them too.
Set `T11G8_MAX_REQUESTS` to send at most that many requests in one run, such as 1 for a test.

A transcription is kept only if it passes the checks in `t11g8_problems`: options A-D or A-E and the answer key of a multiple-choice item, and the scoring-guide category headings of a constructed-response item.
Kept transcriptions are cached per item under `data/cache/item_transcriptions/timss11_g8_math/gemini-3.8-flash-v1/`.
Bump `T11G8_PROMPT_VERSION` after editing the prompt or schema to re-extract everything, or delete one item's file to re-extract only that item.

In [ ]:
from pydantic import BaseModel, Field

T11G8_LLM_MODEL = "google/gemini-3.8-flash"
T11G8_PROMPT_VERSION = 1
T11G8_LLM_CACHE_DIR = (
    T11G8_DATA_DIR
    / "cache"
    / "item_transcriptions"
    / "timss11_g8_math"
    / f"{T11G8_LLM_MODEL.split('/')[-1]}-v{T11G8_PROMPT_VERSION}"
)
T11G8_ITEMS_PER_REQUEST = 4
T11G8_MAX_REQUESTS = None  # a number sends at most that many requests in one run, such as 1 for a test
T11G8_ITEM_BOX_CLIP = pymupdf.Rect(40, 120, T11G8_RIGHT_COLUMN_X, 522)  # item label and item box
T11G8_SCORING_CLIP_BOTTOM = 770  # the scoring area runs from the "Item Number:" line to here
T11G8_SCORING_HEADINGS = ["Correct Response", "Partially Correct Response", "Incorrect Response"]


class T11G8Option(BaseModel):
    label: str = Field(description="Option letter as printed, e.g. 'A'.")
    text: str = Field(description="Option content in the same notation as the stem.")


class T11G8ScoringCategory(BaseModel):
    category: str = Field(description="Category heading as printed, e.g. 'Correct Response'.")
    descriptors: list[str] = Field(description="Each bullet under the heading, in printed order.")


class T11G8Transcription(BaseModel):
    item_id: str
    stem: str
    options: list[T11G8Option]
    answer_key: str | None
    scoring_guide: list[T11G8ScoringCategory]
    scoring_notes: str | None
    has_figure: bool
    figure_description: str | None


T11G8_TRANSCRIPTION_INSTRUCTIONS = """\
You transcribe items from the TIMSS 2011 grade 8 released mathematics booklet into structured data.

Each request covers several items, listed by ID. For each item you receive:
1. Two images of the item page. The first shows the item label and the boxed item exactly as students saw it. The \
second shows the item number and either the multiple-choice answer key ("Correct Response: X") or the scoring guide.
2. The official accessibility transcription of the item, written for screen readers (math is spelled out, e.g. \
"one-half", "x squared", "Answer colon blank"), and accessibility descriptions of any pictures in the scoring guide.

The images are the authority for wording, numbers, and layout. Use the accessibility text to confirm hard-to-read \
content and as the source for picture descriptions. Never add content found in neither.

Return one entry per item, in the order given:
- item_id: The item ID as given.
- stem: Everything inside the item box except multiple-choice response options, in reading order, with the wording \
verbatim. Write math in plain-text notation: fractions as 3/8, mixed numbers as 3 5/6, powers as x^2, square roots \
as sqrt(x), and the symbols ×, ÷, −, =, <, >, ≤, ≥, °, π. Write an empty box in a number sentence as □ and an answer \
blank as ____, keeping labels such as "Answer:". Write tables as Markdown tables and bulleted lists as "- " lines. \
Keep part labels of multi-part items (e.g. "A.", "B.") in the stem. Replace each picture, diagram, graph, grid, or \
map with "[Image: <description>]", using the wording of the matching accessibility description where one exists. \
Separate paragraphs with a newline.
- options: For multiple-choice items, one entry per response option in printed order, in the same notation as the \
stem; a pictured option is "[Image: <description>]". Empty for constructed-response items.
- answer_key: The letter printed in the "Correct Response:" box of a multiple-choice item; null for \
constructed-response items.
- scoring_guide: For constructed-response items, each category under "SCORING" in printed order, with its heading \
(e.g. "Correct Response", "Partially Correct Response", "Incorrect Response") and each bullet verbatim as a \
descriptor, in the same notation as the stem. If the guide is split into parts headed by item IDs (such as \
"M042300A"), start each heading with that ID and a space, e.g. "M042300A Correct Response". Write a picture of a \
sample answer, whether it is a bullet of its own or follows a bullet's text, as "[Image: <description>]", using the \
scoring-guide accessibility text where available. Empty for multiple-choice items.
- scoring_notes: Any scoring-guide text that is not a heading or bullet (e.g. "Note: ..."); null if none.
- has_figure: Whether the item box (including the options) contains any picture, diagram, graph, grid, map, or \
drawn shape. A table of text or numbers alone does not count.
- figure_description: If has_figure, a precise, objective description of every visual element, giving all the \
information a student would need to answer without seeing it: counts, labels, measurements, positions, grid \
coordinates, bar heights, shading, and orientation. Describe what is drawn; do not solve the item or reveal the \
answer. Null if has_figure is false.
"""


def t11g8_scoring_area(page):
    """The scoring area of an item page, from the "Item Number:" line down, and the scoring-guide category headings
    in it; a heading under a part ID (such as "M042300A") starts with that ID."""
    words = [w for w in page.get_text("words", flags=ri.VISIBLE_WORDS) if w[0] < T11G8_RIGHT_COLUMN_X and w[1] > 500]
    page_lines = ri.lines(words)
    top = next(line[0][1] for line in page_lines if ri.text(line[:2]) == "Item Number:")
    headings, part = [], None
    for line in page_lines:
        line_text = ri.text(line)
        if line[0][1] < top:
            continue
        if re.fullmatch(r"M\d{6}[A-Z]", line_text):
            part = line_text
        elif line_text in T11G8_SCORING_HEADINGS:
            headings.append(f"{part} {line_text}" if part else line_text)
    return pymupdf.Rect(40, top - 2, T11G8_RIGHT_COLUMN_X, T11G8_SCORING_CLIP_BOTTOM), headings


def t11g8_request_content(batch):
    content = [ri.text_part(f"Transcribe these {len(batch)} items, in this order: {', '.join(i['item_id'] for i in batch)}.")]
    for item in batch:
        page = t11g8_pdf[item["pdf_page"] - 1]
        item_type = item["item_type"].replace("_", "-")
        content += [
            ri.text_part(
                f"Item {item['item_id']} ({item_type}), labeled \"{item['item_label']}\".\n\n"
                f"Accessibility transcription of the item:\n<<<\n{item['accessibility_text']}\n>>>\n\n"
                "Accessibility descriptions of pictures in the scoring guide:\n"
                f"<<<\n{item['_scoring_accessibility_text'] or '(none)'}\n>>>\n\nItem box:"
            ),
            ri.page_area_part(page, T11G8_ITEM_BOX_CLIP),
            ri.text_part("Answer key or scoring guide:"),
            ri.page_area_part(page, item["_scoring_clip"]),
        ]
    return content


def t11g8_problems(item, t, returned):
    """Disagreements between a transcription and the deterministic parse; empty if the transcription is usable."""
    problems = []
    if not t.stem.strip():
        problems.append("empty stem")
    labels = [o.label for o in t.options]
    if item["item_type"] == "multiple_choice":
        if labels not in (list("ABCD"), list("ABCDE")):
            problems.append(f"option labels {labels}")
        if t.answer_key != item["correct_option"]:
            problems.append(f"answer key {t.answer_key!r}, expected {item['correct_option']!r}")
        if t.scoring_guide:
            problems.append("multiple-choice item has a scoring guide")
    else:
        if t.options or t.answer_key:
            problems.append("constructed-response item has options or an answer key")
        categories = [c.category for c in t.scoring_guide]
        if categories != item["_scoring_headings"]:
            problems.append(f"scoring categories {categories}, expected {item['_scoring_headings']}")
        if any(not c.descriptors for c in t.scoring_guide):
            problems.append("scoring category without descriptors")
    if t.has_figure != bool(t.figure_description):
        problems.append(f"has_figure={t.has_figure} but figure_description={t.figure_description!r}")
    return problems


for item in t11g8_items:
    item["_scoring_clip"], item["_scoring_headings"] = t11g8_scoring_area(t11g8_pdf[item["pdf_page"] - 1])
    if item["item_type"] == "multiple_choice":
        assert not item["_scoring_headings"], f"{item['item_id']}: multiple-choice item with a scoring guide"
    else:
        assert item["_scoring_headings"], f"{item['item_id']}: constructed-response item without scoring headings"
    # 2 points with a partial-credit category, or for a derived item scored from several parts (M042300Z).
    split = len({heading.split()[0] for heading in item["_scoring_headings"] if heading.startswith("M")}) > 1
    item["max_points"] = 2 if "Partially Correct Response" in item["_scoring_headings"] or split else 1

t11g8_transcriber = ri.Transcriber(
    api_key=globals().get("openrouter_api_key"),
    model=T11G8_LLM_MODEL,
    cache_dir=T11G8_LLM_CACHE_DIR,
    schema=T11G8Transcription,
    instructions=T11G8_TRANSCRIPTION_INSTRUCTIONS,
)
t11g8_transcriptions = t11g8_transcriber.run(
    t11g8_items,
    request_content=t11g8_request_content,
    problems=t11g8_problems,
    items_per_request=T11G8_ITEMS_PER_REQUEST,
    max_requests=T11G8_MAX_REQUESTS,
)
print(f"All {len(t11g8_transcriptions)} items are transcribed")

### Validate and write the JSONL

The coverage report lists words drawn in each item box that are missing from the item's transcribed stem and options.
Misses are usually figure labels (axis ticks, protractor degrees), but anything else deserves a look at the page.
`ri.write_jsonl` checks that every record has the fields of the 2011 grade 4 file, in the same order, and that no item ID repeats.

In [ ]:
t11g8_coverage = ri.coverage(
    {item["item_id"]: item["_item_box_text"] for item in t11g8_items},
    {
        item_id: " ".join([t.stem, *(o.text for o in t.options)])
        for item_id, t in t11g8_transcriptions.items()
    },
)
print(f"{len(t11g8_coverage)} items have drawn words missing from the transcription")
if not t11g8_coverage.empty:
    display(t11g8_coverage)

t11g8_records = []
for item in t11g8_items:
    t = t11g8_transcriptions[item["item_id"]]
    t11g8_records.append(
        {
            "item_id": item["item_id"],
            "item_label": item["item_label"],
            "assessment": "TIMSS 2011",
            "grade": 8,
            "subject": "Mathematics",
            "content_domain": item["content_domain"],
            "main_topic": item["main_topic"],
            "cognitive_domain": item["cognitive_domain"],
            "item_type": item["item_type"],
            "max_points": item["max_points"],
            "stem": ri.tidy(t.stem),
            "options": [{"label": o.label, "text": ri.tidy(o.text)} for o in t.options] or None,
            "correct_option": item["correct_option"],
            "scoring_guide": [
                {"category": c.category, "descriptors": [ri.tidy(d) for d in c.descriptors]} for c in t.scoring_guide
            ]
            or None,
            "scoring_notes": t.scoring_notes,
            "has_figure": t.has_figure,
            "figure_description": t.figure_description,
            "accessibility_text": item["accessibility_text"],
            "pct_correct_intl_avg": item["pct_correct_intl_avg"],
            "pct_correct_usa": item["pct_correct_usa"],
            "pct_correct": item["pct_correct"],
            "booklet_page": item["booklet_page"],
            "pdf_page": item["pdf_page"],
            "source_url": T11G8_PDF_URL,
            "transcription_model": T11G8_LLM_MODEL,
        }
    )

t11g8_released_items = ri.write_jsonl(t11g8_records, T11G8_OUTPUT_PATH)
print(f"Wrote {len(t11g8_released_items)} items to {T11G8_OUTPUT_PATH.resolve()}")
display(t11g8_released_items[["item_id", "content_domain", "main_topic", "cognitive_domain", "item_type", "stem", "options"]].head())

## TIMSS 2007 grade 8 released mathematics items

Build `data/timss07_g8_math_released_items.jsonl` from the zip linked as "Mathematics" under "Grade 8 Released Items" on the [TIMSS 2007 Released Items](https://timssandpirls.bc.edu/timss2007/items.html) page, which the NCES page links to for TIMSS 2007.

The PDF is laid out like the grade 4 one: each item page carries the item documentation as text (item ID, block, block sequence, content domain, cognitive domain, maximum points, and key) and the item as one scanned image, and each constructed-response item is followed by a page whose image is its scoring guide, a table of score codes.
MP32753 and MP32754 are shared introduction pages, not items.
Unlike the 2007 grade 4 section, this one fills the fields that the PDF lacks from two official files:

- `item_label`, `main_topic` (the 2007 framework's topic area), and the number of response options come from `T07_G8_ItemInformation.xls` in `T07_Items.zip` (the "Items" link on the [TIMSS 2007 International Database and User Guide](https://timssandpirls.bc.edu/TIMSS2007/idb_ug.html) page), which must agree with the booklet on the content domain, cognitive domain, maximum points, and key. Topic areas use the framework's names (`ri.TIMSS_2007_TOPIC_AREAS`), since the file spells some several ways.
- `pct_correct_intl_avg`, `pct_correct_usa`, and `pct_correct` come from NCES's "TIMSS Grade 8 mathematics item statistics" workbook (TIMSS 2007 on the NCES page), read by `ri.nces_2007_percent_correct`. The workbook marks each result as above or below the U.S. average by cell color, not against the international average, so `vs_intl_avg` is null. It has no statistics for M042273, which TIMSS did not use in scaling.

Extraction is hybrid:

- **Deterministic, from the PDF and the two files:** the item documentation, cross-checked against the item table on PDF pages 4-5; the item and scoring-guide images, taken from the PDF as embedded, without the red watermark; and the fields above.
- **`gemini-3.8-flash` through OpenRouter:** `stem`, `options`, `scoring_guide`, `scoring_notes`, `has_figure`, and `figure_description`, read from those images and checked against the documentation before they are kept.

`figure_description` is model-generated.

### Deterministic parse of the item documentation and images

The item table on PDF pages 4-5 lists every item with its subject, grade, block, block sequence, content domain, cognitive domain, maximum points, and key ("See scoring guide" for constructed-response items); each word belongs to the column whose heading starts left of it.
Each item page repeats that documentation in its header and sidebar, and each scoring-guide page names its item in the header.
The largest image on a page is the item or the scoring guide; the others are logos.
The parts of a multi-part item share a block and block sequence, and each part is transcribed from all of their booklet pages, including a shared introduction.

In [ ]:
import re
import zipfile
from bisect import bisect_right
from collections import defaultdict
from pathlib import Path

import pandas as pd
import pymupdf
from IPython.display import display

from timss_1 import released_items as ri

# The "Mathematics" link under "Grade 8 Released Items" on https://timssandpirls.bc.edu/timss2007/items.html, which
# the NCES page links to for TIMSS 2007
T07G8_ZIP_URL = "https://timssandpirls.bc.edu/timss2007/PDF/T07_G8_Released_Items_MAT.zip"
T07G8_DATA_DIR = Path.cwd() / "data"
T07G8_ZIP_PATH = ri.download(T07G8_ZIP_URL, T07G8_DATA_DIR / "raw" / "T07_G8_Released_Items_MAT.zip")
T07G8_PDF_PATH = T07G8_ZIP_PATH.with_suffix("") / "T07_G8_Released_Items_MAT.pdf"
T07G8_OUTPUT_PATH = T07G8_DATA_DIR / "timss07_g8_math_released_items.jsonl"
if not T07G8_PDF_PATH.exists():
    with zipfile.ZipFile(T07G8_ZIP_PATH) as archive:
        member = next(name for name in archive.namelist() if name.endswith(T07G8_PDF_PATH.name))
        T07G8_PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
        T07G8_PDF_PATH.write_bytes(archive.read(member))
t07g8_pdf = pymupdf.open(T07G8_PDF_PATH)

T07G8_VISIBLE_TEXT = pymupdf.TEXTFLAGS_TEXT | pymupdf.TEXT_IGNORE_ACTUALTEXT
T07G8_ITEM_ID = re.compile(r"M(?:P\d{5}|\d{6})[A-Z]?")
T07G8_TABLE_PAGES = range(3, 5)  # PDF pages 4-5
T07G8_FOOTER_Y = 740
T07G8_FIELDS = [
    "Item ID",
    "Subject",
    "Grade",
    "Block",
    "Block Seq",
    "Content Domain",
    "Cognitive Domain",
    "Maximum Points",
    "Key",
]
# First word of each item-table column heading ("Block" starts both "Block" and "Block Seq").
T07G8_COLUMN_HEADINGS = {"Item", "Subject", "Grade", "Block", "Content", "Cognitive", "Maximum", "Key"}
T07G8_CONTENT_DOMAINS = {"Number", "Algebra", "Geometry", "Data and Chance"}
T07G8_COGNITIVE_DOMAINS = {"Knowing", "Applying", "Reasoning"}
T07G8_CONSTRUCTED_RESPONSE_KEY = "See scoring guide"
# The item information file of the TIMSS 2007 International Database ("Items" on
# https://timssandpirls.bc.edu/TIMSS2007/idb_ug.html) gives each item's label, topic area, and number of options.
T07G8_ITEMS_ZIP_URL = "https://timssandpirls.bc.edu/TIMSS2007/PDF/T07_Items.zip"
T07G8_ITEMS_ZIP_PATH = T07G8_DATA_DIR / "raw" / "T07_Items.zip"


def t07g8_table_rows(page):
    """Item-table rows as {field: text}; each word belongs to the column whose heading starts left of it."""
    words = page.get_text("words", flags=ri.VISIBLE_WORDS)
    first_row_y = min(w[1] for w in words if T07G8_ITEM_ID.fullmatch(w[4]))
    column_x = sorted(w[0] for w in words if w[1] < first_row_y - 5 and w[4] in T07G8_COLUMN_HEADINGS)
    assert len(column_x) == len(T07G8_FIELDS), f"item-table columns on page {page.number + 1}: {column_x}"
    rows = []
    for line in ri.lines(w for w in words if first_row_y - 5 <= w[1] < T07G8_FOOTER_Y):
        row = defaultdict(list)
        for w in line:
            row[T07G8_FIELDS[bisect_right(column_x, w[0] + 1) - 1]].append(w[4])
        rows.append({field: " ".join(texts) for field, texts in row.items()})
    return rows


def t07g8_page_fields(page):
    """Documentation on an item or scoring-guide page: the header's label/value pairs and the sidebar's labeled boxes."""
    fields = {}
    for block in page.get_text("blocks", flags=T07G8_VISIBLE_TEXT):
        lines = [line.strip() for line in block[4].splitlines() if line.strip()]
        if lines[:1] == ["Item ID"]:
            fields.update(zip(lines[::2], lines[1::2]))
        elif lines and lines[0] in T07G8_FIELDS:
            fields[lines[0]] = " ".join(lines[1:])
    return fields


def t07g8_booklet_page(page):
    """The printed page number: the first word of the footer on left-hand pages, the last on right-hand ones."""
    footer = sorted((w for w in page.get_text("words", flags=ri.VISIBLE_WORDS) if w[1] >= T07G8_FOOTER_Y), key=lambda w: w[0])
    return int(next(w[4] for w in (footer[0], footer[-1]) if w[4].isdigit()))


def t07g8_main_image(page):
    """xref of the largest image on the page, the item or the scoring guide; the others are logos."""
    return max(page.get_image_info(xrefs=True), key=lambda info: abs(pymupdf.Rect(info["bbox"])))["xref"]


t07g8_table = [row for number in T07G8_TABLE_PAGES for row in t07g8_table_rows(t07g8_pdf[number])]

# Item pages (including shared introductions such as MP32754) and scoring-guide pages, after the item table.
t07g8_pages = {}
t07g8_scoring_images = defaultdict(list)
for page in t07g8_pdf.pages(T07G8_TABLE_PAGES[-1] + 1):
    fields = t07g8_page_fields(page)
    if "Item ID" not in fields:
        continue
    if "Key" in fields:
        assert fields["Item ID"] not in t07g8_pages, f"second item page for {fields['Item ID']}"
        t07g8_pages[fields["Item ID"]] = {
            **fields,
            "pdf_page": page.number + 1,
            "booklet_page": t07g8_booklet_page(page),
            "image": t07g8_main_image(page),
        }
    else:
        t07g8_scoring_images[fields["Item ID"]].append(t07g8_main_image(page))

assert [row["Item ID"] for row in t07g8_table] == list(t07g8_pages), "item table and item pages list different items"
for row in t07g8_table:
    page_fields = t07g8_pages[row["Item ID"]]
    mismatches = [f for f in T07G8_FIELDS if row.get(f, "") != page_fields.get(f, "")]
    assert not mismatches, f"{row['Item ID']}: item table and item page differ in {mismatches}"

# Booklet page images of each multi-part item (by block and block sequence), in page order.
t07g8_unit_images = defaultdict(list)
for doc in t07g8_pages.values():
    images = t07g8_unit_images[doc["Block"], doc["Block Seq"]]
    if doc["image"] not in images:
        images.append(doc["image"])

t07g8_items = []
for item_id, doc in t07g8_pages.items():
    if not doc["Key"]:
        assert item_id.startswith("MP") and not (doc["Content Domain"] or doc["Maximum Points"]), item_id
        continue  # shared introduction of a multi-part item
    unit = [i for i, d in t07g8_pages.items() if (d["Block"], d["Block Seq"]) == (doc["Block"], doc["Block Seq"]) and d["Key"]]
    is_multiple_choice = doc["Key"] != T07G8_CONSTRUCTED_RESPONSE_KEY
    assert (doc["Subject"], doc["Grade"]) == ("M", "8"), item_id
    assert doc["Content Domain"] in T07G8_CONTENT_DOMAINS and doc["Cognitive Domain"] in T07G8_COGNITIVE_DOMAINS, item_id
    assert doc["Maximum Points"] in {"1", "2"}, item_id
    assert not is_multiple_choice or (doc["Key"] in set("ABCDE") and doc["Maximum Points"] == "1"), item_id
    assert len(t07g8_scoring_images[item_id]) == (0 if is_multiple_choice else 1), f"{item_id}: scoring-guide pages"
    assert len(unit) == 1 or item_id[-1].isalpha(), f"{item_id}: part of a multi-part item without a part letter"
    t07g8_items.append(
        {
            "item_id": item_id,
            "block": doc["Block"],
            "block_seq": doc["Block Seq"],
            "parts": unit if len(unit) > 1 else None,
            "part": item_id[-1] if len(unit) > 1 else None,
            "content_domain": doc["Content Domain"],
            "cognitive_domain": doc["Cognitive Domain"],
            "item_type": "multiple_choice" if is_multiple_choice else "constructed_response",
            "max_points": int(doc["Maximum Points"]),
            "correct_option": doc["Key"] if is_multiple_choice else None,
            "booklet_page": doc["booklet_page"],
            "pdf_page": doc["pdf_page"],
            "page_images": t07g8_unit_images[doc["Block"], doc["Block Seq"]],
            "scoring_image": t07g8_scoring_images[item_id][0] if not is_multiple_choice else None,
        }
    )

# Label, topic area, and number of options from the item information file, which must agree with the booklet.
t07g8_information = ri.timss_2007_math_item_information(ri.download(T07G8_ITEMS_ZIP_URL, T07G8_ITEMS_ZIP_PATH), 8)
for item in t07g8_items:
    info = t07g8_information.loc[item["item_id"]]
    assert info["Released"] == "Yes", f"{item['item_id']}: not released in the item information"
    assert (info["Content Domain"], info["Cognitive Domain"], int(info["Maximum Points"])) == (
        item["content_domain"],
        item["cognitive_domain"],
        item["max_points"],
    ), f"{item['item_id']}: item information differs from the booklet"
    assert (info["Key"] if info["Key"] != "-" else None) == item["correct_option"], f"{item['item_id']}: key differs"
    item["item_label"] = info["Item Label"]
    item["main_topic"] = info["Topic Area"]
    item["option_labels"] = list("ABCDE"[: int(info["Options"])]) if info["Options"] != "-" else None

t07g8_items_df = pd.DataFrame(t07g8_items)
print(f"Parsed {len(t07g8_items)} items")
display(t07g8_items_df.groupby(["content_domain", "item_type"]).size().unstack(fill_value=0))
print("Parts of multi-part items:", t07g8_items_df.loc[t07g8_items_df["part"].notna(), "item_id"].tolist())

### Transcribe stems, options, and scoring guides with Gemini

`gemini-3.8-flash` is called through OpenRouter by `ri.Transcriber`, as in the 2011 grade 8 section, with up to `T07G8_ITEMS_PER_REQUEST` items per request in booklet order, keeping the parts of a multi-part item together, which makes 24 requests for the whole set.
For each item it sends its booklet page image (all pages of a multi-part item, once per request) and, for a constructed-response item, its scoring guide, both as embedded in the PDF and trimmed to the printed content.
These scanned images make larger requests than the rendered pages of the NCES booklets.

A transcription is kept only if it passes the checks in `t07g8_problems`: the option letters that the item information gives for a multiple-choice item, the item ID printed on the scoring guide, the scoring categories (matched by their letters, since M032691's guide prints "Non response") and code ranges implied by the maximum points (the first digit of a score code is the score, except 7 for incorrect and 9 for nonresponse), and only the item's own part label in the stem of a part.
Kept transcriptions are cached per item under `data/cache/item_transcriptions/timss07_g8_math/gemini-3.8-flash-v1/`; set `T07G8_MAX_REQUESTS` to send fewer requests, and bump `T07G8_PROMPT_VERSION` after editing the prompt or schema.

In [ ]:
from pydantic import BaseModel, Field

T07G8_LLM_MODEL = "google/gemini-3.8-flash"
T07G8_PROMPT_VERSION = 1
T07G8_LLM_CACHE_DIR = (
    T07G8_DATA_DIR
    / "cache"
    / "item_transcriptions"
    / "timss07_g8_math"
    / f"{T07G8_LLM_MODEL.split('/')[-1]}-v{T07G8_PROMPT_VERSION}"
)
T07G8_ITEMS_PER_REQUEST = 4
T07G8_MAX_REQUESTS = None  # a number sends at most that many requests in one run, such as 1 for a test
# Codes each printed scoring-guide category may hold, by the item's maximum points: the first digit of a TIMSS
# score code is the score, except 7 (incorrect) and 9 (nonresponse).
T07G8_CATEGORY_CODES = {
    "Correct Response": {1: range(10, 20), 2: range(20, 30)},
    "Partial Response": {2: range(10, 20)},
    "Incorrect Response": {1: range(70, 80), 2: range(70, 80)},
    "Nonresponse": {1: range(90, 100), 2: range(90, 100)},
}


def t07g8_category(heading):
    """The category of a printed heading, matched by its letters: M032691's guide prints "Non response"."""
    key = re.sub(r"[^a-z]", "", heading.casefold())
    return next((c for c in T07G8_CATEGORY_CODES if re.sub(r"[^a-z]", "", c.casefold()) == key), heading)


class T07G8Option(BaseModel):
    label: str = Field(description="Option letter as printed, e.g. 'A'.")
    text: str = Field(description="Option content in the same notation as the stem.")


class T07G8ScoringCode(BaseModel):
    code: int = Field(description="Score code as printed, e.g. 10.")
    response: str = Field(description="The code's response description, verbatim.")


class T07G8ScoringCategory(BaseModel):
    category: str = Field(description="Category heading as printed, e.g. 'Correct Response'.")
    codes: list[T07G8ScoringCode] = Field(description="Each code row under the heading, in printed order.")


class T07G8Transcription(BaseModel):
    item_id: str
    stem: str
    options: list[T07G8Option]
    scoring_guide_item_id: str | None
    scoring_guide: list[T07G8ScoringCategory]
    scoring_notes: str | None
    has_figure: bool
    figure_description: str | None


T07G8_TRANSCRIPTION_INSTRUCTIONS = """\
You transcribe items from the TIMSS 2007 grade 8 released mathematics items into structured data.

Each request covers several items, listed by ID. For each item you receive:
1. The booklet page the item appears on, as an image, exactly as students saw it. The parts of a multi-part item \
(IDs ending in A, B, C, ...) share their booklet pages, which are shown once, before the parts. Those pages hold an \
introduction shared by all parts, which may fill a page of its own, followed by every part, each starting with its \
part label (e.g. "B.").
2. For a constructed-response item, its scoring guide as an image: a table of score codes headed by the item ID.

The images are the only source. Transcribe what is printed, verbatim; never add, correct, or solve anything.

Return one entry per item, in the order given:
- item_id: The item ID as given.
- stem: Everything the student sees for this item except the multiple-choice response options, in reading order. \
For a part of a multi-part item, the shared introduction (everything before the first part label) followed by this \
part only, starting with its part label; leave out the other parts. Leave out page furniture: the item ID printed \
sideways in the margin, and navigation lines such as "This ... question continues on the next page." or "End of ... \
section.". Write math in plain-text notation: fractions as 1/2, mixed numbers as 1 1/2, powers as x^2, square roots \
as sqrt(x), and the symbols ×, ÷, −, =, <, >, ≤, ≥, °, π. Write an empty box in a number sentence as □ and an answer \
blank as ____, keeping labels such as "Answer:". Write tables as Markdown tables and bulleted lists as "- " lines. \
Replace each picture, diagram, graph, grid, or map with "[Image: <short description>]". Separate paragraphs with a \
newline.
- options: For a multiple-choice item, one entry per response option in printed order, labeled with its circled \
letter, in the same notation as the stem; a pictured option is "[Image: <description>]". Empty for \
constructed-response items.
- scoring_guide_item_id: The item ID printed in the header of the scoring guide ("Item: ..."); null for \
multiple-choice items.
- scoring_guide: For a constructed-response item, each category of the scoring guide in printed order (e.g. \
"Correct Response", "Partial Response", "Incorrect Response", "Nonresponse"), with its heading verbatim, except that \
"Partially Correct Response" is written "Partial Response", and each code row under it: the score code and its \
response description, verbatim, in the same notation as the stem. A picture that illustrates a code, such as a \
sample answer in the row below it, belongs to that code's response, as "[Image: <description>]". Empty for \
multiple-choice items.
- scoring_notes: Any scoring-guide text outside the code table, such as a note; null if none.
- has_figure: Whether this item's stem or options contain any picture, diagram, graph, grid, map, or drawn shape. A \
table of text or numbers alone does not count.
- figure_description: If has_figure, a precise, objective description of every visual element in this item's stem \
and options, giving all the information a student would need to answer without seeing it: counts, labels, \
measurements, positions, grid coordinates, bar heights, shading, and orientation. Describe what is drawn; do not \
solve the item or reveal the answer. Null if has_figure is false.
"""


def t07g8_image_part(xref):
    return ri.embedded_image_part(t07g8_pdf, xref)


def t07g8_request_content(batch):
    """The user message for a batch: each item's booklet page images (once per multi-part item) and scoring guide."""
    content = [ri.text_part(f"Transcribe these {len(batch)} items, in this order: {', '.join(i['item_id'] for i in batch)}.")]
    shown_units = set()
    for item in batch:
        points = f"{item['max_points']} point{'s' if item['max_points'] > 1 else ''}"
        description = f"{item['item_type'].replace('_', '-')}, worth {points}"
        if item["option_labels"]:
            description += f", options {item['option_labels'][0]} to {item['option_labels'][-1]}"
        if item["parts"]:
            if item["parts"][0] not in shown_units:
                shown_units.add(item["parts"][0])
                content.append(ri.text_part(f"Booklet pages of the multi-part item with parts {', '.join(item['parts'])}, in order:"))
                content += [t07g8_image_part(xref) for xref in item["page_images"]]
            content.append(ri.text_part(f"Item {item['item_id']}: part {item['part']} of that multi-part item, {description}."))
        else:
            content.append(ri.text_part(f"Item {item['item_id']}: {description}. Booklet page:"))
            content.append(t07g8_image_part(item["page_images"][0]))
        if item["scoring_image"]:
            content.append(ri.text_part(f"Scoring guide of item {item['item_id']}:"))
            content.append(t07g8_image_part(item["scoring_image"]))
    return content


def t07g8_problems(item, t, returned):
    """Disagreements between a transcription and the item's documentation; empty if the transcription is usable."""
    problems = []
    if not t.stem.strip():
        problems.append("empty stem")
    if item["item_type"] == "multiple_choice":
        if [o.label for o in t.options] != item["option_labels"]:
            problems.append(f"option labels {[o.label for o in t.options]}, expected {item['option_labels']}")
        if t.scoring_guide or t.scoring_guide_item_id:
            problems.append("multiple-choice item has a scoring guide")
    else:
        if t.options:
            problems.append("constructed-response item has options")
        if t.scoring_guide_item_id != item["item_id"]:
            problems.append(f"scoring guide headed {t.scoring_guide_item_id!r}")
        expected = [c for c, codes in T07G8_CATEGORY_CODES.items() if item["max_points"] in codes]
        categories = [t07g8_category(c.category) for c in t.scoring_guide]
        if categories != expected:
            problems.append(f"scoring categories {categories}, expected {expected} for {item['max_points']} points")
        for c, category in zip(t.scoring_guide, categories):
            allowed = T07G8_CATEGORY_CODES.get(category, {}).get(item["max_points"], range(0))
            codes = [code.code for code in c.codes]
            if not codes or any(code not in allowed for code in codes):
                problems.append(f"{c.category} codes {codes}")
    if item["part"]:
        labels = re.findall(r"(?m)^([A-Z])\.\s", t.stem)
        if labels != [item["part"]]:
            problems.append(f"part labels {labels} in the stem")
    if t.has_figure != bool(t.figure_description):
        problems.append(f"has_figure={t.has_figure} but figure_description={t.figure_description!r}")
    return problems


t07g8_transcriber = ri.Transcriber(
    api_key=globals().get("openrouter_api_key"),
    model=T07G8_LLM_MODEL,
    cache_dir=T07G8_LLM_CACHE_DIR,
    schema=T07G8Transcription,
    instructions=T07G8_TRANSCRIPTION_INSTRUCTIONS,
)
t07g8_transcriptions = t07g8_transcriber.run(
    t07g8_items,
    request_content=t07g8_request_content,
    problems=t07g8_problems,
    items_per_request=T07G8_ITEMS_PER_REQUEST,
    unit=lambda item: (item["block"], item["block_seq"]),
    max_requests=T07G8_MAX_REQUESTS,
)
print(f"All {len(t07g8_transcriptions)} items are transcribed")

### Validate and write the JSONL

`scoring_guide` keeps each printed category with its response descriptions in printed order; the two-digit score codes are used only by the checks above, as in the 2007 grade 4 file.
The cell reports which items have percent correct from the NCES workbook.

In [ ]:
# Percent correct from the NCES statistics workbook for the TIMSS 2007 grade 8 released items ("TIMSS Grade 8
# mathematics item statistics" under TIMSS 2007 on https://nces.ed.gov/timss/released-questions.asp).
T07G8_STATISTICS_URL = "https://nces.ed.gov/timss/xls/TIMSS_Grade8_Math_Released_Item_Statistics.xlsx"
t07g8_statistics = ri.nces_2007_percent_correct(
    ri.download(T07G8_STATISTICS_URL, T07G8_DATA_DIR / "raw" / "TIMSS_Grade8_Math_Released_Item_Statistics.xlsx"),
    "Gr. 8 Mathematics",
)
t07g8_ids = [item["item_id"] for item in t07g8_items]
print(
    f"Percent correct for {len(set(t07g8_ids) & set(t07g8_statistics))} of {len(t07g8_ids)} items; "
    f"none for {sorted(set(t07g8_ids) - set(t07g8_statistics))}; "
    f"in the workbook but not the booklet: {sorted(set(t07g8_statistics) - set(t07g8_ids))}"
)
t07g8_no_statistics = {"pct_correct_intl_avg": None, "pct_correct_usa": None, "pct_correct": None}

t07g8_records = []
for item in t07g8_items:
    t = t07g8_transcriptions[item["item_id"]]
    statistics = t07g8_statistics.get(item["item_id"], t07g8_no_statistics)
    t07g8_records.append(
        {
            "item_id": item["item_id"],
            "item_label": item["item_label"],
            "assessment": "TIMSS 2007",
            "grade": 8,
            "subject": "Mathematics",
            "content_domain": item["content_domain"],
            "main_topic": item["main_topic"],
            "cognitive_domain": item["cognitive_domain"],
            "item_type": item["item_type"],
            "max_points": item["max_points"],
            "stem": ri.tidy(t.stem),
            "options": [{"label": o.label, "text": ri.tidy(o.text)} for o in t.options] or None,
            "correct_option": item["correct_option"],
            # The response descriptions under each heading; the two-digit score codes were used for the checks.
            "scoring_guide": [
                {"category": c.category, "descriptors": [ri.tidy(code.response) for code in c.codes]}
                for c in t.scoring_guide
            ]
            or None,
            "scoring_notes": t.scoring_notes,
            "has_figure": t.has_figure,
            "figure_description": t.figure_description,
            "accessibility_text": None,
            **statistics,
            "booklet_page": item["booklet_page"],
            "pdf_page": item["pdf_page"],
            "source_url": T07G8_ZIP_URL,
            "transcription_model": T07G8_LLM_MODEL,
        }
    )

t07g8_released_items = ri.write_jsonl(t07g8_records, T07G8_OUTPUT_PATH)
print(f"Wrote {len(t07g8_released_items)} items to {T07G8_OUTPUT_PATH.resolve()}")
display(t07g8_released_items[["item_id", "content_domain", "main_topic", "cognitive_domain", "item_type", "stem", "options"]].head())

## TIMSS 2003 grade 8 released mathematics items

Build `data/timss03_g8_math_released_items.jsonl` from NCES's [Complete 2003 TIMSS 8 mathematics set](https://nces.ed.gov/timss/released-questions.asp).

The booklet is laid out like the 2003 grade 4 one, and also has no accessibility transcription (`accessibility_text` is null), with these differences:

- The classification follows the TIMSS 2003 grade 8 framework: `content_domain` is one of Number, Algebra, Measurement, Geometry, and Data, and `cognitive_domain` one of Knowing Facts and Procedures, Using Concepts, Solving Routine Problems, and Reasoning. The header prints them in small capitals that come out of the text layer in mixed case, so they are matched ignoring case.
- The pages of some multi-part items classify each part on a header row of its own, which starts with the part letter.
- Scoring-guide category headings are the bold 9-point lines of the scoring area: "Correct Response" (or, for M022008, "Correct"), "Partial Response" for 2-point items, and "Incorrect Response".
- Response-option letters are drawn in a font of their own (`T03Options`), as circled lower-case letters.
- The page of M032670, "The number closest to 10", shows the box of another item, M032760 (Red and Black Tiles, parts A-C), under M032670's label, classification, and key. M032670's item box comes from IEA's [TIMSS 2003 Released Items](https://timssandpirls.bc.edu/timss2003i/released.html) (`T03_RELEASED_M8.pdf`) instead, whose page shows the item as an image and must classify it as the booklet does.
- Some tables leave out an education system (such as Ghana, or Latvia for M022227A-C, which lists Latvia without results).

Extraction is hybrid, as for grade 4:

- **Deterministic, from the PDF:** item ID, item label (from the item index at the back), content domain, main topic, cognitive domain, answer key, response-option letters, scoring-guide category headings (and from them the maximum points), and percent correct by education system with the significance markers; the item list, content domains, and pages are checked against the item index, and the item type, key, maximum points, and number of options against the 2003 item information (`Bitinfm3.xls` in `t03_items.zip`), which also gives the number of options of M032670.
- **`gemini-3.8-flash` through OpenRouter:** `stem`, `options`, `scoring_guide`, `scoring_notes`, `has_figure`, and `figure_description`, read from rendered images of the item box and of the answer key or scoring guide, then checked against the deterministic fields.

`figure_description` is model-generated.

### Deterministic parse of the item pages

Every item starts on a page with the classification header (y 70-115, with the main topic from x 220 and the cognitive domain from x 391), the item label, the item box from y 148 down to the "Item Number:" line, the answer key or scoring guide below it, and the "Overall Percent Correct" table with its legend in the right column (x > 420).
Continuation pages lack the table: a "SCORING" heading on one continues the item's scoring guide, and otherwise it shows sample student responses, which are skipped.
As for grade 4, words come from `ri.visible_words`, which leaves out any text hidden under opaque boxes.
The item index lists every part of a multi-part item under the content domain of the first part.

In [ ]:
import re
from itertools import groupby
from pathlib import Path

import pandas as pd
import pymupdf
from IPython.display import display

from timss_1 import released_items as ri

# The "Complete 2003 TIMSS 8 mathematics set" link on https://nces.ed.gov/timss/released-questions.asp
T03G8_PDF_URL = "https://nces.ed.gov/timss/pdf/TIMSS2003_G8_Math.pdf"
T03G8_DATA_DIR = Path.cwd() / "data"
T03G8_PDF_PATH = ri.download(T03G8_PDF_URL, T03G8_DATA_DIR / "raw" / "TIMSS2003_G8_Math.pdf")
T03G8_OUTPUT_PATH = T03G8_DATA_DIR / "timss03_g8_math_released_items.jsonl"
t03g8_pdf = pymupdf.open(T03G8_PDF_PATH)
# The item information of the TIMSS 2003 International Database ("Items" on
# https://timssandpirls.bc.edu/timss2003i/userguide.html), which must agree with the booklet.
T03G8_ITEMS_ZIP_URL = "https://timssandpirls.bc.edu/timss2003i/PDF/t03_items.zip"
# IEA's TIMSS 2003 Released Items, "Eighth Grade Mathematics" on https://timssandpirls.bc.edu/timss2003i/released.html
T03G8_IEA_PDF_URL = "https://timssandpirls.bc.edu/PDF/T03_RELEASED_M8.pdf"
# Items whose box comes from IEA's released items: the NCES page of M032670 shows the box of another item, M032760
# (Red and Black Tiles, parts A-C), under M032670's label, classification, and key.
T03G8_IEA_ITEMS = ["M032670"]

T03G8_VISIBLE_DICT = pymupdf.TEXTFLAGS_DICT | pymupdf.TEXT_IGNORE_ACTUALTEXT
T03G8_ITEM_ID = re.compile(r"M\d{6}[A-Z]?")
T03G8_INDEX_PAGES = [t03g8_pdf.page_count - 2, t03g8_pdf.page_count - 1]  # "Item Index 2003" and its continuation
T03G8_INDEX_RIGHT_COLUMN_X = 260  # between the left column's page numbers and the right column's item IDs
# Item page layout, in points from the top-left corner, as in the grade 4 booklet.
T03G8_HEADER_Y = (70, 115)  # classification rows under "Content Domain", "Main Topic", and "Cognitive Domain"
T03G8_TOPIC_X = 220  # left edge of the "Main Topic" column
T03G8_COGNITIVE_X = 391  # left edge of the "Cognitive Domain" column
T03G8_ITEM_BOX_TOP = 148  # just below the item label
T03G8_RIGHT_COLUMN_X = 420  # the percent-correct table and its legend lie to the right
T03G8_PERCENT_X = 520  # left edge of the table's percent column
T03G8_MARKER_X = 540  # left edge of the table's significance-marker column
T03G8_FOOTER_Y = 750  # the footer: copyright notice and page number (scoring guides can reach y 748)

T03G8_CONTENT_DOMAINS = ["Number", "Algebra", "Measurement", "Geometry", "Data"]
# Main topics printed in this booklet, by content domain, used to validate the parsed header.
T03G8_MAIN_TOPICS = {
    "Number": ["Whole Numbers", "Fractions and Decimals", "Integers", "Ratio, Proportions, and Percent"],
    "Algebra": ["Patterns", "Algebraic Expressions", "Equations and Formulas", "Relationships"],
    "Measurement": ["Attributes and Units", "Tools, Techniques, and Formulas"],
    "Geometry": [
        "Lines and Angles",
        "Two- and Three-Dimensional Shapes",
        "Congruence and Similarity",
        "Locations and Spatial Relationships",
        "Symmetry and Transformations",
    ],
    "Data": ["Data Interpretation", "Uncertainty and Probability"],
}
T03G8_COGNITIVE_DOMAINS = ["Knowing Facts and Procedures", "Using Concepts", "Solving Routine Problems", "Reasoning"]
# Category headings of a scoring guide, in printed order: 1-point items, then 2-point items. The headings are the
# bold 9-point lines of the scoring area; M022008's guide prints "Correct" without "Response".
T03G8_SCORING_GUIDES = [
    ["Correct Response", "Incorrect Response"],
    ["Correct", "Incorrect Response"],
    ["Correct Response", "Partial Response", "Incorrect Response"],
]
T03G8_HEADING_SIZE = 9
T03G8_SIGNIFICANCE = {"Higher": "higher", "Not different": None, "Lower": "lower"}  # legend label -> vs_intl_avg


def t03g8_canonical(text, names):
    """The name in `names` that `text` spells. The header's small capitals come out of the PDF in mixed case
    ("AlgebRA"), so case is ignored."""
    match = next((name for name in names if name.casefold() == text.casefold()), None)
    if match is None:
        raise ValueError(f"unexpected classification {text!r}, expected one of {names}")
    return match


def t03g8_classification(words, part):
    """(content domain, main topic, cognitive domain) from the page header. Most pages print one row, whose cells
    may wrap; the pages of some multi-part items print a row per part, starting with its part label ("B.")."""
    header_lines = ri.lines(w for w in words if T03G8_HEADER_Y[0] <= w[1] < T03G8_HEADER_Y[1])
    if len(header_lines) > 1 and all(re.fullmatch(r"[A-Za-z]\.", line[0][4]) for line in header_lines):
        header_lines = [{line[0][4][0].upper(): line[1:] for line in header_lines}[part]]
    cells = []
    for x0, x1 in ((0, T03G8_TOPIC_X), (T03G8_TOPIC_X, T03G8_COGNITIVE_X), (T03G8_COGNITIVE_X, float("inf"))):
        cell = ""
        for line in header_lines:
            line_text = ri.text(w for w in line if x0 <= w[0] < x1)
            if line_text:  # a line ending in a hyphen ("Two- and Three-") continues the word on the next line
                cell += line_text if not cell or cell.endswith("-") else f" {line_text}"
        cells.append(cell)
    content_domain = t03g8_canonical(cells[0], T03G8_CONTENT_DOMAINS)
    main_topic = t03g8_canonical(cells[1], T03G8_MAIN_TOPICS[content_domain])
    return content_domain, main_topic, t03g8_canonical(cells[2], T03G8_COGNITIVE_DOMAINS)


def t03g8_pct_correct(words, item_id):
    """Rows of the "Overall Percent Correct" table, each row's marker translated with the legend below the table."""
    right = [w for w in words if w[0] >= T03G8_RIGHT_COLUMN_X]
    table_top = next(w[3] for w in right if w[4] == "Overall")
    legend_top = next(w[1] for w in right if w[4] == "Country")  # "Country average vs. International average:"
    legend = {}  # marker glyph -> vs_intl_avg, from legend lines such as "Higher ▲"
    for line in ri.lines(w for w in right if w[1] > legend_top):
        if ri.text(line[:-1]) in T03G8_SIGNIFICANCE:
            legend[line[-1][4]] = T03G8_SIGNIFICANCE[ri.text(line[:-1])]
    assert len(legend) == len(T03G8_SIGNIFICANCE), f"{item_id}: legend {legend}"
    rows = []
    for line in ri.lines(w for w in right if table_top < w[1] < legend_top):
        name = ri.text(w for w in line if w[0] < T03G8_PERCENT_X)
        values = [w[4] for w in line if T03G8_PERCENT_X <= w[0] < T03G8_MARKER_X]
        markers = [w[4] for w in line if w[0] >= T03G8_MARKER_X]
        if not values and not markers:  # a country listed without a result (Latvia for M022227A-C)
            rows.append({"education_system": name, "pct_correct": None, "vs_intl_avg": None, "benchmarking": False})
            continue
        assert len(values) == 1 and values[0].isdigit(), f"{item_id}: percent-correct row {ri.text(line)!r}"
        if name != "International average":  # the international average row is compared with nothing
            assert len(markers) == 1 and markers[0] in legend, f"{item_id}: percent-correct row {ri.text(line)!r}"
        rows.append(
            {
                "education_system": name,
                "pct_correct": int(values[0]),
                "vs_intl_avg": legend.get(markers[0]) if markers else None,
                "benchmarking": False,  # the 2003 booklet lists no benchmarking participants
            }
        )
    return rows


def t03g8_option_labels(page, box):
    """Letters of the circled response options in the item box, drawn as a, b, c, ... in the T03Options font."""
    return sorted(
        {
            span["text"].strip().upper()
            for block in page.get_text("dict", flags=T03G8_VISIBLE_DICT, clip=box)["blocks"]
            for line in block.get("lines", [])
            for span in line["spans"]
            if span["font"] == "T03Options" and span["text"].strip() in set("abcde")
        }
    )


def t03g8_scoring_area(page, words, top, right):
    """The answer key or scoring guide on a page, from `top` down to any sample student responses or the footer:
    its clip, as (page index, rectangle), and the scoring-guide category headings in it, the bold 9-point lines."""
    page_lines = ri.lines(w for w in words if top <= w[1] < T03G8_FOOTER_Y and w[0] < right)
    bottom = next(
        (line[0][1] for line in page_lines if ri.text(line).casefold().startswith("student responses")), T03G8_FOOTER_Y
    )
    clip = pymupdf.Rect(40, top, right, bottom)
    headings = []
    for block in page.get_text("dict", flags=T03G8_VISIBLE_DICT, clip=clip)["blocks"]:
        for line in block.get("lines", []):
            spans = [span for span in line["spans"] if span["text"].strip()]
            if spans and all("Bold" in span["font"] and round(span["size"]) == T03G8_HEADING_SIZE for span in spans):
                headings.append("".join(span["text"] for span in spans).strip())
    return (page.number, clip), headings


def t03g8_item_index(pages):
    """The item index at the back of the booklet as {item ID: (content domain, item label, booklet page)}, column
    by column; a section continues from one page to the next."""
    index, domain = {}, None
    for page in pages:
        words = page.get_text("words", flags=ri.VISIBLE_WORDS)
        for column in (
            [w for w in words if w[0] < T03G8_INDEX_RIGHT_COLUMN_X],
            [w for w in words if w[0] >= T03G8_INDEX_RIGHT_COLUMN_X],
        ):
            for line in ri.lines(column):
                line_text = ri.text(line)
                if line_text in T03G8_CONTENT_DOMAINS:
                    domain = line_text
                elif match := re.fullmatch(rf"({T03G8_ITEM_ID.pattern}) (.+) (\d+)", line_text):
                    index[match[1]] = (domain, match[2], int(match[3]))
    return index


def t03g8_item_number(page_lines):
    """The item ID printed after "Item Number:" and the top of its line, or (None, None) if the page has none."""
    for line in page_lines:
        if ri.text(line[:2]) == "Item Number:":
            assert len(line) == 3 and T03G8_ITEM_ID.fullmatch(line[2][4]), ri.text(line)
            return line[2][4], line[0][1]
    return None, None


def t03g8_scoring_top(words):
    """Top of the "SCORING" heading (printed in small capitals as "sCORInG" on some pages), or None."""
    return next((w[1] for w in words if w[4].casefold() == "scoring" and w[0] < 60), None)


# Item pages, each followed by any continuation pages: the rest of a scoring guide, or sample student responses.
t03g8_items = []
for page in t03g8_pdf:
    words = ri.visible_words(page)
    page_lines = ri.lines(w for w in words if w[0] < T03G8_RIGHT_COLUMN_X)  # the left column, item box and below
    item_id, number_y = t03g8_item_number(page_lines)
    if item_id is None:
        continue
    scoring_top = t03g8_scoring_top(words)
    if not any(w[4] == "Overall" for w in words):  # a continuation page, without the "Overall Percent Correct" table
        assert t03g8_items and t03g8_items[-1]["item_id"] == item_id, f"page {page.number + 1} continues {item_id}"
        if scoring_top is not None:
            title_top = next(line[0][1] for line in page_lines if ri.text(line).endswith("(continued)"))
            clip, headings = t03g8_scoring_area(page, words, title_top - 2, page.rect.width - 40)
            t03g8_items[-1]["scoring_clips"].append(clip)
            t03g8_items[-1]["scoring_categories"] += headings
        continue

    content_domain, main_topic, cognitive_domain = t03g8_classification(words, item_id[7:] or None)
    key_line = next((line for line in page_lines if ri.text(line[:2]) == "Correct Response:"), None)
    box = pymupdf.Rect(40, T03G8_ITEM_BOX_TOP, T03G8_RIGHT_COLUMN_X, number_y - 1)
    pct_rows = t03g8_pct_correct(words, item_id)
    intl_avg = next(r for r in pct_rows if r["education_system"] == "International average")
    usa = next(r for r in pct_rows if r["education_system"] == "United States")
    item = {
        "item_id": item_id,
        "content_domain": content_domain,
        "main_topic": main_topic,
        "cognitive_domain": cognitive_domain,
        "item_type": "multiple_choice" if key_line else "constructed_response",
        "option_labels": t03g8_option_labels(page, box),
        "correct_option": key_line[2][4].upper() if key_line else None,  # small capitals, as for the header
        "pct_correct_intl_avg": intl_avg["pct_correct"],
        "pct_correct_usa": usa["pct_correct"],
        "pct_correct": [r for r in pct_rows if r is not intl_avg],
        "booklet_page": int(  # the page number, right of the copyright notice
            max((w for w in words if w[1] >= T03G8_FOOTER_Y and w[0] < T03G8_RIGHT_COLUMN_X), key=lambda w: w[0])[4]
        ),
        "pdf_page": page.number + 1,
        "item_box": (page.number, box),
        "scoring_clips": [],
        "scoring_categories": [],
        # Words in the item box, used later to check the model's transcription.
        "_item_box_text": ri.text(w for line in ri.lines(w for w in words if ri.center(w) in box) for w in line),
    }
    if key_line or scoring_top is not None:  # the answer key or scoring guide starts on the item page
        clip, headings = t03g8_scoring_area(page, words, number_y - 1, T03G8_RIGHT_COLUMN_X)
        item["scoring_clips"].append(clip)
        item["scoring_categories"] += headings
    t03g8_items.append(item)

# The parts of a multi-part item (IDs ending in A, B, C, ...) share one item box, printed on every part's page.
for _, unit in groupby(t03g8_items, key=lambda item: item["item_id"][:7]):
    unit = list(unit)
    for item in unit:
        item["parts"] = [i["item_id"] for i in unit] if len(unit) > 1 else None
        item["part"] = item["item_id"][7:] if len(unit) > 1 else None
    assert len(unit) == 1 or all(item["part"] for item in unit), f"{unit[0]['item_id']}: parts without part letters"
    assert len({item["_item_box_text"] for item in unit}) == 1, f"{unit[0]['item_id']}: parts with different item boxes"

# The item index gives each item's label and checks the item pages against it; it lists every part of a
# multi-part item under the content domain of the first part.
t03g8_index = t03g8_item_index([t03g8_pdf[number] for number in T03G8_INDEX_PAGES])
assert [item["item_id"] for item in t03g8_items] == list(t03g8_index), "item pages and item index list different items"
t03g8_domains = {item["item_id"]: item["content_domain"] for item in t03g8_items}
for item in t03g8_items:
    domain, label, booklet_page = t03g8_index[item["item_id"]]
    unit_domain = t03g8_domains[(item["parts"] or [item["item_id"]])[0]]
    assert (domain, booklet_page) == (unit_domain, item["booklet_page"]), f"{item['item_id']}: index differs"
    item["item_label"] = label
    if item["item_type"] == "multiple_choice":
        assert item["option_labels"] in ([], list("ABCD"), list("ABCDE")), f"{item['item_id']}: {item['option_labels']}"
        assert not item["option_labels"] or item["correct_option"] in item["option_labels"], item["item_id"]
        assert not item["scoring_categories"], f"{item['item_id']}: multiple-choice item with a scoring guide"
    else:
        assert item["scoring_categories"] in T03G8_SCORING_GUIDES, f"{item['item_id']}: {item['scoring_categories']}"
    item["max_points"] = 2 if "Partial Response" in item["scoring_categories"] else 1

# The item information must agree on each item's type, key, maximum points, and number of printed options. It also
# gives the number of options of the items that come from IEA's released items, whose page must classify them as the
# booklet does.
t03g8_information = ri.timss_2003_math_item_information(
    ri.download(T03G8_ITEMS_ZIP_URL, T03G8_DATA_DIR / "raw" / "t03_items.zip"), 8
).set_index("Unique ID")
t03g8_iea_pdf = pymupdf.open(ri.download(T03G8_IEA_PDF_URL, T03G8_DATA_DIR / "raw" / "T03_RELEASED_M8.pdf"))
for item in t03g8_items:
    info = t03g8_information.loc[item["item_id"]]
    assert info["Released Item"] == "Yes", f"{item['item_id']}: not released in the item information"
    assert (info["ItemType"], info["Key"], int(info["Maximum Points"])) == (
        "MC" if item["item_type"] == "multiple_choice" else "CR",
        item["correct_option"] or "X",
        item["max_points"],
    ), f"{item['item_id']}: item information differs from the booklet"
    option_count = int(info["Response Options"])
    assert len(item["option_labels"]) in (0, option_count), f"{item['item_id']}: {option_count} options in the item information"
    if item["item_id"] in T03G8_IEA_ITEMS:
        fields, item["iea_images"] = ri.iea_2003_item(t03g8_iea_pdf, item["item_id"])
        booklet = [item["content_domain"], item["main_topic"], item["cognitive_domain"], item["correct_option"]]
        assert [fields[name].casefold() for name in ri.IEA_2003_FIELDS] == [value.casefold() for value in booklet], fields
        item["option_labels"] = list("ABCDE"[:option_count])
        item["_item_box_text"] = ""  # the booklet's box shows another item

t03g8_systems = {r["education_system"] for item in t03g8_items for r in item["pct_correct"]}
t03g8_missing_systems = {
    item["item_id"]: sorted(t03g8_systems - {r["education_system"] for r in item["pct_correct"]})
    for item in t03g8_items
    if len(item["pct_correct"]) < len(t03g8_systems)
}
t03g8_items_df = pd.DataFrame(t03g8_items)
print(f"Parsed {len(t03g8_items)} items")
display(t03g8_items_df.groupby(["content_domain", "item_type"]).size().unstack(fill_value=0))
display(t03g8_items_df.groupby(["item_type", "max_points"]).size())
print("Items whose box comes from IEA's released items:", T03G8_IEA_ITEMS)
print(
    "Multiple-choice items whose response options are not printed in the booklet:",
    [item["item_id"] for item in t03g8_items if item["item_type"] == "multiple_choice" and not item["option_labels"]],
)
print(f"{len(t03g8_systems)} education systems; tables without some of them: {t03g8_missing_systems}")

### Transcribe stems, options, and scoring guides with Gemini

`gemini-3.8-flash` is called through OpenRouter by `ri.Transcriber`, as in the 2011 grade 8 section, with up to `T03G8_ITEMS_PER_REQUEST` items per request in booklet order, keeping the parts of a multi-part item together, which makes 26 requests for the whole set (about $0.61 in total on 2026-10-03).
For each item it sends a short description (type, points, and option letters) and the item box (once per multi-part item) and the answer key or scoring guide, rendered at 300 DPI and trimmed to the printed content; for M032670 it sends the item's image from IEA's released items instead of the booklet's box.
Cached transcriptions are checked again on every run, so M032670's first transcription, made from the wrong box, fails the option check and is requested again.

A transcription is kept only if it passes the checks in `t03g8_problems`: the option letters and answer key of a multiple-choice item, the scoring-guide category headings of a constructed-response item, and only the item's own part label in the stem of a part.
Kept transcriptions are cached per item under `data/cache/item_transcriptions/timss03_g8_math/gemini-3.8-flash-v1/`; set `T03G8_MAX_REQUESTS` to send fewer requests, and bump `T03G8_PROMPT_VERSION` after editing the prompt or schema.

In [ ]:
from pydantic import BaseModel, Field

T03G8_LLM_MODEL = "google/gemini-3.8-flash"
T03G8_PROMPT_VERSION = 1
T03G8_LLM_CACHE_DIR = (
    T03G8_DATA_DIR
    / "cache"
    / "item_transcriptions"
    / "timss03_g8_math"
    / f"{T03G8_LLM_MODEL.split('/')[-1]}-v{T03G8_PROMPT_VERSION}"
)
T03G8_ITEMS_PER_REQUEST = 4
T03G8_MAX_REQUESTS = None  # a number sends at most that many requests in one run, such as 1 for a test


class T03G8Option(BaseModel):
    label: str = Field(description="Option letter as printed, e.g. 'A'.")
    text: str = Field(description="Option content in the same notation as the stem.")


class T03G8ScoringCategory(BaseModel):
    category: str = Field(description="Category heading as printed, e.g. 'Correct Response'.")
    descriptors: list[str] = Field(description="Each bullet under the heading, in printed order.")


class T03G8Transcription(BaseModel):
    item_id: str
    stem: str
    options: list[T03G8Option]
    answer_key: str | None
    scoring_guide: list[T03G8ScoringCategory]
    scoring_notes: str | None
    has_figure: bool
    figure_description: str | None


T03G8_TRANSCRIPTION_INSTRUCTIONS = """\
You transcribe items from the TIMSS 2003 grade 8 released mathematics booklet into structured data.

Each request covers several items, listed by ID. For each item you receive:
1. An image of the boxed item exactly as students saw it. The parts of a multi-part item (IDs ending in A, B, C, ...) \
share one item box, which is shown once, before the parts: an introduction shared by all parts, followed by every \
part, each starting with its part label (e.g. "B.").
2. One or more images of what the booklet prints below the item box: the item number and either the multiple-choice \
answer key ("Correct Response: X") or the scoring guide under "SCORING".

The images are the only source. Transcribe what is printed, verbatim; never add, correct, or solve anything.

Return one entry per item, in the order given:
- item_id: The item ID as given.
- stem: Everything in the item box for this item except multiple-choice response options, in reading order, with the \
wording verbatim. For a part of a multi-part item, the shared introduction (everything before the first part label) \
followed by this part only, starting with its part label; leave out the other parts. Write math in plain-text \
notation: fractions as 1/2, mixed numbers as 1 1/2, powers as x^2, square roots as sqrt(x), and the symbols ×, ÷, −, \
=, <, >, ≤, ≥, °, π. Write an empty box in a number sentence as □ and an answer blank as ____, keeping labels such as \
"Answer:". Write tables as Markdown tables and bulleted lists as "- " lines. Replace each picture, diagram, graph, \
grid, or map with "[Image: <short description>]". Separate paragraphs with a newline.
- options: For a multiple-choice item, one entry per response option in printed order, labeled with its circled \
letter (A, B, C, ...), in the same notation as the stem; a pictured option is "[Image: <description>]". Empty for \
constructed-response items, and for a multiple-choice item whose options are not printed.
- answer_key: The letter printed in the "Correct Response:" box of a multiple-choice item; null for \
constructed-response items.
- scoring_guide: For a constructed-response item, each category under "SCORING" in printed order, with its heading \
verbatim (e.g. "Correct Response", "Partial Response", "Incorrect Response") and each bullet verbatim as a \
descriptor, in the same notation as the stem. Write a picture of a sample answer, whether it is a bullet of its own \
or follows a bullet's text, as "[Image: <description>]". Empty for multiple-choice items.
- scoring_notes: Any scoring-guide text that is not a heading or bullet; null if none.
- has_figure: Whether this item's stem or options contain any picture, diagram, graph, grid, map, or drawn shape. A \
table of text or numbers alone does not count.
- figure_description: If has_figure, a precise, objective description of every visual element in this item's stem \
and options, giving all the information a student would need to answer without seeing it: counts, labels, \
measurements, positions, grid coordinates, bar heights, shading, and orientation. Describe what is drawn; do not \
solve the item or reveal the answer. Null if has_figure is false.
"""


def t03g8_page_part(clip):
    page_index, rect = clip
    return ri.page_area_part(t03g8_pdf[page_index], rect)


def t03g8_item_box(item):
    """The item box as image parts: the booklet's, or the item's images in IEA's released items."""
    if item.get("iea_images"):
        return [ri.embedded_image_part(t03g8_iea_pdf, xref) for xref in item["iea_images"]]
    return [t03g8_page_part(item["item_box"])]


def t03g8_description(item):
    if item["item_type"] == "constructed_response":
        return f"constructed-response, worth {item['max_points']} point{'s' if item['max_points'] > 1 else ''}"
    labels = item["option_labels"]
    options = f"options {labels[0]} to {labels[-1]}" if labels else "its response options are not printed"
    return f"multiple-choice ({options}), worth 1 point"


def t03g8_request_content(batch):
    """The user message for a batch: each item's box (once per multi-part item) and what is printed below it."""
    content = [ri.text_part(f"Transcribe these {len(batch)} items, in this order: {', '.join(i['item_id'] for i in batch)}.")]
    shown_units = set()
    for item in batch:
        if item["parts"]:
            if item["parts"][0] not in shown_units:
                shown_units.add(item["parts"][0])
                content.append(ri.text_part(f"Item box of the multi-part item with parts {', '.join(item['parts'])}:"))
                content += t03g8_item_box(item)
            content.append(
                ri.text_part(
                    f"Item {item['item_id']}: part {item['part']} of that multi-part item, {t03g8_description(item)}. "
                    "Printed below its item box:"
                )
            )
        else:
            content.append(ri.text_part(f"Item {item['item_id']}: {t03g8_description(item)}. Item box:"))
            content += t03g8_item_box(item)
            content.append(ri.text_part("Printed below the item box:"))
        content += [t03g8_page_part(clip) for clip in item["scoring_clips"]]
    return content


def t03g8_problems(item, t, returned):
    """Disagreements between a transcription and the deterministic parse; empty if the transcription is usable."""
    problems = []
    if not t.stem.strip():
        problems.append("empty stem")
    if item["item_type"] == "multiple_choice":
        if [o.label for o in t.options] != item["option_labels"]:
            problems.append(f"option labels {[o.label for o in t.options]}, expected {item['option_labels']}")
        if t.answer_key != item["correct_option"]:
            problems.append(f"answer key {t.answer_key!r}, expected {item['correct_option']!r}")
        if t.scoring_guide:
            problems.append("multiple-choice item has a scoring guide")
    else:
        if t.options or t.answer_key:
            problems.append("constructed-response item has options or an answer key")
        categories = [c.category for c in t.scoring_guide]
        if categories != item["scoring_categories"]:
            problems.append(f"scoring categories {categories}, expected {item['scoring_categories']}")
        if any(not c.descriptors for c in t.scoring_guide):
            problems.append("scoring category without descriptors")
    if item["part"]:
        labels = re.findall(r"(?m)^([A-Z])\.\s", t.stem)
        if labels != [item["part"]]:
            problems.append(f"part labels {labels} in the stem")
    if t.has_figure != bool(t.figure_description):
        problems.append(f"has_figure={t.has_figure} but figure_description={t.figure_description!r}")
    return problems


t03g8_transcriber = ri.Transcriber(
    api_key=globals().get("openrouter_api_key"),
    model=T03G8_LLM_MODEL,
    cache_dir=T03G8_LLM_CACHE_DIR,
    schema=T03G8Transcription,
    instructions=T03G8_TRANSCRIPTION_INSTRUCTIONS,
)
t03g8_transcriptions = t03g8_transcriber.run(
    t03g8_items,
    request_content=t03g8_request_content,
    problems=t03g8_problems,
    items_per_request=T03G8_ITEMS_PER_REQUEST,
    unit=lambda item: item["item_id"][:7],
    max_requests=T03G8_MAX_REQUESTS,
)
print(f"All {len(t03g8_transcriptions)} items are transcribed")

### Validate and write the JSONL

The coverage report lists words drawn in each item box (all parts of a multi-part item together) that are missing from the transcribed stems and options; misses are usually figure labels.

In [ ]:
# Words drawn in each item box (all parts of a multi-part item together) that the transcriptions lack.
t03g8_units = {}
for item in t03g8_items:
    t = t03g8_transcriptions[item["item_id"]]
    unit = t03g8_units.setdefault(item["item_id"][:7], {"drawn": item["_item_box_text"], "transcribed": []})
    unit["transcribed"] += [t.stem, *(o.text for o in t.options)]
t03g8_coverage = ri.coverage(
    {unit_id: unit["drawn"] for unit_id, unit in t03g8_units.items()},
    {unit_id: " ".join(unit["transcribed"]) for unit_id, unit in t03g8_units.items()},
)
print(f"{len(t03g8_coverage)} item boxes have drawn words missing from the transcription")
if not t03g8_coverage.empty:
    display(t03g8_coverage)

t03g8_records = []
for item in t03g8_items:
    t = t03g8_transcriptions[item["item_id"]]
    t03g8_records.append(
        {
            "item_id": item["item_id"],
            "item_label": item["item_label"],
            "assessment": "TIMSS 2003",
            "grade": 8,
            "subject": "Mathematics",
            "content_domain": item["content_domain"],
            "main_topic": item["main_topic"],
            "cognitive_domain": item["cognitive_domain"],
            "item_type": item["item_type"],
            "max_points": item["max_points"],
            "stem": ri.tidy(t.stem),
            "options": [{"label": o.label, "text": ri.tidy(o.text)} for o in t.options] or None,
            "correct_option": item["correct_option"],
            "scoring_guide": [
                {"category": c.category, "descriptors": [ri.tidy(d) for d in c.descriptors]} for c in t.scoring_guide
            ]
            or None,
            "scoring_notes": t.scoring_notes,
            "has_figure": t.has_figure,
            "figure_description": t.figure_description,
            "accessibility_text": None,
            "pct_correct_intl_avg": item["pct_correct_intl_avg"],
            "pct_correct_usa": item["pct_correct_usa"],
            "pct_correct": item["pct_correct"],
            "booklet_page": item["booklet_page"],
            "pdf_page": item["pdf_page"],
            "source_url": T03G8_PDF_URL,
            "transcription_model": T03G8_LLM_MODEL,
        }
    )

t03g8_released_items = ri.write_jsonl(t03g8_records, T03G8_OUTPUT_PATH)
print(f"Wrote {len(t03g8_released_items)} items to {T03G8_OUTPUT_PATH.resolve()}")
display(t03g8_released_items[["item_id", "content_domain", "main_topic", "cognitive_domain", "item_type", "stem", "options"]].head())

## TIMSS 1999 grade 8 released mathematics items

Build `data/timss99_g8_math_released_items.jsonl` from NCES's [Complete 1999 TIMSS 8 mathematics set](https://nces.ed.gov/timss/released-questions.asp).

The booklet is laid out like the 1995 grade 4 one, one page per item, with the answer key or scoring guide below the item box, and has no accessibility transcription (`accessibility_text` is null).

- The classification follows the TIMSS 1999 framework, which has no topic areas (`main_topic` is null): `content_domain` is one of Fractions and Number Sense, Algebra, Measurement, Geometry, and Data Representation, Analysis, and Probability, and `cognitive_domain` (the performance expectation) one of Knowing, Using Routine Procedures, Using Complex Procedures, Investigating and Solving Problems, and Communicating and Reasoning.
- Item IDs are the 1999 booklet IDs: cluster letter, position, and part letter, such as B08 and T02A. The parameter section maps them to permanent IDs.
- The parts of a multi-part item share one item box, printed on every part's page, and each part's scoring guide starts with "A. Scoring for ...", "B. Scoring for ...".
- Many items are scanned pictures, with the option letters of a multiple-choice item inside the picture.
- The percent-correct markers are Wingdings arrows coded as control characters, which word extraction drops, so the table is read span by span. The booklet lists no benchmarking participants.

Extraction is hybrid, as for the 1995 grade 4 booklet:

- **Deterministic, from the PDF:** item ID, item label, content domain, cognitive domain, answer key, response-option letters printed as text, scoring-guide category headings (and from them the maximum points), and percent correct by education system with the significance markers; the item list, content domains, and pages are checked against the item index on the last page.
- **`gemini-3.8-flash` through OpenRouter:** `stem`, `options`, `scoring_guide`, `scoring_notes`, `has_figure`, and `figure_description`, read from rendered images of the item box and of the answer key or scoring guide, then checked against the deterministic fields.

`figure_description` is model-generated.

### Deterministic parse of the item pages

Every item page has the layout of the 1995 grade 4 booklet (coordinates in points, origin top-left): the classification header (y 75-115) with the content domain left of x 224 and the cognitive domain right of it, the item label (y 120-150), the item box from y 150 down to the "Item Number:" line, the answer key or scoring guide below it, and the "Overall Percent Correct" table with its legend in the right column (x > 420).
As in that booklet, the fonts declare very tall ascenders, so word boxes are fitted to the glyphs (`TEXT_ACCURATE_BBOXES`), and the item pages leave out the serial comma in "Analysis, and Probability", so classifications are matched ignoring case and punctuation.
Scoring-guide category headings are the bold 9-point lines of the scoring area: "Correct Response", "Partial Response" for 2-point items, and "Incorrect Response".

In [ ]:
import re
from itertools import groupby
from pathlib import Path

import pandas as pd
import pymupdf
from IPython.display import display

from timss_1 import released_items as ri

# The "Complete 1999 TIMSS 8 mathematics set" link on https://nces.ed.gov/timss/released-questions.asp
T99_PDF_URL = "https://nces.ed.gov/timss/pdf/TIMSS1999_G8_Math.pdf"
T99_DATA_DIR = Path.cwd() / "data"
T99_PDF_PATH = ri.download(T99_PDF_URL, T99_DATA_DIR / "raw" / "TIMSS1999_G8_Math.pdf")
T99_OUTPUT_PATH = T99_DATA_DIR / "timss99_g8_math_released_items.jsonl"
t99_pdf = pymupdf.open(T99_PDF_PATH)

# As in the 1995 booklet, the fonts declare very tall ascenders and descenders, so word boxes are fitted to the glyphs.
T99_WORDS = ri.VISIBLE_WORDS | pymupdf.TEXT_ACCURATE_BBOXES
T99_DICT = pymupdf.TEXTFLAGS_DICT | pymupdf.TEXT_IGNORE_ACTUALTEXT
T99_ITEM_ID = re.compile(r"[A-Z]\d{2}[A-Z]?")  # 1999 item IDs: cluster letter, position, and any part letter
T99_INDEX_PAGE = t99_pdf.page_count - 1  # "Item Index 1999"
T99_INDEX_RIGHT_COLUMN_X = 260  # between the left column's page numbers and the right column's item IDs
# Item page layout, in points from the top-left corner, as in the 1995 grade 4 booklet.
T99_HEADER_Y = (75, 115)  # classification row under "Content Domain" and "Cognitive Domain"
T99_COGNITIVE_X = 224  # the line between the two columns
T99_LABEL_Y = (120, 150)  # item label
T99_ITEM_BOX_TOP = 150  # just below the item label
T99_RIGHT_COLUMN_X = 420  # the percent-correct table and its legend lie to the right
T99_PERCENT_X = 510  # left edge of the table's percent column
T99_MARKER_X = 535  # left edge of the table's significance-marker column
T99_FOOTER_Y = 750

# Spelled as in the item index; the item pages leave out the serial comma in "Analysis, and Probability".
T99_CONTENT_DOMAINS = [
    "Fractions and Number Sense",
    "Algebra",
    "Measurement",
    "Geometry",
    "Data Representation, Analysis, and Probability",
]
T99_DOMAINS_BY_LETTERS = {re.sub(r"[^a-z]", "", domain.casefold()): domain for domain in T99_CONTENT_DOMAINS}
T99_COGNITIVE_DOMAINS = [
    "Knowing",
    "Using Routine Procedures",
    "Using Complex Procedures",
    "Investigating and Solving Problems",
    "Communicating and Reasoning",
]
# Category headings of a scoring guide, in printed order (the bold 9-point lines): 1-point items, then 2-point items.
T99_SCORING_GUIDES = [
    ["Correct Response", "Incorrect Response"],
    ["Correct Response", "Partial Response", "Incorrect Response"],
]
T99_HEADING_SIZE = 9
T99_SIGNIFICANCE = {"Higher": "higher", "Not different": None, "Lower": "lower"}  # legend label -> vs_intl_avg


def t99_canonical(text, names):
    """The name in `names` that `text` spells, ignoring case and punctuation (the pages and the index differ in commas)."""
    letters = re.sub(r"[^a-z]", "", text.casefold())
    match = next((name for name in names if re.sub(r"[^a-z]", "", name.casefold()) == letters), None)
    if match is None:
        raise ValueError(f"unexpected classification {text!r}, expected one of {names}")
    return match


def t99_classification(words):
    """(content domain, cognitive domain) from the page header, whose cells may wrap onto a second line."""
    header_lines = ri.lines(w for w in words if T99_HEADER_Y[0] <= w[1] < T99_HEADER_Y[1])
    content = " ".join(ri.text(w for w in line if w[0] < T99_COGNITIVE_X) for line in header_lines)
    cognitive = " ".join(ri.text(w for w in line if T99_COGNITIVE_X <= w[0] < T99_RIGHT_COLUMN_X) for line in header_lines)
    return t99_canonical(content, T99_CONTENT_DOMAINS), t99_canonical(cognitive, T99_COGNITIVE_DOMAINS)


def t99_pct_correct(page, item_id):
    """Rows of the "Overall Percent Correct" table, each row's marker translated with the legend below the table.
    The markers are Wingdings 3 arrows coded as control characters, which word extraction drops, so the table is read
    span by span: a country name, its percent, and its marker are separate spans."""
    right = [
        (*span["bbox"], span["text"].strip())
        for block in page.get_text("dict", flags=T99_DICT | pymupdf.TEXT_ACCURATE_BBOXES)["blocks"]
        for line in block.get("lines", [])
        for span in line["spans"]
        if span["text"].strip() and span["bbox"][0] >= T99_RIGHT_COLUMN_X
    ]
    table_top = next(w[3] for w in right if w[4].startswith("Overall Percent Correct"))
    legend_top = next(w[1] for w in right if w[4].startswith("Country average"))
    legend = {}  # marker glyph -> vs_intl_avg, from legend lines such as "Higher ▲"
    for line in ri.lines(w for w in right if w[1] > legend_top):
        if ri.text(line[:-1]) in T99_SIGNIFICANCE:
            legend[line[-1][4]] = T99_SIGNIFICANCE[ri.text(line[:-1])]
    assert len(legend) == len(T99_SIGNIFICANCE), f"{item_id}: legend {legend}"
    rows = []
    for line in ri.lines(w for w in right if table_top < w[1] < legend_top):
        name = ri.text(w for w in line if w[0] < T99_PERCENT_X)
        values = [w[4] for w in line if T99_PERCENT_X <= w[0] < T99_MARKER_X]
        markers = [w[4] for w in line if w[0] >= T99_MARKER_X]
        assert len(values) == 1 and values[0].isdigit(), f"{item_id}: percent-correct row {ri.text(line)!r}"
        if name.casefold() != "international average":  # compared with nothing
            assert len(markers) == 1 and markers[0] in legend, f"{item_id}: percent-correct row {ri.text(line)!r}"
        rows.append(
            {
                "education_system": name,
                "pct_correct": int(values[0]),
                "vs_intl_avg": legend.get(markers[0]) if markers else None,
                "benchmarking": False,  # the 1999 booklet lists no benchmarking participants
            }
        )
    return rows


def t99_item_number(page_lines):
    """The item ID printed after "Item Number:" and the top of its line, or (None, None) if the page has none."""
    for line in page_lines:
        if ri.text(line[:2]) == "Item Number:":
            assert len(line) == 3 and T99_ITEM_ID.fullmatch(line[2][4]), ri.text(line)
            return line[2][4], line[0][1]
    return None, None


def t99_scoring_headings(page, clip):
    """The scoring-guide category headings in an area of a page: its bold 9-point lines."""
    headings = []
    for block in page.get_text("dict", flags=T99_DICT, clip=clip)["blocks"]:
        for line in block.get("lines", []):
            spans = [span for span in line["spans"] if span["text"].strip()]
            if spans and all("Bold" in span["font"] and round(span["size"]) == T99_HEADING_SIZE for span in spans):
                headings.append("".join(span["text"] for span in spans).strip())
    return headings


def t99_item_index(page):
    """The item index on the last page as {item ID: (content domain, item label, booklet page)}, column by column;
    the right column starts by continuing the left column's last section ("Algebra (continued)")."""
    words = page.get_text("words", flags=T99_WORDS)
    index, domain = {}, None
    for column in (
        [w for w in words if w[0] < T99_INDEX_RIGHT_COLUMN_X],
        [w for w in words if w[0] >= T99_INDEX_RIGHT_COLUMN_X],
    ):
        for line in ri.lines(column):
            line_text = re.sub(r" \(continued\)$", "", ri.text(line))
            letters = re.sub(r"[^a-z]", "", line_text.casefold())
            if letters in T99_DOMAINS_BY_LETTERS:  # a section heading, spelled with or without the serial comma
                domain = T99_DOMAINS_BY_LETTERS[letters]
            elif match := re.fullmatch(rf"({T99_ITEM_ID.pattern}) (.+) (\d+)", line_text):
                index[match[1]] = (domain, match[2], int(match[3]))
    return index


# Every item has one page, with its answer key or scoring guide below the item box.
t99_items = []
for page in t99_pdf:
    words = page.get_text("words", flags=T99_WORDS)
    page_lines = ri.lines(w for w in words if w[0] < T99_RIGHT_COLUMN_X)  # the left column, item box and below
    item_id, number_y = t99_item_number(page_lines)
    if item_id is None:
        continue
    content_domain, cognitive_domain = t99_classification(words)
    key_line = next((line for line in page_lines if ri.text(line[:2]) == "Correct Response:"), None)
    box = pymupdf.Rect(40, T99_ITEM_BOX_TOP, T99_RIGHT_COLUMN_X, number_y - 1)
    box_words = [w for w in words if ri.center(w) in box]
    scoring_words = [w for w in words if w[0] < T99_RIGHT_COLUMN_X and number_y - 1 <= w[1] < T99_FOOTER_Y]
    scoring = pymupdf.Rect(40, number_y - 1, T99_RIGHT_COLUMN_X, max(T99_FOOTER_Y, max(w[3] for w in scoring_words) + 0.5))
    scoring_lines = [ri.text(line) for line in ri.lines(scoring_words)]
    scored_parts = [m[1] for text in scoring_lines if (m := re.match(r"([A-C])[.:] Scoring for ", text))]
    assert key_line or "SCORING" in scoring_lines, f"{item_id}: neither an answer key nor a scoring guide"
    pct_rows = t99_pct_correct(page, item_id)
    intl_avg = next(r for r in pct_rows if r["education_system"].casefold() == "international average")
    usa = next(r for r in pct_rows if r["education_system"] == "United States")
    t99_items.append(
        {
            "item_id": item_id,
            "page_label": ri.text(w for w in words if T99_LABEL_Y[0] <= w[1] < T99_LABEL_Y[1] and w[0] < T99_RIGHT_COLUMN_X),
            "content_domain": content_domain,
            "cognitive_domain": cognitive_domain,
            "item_type": "multiple_choice" if key_line else "constructed_response",
            # Letters of the response options printed as text ("A. 90"); scanned items have them in the picture.
            "option_labels": [line[0][4][0] for line in ri.lines(box_words) if re.fullmatch(r"[A-E]\.", line[0][4])],
            "correct_option": key_line[2][4] if key_line else None,
            "pct_correct_intl_avg": intl_avg["pct_correct"],
            "pct_correct_usa": usa["pct_correct"],
            "pct_correct": [r for r in pct_rows if r is not intl_avg],
            "booklet_page": int(max((w for w in words if w[1] >= T99_FOOTER_Y), key=lambda w: w[0])[4]),
            "pdf_page": page.number + 1,
            "item_box": (page.number, box),
            "scoring_clip": (page.number, scoring),
            "scoring_categories": t99_scoring_headings(page, scoring),
            "scored_part": scored_parts[0] if scored_parts else None,  # "A. Scoring for ..."
            "_item_box_text": ri.text(w for line in ri.lines(box_words) for w in line),
        }
    )

# The parts of a multi-part item (IDs ending in A, B, C) share one item box, printed on every part's page, and each
# part's scoring guide starts with "A. Scoring for ...", "B. Scoring for ...".
for _, unit in groupby(t99_items, key=lambda item: item["item_id"][:3]):
    unit = list(unit)
    for item in unit:
        item["parts"] = [i["item_id"] for i in unit] if len(unit) > 1 else None
        item["part"] = item["item_id"][3:] if len(unit) > 1 else None
        assert item["scored_part"] == item["part"], f"{item['item_id']}: scoring guide for part {item['scored_part']}"

# The item index must agree with the item pages.
t99_index = t99_item_index(t99_pdf[T99_INDEX_PAGE])
assert [item["item_id"] for item in t99_items] == list(t99_index), "item pages and item index list different items"
t99_label_differences = {}
for item in t99_items:
    domain, label, booklet_page = t99_index[item["item_id"]]
    assert (domain, booklet_page) == (item["content_domain"], item["booklet_page"]), f"{item['item_id']}: index differs"
    if label != item["page_label"]:
        t99_label_differences[item["item_id"]] = (item["page_label"], label)
    item["item_label"] = item["page_label"]

    if item["item_type"] == "multiple_choice":
        assert item["option_labels"] in ([], list("ABCD"), list("ABCDE")), f"{item['item_id']}: {item['option_labels']}"
        assert not item["option_labels"] or item["correct_option"] in item["option_labels"], item["item_id"]
        assert not item["scoring_categories"], f"{item['item_id']}: multiple-choice item with a scoring guide"
        item["option_labels"] = item["option_labels"] or None  # None: the letters are printed inside the picture
    else:
        assert not item["option_labels"], f"{item['item_id']}: constructed-response item with options"
        assert item["scoring_categories"] in T99_SCORING_GUIDES, f"{item['item_id']}: {item['scoring_categories']}"
    item["max_points"] = 2 if "Partial Response" in item["scoring_categories"] else 1

t99_systems = {r["education_system"] for item in t99_items for r in item["pct_correct"]}
t99_missing_systems = {
    item["item_id"]: sorted(t99_systems - {r["education_system"] for r in item["pct_correct"]})
    for item in t99_items
    if len(item["pct_correct"]) < len(t99_systems)
}
t99_items_df = pd.DataFrame(t99_items)
print(f"Parsed {len(t99_items)} items; labels that differ in the index (page, index): {t99_label_differences}")
display(t99_items_df.groupby(["content_domain", "item_type"]).size().unstack(fill_value=0))
display(t99_items_df.groupby(["item_type", "max_points"]).size())
print(
    "Multiple-choice items whose option letters are printed inside the pictures:",
    [item["item_id"] for item in t99_items if item["item_type"] == "multiple_choice" and not item["option_labels"]],
)
print(f"{len(t99_systems)} education systems; tables without some of them: {t99_missing_systems}")

### Transcribe stems, options, and scoring guides with Gemini

`gemini-3.8-flash` is called through OpenRouter by `ri.Transcriber`, as in the 2011 grade 8 section, with up to `T99_ITEMS_PER_REQUEST` items per request in booklet order, keeping the parts of a multi-part item together, which makes 21 requests for the whole set.
For each item it sends a short description (label, type, points, option letters, and part) and the item box and the area below it, rendered at 300 DPI and trimmed to the printed content.

A transcription is kept only if it passes the checks in `t99_problems`: the option letters (A-D or A-E when the letters are inside the picture) and answer key of a multiple-choice item, the scoring-guide category headings of a constructed-response item, and a stem that differs from those of the item's other parts.
Kept transcriptions are cached per item under `data/cache/item_transcriptions/timss99_g8_math/gemini-3.8-flash-v1/`; set `T99_MAX_REQUESTS` to send fewer requests, and bump `T99_PROMPT_VERSION` after editing the prompt or schema.

In [ ]:
from pydantic import BaseModel, Field

T99_LLM_MODEL = "google/gemini-3.8-flash"
T99_PROMPT_VERSION = 1
T99_LLM_CACHE_DIR = (
    T99_DATA_DIR / "cache" / "item_transcriptions" / "timss99_g8_math" / f"{T99_LLM_MODEL.split('/')[-1]}-v{T99_PROMPT_VERSION}"
)
T99_ITEMS_PER_REQUEST = 4
T99_MAX_REQUESTS = None  # a number sends at most that many requests in one run, such as 1 for a test


class T99Option(BaseModel):
    label: str = Field(description="Option letter as printed, without the period, e.g. 'A'.")
    text: str = Field(description="Option content in the same notation as the stem.")


class T99ScoringCategory(BaseModel):
    category: str = Field(description="Category heading as printed, e.g. 'Correct Response'.")
    descriptors: list[str] = Field(description="Each bullet under the heading, in printed order.")


class T99Transcription(BaseModel):
    item_id: str
    stem: str
    options: list[T99Option]
    answer_key: str | None
    scoring_guide: list[T99ScoringCategory]
    scoring_notes: str | None
    has_figure: bool
    figure_description: str | None


T99_TRANSCRIPTION_INSTRUCTIONS = """\
You transcribe items from the TIMSS 1999 grade 8 released mathematics booklet into structured data.

Each request covers several items, listed by ID with their labels. For each item you receive:
1. An image of the boxed item exactly as students saw it; many items are scans of the printed test.
2. An image of what the booklet prints below the item box: the item number and either the multiple-choice answer key \
("Correct Response: X") or the scoring guide under "SCORING".

The parts of a multi-part item (IDs ending in A, B, C) share one item box, whose image is given with each part: an \
introduction shared by all parts, followed by every part, each starting with its part label (such as "a)" or "b."). \
Each part's scoring guide starts with a line such as "A. Scoring for ...".

The images are the only source. Transcribe what is printed, verbatim; never add, correct, or solve anything.

Return one entry per item, in the order given:
- item_id: The item ID as given.
- stem: Everything in the item box for this item except multiple-choice response options, in reading order, with the \
wording verbatim. For a part of a multi-part item, the shared introduction followed by this part only, starting with \
its part label as printed; leave out the other parts. Write math in plain-text notation: fractions as 1/2, mixed \
numbers as 1 1/2, powers as x^2, square roots as sqrt(x), and the symbols ×, ÷, −, =, <, >, ≤, ≥, °, π. Write an empty \
box in a number sentence as □ and an answer blank as ____, keeping labels such as "Answer:". Write tables as Markdown \
tables and bulleted lists as "- " lines. Replace each picture, diagram, graph, grid, or map with \
"[Image: <short description>]". Separate paragraphs with a newline.
- options: For a multiple-choice item, one entry per response option in printed order, labeled with its letter \
(A, B, C, ...) without the period, in the same notation as the stem; a pictured option is "[Image: <description>]", \
and its letter may be printed inside the picture. Empty for constructed-response items.
- answer_key: The letter printed in the "Correct Response:" box of a multiple-choice item; null for \
constructed-response items.
- scoring_guide: For a constructed-response item, each category of the scoring guide in printed order, with its \
heading verbatim (e.g. "Correct Response", "Partial Response", "Incorrect Response") and each bullet verbatim as a \
descriptor, in the same notation as the stem; leave out the "A. Scoring for ..." line of a part. Write a picture of a \
sample answer, whether it is a bullet of its own or follows a bullet's text, as "[Image: <description>]". Empty for \
multiple-choice items.
- scoring_notes: Any scoring-guide text that is not a heading or bullet, such as a note; null if none.
- has_figure: Whether this item's stem or options contain any picture, diagram, graph, grid, map, or drawn shape. A \
table of text or numbers alone does not count.
- figure_description: If has_figure, a precise, objective description of every visual element in this item's stem \
and options, giving all the information a student would need to answer without seeing it: counts, labels, \
measurements, positions, grid coordinates, bar heights, shading, and orientation. Describe what is drawn; do not \
solve the item or reveal the answer. Null if has_figure is false.
"""


def t99_page_part(clip):
    page_index, rect = clip
    return ri.page_area_part(t99_pdf[page_index], rect)


def t99_description(item):
    if item["item_type"] == "constructed_response":
        kind = f"constructed-response, worth {item['max_points']} point{'s' if item['max_points'] > 1 else ''}"
    else:
        labels = item["option_labels"]
        options = f"options {labels[0]} to {labels[-1]}" if labels else "option letters printed in the picture"
        kind = f"multiple-choice ({options}), worth 1 point"
    if item["parts"]:
        kind = f"part {item['part']} of the multi-part item with parts {', '.join(item['parts'])}, {kind}"
    return f"Item {item['item_id']} (\"{item['item_label']}\"): {kind}."


def t99_request_content(batch):
    """The user message for a batch: each item's box and what is printed below it."""
    content = [ri.text_part(f"Transcribe these {len(batch)} items, in this order: {', '.join(i['item_id'] for i in batch)}.")]
    for item in batch:
        content += [
            ri.text_part(f"{t99_description(item)} Item box:"),
            t99_page_part(item["item_box"]),
            ri.text_part("Printed below the item box:"),
            t99_page_part(item["scoring_clip"]),
        ]
    return content


def t99_problems(item, t, returned):
    """Disagreements between a transcription and the deterministic parse, or with the transcriptions of the other
    parts of its multi-part item in `returned`; empty if the transcription is usable."""
    problems = []
    if not t.stem.strip():
        problems.append("empty stem")
    labels = [o.label for o in t.options]
    if item["item_type"] == "multiple_choice":
        if labels not in ([item["option_labels"]] if item["option_labels"] else [list("ABCD"), list("ABCDE")]):
            problems.append(f"option labels {labels}, expected {item['option_labels'] or 'A-D or A-E'}")
        if t.answer_key != item["correct_option"] or t.answer_key not in labels:
            problems.append(f"answer key {t.answer_key!r}, expected {item['correct_option']!r}")
        if t.scoring_guide:
            problems.append("multiple-choice item has a scoring guide")
    else:
        if t.options or t.answer_key:
            problems.append("constructed-response item has options or an answer key")
        categories = [c.category for c in t.scoring_guide]
        if categories != item["scoring_categories"]:
            problems.append(f"scoring categories {categories}, expected {item['scoring_categories']}")
        if any(not c.descriptors for c in t.scoring_guide):
            problems.append("scoring category without descriptors")
    for other in item["parts"] or []:  # each part is its own question
        if other != item["item_id"] and other in returned and returned[other].stem == t.stem:
            problems.append(f"same stem as {other}")
    if t.has_figure != bool(t.figure_description):
        problems.append(f"has_figure={t.has_figure} but figure_description={t.figure_description!r}")
    return problems


t99_transcriber = ri.Transcriber(
    api_key=globals().get("openrouter_api_key"),
    model=T99_LLM_MODEL,
    cache_dir=T99_LLM_CACHE_DIR,
    schema=T99Transcription,
    instructions=T99_TRANSCRIPTION_INSTRUCTIONS,
)
t99_transcriptions = t99_transcriber.run(
    t99_items,
    request_content=t99_request_content,
    problems=t99_problems,
    items_per_request=T99_ITEMS_PER_REQUEST,
    unit=lambda item: item["item_id"][:3],
    max_requests=T99_MAX_REQUESTS,
)
print(f"All {len(t99_transcriptions)} items are transcribed")

### Validate and write the JSONL

The coverage report lists words drawn in each item box (all parts of a multi-part item together) that are missing from the transcriptions; scanned items have no words to check.

In [ ]:
# Words in each item box (all parts of a multi-part item together) that the transcriptions lack. Many 1999 items
# are scanned images, which have no words to check.
t99_units = {}
for item in t99_items:
    t = t99_transcriptions[item["item_id"]]
    unit = t99_units.setdefault(item["item_id"][:3], {"drawn": item["_item_box_text"], "transcribed": []})
    unit["transcribed"] += [t.stem, *(o.text for o in t.options)]
t99_coverage = ri.coverage(
    {unit_id: unit["drawn"] for unit_id, unit in t99_units.items()},
    {unit_id: " ".join(unit["transcribed"]) for unit_id, unit in t99_units.items()},
)
print(f"{len(t99_coverage)} item boxes have drawn words missing from the transcription")
if not t99_coverage.empty:
    display(t99_coverage)

t99_records = []
for item in t99_items:
    t = t99_transcriptions[item["item_id"]]
    t99_records.append(
        {
            "item_id": item["item_id"],
            "item_label": item["item_label"],
            "assessment": "TIMSS 1999",
            "grade": 8,
            "subject": "Mathematics",
            "content_domain": item["content_domain"],
            "main_topic": None,  # the 1999 booklet classifies items by content domain and performance expectation only
            "cognitive_domain": item["cognitive_domain"],
            "item_type": item["item_type"],
            "max_points": item["max_points"],
            "stem": ri.tidy(t.stem),
            "options": [{"label": o.label, "text": ri.tidy(o.text)} for o in t.options] or None,
            "correct_option": item["correct_option"],
            "scoring_guide": [
                {"category": c.category, "descriptors": [ri.tidy(d) for d in c.descriptors]} for c in t.scoring_guide
            ]
            or None,
            "scoring_notes": t.scoring_notes,
            "has_figure": t.has_figure,
            "figure_description": t.figure_description,
            "accessibility_text": None,
            "pct_correct_intl_avg": item["pct_correct_intl_avg"],
            "pct_correct_usa": item["pct_correct_usa"],
            "pct_correct": item["pct_correct"],
            "booklet_page": item["booklet_page"],
            "pdf_page": item["pdf_page"],
            "source_url": T99_PDF_URL,
            "transcription_model": T99_LLM_MODEL,
        }
    )

t99_released_items = ri.write_jsonl(t99_records, T99_OUTPUT_PATH)
print(f"Wrote {len(t99_released_items)} items to {T99_OUTPUT_PATH.resolve()}")
display(t99_released_items[["item_id", "content_domain", "cognitive_domain", "item_type", "max_points", "stem", "options"]].head())

## TIMSS Advanced 2008 released mathematics items

Build `data/timssadv08_g12_math_released_items.jsonl` from `TA08_MAT_Released_Items.pdf` in `TA08_Items.zip`, the "Items" download of the [TIMSS Advanced 2008 International Database](https://timssandpirls.bc.edu/timss_advanced/idb.html), which the NCES page links to as the complete 2008 TIMSS Advanced mathematics set.

TIMSS Advanced assesses students in the final year of secondary school who take advanced mathematics, so `assessment` is "TIMSS Advanced 2008", `grade` 12, and `subject` "Advanced Mathematics".
The PDF is laid out like the 2007 booklets: each item page carries its documentation as text (item ID and block position in the header; content domain, cognitive domain, maximum points, and key in the sidebar) and the item as one embedded image, and each constructed-response item is followed by a scoring-guide page with its code tables as text.
The parts of a multi-part item share one item image and block position, and each part's scoring-guide page shows the code tables of all parts.
The booklet has no item statistics, so `pct_correct_intl_avg`, `pct_correct_usa`, and `pct_correct` are null, and no accessibility transcription.

`TA08_MAT_Item_Information.xls`, in the same zip, gives each item's label, content topic area (`main_topic`), and number of options, and must agree with the booklet on the domains, maximum points, block position, and key.
It marks four more items as released (MA13005, MA13010, MA13022, and MA13023); they were not used in scaling and are not in the booklet.

Extraction is hybrid:

- **Deterministic, from the PDF and the item information:** the item documentation and the fields above, and the item images as embedded.
- **`gemini-3.8-flash` through OpenRouter:** `stem`, `options`, `scoring_guide`, `scoring_notes`, `has_figure`, and `figure_description`, read from the item image and the rendered scoring-guide page, then checked against the documentation.

`figure_description` is model-generated.

### Deterministic parse of the item documentation and images

Each sidebar label ("Content Domain") is a text block of its own, with its value in the next block below it.
The item information gives a part its own block position ("05A"), and the booklet the item's ("M3_05").

In [ ]:
import re
import zipfile
from collections import defaultdict
from pathlib import Path

import pandas as pd
import pymupdf
from IPython.display import display

from timss_1 import released_items as ri

# "Complete 2008 TIMSS Advanced mathematics set" on https://nces.ed.gov/timss/released-questions.asp links to this
# zip on the TIMSS & PIRLS International Study Center site; the booklet is TA08_MAT_Released_Items.pdf in it.
TA08_ZIP_URL = "https://timss.bc.edu/timss_advanced/downloads/TA08_Items.zip"
TA08_DATA_DIR = Path.cwd() / "data"
TA08_ZIP_PATH = ri.download(TA08_ZIP_URL, TA08_DATA_DIR / "raw" / "TA08_Items.zip")
TA08_DIR = TA08_ZIP_PATH.with_suffix("")  # the zip holds a TA08_Items/ folder
TA08_OUTPUT_PATH = TA08_DATA_DIR / "timssadv08_g12_math_released_items.jsonl"
if not (TA08_DIR / "TA08_MAT_Released_Items.pdf").exists():
    with zipfile.ZipFile(TA08_ZIP_PATH) as archive:
        archive.extractall(TA08_ZIP_PATH.parent)
ta08_pdf = pymupdf.open(TA08_DIR / "TA08_MAT_Released_Items.pdf")

TA08_VISIBLE_TEXT = pymupdf.TEXTFLAGS_TEXT | pymupdf.TEXT_IGNORE_ACTUALTEXT
TA08_ITEM_ID = re.compile(r"MA\d{5}[A-Z]?")
TA08_FIRST_ITEM_PAGE = 5  # PDF page 6; pages 4-5 hold the item table
TA08_SIDEBAR_FIELDS = ["Content Domain", "Cognitive Domain", "Maximum Points", "Key"]
TA08_CONTENT_DOMAINS = {"Algebra", "Calculus", "Geometry"}
TA08_COGNITIVE_DOMAINS = {"Knowing", "Applying", "Reasoning"}
TA08_CONSTRUCTED_RESPONSE_KEY = "See scoring guide"
TA08_SCORING_CLIP_TOP = 88  # below the item header bar
TA08_SIDEBAR_X = 410  # the sidebar of an item page, right of the item image


def ta08_page_fields(page):
    """The header's item ID and block position, and the sidebar's labeled boxes, of an item or scoring-guide page.
    Each sidebar label ("Content Domain") is a text block of its own, with its value in the next block below it."""
    fields = {}
    blocks = page.get_text("blocks", flags=TA08_VISIBLE_TEXT)
    for block in blocks:
        block_lines = [line.strip() for line in block[4].splitlines() if line.strip()]
        if block_lines and block_lines[0].startswith("Item ID "):
            fields["Item ID"] = block_lines[0].removeprefix("Item ID ")
            fields["Block Position"] = block_lines[-1].removeprefix("Block_Sequence ")
    sidebar = sorted((b for b in blocks if b[0] > TA08_SIDEBAR_X and b[1] > TA08_SCORING_CLIP_TOP), key=lambda b: b[1])
    for label, value in zip(sidebar, sidebar[1:]):
        if label[4].strip() in TA08_SIDEBAR_FIELDS:
            fields[label[4].strip()] = " ".join(value[4].split())
    return fields


def ta08_main_image(page):
    """xref of the largest image on the page, the item; the others are logos."""
    return max(page.get_image_info(xrefs=True), key=lambda info: abs(pymupdf.Rect(info["bbox"])))["xref"]


def ta08_booklet_page(page):
    return int(page.get_text(flags=TA08_VISIBLE_TEXT).split()[0])


ta08_items, ta08_scoring_pages = [], defaultdict(list)
for page in ta08_pdf.pages(TA08_FIRST_ITEM_PAGE):
    fields = ta08_page_fields(page)
    if "Item ID" not in fields:
        continue
    if "Key" not in fields:  # a scoring-guide page: code tables, as text
        ta08_scoring_pages[fields["Item ID"]].append(page.number)
        continue
    is_multiple_choice = fields["Key"] != TA08_CONSTRUCTED_RESPONSE_KEY
    assert TA08_ITEM_ID.fullmatch(fields["Item ID"]), fields
    assert fields["Content Domain"] in TA08_CONTENT_DOMAINS and fields["Cognitive Domain"] in TA08_COGNITIVE_DOMAINS, fields
    assert not is_multiple_choice or fields["Key"] in set("ABCDE"), fields
    ta08_items.append(
        {
            "item_id": fields["Item ID"],
            "block_position": fields["Block Position"],
            "content_domain": fields["Content Domain"],
            "cognitive_domain": fields["Cognitive Domain"],
            "item_type": "multiple_choice" if is_multiple_choice else "constructed_response",
            "max_points": int(fields["Maximum Points"]),
            "correct_option": fields["Key"] if is_multiple_choice else None,
            "booklet_page": ta08_booklet_page(page),
            "pdf_page": page.number + 1,
            "image": ta08_main_image(page),
        }
    )

# The parts of a multi-part item share their block position and item image; each part's scoring-guide page shows
# the code tables of all parts.
ta08_units = defaultdict(list)
for item in ta08_items:
    ta08_units[item["block_position"]].append(item["item_id"])
for item in ta08_items:
    unit = ta08_units[item["block_position"]]
    item["parts"] = unit if len(unit) > 1 else None
    item["part"] = item["item_id"][-1] if len(unit) > 1 else None
    item["scoring_pages"] = ta08_scoring_pages.get(item["item_id"], [])
    is_multiple_choice = item["item_type"] == "multiple_choice"
    assert len(item["scoring_pages"]) == (0 if is_multiple_choice else 1), f"{item['item_id']}: scoring-guide pages"
    assert len(unit) == 1 or item["item_id"][-1].isalpha(), f"{item['item_id']}: part without a part letter"

# The item information file in the same zip gives the label, topic area, and number of options, and must agree.
ta08_information = pd.read_excel(TA08_DIR / "TA08_MAT_Item_Information.xls", dtype=str)
ta08_information.columns = [" ".join(str(column).split()) for column in ta08_information.columns]
ta08_information = ta08_information.apply(lambda column: column.str.strip()).set_index("Item ID")
# Four items marked released there (MA13005, MA13010, MA13022, MA13023) were not used in scaling and are not in the
# booklet.
released = set(ta08_information.index[ta08_information["Released"] == "Yes"])
ta08_not_in_booklet = sorted(released - {item["item_id"] for item in ta08_items})
assert {item["item_id"] for item in ta08_items} <= released, "booklet items that the item information does not release"
assert (ta08_information.loc[ta08_not_in_booklet, "In Scaling"] == "No").all(), "scaled released items not in the booklet"
for item in ta08_items:
    info = ta08_information.loc[item["item_id"]]
    assert (info["Content Domain"], info["Cognitive Domain"], int(info["Maximum Points"])) == (
        item["content_domain"],
        item["cognitive_domain"],
        item["max_points"],
    ), f"{item['item_id']}: item information differs from the booklet"
    # The item information gives a part its own position ("05A"); the booklet prints the item's ("M3_05").
    assert f"{info['TIMSA 2008 Block']}_{info['TIMSA 2008 Block Seq'][:2]}" == item["block_position"], item["item_id"]
    assert info["Key"] == (item["correct_option"] or "-"), f"{item['item_id']}: key differs"
    item["item_label"] = info["Item Label"]
    item["main_topic"] = info["Content Topic Area"]
    item["option_labels"] = list("ABCDE"[: int(info["Options"])]) if info["Item Type"] == "MC" else None

ta08_items_df = pd.DataFrame(ta08_items)
print(f"Parsed {len(ta08_items)} items; released items not in the booklet (not scaled): {ta08_not_in_booklet}")
display(ta08_items_df.groupby(["content_domain", "item_type"]).size().unstack(fill_value=0))
display(ta08_items_df.groupby(["item_type", "max_points"]).size())
print("Parts of multi-part items:", ta08_items_df.loc[ta08_items_df["part"].notna(), "item_id"].tolist())

### Transcribe stems, options, and scoring guides with Gemini

`gemini-3.8-flash` is called through OpenRouter by `ri.Transcriber`, as in the 2011 grade 8 section, with up to `TA08_ITEMS_PER_REQUEST` items per request in booklet order, keeping the parts of a multi-part item together, which makes 10 requests for the whole set (about $0.22 in total on 2026-10-03).
For each item it sends the item image (once per multi-part item) and, for a constructed-response item, its scoring-guide page rendered at 300 DPI.

A transcription is kept only if it passes the checks in `ta08_problems`: the option letters that the item information gives for a multiple-choice item, the item ID on the scoring guide, the scoring categories and code ranges implied by the maximum points, and a stem that differs from those of the item's other parts.
Kept transcriptions are cached per item under `data/cache/item_transcriptions/timssadv08_math/gemini-3.8-flash-v1/`; set `TA08_MAX_REQUESTS` to send fewer requests, and bump `TA08_PROMPT_VERSION` after editing the prompt or schema.

In [ ]:
from pydantic import BaseModel, Field

TA08_LLM_MODEL = "google/gemini-3.8-flash"
TA08_PROMPT_VERSION = 1
TA08_LLM_CACHE_DIR = (
    TA08_DATA_DIR
    / "cache"
    / "item_transcriptions"
    / "timssadv08_math"
    / f"{TA08_LLM_MODEL.split('/')[-1]}-v{TA08_PROMPT_VERSION}"
)
TA08_ITEMS_PER_REQUEST = 4
TA08_MAX_REQUESTS = None  # a number sends at most that many requests in one run, such as 1 for a test
# Codes each scoring-guide category may hold, by the item's maximum points, as in TIMSS 2007: the first digit of a
# score code is the score, except 7 (incorrect) and 9 (nonresponse).
TA08_CATEGORY_CODES = {
    "Correct Response": {1: range(10, 20), 2: range(20, 30)},
    "Partial Response": {2: range(10, 20)},
    "Incorrect Response": {1: range(70, 80), 2: range(70, 80)},
    "Nonresponse": {1: range(90, 100), 2: range(90, 100)},
}


class TA08Option(BaseModel):
    label: str = Field(description="Option letter as printed, e.g. 'A'.")
    text: str = Field(description="Option content in the same notation as the stem.")


class TA08ScoringCode(BaseModel):
    code: int = Field(description="Score code as printed, e.g. 10.")
    response: str = Field(description="The code's response description, verbatim.")


class TA08ScoringCategory(BaseModel):
    category: str = Field(description="Category heading as printed, e.g. 'Correct Response'.")
    codes: list[TA08ScoringCode] = Field(description="Each code row under the heading, in printed order.")


class TA08Transcription(BaseModel):
    item_id: str
    stem: str
    options: list[TA08Option]
    scoring_guide_item_id: str | None
    scoring_guide: list[TA08ScoringCategory]
    scoring_notes: str | None
    has_figure: bool
    figure_description: str | None


TA08_TRANSCRIPTION_INSTRUCTIONS = """\
You transcribe items from the TIMSS Advanced 2008 released advanced mathematics items (final year of secondary \
school) into structured data.

Each request covers several items, listed by ID. For each item you receive:
1. The item as printed in the test booklet, as an image. The parts of a multi-part item (IDs ending in A, B, ...) share \
it, and it is shown once, before the parts: an introduction shared by all parts, followed by every part, each \
starting with its part label (e.g. "a)" or "B.").
2. For a constructed-response item, its scoring-guide page as an image: one table of score codes per item part, each \
headed "Item: <item ID>". Use only the table headed by the item's own ID.

The images are the only source. Transcribe what is printed, verbatim; never add, correct, or solve anything.

Return one entry per item, in the order given:
- item_id: The item ID as given.
- stem: Everything the student sees for this item except the multiple-choice response options, in reading order. \
For a part of a multi-part item, the shared introduction followed by this part only, starting with its part label as \
printed; leave out the other parts. Leave out page furniture such as the item ID printed sideways in the margin. \
Write math in plain-text notation: fractions as a/b (with parentheses where needed), powers as x^2, roots as sqrt(x), \
functions as f(x), derivatives as f'(x), integrals as integral from a to b of f(x) dx, limits as lim_(x->a) f(x), \
and the symbols ×, ÷, −, =, <, >, ≤, ≥, ≠, ∞, π, °. Write an answer blank as ____. Write tables as Markdown tables. \
Replace each graph, diagram, or other picture with "[Image: <short description>]". Separate paragraphs with a newline.
- options: For a multiple-choice item, one entry per response option in printed order, labeled with its circled \
letter, in the same notation as the stem; a pictured option is "[Image: <description>]". Empty for \
constructed-response items.
- scoring_guide_item_id: The item ID in the header of the code table used ("Item: ..."); null for multiple-choice \
items.
- scoring_guide: For a constructed-response item, each category of that table in printed order (e.g. "Correct \
Response", "Partial Response", "Incorrect Response", "Nonresponse"), with its heading verbatim, except that \
"Partially Correct Response" is written "Partial Response", and each code row under it: the score code and its \
response description, verbatim (including examples and notes in the row), in the same notation as the stem. Empty \
for multiple-choice items.
- scoring_notes: Any scoring-guide text outside the code table, such as a note; null if none.
- has_figure: Whether this item's stem or options contain any graph, diagram, or other picture. A table of text or \
numbers alone does not count.
- figure_description: If has_figure, a precise, objective description of every visual element in this item's stem \
and options, giving all the information a student would need to answer without seeing it: axes, scales, labels, \
coordinates of marked points, shapes, and measurements. Describe what is drawn; do not solve the item or reveal the \
answer. Null if has_figure is false.
"""


def ta08_request_content(batch):
    """The user message for a batch: each item's image (once per multi-part item) and its scoring-guide page."""
    content = [ri.text_part(f"Transcribe these {len(batch)} items, in this order: {', '.join(i['item_id'] for i in batch)}.")]
    shown_images = set()
    for item in batch:
        description = f"{item['item_type'].replace('_', '-')}, worth {item['max_points']} point{'s' if item['max_points'] > 1 else ''}"
        if item["option_labels"]:
            description += f", options {item['option_labels'][0]} to {item['option_labels'][-1]}"
        if item["parts"]:
            if item["image"] not in shown_images:
                shown_images.add(item["image"])
                content.append(ri.text_part(f"The item with parts {', '.join(item['parts'])}:"))
                content.append(ri.embedded_image_part(ta08_pdf, item["image"]))
            content.append(ri.text_part(f"Item {item['item_id']}: part {item['part']} of that item, {description}."))
        else:
            content.append(ri.text_part(f"Item {item['item_id']}: {description}."))
            content.append(ri.embedded_image_part(ta08_pdf, item["image"]))
        for page_number in item["scoring_pages"]:
            page = ta08_pdf[page_number]
            content.append(ri.text_part(f"Scoring-guide page of item {item['item_id']}:"))
            content.append(ri.page_area_part(page, pymupdf.Rect(0, TA08_SCORING_CLIP_TOP, page.rect.width, page.rect.height)))
    return content


def ta08_problems(item, t, returned):
    """Disagreements between a transcription and the item's documentation; empty if the transcription is usable."""
    problems = []
    if not t.stem.strip():
        problems.append("empty stem")
    if item["item_type"] == "multiple_choice":
        if [o.label for o in t.options] != item["option_labels"]:
            problems.append(f"option labels {[o.label for o in t.options]}, expected {item['option_labels']}")
        if t.scoring_guide or t.scoring_guide_item_id:
            problems.append("multiple-choice item has a scoring guide")
    else:
        if t.options:
            problems.append("constructed-response item has options")
        if t.scoring_guide_item_id != item["item_id"]:
            problems.append(f"scoring guide headed {t.scoring_guide_item_id!r}")
        expected = [c for c, codes in TA08_CATEGORY_CODES.items() if item["max_points"] in codes]
        categories = [c.category for c in t.scoring_guide]
        if categories != expected:
            problems.append(f"scoring categories {categories}, expected {expected} for {item['max_points']} points")
        for c in t.scoring_guide:
            allowed = TA08_CATEGORY_CODES.get(c.category, {}).get(item["max_points"], range(0))
            codes = [code.code for code in c.codes]
            if not codes or any(code not in allowed for code in codes):
                problems.append(f"{c.category} codes {codes}")
    for other in item["parts"] or []:  # each part is its own question
        if other != item["item_id"] and other in returned and returned[other].stem == t.stem:
            problems.append(f"same stem as {other}")
    if t.has_figure != bool(t.figure_description):
        problems.append(f"has_figure={t.has_figure} but figure_description={t.figure_description!r}")
    return problems


ta08_transcriber = ri.Transcriber(
    api_key=globals().get("openrouter_api_key"),
    model=TA08_LLM_MODEL,
    cache_dir=TA08_LLM_CACHE_DIR,
    schema=TA08Transcription,
    instructions=TA08_TRANSCRIPTION_INSTRUCTIONS,
)
ta08_transcriptions = ta08_transcriber.run(
    ta08_items,
    request_content=ta08_request_content,
    problems=ta08_problems,
    items_per_request=TA08_ITEMS_PER_REQUEST,
    unit=lambda item: item["block_position"],
    max_requests=TA08_MAX_REQUESTS,
)
print(f"All {len(ta08_transcriptions)} items are transcribed")

### Write the JSONL

The items are images, so there are no drawn words to compare with the transcriptions.
`scoring_guide` keeps each category's response descriptions; the score codes are used only by the checks above.

In [ ]:
ta08_records = []
for item in ta08_items:
    t = ta08_transcriptions[item["item_id"]]
    ta08_records.append(
        {
            "item_id": item["item_id"],
            "item_label": item["item_label"],
            "assessment": "TIMSS Advanced 2008",
            "grade": 12,  # final year of secondary school
            "subject": "Advanced Mathematics",
            "content_domain": item["content_domain"],
            "main_topic": item["main_topic"],
            "cognitive_domain": item["cognitive_domain"],
            "item_type": item["item_type"],
            "max_points": item["max_points"],
            "stem": ri.tidy(t.stem),
            "options": [{"label": o.label, "text": ri.tidy(o.text)} for o in t.options] or None,
            "correct_option": item["correct_option"],
            # The response descriptions under each heading; the two-digit score codes were used for the checks.
            "scoring_guide": [
                {"category": c.category, "descriptors": [ri.tidy(code.response) for code in c.codes]}
                for c in t.scoring_guide
            ]
            or None,
            "scoring_notes": t.scoring_notes,
            "has_figure": t.has_figure,
            "figure_description": t.figure_description,
            "accessibility_text": None,
            "pct_correct_intl_avg": None,  # the booklet has no item statistics
            "pct_correct_usa": None,
            "pct_correct": None,
            "booklet_page": item["booklet_page"],
            "pdf_page": item["pdf_page"],
            "source_url": TA08_ZIP_URL,
            "transcription_model": TA08_LLM_MODEL,
        }
    )

ta08_released_items = ri.write_jsonl(ta08_records, TA08_OUTPUT_PATH)
print(f"Wrote {len(ta08_released_items)} items to {TA08_OUTPUT_PATH.resolve()}")
display(ta08_released_items[["item_id", "content_domain", "main_topic", "cognitive_domain", "item_type", "stem", "options"]].head())

## TIMSS Advanced 1995 released mathematics items

Build `data/timssadv95_g12_math_released_items.jsonl` from the [TIMSS 1995 Advanced Mathematics released items](https://timss.bc.edu/timss1995i/TIMSSPDF/CitemAdM.pdf) (`CitemAdM.pdf`), which the NCES page links to as the complete 1995 TIMSS Advanced mathematics set.
As for 2008, `assessment` is "TIMSS Advanced 1995", `grade` 12, and `subject` "Advanced Mathematics".

The booklet differs from the others:

- Each item fills a page, with its ID in a box at the top right ("K-1", "L-15a"; recorded as K1 and L15A) and a table at the foot giving the subject, answer key, content category, performance expectation, international average percent correct, and difficulty index.
- `content_domain` is the 1995 content category (Numbers, Equations and Functions; Calculus; Geometry; Probability & Statistics; Validation and Structure), and `cognitive_domain` the performance expectation (Knowing, Routine Procedures, Complex Procedures, Solving Problems, Justifying and Proving, Communicating).
- Items have no labels and no topic areas, so `item_label` and `main_topic` are null, and only the international average percent correct is given, so `pct_correct_usa` and `pct_correct` are null.
- Each constructed-response item is followed by a coding guide of one or two pages with score codes under "Correct Response", "Partial Response", "Minimal Response", "Incorrect Response", and "Nonresponse". The first digit of a correct-response code gives the maximum points: some items are worth 3 points.
- A red copyright watermark, which Acrobat stamped over the item and coding-guide pages, hides the text under it. `ri.remove_watermarks` removes it from the opened document, so neither the parse nor the images sent to the model contain it.

Extraction is hybrid:

- **Deterministic, from the PDF:** item ID, content domain, cognitive domain, answer key, percent correct, coding-guide category headings, and the maximum points.
- **`gemini-3.8-flash` through OpenRouter:** `stem`, `options`, `scoring_guide`, `scoring_notes`, `has_figure`, and `figure_description`, read from rendered images of the item and its coding guide, then checked against the deterministic fields.

`figure_description` is model-generated.

### Deterministic parse of the item pages and coding guides

An item page is a page with an item ID box right of x 540 and the "Difficulty" heading of the table at the foot (values at y 705-738, each assigned to the column whose left edge precedes it).
A coding-guide page is headed "K-14 Coding Guide" (or "... (Continued)").
Coding-guide headings are matched by their letters, without a final "s", since the guides also print "Incorrect Responses" and, in the text layer, "IncorrectResponse".

In [ ]:
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
import pymupdf
from IPython.display import display

from timss_1 import released_items as ri

# "Complete 1995 TIMSS Advanced mathematics set" on https://nces.ed.gov/timss/released-questions.asp
T95ADV_PDF_URL = "https://timss.bc.edu/timss1995i/TIMSSPDF/CitemAdM.pdf"
T95ADV_DATA_DIR = Path.cwd() / "data"
T95ADV_PDF_PATH = ri.download(T95ADV_PDF_URL, T95ADV_DATA_DIR / "raw" / "CitemAdM.pdf")
T95ADV_OUTPUT_PATH = T95ADV_DATA_DIR / "timssadv95_g12_math_released_items.jsonl"
t95adv_pdf = pymupdf.open(T95ADV_PDF_PATH)
# The red copyright watermark over the pages hides the text under it; remove it before the parse and the images.
print(f"Removed the watermark from {ri.remove_watermarks(t95adv_pdf)} pages")

T95ADV_DICT = pymupdf.TEXTFLAGS_DICT | pymupdf.TEXT_IGNORE_ACTUALTEXT
T95ADV_LABEL = re.compile(r"([A-Z])-(\d+)([a-z]?)")  # the item ID box, such as "K-1" or "L-15a"
T95ADV_LABEL_X = 540  # the box lies right of this
T95ADV_TABLE_Y = (705, 738)  # the row of values under "Subject  Item Key  Content Category ..."
T95ADV_TABLE_COLUMNS = {  # column -> left edge of its values
    "subject": 100,
    "key": 160,
    "content_category": 195,
    "performance_expectation": 295,
    "percent_correct": 395,
    "difficulty_index": 480,
}
T95ADV_TABLE_RIGHT = 545
T95ADV_CONTENT_DOMAINS = [
    "Numbers, Equations and Functions",
    "Calculus",
    "Geometry",
    "Probability & Statistics",
    "Validation and Structure",
]
T95ADV_COGNITIVE_DOMAINS = [
    "Knowing",
    "Routine Procedures",
    "Complex Procedures",
    "Solving Problems",
    "Justifying and Proving",
    "Communicating",
]
# Coding-guide categories, in printed order; a code's first digit is the score, except 7 (incorrect) and 9. The
# guides also print "Incorrect Responses" (L14) and, in the text layer, "IncorrectResponse", so headings are matched
# by their letters, without a final "s".
T95ADV_SCORING_CATEGORIES = ["Correct Response", "Partial Response", "Minimal Response", "Incorrect Response", "Nonresponse"]


def t95adv_heading_key(heading):
    return re.sub(r"[^a-z]", "", heading.casefold()).removesuffix("s")


T95ADV_CATEGORY_KEYS = {t95adv_heading_key(category): category for category in T95ADV_SCORING_CATEGORIES}


def t95adv_item_id(label):
    """"K-1" -> "K1", "L-15a" -> "L15A"."""
    match = T95ADV_LABEL.fullmatch(label)
    return f"{match[1]}{match[2]}{match[3].upper()}"


def t95adv_table(words):
    """The values under the column headings at the foot of an item page."""
    cells = defaultdict(list)
    edges = list(T95ADV_TABLE_COLUMNS.items())
    for w in sorted(words, key=lambda w: (round(w[1]), w[0])):
        if T95ADV_TABLE_Y[0] <= w[1] < T95ADV_TABLE_Y[1] and edges[0][1] <= w[0] < T95ADV_TABLE_RIGHT:
            column = next(name for name, x in reversed(edges) if w[0] >= x)
            cells[column].append(w[4])
    return {column: " ".join(cells[column]) for column in T95ADV_TABLE_COLUMNS}


def t95adv_spans(page, clip=None):
    """The text spans of a page, or of an area of it, as (x0, y0, x1, y1, text) tuples."""
    return [
        (*span["bbox"], " ".join(span["text"].split()))
        for block in page.get_text("dict", flags=T95ADV_DICT, clip=clip)["blocks"]
        for line in block.get("lines", [])
        for span in line["spans"]
        if span["text"].strip()
    ]


def t95adv_scoring_categories(page):
    """The category headings and the score codes of a coding-guide page, in printed order: a code is a two-digit
    number at the start of a row of the code table."""
    headings, codes = [], []
    for line in ri.lines(w for w in t95adv_spans(page) if 40 <= w[1] < 738):
        key = t95adv_heading_key(ri.text(line))
        if key in T95ADV_CATEGORY_KEYS:
            headings.append(T95ADV_CATEGORY_KEYS[key])
        elif re.fullmatch(r"\d{2}", line[0][4]) and line[0][0] < 160:
            codes.append((headings[-1] if headings else None, int(line[0][4])))
    return headings, codes


t95adv_items, t95adv_guides = [], defaultdict(list)
for page in t95adv_pdf:
    words = page.get_text("words", flags=ri.VISIBLE_WORDS)
    page_text = page.get_text()
    guide = re.search(r"([A-Z]-\d+[a-z]?) Coding Guide", page_text)
    if guide:
        t95adv_guides[t95adv_item_id(guide[1])].append(page.number)
        continue
    labels = [w for w in words if w[0] >= T95ADV_LABEL_X and T95ADV_LABEL.fullmatch(w[4])]
    if not labels or not any(w[4] == "Difficulty" for w in words):
        continue  # front matter
    item_id = t95adv_item_id(labels[0][4])
    table = t95adv_table(words)
    assert table["subject"] == "Advanced Mathematics", (item_id, table)
    content = " ".join(table["content_category"].split())
    assert content in T95ADV_CONTENT_DOMAINS, (item_id, content)
    assert table["performance_expectation"] in T95ADV_COGNITIVE_DOMAINS, (item_id, table)
    is_multiple_choice = table["key"] in set("ABCDE")
    assert is_multiple_choice or table["key"] == "next page", (item_id, table)
    reproduced_y = next(w[1] for w in words if w[4] == "Reproduced")
    clip = pymupdf.Rect(40, 40, T95ADV_LABEL_X, reproduced_y - 2)
    t95adv_items.append(
        {
            "item_id": item_id,
            "content_domain": content,
            "cognitive_domain": table["performance_expectation"],
            "item_type": "multiple_choice" if is_multiple_choice else "constructed_response",
            "correct_option": table["key"] if is_multiple_choice else None,
            "pct_correct_intl_avg": int(table["percent_correct"].rstrip("%")),
            "difficulty_index": int(table["difficulty_index"]),
            "booklet_page": int(max((w for w in words if w[1] > 738), key=lambda w: w[0])[4]),
            "pdf_page": page.number + 1,
            "item_clip": (page.number, clip),
            "_item_text": ri.text(w for line in ri.lines(t95adv_spans(page, clip)) for w in line),
        }
    )

# Each constructed-response item is followed by its coding guide, one or two pages; its correct-response codes give
# the maximum points (10 for 1 point, 20 for 2, 30 for 3).
for item in t95adv_items:
    item["guide_pages"] = t95adv_guides.get(item["item_id"], [])
    headings, codes = [], []
    for number in item["guide_pages"]:
        page_headings, page_codes = t95adv_scoring_categories(t95adv_pdf[number])
        headings += page_headings
        codes += [(category or (headings[-1] if headings else None), code) for category, code in page_codes]
    item["scoring_categories"] = headings
    if item["item_type"] == "multiple_choice":
        assert not item["guide_pages"], f"{item['item_id']}: multiple-choice item with a coding guide"
        item["max_points"] = 1
    else:
        assert item["guide_pages"], f"{item['item_id']}: no coding guide"
        correct = {code // 10 for category, code in codes if category == "Correct Response"}
        assert len(correct) == 1, f"{item['item_id']}: correct-response codes {codes}"
        item["max_points"] = correct.pop()
        assert headings[0] == "Correct Response" and headings[-1] == "Nonresponse", (item["item_id"], headings)
assert set(t95adv_guides) <= {item["item_id"] for item in t95adv_items}, "coding guides of unknown items"

t95adv_items_df = pd.DataFrame(t95adv_items)
print(f"Parsed {len(t95adv_items)} items")
display(t95adv_items_df.groupby(["content_domain", "item_type"]).size().unstack(fill_value=0))
display(t95adv_items_df.groupby(["item_type", "max_points"]).size())

### Transcribe stems, options, and coding guides with Gemini

`gemini-3.8-flash` is called through OpenRouter by `ri.Transcriber`, as in the 2011 grade 8 section, with up to `T95ADV_ITEMS_PER_REQUEST` items per request in booklet order, which makes 10 requests for the whole set.
For each item it sends the item page and its coding-guide pages, rendered at 300 DPI.

A transcription is kept only if it passes the checks in `t95adv_problems`: options A-D or A-E that include the answer key for a multiple-choice item, the coding-guide categories and the points of the correct-response codes for a constructed-response item, and every printed word of the item in its stem or options or, for a figure label, its figure description.
The word check catches what the others cannot: when the images were made by whitening the watermark's pixels, which also whitened the text under it, the model filled in the hidden text, turning option B of L4 ("11C8 = 165") into "11C9 = 55" and option D of L11 ("none need have told the truth") into "none of the others told the truth".
Kept transcriptions are cached per item under `data/cache/item_transcriptions/timssadv95_math/gemini-3.8-flash-v2/`; set `T95ADV_MAX_REQUESTS` to send fewer requests, and bump `T95ADV_PROMPT_VERSION` after editing the prompt, the schema, or the images.

In [ ]:
from pydantic import BaseModel, Field

T95ADV_LLM_MODEL = "google/gemini-3.8-flash"
T95ADV_PROMPT_VERSION = 2  # 2: images without the watermark instead of with its pixels whitened
T95ADV_LLM_CACHE_DIR = (
    T95ADV_DATA_DIR
    / "cache"
    / "item_transcriptions"
    / "timssadv95_math"
    / f"{T95ADV_LLM_MODEL.split('/')[-1]}-v{T95ADV_PROMPT_VERSION}"
)
T95ADV_ITEMS_PER_REQUEST = 4
T95ADV_MAX_REQUESTS = None  # a number sends at most that many requests in one run, such as 1 for a test
T95ADV_GUIDE_CLIP = pymupdf.Rect(40, 40, 580, 738)  # a coding-guide page without its page number


class T95AdvOption(BaseModel):
    label: str = Field(description="Option letter as printed, e.g. 'A'.")
    text: str = Field(description="Option content in the same notation as the stem.")


class T95AdvScoringCode(BaseModel):
    code: int = Field(description="Score code as printed, e.g. 10.")
    response: str = Field(description="The code's response description, verbatim.")


class T95AdvScoringCategory(BaseModel):
    category: str = Field(description="Category heading as printed, e.g. 'Correct Response'.")
    codes: list[T95AdvScoringCode] = Field(description="Each code row under the heading, in printed order.")


class T95AdvTranscription(BaseModel):
    item_id: str
    stem: str
    options: list[T95AdvOption]
    scoring_guide: list[T95AdvScoringCategory]
    scoring_notes: str | None
    has_figure: bool
    figure_description: str | None


T95ADV_TRANSCRIPTION_INSTRUCTIONS = """\
You transcribe items from the TIMSS 1995 released advanced mathematics items (final year of secondary school) into \
structured data.

Each request covers several items, listed by ID. For each item you receive:
1. The item as printed, as an image (a red copyright watermark has been removed from it).
2. For a constructed-response item, its coding guide, one or two pages: a table of score codes under category \
headings, after a small copy of the item.

The images are the only source. Transcribe what is printed, verbatim; never add, correct, or solve anything.

Return one entry per item, in the order given:
- item_id: The item ID as given.
- stem: Everything the student sees for this item except the multiple-choice response options, in reading order, \
without the item number that starts it (such as "K3."). Write math in plain-text notation: fractions as a/b (with \
parentheses where needed), powers as x^2, roots as sqrt(x), functions as f(x), derivatives as f'(x), integrals as \
integral from a to b of f(x) dx, limits as lim_(x->a) f(x), vectors as vector AB, and the symbols ×, ÷, −, =, <, >, \
≤, ≥, ≠, ∞, π, °, ∠. Write an answer blank as ____. Write tables as Markdown tables. Replace each graph, diagram, or \
other picture with "[Image: <short description>]". Separate paragraphs with a newline.
- options: For a multiple-choice item, one entry per response option in printed order, labeled with its letter, in \
the same notation as the stem; a pictured option is "[Image: <description>]". Empty for constructed-response items.
- scoring_guide: For a constructed-response item, each category of the coding guide in printed order (e.g. "Correct \
Response", "Partial Response", "Minimal Response", "Incorrect Response", "Nonresponse"), with its heading verbatim, \
and each code row under it: the score code and its response description, verbatim (including any method, note, or \
example in the row), in the same notation as the stem. Leave out the small copy of the item. Empty for \
multiple-choice items.
- scoring_notes: Any coding-guide text outside the code table, such as a note or a heading for one part of the \
item; null if none.
- has_figure: Whether this item's stem or options contain any graph, diagram, or other picture. A table of text or \
numbers alone does not count.
- figure_description: If has_figure, a precise, objective description of every visual element in this item's stem \
and options, giving all the information a student would need to answer without seeing it: axes, scales, labels, \
coordinates of marked points, shapes, and measurements. Describe what is drawn; do not solve the item or reveal the \
answer. Null if has_figure is false.
"""


def t95adv_request_content(batch):
    """The user message for a batch: each item's picture and its coding-guide pages."""
    content = [ri.text_part(f"Transcribe these {len(batch)} items, in this order: {', '.join(i['item_id'] for i in batch)}.")]
    for item in batch:
        page_index, clip = item["item_clip"]
        kind = item["item_type"].replace("_", "-")
        content += [
            ri.text_part(f"Item {item['item_id']}: {kind}, worth {item['max_points']} point{'s' if item['max_points'] > 1 else ''}."),
            ri.page_area_part(t95adv_pdf[page_index], clip),
        ]
        for number, page_number in enumerate(item["guide_pages"], 1):
            content.append(ri.text_part(f"Coding guide of item {item['item_id']}, page {number}:"))
            content.append(ri.page_area_part(t95adv_pdf[page_number], T95ADV_GUIDE_CLIP))
    return content


def t95adv_problems(item, t, returned):
    """Disagreements between a transcription and the deterministic parse; empty if the transcription is usable."""
    problems = []
    if not t.stem.strip():
        problems.append("empty stem")
    labels = [o.label for o in t.options]
    if item["item_type"] == "multiple_choice":
        if labels not in (list("ABCD"), list("ABCDE")) or item["correct_option"] not in labels:
            problems.append(f"option labels {labels} for key {item['correct_option']}")
        if t.scoring_guide:
            problems.append("multiple-choice item has a scoring guide")
    else:
        if t.options:
            problems.append("constructed-response item has options")
        categories = [T95ADV_CATEGORY_KEYS.get(t95adv_heading_key(c.category)) for c in t.scoring_guide]
        if categories != item["scoring_categories"]:
            problems.append(f"scoring categories {[c.category for c in t.scoring_guide]}, expected {item['scoring_categories']}")
        correct = {code.code // 10 for c in t.scoring_guide if c.category == "Correct Response" for code in c.codes}
        if correct != {item["max_points"]}:
            problems.append(f"correct-response codes for {correct} points, expected {item['max_points']}")
    if t.has_figure != bool(t.figure_description):
        problems.append(f"has_figure={t.has_figure} but figure_description={t.figure_description!r}")
    # The item text starts with its number, and figure labels may be only in the figure description.
    transcribed = " ".join([item["item_id"], t.stem, *(o.text for o in t.options), t.figure_description or ""])
    missing = ri.coverage({item["item_id"]: item["_item_text"]}, {item["item_id"]: transcribed})
    if not missing.empty:
        problems.append(f"printed words missing from the transcription: {missing['missing_words'].iloc[0]}")
    return problems


t95adv_transcriber = ri.Transcriber(
    api_key=globals().get("openrouter_api_key"),
    model=T95ADV_LLM_MODEL,
    cache_dir=T95ADV_LLM_CACHE_DIR,
    schema=T95AdvTranscription,
    instructions=T95ADV_TRANSCRIPTION_INSTRUCTIONS,
)
t95adv_transcriptions = t95adv_transcriber.run(
    t95adv_items,
    request_content=t95adv_request_content,
    problems=t95adv_problems,
    items_per_request=T95ADV_ITEMS_PER_REQUEST,
    max_requests=T95ADV_MAX_REQUESTS,
)
print(f"All {len(t95adv_transcriptions)} items are transcribed")

### Write the JSONL

In [ ]:
t95adv_records = []
for item in t95adv_items:
    t = t95adv_transcriptions[item["item_id"]]
    t95adv_records.append(
        {
            "item_id": item["item_id"],
            "item_label": None,  # the 1995 booklet gives items no labels
            "assessment": "TIMSS Advanced 1995",
            "grade": 12,  # final year of secondary school
            "subject": "Advanced Mathematics",
            "content_domain": item["content_domain"],
            "main_topic": None,
            "cognitive_domain": item["cognitive_domain"],
            "item_type": item["item_type"],
            "max_points": item["max_points"],
            "stem": ri.tidy(t.stem),
            "options": [{"label": o.label, "text": ri.tidy(o.text)} for o in t.options] or None,
            "correct_option": item["correct_option"],
            # The response descriptions under each heading; the two-digit score codes were used for the checks.
            "scoring_guide": [
                {"category": c.category, "descriptors": [ri.tidy(code.response) for code in c.codes]}
                for c in t.scoring_guide
            ]
            or None,
            "scoring_notes": t.scoring_notes,
            "has_figure": t.has_figure,
            "figure_description": t.figure_description,
            "accessibility_text": None,
            "pct_correct_intl_avg": item["pct_correct_intl_avg"],
            "pct_correct_usa": None,  # the booklet gives the international average only
            "pct_correct": None,
            "booklet_page": item["booklet_page"],
            "pdf_page": item["pdf_page"],
            "source_url": T95ADV_PDF_URL,
            "transcription_model": T95ADV_LLM_MODEL,
        }
    )

t95adv_released_items = ri.write_jsonl(t95adv_records, T95ADV_OUTPUT_PATH)
print(f"Wrote {len(t95adv_released_items)} items to {T95ADV_OUTPUT_PATH.resolve()}")
display(t95adv_released_items[["item_id", "content_domain", "cognitive_domain", "item_type", "max_points", "stem", "options"]].head())

## Combine the released mathematics items

Concatenate the per-set files (`data/timssYY_gG_math_released_items.jsonl` and, for TIMSS Advanced, `data/timssadvYY_g12_math_released_items.jsonl`) into `data/timss_math_released_items.jsonl`, one item per line, oldest assessment first, then by grade, each file in its own item order.
The grade 4 items alone also go to `data/timss_g4_math_released_items.jsonl`, as before the other grades were added.
The `assessment` and `grade` fields tell the sets apart.
The cell checks that every file has the fields of the 2011 grade 4 file, in the same order (`ri.RECORD_FIELDS`), that each file holds a single assessment and grade, and that no (assessment, grade, item ID) repeats.
Item IDs alone are not unique: the short IDs of the 1995 and 1999 booklets (such as K1 and L10) name different items in different sets, and the cell lists them.
The table at the end counts, per set, the items with each field filled, since some booklets lack some fields (for example, the 1995 and 1999 frameworks have no main topics, and only the 2011 booklets have accessibility text).

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

from timss_1 import released_items as ri

COMBINED_DATA_DIR = Path.cwd() / "data"
# One file per assessment and grade, such as timss11_g4_math_released_items.jsonl or
# timssadv95_g12_math_released_items.jsonl.
COMBINED_FILE_NAME = re.compile(r"timss(?:adv)?\d{2}_g\d+_math_released_items\.jsonl")
COMBINED_OUTPUT_PATH = COMBINED_DATA_DIR / "timss_math_released_items.jsonl"
COMBINED_G4_OUTPUT_PATH = COMBINED_DATA_DIR / "timss_g4_math_released_items.jsonl"
COMBINED_KEY = ["assessment", "grade", "item_id"]


def combined_read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def combined_write_jsonl(records, path):
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


combined_files = {}  # (year, grade, assessment) -> (path, records)
for path in sorted(p for p in COMBINED_DATA_DIR.glob("timss*.jsonl") if COMBINED_FILE_NAME.fullmatch(p.name)):
    records = combined_read_jsonl(path)
    sets = {(record["assessment"], record["grade"]) for record in records}
    assert records and len(sets) == 1, f"{path.name}: assessments and grades {sets}"
    assert all(list(record) == ri.RECORD_FIELDS for record in records), f"{path.name}: fields differ from the 2011 file"
    assessment, grade = sets.pop()
    key = (int(assessment.split()[-1]), grade, assessment)  # the year ends "TIMSS 1995" and "TIMSS Advanced 2008"
    assert key not in combined_files, f"{path.name} and {combined_files[key][0].name} are both {assessment} grade {grade}"
    combined_files[key] = (path, records)

# Oldest assessment first, then by grade.
combined_files = dict(sorted(combined_files.items()))
combined_records = [record for _, records in combined_files.values() for record in records]
combined_keys = pd.DataFrame([{field: record[field] for field in COMBINED_KEY} for record in combined_records])
assert not combined_keys.duplicated().any(), f"repeated records: {combined_keys[combined_keys.duplicated()].to_dict('records')}"
combined_write_jsonl(combined_records, COMBINED_OUTPUT_PATH)
# The grade 4 items alone, as before the other grades were added.
combined_write_jsonl([record for record in combined_records if record["grade"] == 4], COMBINED_G4_OUTPUT_PATH)

combined_items = pd.read_json(COMBINED_OUTPUT_PATH, lines=True)
print(f"Wrote {len(combined_items)} items to {COMBINED_OUTPUT_PATH.resolve()} from:")
for path, records in combined_files.values():
    print(f"  {path.name} ({len(records)})")
print(f"and the {sum(combined_items['grade'] == 4)} grade 4 items to {COMBINED_G4_OUTPUT_PATH.resolve()}")
# Short IDs, such as K1 in TIMSS 1995 and in TIMSS Advanced 1995, name different items in different sets.
combined_sets = combined_items.groupby("item_id")["assessment"].agg(" and ".join)
combined_sets = combined_sets[combined_items["item_id"].value_counts()[combined_sets.index] > 1]
print("Item IDs in more than one set:", {sets: ids.index.tolist() for sets, ids in combined_sets.groupby(combined_sets)})
display(
    combined_items.groupby(["assessment", "grade", "item_type"], sort=False).size().unstack(fill_value=0)
)
print("Items with each field filled:")
display(
    combined_items.groupby(["assessment", "grade"], sort=False)[
        [
            "item_label",
            "main_topic",
            "options",
            "scoring_guide",
            "figure_description",
            "accessibility_text",
            "pct_correct_intl_avg",
            "pct_correct",
        ]
    ].count()
)

## Item parameters of the released mathematics items

Find the IRT item parameters of every item in `data/timss_math_released_items.jsonl` in official TIMSS documents, and export one row per item, with the item's metadata, to `data/timss_math_released_item_parameters.xlsx`, and the grade 4 items alone to `data/timss_g4_math_released_item_parameters.xlsx`.
The parameters are the slope (aj), location (bj), guessing (cj), and step parameters (dj1, dj2, and, for 3-point TIMSS Advanced items, dj3) of the IRT models TIMSS uses: the 3PL model for multiple-choice items, the 2PL model for 1-point constructed-response items, and the generalized partial credit model (GPCM) for items worth 2 or 3 points.
Each parameter's standard error is kept as well (`*_se`).
All sources are on the sites of the TIMSS & PIRLS International Study Center (timssandpirls.bc.edu and timss.bc.edu), and the cells download them to `data/raw/` unless they are already there.

How the parameters were found, by assessment:

- **TIMSS 2011 (73 grade 4 and 88 grade 8 items).** The [TIMSS 2011 International Database](https://timssandpirls.bc.edu/timss2011/international-database.html) offers `T11_ItemParameters.zip`. The `MAT` sheets of its `T11_G4_ItemParameters.xlsx` and `T11_G8_ItemParameters.xlsx` list the parameters from the concurrent calibration of the 2007 and 2011 data, and every released item is in them.
- **TIMSS 2007 (74 grade 4 and 89 grade 8 items).** The "Items" link on the [TIMSS 2007 International Database and User Guide](https://timssandpirls.bc.edu/TIMSS2007/idb_ug.html) page downloads `T07_Items.zip`. The `G4-MAT` and `G8-MAT` sheets of its `T07_ItemParameters_Overall.xls` list the parameters from the concurrent calibration of the 2003 data, the 2007 bridging data, and the 2007 data. Their rows with IDs starting `MZ` are the items in the 2007 assessment booklets (MZ31286 is M031286), and every released item is among them except M042273, which the grade 8 item information marks as not used in scaling. The other rows are the items of the 2003 booklets under their 2003 IDs (`M0`, `MF`, and, at grade 8, five `MC` rows). The 2011 `MAT` sheets also list the items released in 2007, with parameters from the 2011 calibration; those are not used.
- **TIMSS 2003 (79 grade 4 and 99 grade 8 items).** The [TIMSS 2003 International Database](https://timssandpirls.bc.edu/timss2003i/userguide.html) has no parameter file; its `t03_items.zip` holds item information only (`Aitinfm3.xls` for grade 4 and `Bitinfm3.xls` for grade 8). The parameters are published in Appendix D, "Item Parameters for IRT Analyses of TIMSS 2003 Data", of the [TIMSS 2003 Technical Report](https://timssandpirls.bc.edu/timss2003i/technicalD.html) (`T03_TR_AppD.pdf`): Exhibit D.3, "IRT Parameters for TIMSS Joint 1995-2003 Fourth-Grade Mathematics", and Exhibit D.1, "IRT Parameters for TIMSS Joint 1999-2003 Eighth-Grade Mathematics", hold the overall mathematics parameters (the later exhibits are for the content-domain scales). They list many items twice: the `M0` row is the item in its regular block positions, and the `MF` row the item in the third and sixth block positions, which the calibration scaled as a separate item because many students did not reach those positions (Technical Report, chapter 11, section 11.3.1). The `M0` rows are used, and every released item is among them.
- **TIMSS 1999 (82 grade 8 items).** The [TIMSS 1999 International Database](https://timssandpirls.bc.edu/timss1999i/database.html) has no parameter file. Because the 2003 calibration included the 1999 data, Exhibit D.1 also lists the released 1999 items, under their permanent IDs (such as M012008 and M022262A) instead of the booklet IDs (B08, V04A); the achievement codebook of the 1999 database, `BSACBKM2.CDT` in `bm2_cdbks.zip`, gives the permanent ID of every booklet position. The 1999 [Technical Report](https://timss.bc.edu/timss1999i/tech_report.html) has its own exhibit, Exhibit E.3 of Appendix E (`T99_TR_AppenE.pdf`, joint 1995-1999 calibration), but it lists no items between M022070 and M022165 and so lacks 17 released items. Exhibit D.1 is used for all 82, so that they share one calibration, and Exhibit E.3 serves as a check.
- **TIMSS 1995 (70 grade 4 items).** The [TIMSS 1995 International Database](https://timssandpirls.bc.edu/timss1995i/database.html) has no parameter file, and the 1995 scaling used a different model. But because the 2003 calibration included the 1995 data, Exhibit D.3 also lists the 70 released 1995 items, under 2003-style IDs instead of the 1995 IDs (cluster letter and item number, such as I3 and V4A) of the released-items booklet. No official crosswalk between the two was found, so it is derived from official documents and checked against the parameters (see below).
- **TIMSS Advanced 2008 and 1995 (40 and 37 items).** `TA08_Items.zip`, the "Items" download of the [TIMSS Advanced 2008 International Database](https://timssandpirls.bc.edu/timss_advanced/idb.html), holds `TA08_Item_Parameters.xls`, whose `Adv. Math` sheet lists the parameters from the concurrent scaling of the 1995 and 2008 data (Exhibit C.3 of Appendix C of the [TIMSS Advanced 2008 Technical Report](https://timssandpirls.bc.edu/timss_advanced/tr.html); Exhibit C.1 has parameters from a rescaling of the 1995 data alone, not used). It lists the 2008 items under their IDs and 2008 block positions, and the 1995 released items under IDs that the 1995 booklet does not print, with their 1995 positions (K01 for item K1). Six 1995 items (K8, K9, L10, L11, L15A, and L15B) are not in it: the rescaling of the 1995 data left out 11 of the 68 advanced mathematics items of 1995, 10 of them because they did not fit the 2008 framework (Technical Report, section 8.4.1). K14 and L18, 2-point items in the 1995 coding guides, have 1-point (2PL) parameters: the Technical Report (section 8.4.2) counts the 31 scaled 1995 released items as worth 40 score points, which holds only with K14 and L18 at 1 point.

Notes:

- Parameters from different calibrations are on different scales, so they are not directly comparable across them: the 1995 and 2003 grade 4 items share one calibration, the 1999 and 2003 grade 8 items another, the 1995 and 2008 TIMSS Advanced items a third, and the 2007 and the 2011 items of each grade have their own. Each database documents the transformation of its calibration to the reporting scale (for example `T11_G4_Transform.xlsx` in `T11_ItemParameters.zip`).
- Exhibits D.1, D.3, and E.3 print a guessing parameter of 0.000 with a standard error of 0.000 for constructed-response items, which the calibration did not estimate; it is left blank, as in the workbooks.
- `irt_model` follows from which parameters are present, and the last cell checks it against each item's type and maximum points, checks that a GPCM item has one step per point and that its steps add up to 0, and reports the rank correlation of the location with the international average percent correct in each set.
- The workbook has four sheets: `item_parameters` (one row per item, with the source of its parameters and a `note` on items without parameters or scaled with fewer points), `sources` (this documentation, by assessment and grade, with the correlations), `crosswalk_1995_g4`, and `crosswalk_1999_g8`. The grade 4 workbook has the grade 4 rows and `crosswalk_1995_g4`.

In [ ]:
import io
import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf
from IPython.display import display

from timss_1 import released_items as ri

IP_DATA_DIR = Path.cwd() / "data"
IP_RAW_DIR = IP_DATA_DIR / "raw"
IP_RELEASED_ITEMS_PATH = IP_DATA_DIR / "timss_math_released_items.jsonl"
IP_OUTPUT_PATH = IP_DATA_DIR / "timss_math_released_item_parameters.xlsx"
IP_G4_OUTPUT_PATH = IP_DATA_DIR / "timss_g4_math_released_item_parameters.xlsx"  # the grade 4 items alone
IP_KEY = ["assessment", "grade", "item_id"]  # identifies a released item; short IDs such as L10 repeat across sets
# Official files: local name -> (download URL, the page that links to it).
IP_SOURCES = {
    "T11_ItemParameters.zip": (
        "https://timssandpirls.bc.edu/timss2011/downloads/T11_ItemParameters.zip",
        "https://timssandpirls.bc.edu/timss2011/international-database.html",
    ),
    "T07_Items.zip": (
        "https://timssandpirls.bc.edu/TIMSS2007/PDF/T07_Items.zip",
        "https://timssandpirls.bc.edu/TIMSS2007/idb_ug.html",
    ),
    "T03_TR_AppD.pdf": (
        "https://timssandpirls.bc.edu/PDF/t03_download/T03_TR_AppD.pdf",
        "https://timssandpirls.bc.edu/timss2003i/technicalD.html",
    ),
    "t03_items.zip": (
        "https://timssandpirls.bc.edu/timss2003i/PDF/t03_items.zip",
        "https://timssandpirls.bc.edu/timss2003i/userguide.html",
    ),
    "T99_TR_AppenE.pdf": (
        "https://timss.bc.edu/timss1999i/pdf/T99_TR_AppenE.pdf",
        "https://timss.bc.edu/timss1999i/tech_report.html",
    ),
    "T99_bm2_cdbks.zip": (
        "https://timssandpirls.bc.edu/timss1999i/data/bm2_cdbks.zip",
        "https://timssandpirls.bc.edu/timss1999i/database.html",
    ),
    "T95_UG_1and2.pdf": (
        "https://timssandpirls.bc.edu/timss1995i/database/UG_1and2.pdf",
        "https://timssandpirls.bc.edu/timss1995i/database.html",
    ),
    "T95_BMItems.pdf": (
        "https://timssandpirls.bc.edu/timss1995i/TIMSSPDF/BMItems.pdf",
        "https://timssandpirls.bc.edu/timss1995i/Items.html",
    ),
    "TA08_Items.zip": (
        "https://timss.bc.edu/timss_advanced/downloads/TA08_Items.zip",
        "https://timssandpirls.bc.edu/timss_advanced/idb.html",
    ),
}
# Sheet heading -> column name. Each parameter spans two columns: the estimate and its standard error.
IP_PARAMETER_HEADINGS = {
    "Slope (aj)": "slope",
    "Location (bj)": "location",
    "Guessing (cj)": "guessing",
    "Step 1 (dj1)": "step1",
    "Step 2 (dj2)": "step2",
    "Step 3 (dj3)": "step3",  # 3-point items, in TIMSS Advanced only
}
IP_PARAMETERS = list(IP_PARAMETER_HEADINGS.values())
IP_STEPS = [name for name in IP_PARAMETERS if name.startswith("step")]
IP_VALUE_COLUMNS = [column for name in IP_PARAMETERS for column in (name, f"{name}_se")]


def ip_download(name):
    """The local copy of an official file in data/raw, downloaded first if it is not there yet."""
    return ri.download(IP_SOURCES[name][0], IP_RAW_DIR / name)


def ip_zip_member(name, member):
    """A file inside a downloaded zip, as an in-memory file for pandas."""
    with zipfile.ZipFile(ip_download(name)) as archive:
        return io.BytesIO(archive.read(member))


def ip_released_ids(assessment, grade):
    """The IDs of the released items of one assessment and grade."""
    return set(ip_released.loc[(ip_released["assessment"] == assessment) & (ip_released["grade"] == grade), "item_id"])


with IP_RELEASED_ITEMS_PATH.open(encoding="utf-8") as f:
    ip_released = pd.DataFrame([json.loads(line) for line in f])
assert not ip_released.duplicated(IP_KEY).any(), "released items repeat"
for name in IP_SOURCES:
    print(f"{name}: {ip_download(name).stat().st_size / 1e6:.1f} MB")
display(ip_released.groupby(["assessment", "grade"], sort=False).size().rename("items").to_frame())

### TIMSS 2011, 2007, and TIMSS Advanced: the item-parameter workbooks

Each sheet has a title row, a heading row, and one row per item with the estimate and standard error of each parameter; the 2011 sheets also have group rows such as "Items Introduced in 2011:".
The 2011 sheets give the block position first and the item ID second, and the 2007 and TIMSS Advanced sheets the other way around.
The TIMSS Advanced sheet has a third step parameter; `ip_parameter_sheet` reads whichever parameters a sheet has, in the order of `IP_PARAMETER_HEADINGS`.
The cell also lists the released 2007 items that the item information marks as not used in scaling.

In [ ]:
def ip_parameter_sheet(workbook, sheet_name, id_column):
    """The item rows of a TIMSS item-parameter sheet: the item ID as listed, the block position, and each parameter's
    estimate and standard error. Row 1 is the title and row 2 the headings, the first parameters of
    IP_PARAMETER_HEADINGS in order; group rows, such as "Items Introduced in 2011:", fill only the first column. The
    ID is in column `id_column` (0 or 1) and the position in the other."""
    sheet = pd.read_excel(workbook, sheet_name=sheet_name, header=None)
    headings = sheet.iloc[1].tolist()
    parameters = [IP_PARAMETER_HEADINGS.get(heading) for heading in headings[2::2]]
    if headings[0] != "Item" or parameters != IP_PARAMETERS[: len(parameters)] or sheet.shape[1] != 2 + 2 * len(parameters):
        raise ValueError(f"Unexpected headings in the {sheet_name} sheet: {headings}")
    body = sheet.iloc[2:].dropna(how="all")
    body = body[body[1].notna()]
    values = {}
    for k, name in enumerate(parameters):
        values[name] = pd.to_numeric(body[2 + 2 * k])
        values[f"{name}_se"] = pd.to_numeric(body[3 + 2 * k])
    rows = pd.DataFrame(
        {
            "source_item_id": body[id_column].astype(str).str.strip(),
            "block_position": body[1 - id_column].astype(str).str.strip(),
            **values,
        }
    )
    return rows.reindex(columns=["source_item_id", "block_position", *IP_VALUE_COLUMNS]).reset_index(drop=True)


ip_workbook_sets = []

# TIMSS 2011: every item of the 2007-2011 calibration, listed under its ID.
for grade in [4, 8]:
    sheet = ip_parameter_sheet(
        ip_zip_member("T11_ItemParameters.zip", f"T11_G{grade}_ItemParameters.xlsx"), "MAT", id_column=1
    )
    ip_workbook_sets.append(
        sheet.assign(
            assessment="TIMSS 2011",
            grade=grade,
            item_id=sheet["source_item_id"],
            source_file=f"T11_ItemParameters.zip > T11_G{grade}_ItemParameters.xlsx",
            source_location="sheet MAT",
        )
    )

# TIMSS 2007: the MZ rows are the items in the 2007 assessment booklets (MZ31286 is M031286). The other rows are the
# items in the 2003 booklets, from the 2003 data and the 2007 bridging study, under their 2003 unique IDs: M0, MF
# (copies in block positions 3 and 6), and, at grade 8, MC (five items that the 2003 item information lists twice).
for grade in [4, 8]:
    sheet = ip_parameter_sheet(
        ip_zip_member("T07_Items.zip", "T07_Items/T07_ItemParameters_Overall.xls"), f"G{grade}-MAT", id_column=0
    )
    assert sheet["source_item_id"].str.fullmatch(r"M[0FZC]\d{5}[A-Z]?").all(), f"unexpected IDs in the G{grade}-MAT sheet"
    sheet = sheet[sheet["source_item_id"].str.startswith("MZ")]
    ip_workbook_sets.append(
        sheet.assign(
            assessment="TIMSS 2007",
            grade=grade,
            item_id="M0" + sheet["source_item_id"].str[2:],
            source_file="T07_Items.zip > T07_Items/T07_ItemParameters_Overall.xls",
            source_location=f"sheet G{grade}-MAT, rows with MZ IDs",
        )
    )
# Released items that the 2007 item information marks as not used in scaling, which have no parameters.
ip_t07_unscaled = {
    grade: set(information.index[information["In Scaling"] == "No"]) & ip_released_ids("TIMSS 2007", grade)
    for grade in [4, 8]
    for information in [ri.timss_2007_math_item_information(ip_download("T07_Items.zip"), grade)]
}

# TIMSS Advanced: the 1995-2008 concurrent scaling lists the 2008 items under their IDs and 2008 block positions
# (such as M1_01), and the items given only in 1995 under IDs that the 1995 booklet does not print, with their 1995
# positions (K01 for item K1, the first item of block K).
ip_adv = ip_parameter_sheet(
    ip_zip_member("TA08_Items.zip", "TA08_Items/TA08_Item_Parameters.xls"), "Adv. Math", id_column=0
)
ip_adv_1995 = ip_adv["block_position"].str.fullmatch(r"[KL]\d{2}[A-Z]?")
assert ip_adv.loc[~ip_adv_1995, "block_position"].str.fullmatch(r"M\d_\d{2}[A-Z]?").all(), "unexpected positions"
ip_workbook_sets += [
    ip_adv[~ip_adv_1995].assign(
        assessment="TIMSS Advanced 2008",
        grade=12,
        item_id=ip_adv.loc[~ip_adv_1995, "source_item_id"],
        source_file="TA08_Items.zip > TA08_Items/TA08_Item_Parameters.xls",
        source_location="sheet Adv. Math",
    ),
    ip_adv[ip_adv_1995].assign(
        assessment="TIMSS Advanced 1995",
        grade=12,
        item_id=[
            f"{position[0]}{int(position[1:3])}{position[3:]}" for position in ip_adv.loc[ip_adv_1995, "block_position"]
        ],
        source_file="TA08_Items.zip > TA08_Items/TA08_Item_Parameters.xls",
        source_location="sheet Adv. Math, by the 1995 block position",
    ),
]

for sheet in ip_workbook_sets:
    assessment, grade = sheet["assessment"].iloc[0], sheet["grade"].iloc[0]
    assert sheet["item_id"].is_unique, f"{assessment} grade {grade}: duplicate item IDs"
    released = ip_released_ids(assessment, grade)
    missing = sorted(released - set(sheet["item_id"]))
    print(
        f"{assessment} grade {grade} ({sheet['source_location'].iloc[0]}): {len(sheet)} items, "
        f"{len(released) - len(missing)} of the {len(released)} released items; not listed: {missing or 'none'}"
    )
print("Released TIMSS 2007 items not used in scaling:", {grade: sorted(ids) for grade, ids in ip_t07_unscaled.items()})

### TIMSS 2003: Exhibits D.3 and D.1 of the technical report

Each page of an exhibit is a table with the item ID and ten value columns (each parameter and its standard error), and a parameter that does not apply to an item is not printed.
The row of subheadings under the column headings ("(aj) (aj) (bj) (bj) ...") names each value column once, so each value is assigned to the column whose subheading is nearest horizontally.
The cell checks that every mathematics item of the grade that the 2003 item information marks as used in scaling is in the exhibit.
Each exhibit also lists items that are not in the 2003 assessment: the released 1995 items (Exhibit D.3) and the 1999 items that 2003 did not reuse (Exhibit D.1).

In [ ]:
# Exhibits of the TIMSS 2003 Technical Report, Appendix D, with the overall mathematics parameters, by grade.
IP_D_EXHIBITS = {
    4: ("Exhibit D.3", "IRT Parameters for TIMSS Joint 1995-2003 Fourth-Grade Mathematics"),
    8: ("Exhibit D.1", "IRT Parameters for TIMSS Joint 1999-2003 Eighth-Grade Mathematics"),
}
IP_EXHIBIT_ITEM_ID = re.compile(r"M[0FC]\d{5}[A-Z]?")
IP_SUBHEADINGS = ["(aj)", "(bj)", "(cj)", "(dj1)", "(dj2)"]  # printed under each parameter and its standard error
IP_D_VALUE_COLUMNS = IP_VALUE_COLUMNS[: 2 * len(IP_SUBHEADINGS)]  # no third step


def ip_exhibit_rows(page):
    """The item rows on one page of an Appendix D exhibit. The row of subheadings, "(aj) (aj) (bj) (bj) ...", names
    each of the ten value columns once, and every value goes to the column whose subheading is nearest horizontally;
    a parameter that does not apply to the item is not printed."""
    words = page.get_text("words")
    subheadings = sorted((w for w in words if w[4] in IP_SUBHEADINGS), key=lambda w: w[0])
    assert len(subheadings) == len(IP_D_VALUE_COLUMNS), f"PDF page {page.number + 1}: {len(subheadings)} subheadings"
    centers = [(w[0] + w[2]) / 2 for w in subheadings]
    report_page = int(re.search(r"BOSTON COLLEGE\s+(\d+)", page.get_text())[1])
    rows = []
    for line in ri.lines((w for w in words if w[1] > max(w[3] for w in subheadings)), tolerance=3):
        if not IP_EXHIBIT_ITEM_ID.fullmatch(line[0][4]):
            continue
        row = {"source_item_id": line[0][4], "report_page": report_page}
        for w in line[1:]:
            column = IP_D_VALUE_COLUMNS[min(range(len(centers)), key=lambda k: abs(centers[k] - (w[0] + w[2]) / 2))]
            assert column not in row, f"{line[0][4]}: two values under {column}"
            row[column] = float(w[4])
        rows.append(row)
    return rows


def ip_exhibit_checks(rows, name):
    """Problems with the parameters read from an exhibit: estimates and standard errors not printed together, missing
    columns, or a lone step parameter."""
    problems = []
    if not rows["source_item_id"].is_unique:
        problems.append(f"{name}: duplicate IDs")
    for parameter in IP_PARAMETERS:
        if not rows[parameter].notna().equals(rows[f"{parameter}_se"].notna()):
            problems.append(f"{name}: {parameter} estimates and standard errors are not printed for the same items")
    if rows[["slope", "location", "guessing"]].isna().any(axis=None):
        problems.append(f"{name}: items without a slope, location, or guessing column")
    if not rows["step1"].notna().equals(rows["step2"].notna()):
        problems.append(f"{name}: items with only one step parameter")
    return problems


def ip_blank_fixed_guessing(rows):
    """The guessing parameter of constructed-response items is printed as 0.000 with a standard error of 0.000: fixed,
    not estimated. It is left blank, as the 2007 and 2011 workbooks do."""
    fixed = (rows["guessing"] == 0) & (rows["guessing_se"] == 0)
    rows.loc[fixed, ["guessing", "guessing_se"]] = np.nan
    return rows


ip_appendix_d = pymupdf.open(ip_download("T03_TR_AppD.pdf"))
ip_t03_information_by_grade, ip_d, ip_d_not_2003, ip_exhibit_sets = {}, {}, {}, []
for grade, (exhibit, title) in IP_D_EXHIBITS.items():
    pages = [page for page in ip_appendix_d if re.search(rf"{re.escape(exhibit)}\b", page.get_text())]
    assert all(title in " ".join(page.get_text().split()) for page in pages), f"{exhibit} is not titled {title}"
    rows = pd.DataFrame(
        [row for page in pages for row in ip_exhibit_rows(page)],
        columns=["source_item_id", "report_page", *IP_VALUE_COLUMNS],
    )
    problems = ip_exhibit_checks(rows, exhibit)
    assert not problems, "\n".join(problems)
    rows = ip_blank_fixed_guessing(rows)

    # Every 2003 mathematics item that the item information marks as used in scaling is in the exhibit, under its
    # Unique ID: M0 for the regular rows, MF for the end-of-session copies, and MC for five grade 8 items listed twice.
    information = ri.timss_2003_math_item_information(ip_download("t03_items.zip"), grade)
    ip_t03_information_by_grade[grade] = information
    unlisted = sorted(set(information.loc[information["In Scaling"] == "Yes", "Unique ID"]) - set(rows["source_item_id"]))
    assert not unlisted, f"2003 grade {grade} items used in scaling but not in {exhibit}: {unlisted}"
    ip_d_not_2003[grade] = sorted(set(rows["source_item_id"]) - set(information["Unique ID"]))

    rows = rows[rows["source_item_id"].str.startswith("M0")].copy()  # the regular rows
    ip_d[grade] = rows.assign(
        source_file="T03_TR_AppD.pdf (TIMSS 2003 Technical Report, Appendix D)",
        source_location=[f"{exhibit}, report page {page}" for page in rows["report_page"]],
    )
    released = ip_released_ids("TIMSS 2003", grade)
    found = ip_d[grade][ip_d[grade]["source_item_id"].isin(released)]
    ip_exhibit_sets.append(found.assign(assessment="TIMSS 2003", grade=grade, item_id=found["source_item_id"]))
    missing = sorted(released - set(found["source_item_id"]))
    assert not missing, f"no parameters for the released TIMSS 2003 grade {grade} items {missing}"
    print(
        f"{exhibit} ({title}): {len(rows)} regular rows on report pages {rows['report_page'].min()}-"
        f"{rows['report_page'].max()}, including all {len(found)} TIMSS 2003 grade {grade} released items; "
        f"{len(ip_d_not_2003[grade])} items are not in the 2003 assessment"
    )

### TIMSS 1995: map the 1995 item IDs to the IDs in Exhibit D.3

The IDs that TIMSS 2003 gave to 1995 items follow two rules, which the 2003 item information shows: its "Cluster ID" column gives the 1995 ID of each item that the 2003 assessment reused.

1. An item that 1995 gave to both population 1 (grades 3 and 4) and population 2 (grades 7 and 8) keeps its population 2 number: `M012` and its place among the 151 population 2 mathematics items in the order of their IDs (A01 = M012001, E06 = M012030), as listed in Table 7.9 of the [1995 user guide](https://timssandpirls.bc.edu/timss1995i/database.html) (`UG_1and2.pdf`). All 2003 grade 8 trend items follow this rule.
2. The other population 1 items are numbered `M011001` upward in the order of their 1995 IDs among the 102 population 1 mathematics items in Table 7.7 of the user guide. The 28 such items that 2003 reused at grade 4, A01 to H09, are M011001 to M011028.

The released 1995 items are clusters I to V, which follow A to H, so rule 2 numbers them from M011029, and the parts of a multi-part item share a number (V4A and V4B are M011086A and M011086B).
Which of them also went to population 2 is found by matching them with the [released population 2 items](https://timssandpirls.bc.edu/timss1995i/Items.html) (`BMItems.pdf`): I9, L2, L4, L8, and U2 are word for word the population 2 items R12, M3, L13, L12, and I6.
K7, "Length of rectangle", is population 2 item E06, which 2003 reused at grade 8 as M012030, "Length of rectangle from width/perimeter".

The cell checks the derivation:

- Both rules reproduce the IDs that the 2003 item information gives the reused items.
- The 70 derived IDs are exactly the 70 items in Exhibit D.3 that are not in the 2003 assessment.
- The IRT model of every item (3PL, 2PL, or GPCM) matches its type and maximum points in the 1995 booklet, and the location parameters agree with the 1995 results: their rank correlation with the international percent correct is about -0.9 (-0.97 for multiple-choice items). Both are checked with the other sets in the last cell.

In [ ]:
# Released population 1 items that TIMSS 1995 also gave to population 2, with their population 2 twin: found by
# matching the items with the released population 2 items (BMItems.pdf), except E06, a secure population 2 item that
# 2003 reused at grade 8 and whose label the 2003 grade 8 item information gives.
IP_1995_SHARED = {"I09": "R12", "K07": "E06", "L02": "M03", "L04": "L13", "L08": "L12", "U02": "I06"}
IP_UG_POPULATION_1_PAGE = 96  # Table 7.7 of the 1995 user guide: population 1 mathematics items by content category
IP_UG_POPULATION_2_PAGE = 98  # Table 7.9: population 2, with a footnote on M09, which was dropped from the analysis


def ip_1995_unit(item_id):
    """A 1995 item ID with a two-digit item number and without any part letter: "V4A" -> "V04"."""
    match = re.fullmatch(r"([A-Z])(\d{1,2})[A-Z]?", item_id)
    return f"{match[1]}{int(match[2]):02d}"


def ip_user_guide_items(pdf_page):
    """The item IDs in a table of the 1995 user guide, in ID order, each multi-part item once."""
    text = pymupdf.open(ip_download("T95_UG_1and2.pdf"))[pdf_page - 1].get_text()
    return sorted(set(re.findall(r"\b([A-Z]\d{2})[A-Z]?\b", text)), key=lambda unit: (unit[0], int(unit[1:])))


def ip_tokens(text):
    return set(re.findall(r"[a-z]{3,}|\d+", text.lower().replace(",", "")))


population_1 = ip_user_guide_items(IP_UG_POPULATION_1_PAGE)
population_2 = ip_user_guide_items(IP_UG_POPULATION_2_PAGE)
assert len(population_1) == 102 and len(population_2) == 151, (len(population_1), len(population_2))
population_2_number = {unit: number for number, unit in enumerate(population_2, 1)}

# The 2003 item information gives the 1995 ID ("Cluster ID") of each item that 2003 reused from 1995.
t03_g4_information = ip_t03_information_by_grade[4]
trend_g4 = t03_g4_information[(t03_g4_information["Year"] == "1") & (t03_g4_information["Scaling Status"] == "Regular")]
trend_g4_ids = dict(zip(trend_g4["Cluster ID"], trend_g4["Unique ID"]))
t03_g8_information = ip_t03_information_by_grade[8]
trend_g8 = t03_g8_information[(t03_g8_information["Year"] == "1") & (t03_g8_information["Scaling Status"] == "Regular")]

# Rule 1: an item given to both populations keeps its population 2 number, M012 + its place in the population 2 order.
# The 2003 grade 8 trend items show it.
wrong = {
    unit: item_id
    for unit, item_id in zip(trend_g8["Cluster ID"], trend_g8["Unique ID"])
    if item_id[:7] != f"M012{population_2_number[ip_1995_unit(unit)]:03d}"
}
assert not wrong, f"the population 2 numbering rule does not hold for the 2003 grade 8 trend items {wrong}"
# Rule 2: the other population 1 items are numbered M011001 upward in the order of their 1995 IDs.
shared = {unit for unit, item_id in trend_g4_ids.items() if item_id.startswith("M012")} | set(IP_1995_SHARED)
population_1_only = [unit for unit in population_1 if unit not in shared]
ip_1995_ids = {unit: f"M011{number:03d}" for number, unit in enumerate(population_1_only, 1)}
ip_1995_ids |= {unit: f"M012{population_2_number[twin]:03d}" for unit, twin in IP_1995_SHARED.items()}
wrong = {
    unit: item_id
    for unit, item_id in trend_g4_ids.items()
    if item_id.startswith("M011") and ip_1995_ids[unit] != item_id
}
assert not wrong, f"the population 1 numbering rule does not hold for the 2003 grade 4 trend items {wrong}"

# The shared items: the released twins match the 1995 items word for word, and E06 is the grade 8 trend item M012030.
population_2_pages = [page.get_text() for page in pymupdf.open(ip_download("T95_BMItems.pdf"))]
released_1995 = ip_released[(ip_released["assessment"] == "TIMSS 1995") & (ip_released["grade"] == 4)]
shared_rows = []
for unit, twin in IP_1995_SHARED.items():
    item = released_1995[released_1995["item_id"].map(ip_1995_unit) == unit].iloc[0]
    item_tokens = ip_tokens(" ".join([item["stem"], *(option["text"] for option in item["options"] or [])]))
    box_label = re.compile(rf"^{twin[0]}-{int(twin[1:])}$", re.MULTILINE)  # the item ID box, such as "R-12"
    twin_pages = [text for text in population_2_pages if box_label.search(text)]
    if twin_pages:
        overlap = len(item_tokens & ip_tokens(twin_pages[0])) / len(item_tokens)
        assert overlap >= 0.8, f"{item['item_id']} and population 2 item {twin} differ ({overlap:.0%} shared words)"
        evidence = f"same item as {twin} in the released population 2 items ({overlap:.0%} of its words)"
    else:
        g8 = trend_g8[trend_g8["Cluster ID"] == twin].iloc[0]
        evidence = f"population 2 item {twin} is the 2003 grade 8 trend item {g8['Unique ID']}, \"{g8['Item Label']}\""
    shared_rows.append({"item_id": item["item_id"], "population_2_item": twin, "evidence": evidence})

# The released 1995 items, parts and all, under the IDs of Exhibit D.3.
ip_1995_crosswalk = released_1995[["item_id", "item_label", "item_type", "max_points", "pct_correct_intl_avg"]].copy()
ip_1995_crosswalk["source_item_id"] = [
    ip_1995_ids[ip_1995_unit(item_id)] + re.fullmatch(r"[A-Z]\d{1,2}([A-Z]?)", item_id)[1]
    for item_id in ip_1995_crosswalk["item_id"]
]
ip_1995_crosswalk = ip_1995_crosswalk.merge(pd.DataFrame(shared_rows), on="item_id", how="left")
ip_1995_crosswalk["rule"] = np.where(
    ip_1995_crosswalk["population_2_item"].notna(), "shared with population 2", "population 1 only"
)
assert sorted(ip_1995_crosswalk["source_item_id"]) == ip_d_not_2003[4], (
    f"the derived IDs are not the {len(ip_d_not_2003[4])} items of Exhibit D.3 outside the 2003 assessment"
)

ip_t95 = ip_1995_crosswalk[["item_id", "source_item_id"]].merge(ip_d[4], on="source_item_id")
ip_exhibit_sets.append(ip_t95.assign(assessment="TIMSS 1995", grade=4))
print(f"{len(ip_t95)} TIMSS 1995 grade 4 items mapped to Exhibit D.3")
display(
    ip_1995_crosswalk[ip_1995_crosswalk["population_2_item"].notna()][
        ["item_id", "item_label", "source_item_id", "evidence"]
    ]
)

### TIMSS 1999: map the booklet IDs to permanent IDs

The 1999 achievement codebook (`BSACBKM2.CDT` in `bm2_cdbks.zip`, from the [TIMSS 1999 International Database](https://timssandpirls.bc.edu/timss1999i/database.html)) describes each item variable on a line such as `37 B08 M012008 HOW MANY CALORIES IN 30G FOOD (A)`: the variable number, the booklet position (written T2A for the booklet's T02A), the permanent ID, an abbreviated label, and the answer key ("X" for a constructed-response item).
The cell maps every released 1999 item to its permanent ID and checks:

- The codebook's key is the booklet's key for every item (X for every constructed-response item).
- Every permanent ID is among the Exhibit D.1 items that are not in the 2003 assessment, as an item released after 1999 must be; the one other such item, M012018, is a 1995 item that 1999 used and 2003 did not.
- Exhibit E.3 of the TIMSS 1999 Technical Report, the 1999 calibration, agrees with Exhibit D.1 for the 65 items it lists: the same IRT model for every item, and slopes and locations correlated 0.96 and 0.97. Its item lines give the slope, location, and guessing parameters with their standard errors, and the step parameters of a 2- or 3-point item follow on lines of their own ("d1", "d2", ...).

In [ ]:
IP_CODEBOOK = "BSACBKM2.CDT"  # the codebook of the grade 8 student achievement file, in T99_bm2_cdbks.zip
# A codebook line of an achievement item: variable number, location (booklet ID), permanent ID, abbreviated label, and
# the answer key in parentheses ("X" for a constructed-response item), as in
# "  37 B08          M012008   HOW MANY CALORIES IN 30G FOOD (A)".
IP_CODEBOOK_ITEM = re.compile(
    r"^\s*\d+\s+(?P<location>[A-Z]\d{1,2}[A-Z]?)\s+(?P<source_item_id>M\d{6}[A-Z]?)\s+(?P<codebook_label>.+?)\s*"
    r"\((?P<codebook_key>[A-EX])\)",
    re.MULTILINE,
)
IP_E_EXHIBIT = "Exhibit E.3"
IP_E_TITLE = "IRT Parameters for TIMSS Joint 1995-1999 Eighth Grade Mathematics"
IP_E_ITEM_ID = re.compile(r"M0\d{5}[A-Z]?")
IP_E_COLUMNS = ["slope", "slope_se", "location", "location_se", "guessing", "guessing_se"]  # the item line's values


def ip_1999_location(location):
    """A codebook location as the booklet prints the item ID: "T2A" -> "T02A", "B08" stays."""
    match = re.fullmatch(r"([A-Z])(\d{1,2})([A-Z]?)", location)
    return f"{match[1]}{int(match[2]):02d}{match[3]}"


def ip_e_rows(pages):
    """The item rows of an Appendix E exhibit. Each item has a line with its ID and the slope, location, and guessing
    parameters, each followed by its standard error; a constructed-response item worth more than 1 point has one more
    line per step ("d1", "d2", ...) with the step parameter and its standard error, after an empty "d0" line."""
    rows = []
    for page in pages:
        page_label = "E·" + re.search(r"E·(\d+)", page.get_text())[1]  # the appendix page, such as E·11
        for line in ri.lines(page.get_text("words"), tolerance=3):
            first, values = line[0][4], [w[4] for w in line[1:]]
            if IP_E_ITEM_ID.fullmatch(first):
                assert len(values) == len(IP_E_COLUMNS), f"{first}: {values}"
                rows.append({"source_item_id": first, "report_page": page_label, **dict(zip(IP_E_COLUMNS, map(float, values)))})
            elif step := re.fullmatch(r"d(\d)", first):
                if step[1] == "0":
                    assert not values, f"values on a d0 line after {rows[-1]['source_item_id']}"
                    continue
                assert len(values) == 2, f"step line {step[0]} after {rows[-1]['source_item_id']}: {values}"
                rows[-1] |= {f"step{step[1]}": float(values[0]), f"step{step[1]}_se": float(values[1])}
    return pd.DataFrame(rows, columns=["source_item_id", "report_page", *IP_VALUE_COLUMNS])


# The released 1999 items under their permanent IDs, from the achievement codebook.
with zipfile.ZipFile(ip_download("T99_bm2_cdbks.zip")) as archive:
    codebook = archive.read(IP_CODEBOOK).decode("latin-1")
ip_codebook = pd.DataFrame([match.groupdict() for match in IP_CODEBOOK_ITEM.finditer(codebook)])
ip_codebook["item_id"] = ip_codebook["location"].map(ip_1999_location)
assert ip_codebook["item_id"].is_unique and ip_codebook["source_item_id"].is_unique, "codebook items repeat"
released_1999 = ip_released[(ip_released["assessment"] == "TIMSS 1999") & (ip_released["grade"] == 8)]
ip_1999_crosswalk = released_1999[["item_id", "item_label", "item_type", "max_points", "correct_option"]].merge(
    ip_codebook[["item_id", "location", "source_item_id", "codebook_label", "codebook_key"]],
    on="item_id",
    how="left",
    validate="one_to_one",
)
problems = []
missing = ip_1999_crosswalk.loc[ip_1999_crosswalk["source_item_id"].isna(), "item_id"].tolist()
if missing:
    problems.append(f"released items not in the codebook: {missing}")
# The codebook's key must be the booklet's: the letter of a multiple-choice item, X for a constructed-response item.
wrong_key = ip_1999_crosswalk["codebook_key"] != ip_1999_crosswalk["correct_option"].fillna("X")
if wrong_key.any():
    problems.append(f"codebook keys that differ from the booklet: {ip_1999_crosswalk.loc[wrong_key, 'item_id'].tolist()}")
# Released after 1999, these items did not return in 2003, so they must be among the Exhibit D.1 items outside the 2003
# assessment.
in_2003 = sorted(set(ip_1999_crosswalk["source_item_id"]) - set(ip_d_not_2003[8]))
if in_2003:
    problems.append(f"permanent IDs that are not Exhibit D.1 items outside the 2003 assessment: {in_2003}")
assert not problems, "\n".join(problems)
ip_1999_crosswalk = ip_1999_crosswalk.drop(columns="correct_option")
ip_t99 = ip_1999_crosswalk[["item_id", "source_item_id"]].merge(ip_d[8], on="source_item_id", validate="one_to_one")
ip_exhibit_sets.append(ip_t99.assign(assessment="TIMSS 1999", grade=8))

# Check against Exhibit E.3 of the TIMSS 1999 Technical Report, the joint 1995-1999 calibration. It lists no items
# between M022070 and M022165 (the content-domain exhibits E.5 to E.9 do), so it lacks some released items.
ip_appendix_e = pymupdf.open(ip_download("T99_TR_AppenE.pdf"))
pages = [page for page in ip_appendix_e if re.search(rf"{re.escape(IP_E_EXHIBIT)}\b", page.get_text())]
assert all(IP_E_TITLE in " ".join(page.get_text().split()) for page in pages), f"{IP_E_EXHIBIT} is not titled {IP_E_TITLE}"
ip_e3 = ip_e_rows(pages)
problems = ip_exhibit_checks(ip_e3, IP_E_EXHIBIT)
assert not problems, "\n".join(problems)
ip_e3 = ip_blank_fixed_guessing(ip_e3)
both = ip_t99.merge(ip_e3, on="source_item_id", suffixes=("", "_e3"))
same_model = all(both[name].notna().equals(both[f"{name}_e3"].notna()) for name in ["guessing", *IP_STEPS])
assert same_model, f"items whose IRT model differs between Exhibit D.1 and {IP_E_EXHIBIT}"
print(
    f"{len(ip_t99)} TIMSS 1999 grade 8 items mapped to Exhibit D.1 through the codebook, with matching keys. "
    f"{len(both)} of them are in {IP_E_EXHIBIT}, with the same IRT models; correlation of the two calibrations' "
    f"slopes {both['slope'].corr(both['slope_e3']):.2f}, locations {both['location'].corr(both['location_e3']):.2f}. "
    f"Not in {IP_E_EXHIBIT}: {sorted(set(ip_t99['item_id']) - set(both['item_id']))}"
)
print(
    "Exhibit D.1 items outside the 2003 assessment that are not released 1999 items:",
    sorted(set(ip_d_not_2003[8]) - set(ip_1999_crosswalk["source_item_id"])),
)
display(ip_1999_crosswalk.head())

### Combine and export

Join the parameters with the metadata of the released items (assessment, grade, subject, item ID, label, domains, type, and maximum points), check them, and export the workbooks.
`source_item_id` is the ID under which the source lists the item (such as MZ31286, M011029, M012008, or MA13064), and `source_file`, `source_location`, `source_url`, and `source_page` say where its parameters come from.
`IP_NOT_SCALED` lists the released items without parameters and `IP_SCALED_POINTS` those scaled with fewer points than their scoring guide gives, each with the reason, which goes to the `note` column; any other item without parameters or with an unexpected model stops the cell.

In [ ]:
IP_METADATA = [
    "assessment",
    "grade",
    "subject",
    "item_id",
    "item_label",
    "content_domain",
    "main_topic",
    "cognitive_domain",
    "item_type",
    "max_points",
]
# Where each set's parameters come from, by (assessment, grade); "file" is a key of IP_SOURCES.
IP_SET_SOURCES = [
    {
        "assessment": "TIMSS 1995",
        "grade": 4,
        "file": "T03_TR_AppD.pdf",
        "where": "Exhibit D.3, IRT Parameters for TIMSS Joint 1995-2003 Fourth-Grade Mathematics, under 2003-style IDs "
        "(see sheet crosswalk_1995_g4).",
        "calibration": "Concurrent calibration of the TIMSS 1995 and 2003 fourth-grade mathematics data.",
        "notes": "The TIMSS 1995 International Database has no item-parameter file, and the 1995 scaling used a "
        "different model. No official crosswalk from the 1995 item IDs to the IDs in Exhibit D.3 was found; it is "
        "derived from the 1995 user guide, the 1995 released population 2 items, and the 2003 item information, "
        "and checked against the parameters.",
    },
    {
        "assessment": "TIMSS Advanced 1995",
        "grade": 12,
        "file": "TA08_Items.zip",
        "where": "TA08_Items/TA08_Item_Parameters.xls, sheet Adv. Math, the rows whose block position is a 1995 "
        "position (K01 for item K1).",
        "calibration": "Concurrent scaling of the TIMSS Advanced 1995 and 2008 advanced mathematics data.",
        "notes": "The sheet matches Exhibit C.3 of the TIMSS Advanced 2008 Technical Report, Appendix C; Exhibit C.1 has "
        "parameters from a rescaling of the 1995 data alone, not used. The sheet lists the 1995 items under IDs that "
        "the 1995 booklet does not print (MA13064 for K1). K8, K9, L10, L11, and L15 (parts A and B) are not in it: "
        "the rescaling left out 11 of the 68 advanced mathematics items of 1995 (Technical Report, section 8.4.1). K14 "
        "and L18, 2-point items in the 1995 coding guides, were scaled as 1-point items.",
    },
    {
        "assessment": "TIMSS 1999",
        "grade": 8,
        "file": "T03_TR_AppD.pdf",
        "where": "Exhibit D.1, IRT Parameters for TIMSS Joint 1999-2003 Eighth-Grade Mathematics, under the permanent "
        "IDs that the TIMSS 1999 achievement codebook (BSACBKM2.CDT in bm2_cdbks.zip) gives for the booklet IDs (see "
        "sheet crosswalk_1999_g8).",
        "calibration": "Concurrent calibration of the TIMSS 1999 and 2003 eighth-grade mathematics data.",
        "notes": "The TIMSS 1999 International Database has no item-parameter file. Exhibit E.3 of the TIMSS 1999 "
        "Technical Report, Appendix E (joint 1995-1999 calibration), lists no items between M022070 and M022165, so "
        "it lacks 17 of the released items; for the other 65 it agrees with Exhibit D.1 (same IRT models, locations "
        "correlated 0.97).",
    },
    {
        "assessment": "TIMSS 2003",
        "grade": 4,
        "file": "T03_TR_AppD.pdf",
        "where": "Exhibit D.3, IRT Parameters for TIMSS Joint 1995-2003 Fourth-Grade Mathematics, the M0 rows. The MF "
        "rows are the same items in the third and sixth block positions, which the calibration scaled separately "
        "because of a position effect (Technical Report, chapter 11).",
        "calibration": "Concurrent calibration of the TIMSS 1995 and 2003 fourth-grade mathematics data.",
        "notes": "The TIMSS 2003 International Database has no item-parameter file. The guessing parameter printed as "
        "0.000 (standard error 0.000) for constructed-response items is fixed and left blank.",
    },
    {
        "assessment": "TIMSS 2003",
        "grade": 8,
        "file": "T03_TR_AppD.pdf",
        "where": "Exhibit D.1, IRT Parameters for TIMSS Joint 1999-2003 Eighth-Grade Mathematics, the M0 rows (MF "
        "rows as for grade 4).",
        "calibration": "Concurrent calibration of the TIMSS 1999 and 2003 eighth-grade mathematics data.",
        "notes": "As for grade 4.",
    },
    {
        "assessment": "TIMSS 2007",
        "grade": 4,
        "file": "T07_Items.zip",
        "where": "T07_Items/T07_ItemParameters_Overall.xls, sheet G4-MAT, rows whose IDs start MZ (the items in the "
        "2007 assessment booklets; MZ31286 is M031286). The other rows are 2003 booklet items.",
        "calibration": "Concurrent calibration of the TIMSS 2003 data, 2007 bridging data, and 2007 data (fourth-grade "
        "mathematics).",
        "notes": "The 2011 workbook also has parameters for these items, from the 2011 calibration; they are not used.",
    },
    {
        "assessment": "TIMSS 2007",
        "grade": 8,
        "file": "T07_Items.zip",
        "where": "T07_Items/T07_ItemParameters_Overall.xls, sheet G8-MAT, rows whose IDs start MZ.",
        "calibration": "Concurrent calibration of the TIMSS 2003 data, 2007 bridging data, and 2007 data (eighth-grade "
        "mathematics).",
        "notes": "M042273 has no parameters: the grade 8 item information marks it as not used in scaling.",
    },
    {
        "assessment": "TIMSS Advanced 2008",
        "grade": 12,
        "file": "TA08_Items.zip",
        "where": "TA08_Items/TA08_Item_Parameters.xls, sheet Adv. Math, under the item IDs.",
        "calibration": "Concurrent scaling of the TIMSS Advanced 1995 and 2008 advanced mathematics data.",
        "notes": "The sheet matches Exhibit C.3 of the TIMSS Advanced 2008 Technical Report, Appendix C.",
    },
    {
        "assessment": "TIMSS 2011",
        "grade": 4,
        "file": "T11_ItemParameters.zip",
        "where": "T11_G4_ItemParameters.xlsx, sheet MAT, under the item IDs.",
        "calibration": "Concurrent calibration of the TIMSS 2007 and 2011 fourth-grade mathematics data.",
        "notes": "The sheet also lists the items released in 2007, with parameters from this calibration; they are "
        "not used for the TIMSS 2007 items.",
    },
    {
        "assessment": "TIMSS 2011",
        "grade": 8,
        "file": "T11_ItemParameters.zip",
        "where": "T11_G8_ItemParameters.xlsx, sheet MAT, under the item IDs.",
        "calibration": "Concurrent calibration of the TIMSS 2007 and 2011 eighth-grade mathematics data.",
        "notes": "As for grade 4.",
    },
]
# Released items without parameters, and why.
IP_NOT_SCALED = {
    ("TIMSS 2007", 8, "M042273"): "Not used in scaling (In Scaling: No in T07_G8_ItemInformation.xls), so not in the "
    "G8-MAT sheet.",
    **{
        ("TIMSS Advanced 1995", 12, item_id): "Not in the Adv. Math sheet: the TIMSS Advanced 2008 rescaling of the "
        "1995 data left out 11 of the 68 advanced mathematics items of 1995, 10 of them because they did not fit the "
        "2008 framework (TIMSS Advanced 2008 Technical Report, section 8.4.1)."
        for item_id in ["K8", "K9", "L10", "L11", "L15A", "L15B"]
    },
}
# Released items scaled with fewer points than their coding guide gives, which the IRT model check allows.
IP_SCALED_POINTS = {
    ("TIMSS Advanced 1995", 12, item_id): 1
    for item_id in ["K14", "L18"]
}
IP_SCALED_POINTS_NOTE = (
    "The 1995 coding guide gives 2 points, but the 1995-2008 concurrent scaling has 1-point (2PL) parameters: the "
    "TIMSS Advanced 2008 Technical Report (section 8.4.2) counts the 31 released 1995 items in it as worth 40 score "
    "points, which holds only with K14 and L18 at 1 point. How their codes were collapsed is not documented."
)
assert {key[2] for key in IP_NOT_SCALED if key[0] == "TIMSS 2007"} == ip_t07_unscaled[8] and not ip_t07_unscaled[4]

ip_set_sources = pd.DataFrame(IP_SET_SOURCES)
ip_found = pd.concat([*ip_workbook_sets, *ip_exhibit_sets], ignore_index=True)
ip_found = ip_found.merge(ip_set_sources[["assessment", "grade", "calibration"]], on=["assessment", "grade"])
ip_item_parameters = ip_released[IP_METADATA].merge(
    ip_found[[*IP_KEY, "source_item_id", *IP_VALUE_COLUMNS, "calibration", "source_file", "source_location"]],
    on=IP_KEY,
    how="left",
    validate="one_to_one",
)
source_files = ip_set_sources.set_index(["assessment", "grade"])["file"]
ip_item_parameters["source_url"] = [IP_SOURCES[source_files[key]][0] for key in zip(ip_item_parameters["assessment"], ip_item_parameters["grade"])]
ip_item_parameters["source_page"] = [IP_SOURCES[source_files[key]][1] for key in zip(ip_item_parameters["assessment"], ip_item_parameters["grade"])]
scaled = ip_item_parameters["slope"].notna()
ip_item_parameters.insert(
    len(IP_METADATA),
    "irt_model",
    np.select(
        [~scaled, ip_item_parameters["guessing"].notna(), ip_item_parameters["step1"].notna()],
        [None, "3PL", "GPCM"],
        default="2PL",
    ),
)
ip_keys = list(zip(*(ip_item_parameters[column] for column in IP_KEY)))
ip_item_parameters["note"] = [
    IP_NOT_SCALED.get(key) or (IP_SCALED_POINTS_NOTE if key in IP_SCALED_POINTS else None) for key in ip_keys
]
ip_item_parameters = ip_item_parameters[
    [
        *IP_METADATA,
        "irt_model",
        *IP_PARAMETERS,
        *[f"{name}_se" for name in IP_PARAMETERS],
        "source_item_id",
        "calibration",
        "source_file",
        "source_location",
        "source_url",
        "source_page",
        "note",
    ]
]

problems = []
unexplained = ip_item_parameters[~scaled & ip_item_parameters["note"].isna()]
if not unexplained.empty:
    problems.append(f"items without parameters: {unexplained[IP_KEY].to_dict('records')}")
if any(key in IP_NOT_SCALED for key, has_parameters in zip(ip_keys, scaled) if has_parameters):
    problems.append("items listed in IP_NOT_SCALED have parameters")
for name in IP_PARAMETERS:
    if not ip_item_parameters[name].notna().equals(ip_item_parameters[f"{name}_se"].notna()):
        problems.append(f"{name} estimates and standard errors are not present for the same items")
# 3PL for multiple-choice items, 2PL for 1-point and GPCM for 2- and 3-point constructed-response items, with one step
# per point; TIMSS centers the steps, so they add up to 0 (within rounding).
scaled_points = pd.Series([IP_SCALED_POINTS.get(key) for key in ip_keys], index=ip_item_parameters.index)
scaled_points = scaled_points.fillna(ip_item_parameters["max_points"])
expected_model = pd.Series(
    np.select([ip_item_parameters["item_type"] == "multiple_choice", scaled_points >= 2], ["3PL", "GPCM"], "2PL"),
    index=ip_item_parameters.index,
)
step_count = ip_item_parameters[IP_STEPS].notna().sum(axis=1)
gpcm = ip_item_parameters["irt_model"] == "GPCM"
mismatched = ip_item_parameters[
    scaled & ((ip_item_parameters["irt_model"] != expected_model) | (gpcm & (step_count != scaled_points)))
]
if not mismatched.empty:
    problems.append(f"IRT models that do not match the item type and maximum points: {mismatched[IP_KEY].to_dict('records')}")
step_sums = ip_item_parameters.loc[gpcm, IP_STEPS].sum(axis=1)
if (step_sums.abs() > 0.002).any():
    problems.append(f"steps that do not add up to 0: {ip_item_parameters.loc[step_sums.index[step_sums.abs() > 0.002], IP_KEY].to_dict('records')}")
assert not problems, "\n".join(problems)


def ip_rank_correlation(rows):
    """Rank correlation of the location with the international average percent correct (negative: harder items
    have higher locations), for all items and multiple-choice items alone."""
    multiple_choice = rows[rows["item_type"] == "multiple_choice"]
    return pd.Series(
        {
            "all_items": rows["location"].corr(rows["pct_correct_intl_avg"], method="spearman"),
            "multiple_choice": multiple_choice["location"].corr(multiple_choice["pct_correct_intl_avg"], method="spearman"),
        }
    )


ip_checks = ip_item_parameters.merge(ip_released[[*IP_KEY, "pct_correct_intl_avg"]], on=IP_KEY)
ip_correlations = ip_checks.groupby(["assessment", "grade"], sort=False)[["item_type", "location", "pct_correct_intl_avg"]].apply(
    ip_rank_correlation
)

ip_sources = ip_set_sources.rename(columns={"file": "source_file"})
ip_sources.insert(2, "items", [len(ip_released_ids(a, g)) for a, g in zip(ip_sources["assessment"], ip_sources["grade"])])
ip_sources.insert(
    3,
    "items_with_parameters",
    [scaled[(ip_item_parameters["assessment"] == a) & (ip_item_parameters["grade"] == g)].sum() for a, g in zip(ip_sources["assessment"], ip_sources["grade"])],
)
ip_sources.insert(4, "source_page", ip_sources["source_file"].map(lambda name: IP_SOURCES[name][1]))
ip_sources = ip_sources.merge(
    ip_correlations.add_prefix("location_vs_pct_correct_").round(2).reset_index(), on=["assessment", "grade"], how="left"
)


def ip_export(path, item_parameters, sources, crosswalks):
    with pd.ExcelWriter(path) as writer:
        item_parameters.to_excel(writer, sheet_name="item_parameters", index=False)
        sources.to_excel(writer, sheet_name="sources", index=False)
        for sheet_name, crosswalk in crosswalks.items():
            crosswalk.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"Exported {len(item_parameters)} items to {path.resolve()}")


ip_export(
    IP_OUTPUT_PATH,
    ip_item_parameters,
    ip_sources,
    {"crosswalk_1995_g4": ip_1995_crosswalk, "crosswalk_1999_g8": ip_1999_crosswalk},
)
ip_export(
    IP_G4_OUTPUT_PATH,
    ip_item_parameters[ip_item_parameters["grade"] == 4],
    ip_sources[ip_sources["grade"] == 4],
    {"crosswalk_1995_g4": ip_1995_crosswalk},
)
display(
    pd.crosstab(
        [ip_item_parameters["assessment"], ip_item_parameters["grade"]],
        ip_item_parameters["irt_model"].fillna("no parameters"),
        margins=True,
    )
)
print("Rank correlation of the location with the international average percent correct:")
display(ip_correlations.round(2))
display(ip_item_parameters.groupby(["assessment", "grade"], sort=False)[IP_PARAMETERS].mean().round(3))